# 00 — YouTube Automation: Kalıcı Master Notebook (Orkestratör)

Bu, normal Colab kullanımında **açman gereken tek notebook**. Notebook 01–06'yı ayrıca yüklemene veya
çalıştırmana gerek yoktur — altı fazın hepsi burada, aynı `local_pipeline` kanonik modülleri üzerinden
gerçekten çalışır.

**Mimari sınır:** Ana üretim hattı yalnızca verilen kaynak videoyu `analiz → script → ses → render`
zinciriyle Shorts'a dönüştürür. YouTube keşfi ve viral tahmin bağımsız araştırma yan koludur;
kaynak videoyu, seçilen hot moment'i veya render kararını otomatik olarak değiştirmez.

## Maliyet politikası (ücretli sağlayıcı fan-out'u yok)

Her ücretli faz **tek bir seçili sağlayıcı/model** çağırır. Fallback kapalıyken bir istek en fazla
**bir** ücretli çağrı yapar; fallback açıkken adaylar **sırayla** denenir ve **ilk geçerli sonuçta
durulur** — bir sonraki aday yalnızca öncekinin gerçekten başarısız olduğu durumda çağrılır.
Karşılaştırma/benchmark amaçlı çok-modelli fan-out **sadece** standalone Notebook 02/03'te,
açık karşılaştırma sarmalayıcıları (`call_*_script`, `call_*_tts`) ile yapılır.

Denenen her sağlayıcı/model ve başarısızlık nedeni artefaktın `provider_attempts` alanına yazılır.

## Onay politikası (üretimden SONRA, içeriğe bağlı)

Üretim fonksiyonları **onay argümanı almaz** ve asla `approved` üretemez. Yeni üretilen içerik her zaman
`review_required` (gTTS için `development_placeholder`) olur — çünkü henüz kimse okumadı/dinlemedi/izlemedi.

Her faz için sıra: **üret → incele (görüntüle/dinle/izle) → ayrı onay hücresi**. Onay hücresi, inceleme
hücresinin yazdırdığı **SHA-256** değerini istemenizi gerektirir; bu, içeriği gerçekten gördüğünüzün
kaydıdır. İçerik sonradan değişirse (yeniden üretim, yeniden render) onay **otomatik olarak geçersiz**
olur (`approval.check_approval`). Hiçbir yerde otomatik `production_ready` yoktur ve
**bu notebook hiçbir şeyi YouTube'a yüklemez veya yayınlamaz.**

## Tek dosya Colab çalışması

`local_pipeline` modülleri bu notebook'a gömülüdür (base64 + SHA-256 doğrulamalı). Yükleme sırası:
repo → Drive/üst dizin kopyası → `PROJECT_REPO_URL` klonu → gömülü hash-doğrulanmış yedek.

## Bağımlılık farkındalığı

Bir faz kapalıysa, sonraki açık faz gerekli artefaktı diskte arar ve şemasını doğrular; bulamazsa
**sadece o fazı** net bir hatayla durdurur. Her faz iki durum bildirir:
`execution_status` (completed/skipped/failed) ve `artifact_status`
(review_required/approved/insufficient_data/failed/development_placeholder).

In [ ]:
!pip -q install requests pandas numpy google-api-python-client faster-whisper scenedetect librosa openai anthropic google-genai elevenlabs gTTS imageio-ffmpeg > /dev/null
print('Kurulum tamam.')

## Ana üretim hattı ve bağımsız araştırma anahtarları

In [ ]:
# --- ANA URETIM HATTI ---------------------------------------------------------
# Varsayilan: kaynak video -> moment secimi -> orijinal ses + gomulu altyazi.
# 'narrated_recap': istege bagli Gemini script -> TTS -> anlatimli render.
# NOT: PIPELINE_MODE artik ZORUNLU Faz 0 isleme planindan TURETILIR
# (direct_clip_series -> original_audio_subtitles, narrated_theory -> narrated_recap).
# Asagidaki deger yalnizca plan yokken gecerli olan guvenli baslangictir.
PIPELINE_MODE = 'original_audio_subtitles'  # 'original_audio_subtitles' | 'narrated_recap'
RUN_VIDEO_ANALYSIS = False
RUN_SCRIPT_GENERATION = False
RUN_TTS = False
RUN_RENDER = False

# --- BAGIMSIZ ARASTIRMA YAN KOLU: ana pipeline'a kaynak/karar BESLEMEZ -------
RUN_YOUTUBE_DATA = False
RUN_VIRAL_PREDICTION = False

# Her faz icin IKI ayri durum tutulur:
#   execution_status: completed | skipped | failed   (faz hucresi calisti mi?)
#   artifact_status : review_required | approved | insufficient_data | failed |
#                     development_placeholder | None
# Temiz calisip 'insufficient_data' artefakti yazan bir faz sadece "ok" gorunmez.
EXECUTION_STATUS_VALUES = {'completed', 'skipped', 'failed'}
ARTIFACT_STATUS_VALUES = {
    'review_required', 'approved', 'insufficient_data', 'failed',
    'development_placeholder', 'experimental', 'production_ready', None,
}
phase_report = {}


def record_phase(name, execution_status, artifact_status=None, detail=''):
    assert execution_status in EXECUTION_STATUS_VALUES, f'gecersiz execution_status: {execution_status!r}'
    assert artifact_status in ARTIFACT_STATUS_VALUES, f'gecersiz artifact_status: {artifact_status!r}'
    phase_report[name] = {
        'execution_status': execution_status,
        'artifact_status': artifact_status,
        'detail': detail,
    }
    suffix = f' | artifact_status={artifact_status}' if artifact_status is not None else ''
    print(f'[faz:{name}] execution_status={execution_status}{suffix}' + (f' | {detail}' if detail else ''))


## Google model router yapilandirmasi (tek merkez)

Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider. Notebook icinde **hicbir**
`genai.Client(...)` / sabit `model='...'` cagrisi yoktur.

**Kota proje bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits).
Bu yuzden ikinci bir hesap/proje anahtari eklemek limitleri artirmaz; router yalnizca **ayni proje
icinde model degistirir**. `ALLOW_ACCOUNT_QUOTA_ROTATION` politika geregi `False`'tir ve
`google_ai.assert_no_account_quota_rotation()` True yapilirsa calismayi durdurur.

**Ucretsiz katman veri politikasi:** gonderdigin icerik Google tarafindan urun gelistirmede
kullanilabilir ve insan gozden gecirenler okuyabilir. Bu yuzden `ALLOW_RAW_MEDIA_TO_FREE_TIER`
varsayilan `False`: ham telifli video/ses **yuklenmez**, yalnizca yerel olarak uretilmis
transcript pencereleri / zaman damgalari / sahne-ses skorlari gonderilir.


In [ ]:
# --- Google model router yapilandirmasi (BURADAN AC/KAPA) --------------------
# Manuel credential profili: kota dolunca otomatik hesap gecisi YOKTUR.
# Profiller farkli proje/kanal ortamlari icindir; kota sinirini asmak icin profil degistirme.
# Profil degistirmek icin yalnizca asagidaki iki etiketi elle degistirip hucreleri yeniden calistir.
GOOGLE_CREDENTIAL_PROFILE = 'primary'
YOUTUBE_CREDENTIAL_PROFILE = 'primary'
GOOGLE_CREDENTIAL_PROFILES = {
    'primary': ('GeminiKey', 'GOOGLE_API_KEY'),
    'secondary': ('GeminiKeySecondary',),
}
YOUTUBE_CREDENTIAL_PROFILES = {
    'primary': ('YoutubeKEY', 'YOUTUBE_DATA_API_KEY'),
    'secondary': ('YoutubeKEYSecondary',),
}

GOOGLE_ROUTE_MODE = 'auto'              # 'auto' | 'explicit'
GOOGLE_TASK_MODEL_OVERRIDES = {}        # ornek: {'script_generation': ['gemini-3.5-flash-lite']}
ENABLE_GOOGLE_MODEL_FALLBACK = True     # False: gorev basina TEK Google modeli denenir
ENABLE_CROSS_PROVIDER_FALLBACK = True   # script/TTS/embedding: Groq/Cloudflare/Azure (kimlik varsa)
MAX_GOOGLE_MODEL_ATTEMPTS = 4
MAX_PROVIDER_ATTEMPTS = 3
RUN_GOOGLE_CONNECTION_TEST = False      # True: SADECE secili birincil modellere minik bir ping
ENABLE_GOOGLE_SEARCH_GROUNDING = False  # Gemini 2.5 arama-temelli arastirma
RUN_ANTIGRAVITY_RESEARCH = False        # deneysel ajan raporu (varsayilan KAPALI)
ALLOW_RAW_MEDIA_TO_FREE_TIER = False    # ham telifli medya ucretsiz katmana YUKLENMEZ

# POLITIKA -- ASLA True yapma. Birden fazla Google hesabi/projesi arasinda anahtar
# dondurerek RPD/RPM/TPM limitlerini asmak kota ihlalidir. Kota proje bazlidir;
# ikinci bir anahtar limiti artirmaz. Router yalnizca AYNI proje icinde MODEL degistirir.
ALLOW_ACCOUNT_QUOTA_ROTATION = False

# --- Her Google yetenegi icin BAGIMSIZ komut anahtari -----------------------
RUN_GOOGLE_HOTMOMENT_REVIEW = False     # Faz 1 adaylarinin nihai LLM incelemesi
RUN_GOOGLE_SCRIPT_GENERATION = True     # Faz 2 script uretimi (Google = kanonik varsayilan)
RUN_GOOGLE_BULK_CLASSIFICATION = False  # baslik/aciklama/hashtag, trend satiri siniflama
RUN_GOOGLE_GROUNDED_RESEARCH = False    # arama-temelli guncellik dogrulamasi
RUN_GOOGLE_TTS = True                   # Ana pipeline seslendirmesi: Gemini TTS (preview -> review_required)
RUN_GOOGLE_EMBEDDINGS = False           # duplikasyon/novelty/kumeleme

# Gemini TTS ayarlari (RUN_GOOGLE_TTS ve TTS_PROVIDER='gemini' ile kullanilir)
GEMINI_TTS_VOICE = 'Kore'               # bkz. google_ai.GEMINI_TTS_VOICES
GEMINI_TTS_LANGUAGE_CODE = 'tr-TR'
GEMINI_TTS_NARRATION_DIRECTION = 'Gergin, merak uyandiran bir tonda, net ve akici oku:'

# Antigravity deneysel arastirma icin ZORUNLU token butcesi (sessiz tuketim yasak)
ANTIGRAVITY_MAX_TOTAL_TOKENS = 8000


## Kurulum — Drive bir kez bağlanır, yollar bir kez tanımlanır, Secrets bir kez okunur

In [ ]:
import os, sys, json
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()

USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/AVRUPA YAKASI/YoutubeOtomationv0"

# Colab Secrets erisim hatalari (panelde 'Notebook erisimi' kapali, secret hic
# eklenmemis, vb.) ACIKLAYICI olmali ama DEGERI asla gostermemeli.
_SECRET_ACCESS_NOTES = []


def get_secret(name):
    """Colab Secrets -> os.environ sirasiyla dener. Deger asla yazdirilmaz/loglanmaz."""
    try:
        from google.colab import userdata
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception as exc:
            _SECRET_ACCESS_NOTES.append(
                f"{name}: Colab Secrets okunamadi ({type(exc).__name__}). "
                "Sol paneldeki anahtar simgesi -> ilgili secret -> 'Notebook erisimi' ACIK olmali."
            )
    except Exception:
        pass
    return os.environ.get(name)


def resolve_alias(*names):
    """Ilk dolu alias'i dondurur: (deger, kullanilan_ad). Deger asla yazdirilmaz."""
    for name in names:
        value = get_secret(name)
        if value:
            return value, name
    return None, None


def resolve_profile_alias(profile, profiles, kind):
    """Yalnizca ELLE secilen profili cozer; baska profile otomatik gecmez."""
    if profile not in profiles:
        raise ValueError(f'Bilinmeyen {kind} profili: {profile!r}; mevcut={sorted(profiles)}')
    names = profiles[profile]
    names = (names,) if isinstance(names, str) else tuple(names)
    return resolve_alias(*names)


# Her servis icin tam olarak bir MANUEL profil secilir. Kota/429 baska profile gecis yaptirmaz.
YOUTUBE_DATA_API_KEY, YOUTUBE_SECRET_ALIAS = resolve_profile_alias(
    YOUTUBE_CREDENTIAL_PROFILE, YOUTUBE_CREDENTIAL_PROFILES, 'YouTube')
GOOGLE_API_KEY, GOOGLE_SECRET_ALIAS = resolve_profile_alias(
    GOOGLE_CREDENTIAL_PROFILE, GOOGLE_CREDENTIAL_PROFILES, 'Gemini')
TMDB_API_KEY, _ = resolve_alias('TMDB_API_KEY')

# Opsiyonel / varsayilan olarak KULLANILMAZ (capraz-saglayici fallback, TTS alternatifleri):
OPENAI_API_KEY, _ = resolve_alias('OPENAI_API_KEY')
ANTHROPIC_API_KEY, _ = resolve_alias('ANTHROPIC_API_KEY')
ELEVENLABS_API_KEY, _ = resolve_alias('ELEVENLABS_API_KEY')


def describe(value, alias=None):
    if not value:
        return 'missing'
    return f'configured ({alias})' if alias else 'configured'


print('--- Zorunlu/onerilen (Google-agirlikli ucretsiz strateji) ---')
print(f'Gemini profil={GOOGLE_CREDENTIAL_PROFILE}:', describe(GOOGLE_API_KEY, GOOGLE_SECRET_ALIAS))
print(f'YouTube profil={YOUTUBE_CREDENTIAL_PROFILE}:', describe(YOUTUBE_DATA_API_KEY, YOUTUBE_SECRET_ALIAS))
print('TMDB                                :', describe(TMDB_API_KEY))
print('--- Opsiyonel (varsayilan kapali) ---')
print('OpenAI     :', describe(OPENAI_API_KEY))
print('Anthropic  :', describe(ANTHROPIC_API_KEY))
print('ElevenLabs :', describe(ELEVENLABS_API_KEY))

if _SECRET_ACCESS_NOTES:
    print()
    print('!!! Colab Secrets erisim uyarilari (deger GOSTERILMEZ) !!!')
    for _note in _SECRET_ACCESS_NOTES:
        print(' -', _note)
if not GOOGLE_API_KEY:
    print()
    print("Gemini anahtari YOK. Duzeltme: Colab sol panel -> anahtar (Secrets) simgesi ->")
    print("  '+ Yeni gizli anahtar' -> Ad: 'GeminiKey' -> Deger: https://aistudio.google.com/apikey")
    print("  -> 'Notebook erisimi' anahtarini AC. Yerelde: ortam degiskeni GeminiKey veya GOOGLE_API_KEY.")


## `local_pipeline` modullerini yukle (tum standalone notebooklarla ayni tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['video_analysis', 'script_generation', 'tts', 'render_utils', 'youtube_data', 'viral_prediction', 'approval', 'google_ai', 'provider_adapters', 'narration_presets', 'source_strategy']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'video_analysis': {
        'sha256': 'bcee971d6885d7e58b40f37514f93ca63c0e10cb44904470d936a7608d6f45f0',
        'encoding': 'zlib+b85',
        'data': (
            'c-qZ<YjfK;w%~XD3XHmHEv6+!c6a*Db&q;0lQ=V}$)hVyy6>#lB~ub*aZHiAeArQPUA4b#zijRQ+VcVkfTXNsy0@mQW@3@V0dR0|'
            'aNg+m`|qNv2<J(pZsTQ?jlwicc12vM$7-FGqfNGn(o!v>GFp^zmZ~JXju)y}hk3MA^IdOpJfns)-EaoYxYan`?$Wsm)1{hBX1lB^'
            't9djFt1{b!2y3<pi!#djvvIHYCM%<PmffhyvAT((ZK2|_P}yCoHdRu_qXj^Umth&Hd6X{Jn=rqL(`)tNr5Y@<%{I*OR}tMtd6@Km'
            'UB|^X%GD-YMu{roPf;;c#Ue^0nM{_$0~}kn#leJ$diClxe39i*UO<B-bhD#>gJEx8m8x7vDqNHmd>JO!Sss_`4Gys$O`%33rw@P&'
            'eN@Xh8*QWTrnik3IArb@;U(Du!145o_!>uHVI2p9b6RCNd=et`B)*N*I?5weWW9M-u2m{nD-?iA2x{~Z#;J;M6R8DEU&SflIfR+6'
            ')FzD668?u^m(@0j7l6-UFHV=yHiG|4%ValN=JBe;FBaK0jtW?s_kc}AWd(GSB8C;T&B7%tPp-DrJc)~SRKT=$<r*6Ph-k^uWT)~7'
            'KMG}{L@B*h3{VgQcYs4^#qYLxbQ?!^YLUcSq&}QT9+g#|0-?p}XcKL+e5YXjLsgV{1uKz9<K8ovG%nb{;2}^zo6xk_W@!;4Wdr`G'
            'FPsvfcgE-~#{VdQpvz1x<KkxA>-YP;Ri16sY__Tp_-v-)%{I$R7%V_1nk{-gd6q}|ucDf7^K20ng?>`Tn}}g;!*ZR(a}DTc_{lGj'
            'j`1`7M*kjC<4;*?z^}ssLFk`2Xfuek+H7|!EL6Jf_0HcvegENn_RG^(AI{FzRDJF#_^%&*gnba>jKicqRQ)BaUy^O{!EBp^i)fuC'
            '%P7ZJyacm6`p+tc{oNljgm4S<yT#wC542RRR`DXn=C~O6a23Lm(rfr|SrOsSaOrmd;vPo!;pd;<zI%W6e0Ki!!@Fl^vtM35KYKen'
            '`;Yf$Z_Z!7ePfMwyxD$_fX18bmjAusfA6CChW-d|V}9}dEkl3t_8Cn3-PvCR(x6|IVP4`#n|QfQBK!-88~<5U1+brf(ChV<(F&;~'
            'OKzi@dF5ex9St4<tH`Q+5zUrW&b+FK7DFH;Xb%&+h9}cEuttoaj{lI7YXF;AnkXK_j0Raheh?CW5963lh8}Euc=G)uP@|tz2|F{o'
            'Si$ZrVZiv$<td{Uc|A=R_T#$Rgz0FN!!j;mn<iO$J)+&Icy_ogfZt|mjC&K{XXz3d0RlSv7#Fmc^p+#egPRb!Hg4QFmnNfmSitrT'
            '^XrPYoO%!2`WcW0@LvM<4E9t8dlWh|e5!y!sjDj++gDd=n^eGq;<N&u!Ek7?6mX_6xeIqik5LtFw$LSv-(uLjn(&Y6CvfPf<B5jU'
            '$DUy#&}no7+ZA4(y?rs(7^8pVm9z3NPBq}A?DvCHvv9OPK!1evfxBkNCntgVRIi--wY{F$0GHG>fkAKwe^L{CX+ZpGswSrn_2gk('
            'MCz9?siLzy&+<Xv8_ZA8L;)Q%d;JN4Cz~*d`*qjMhPnHp>9`d&h)F<?vnA`nH3qio&*LOcH_<Li9YCk*v;FfA`Mr#%%y$;X-A!t+'
            '*|K@<+5Be}0ii8t53B-ENT436A131oHI6<mqHU=L?{`}s{?M9AfLs{9*fZHaZ?FQ6mLf*<BiD&P=?6u)iwmGtjebSv1c20t_GjxZ'
            'e6zdiQ<d_OtFmQy9V_74sk#YwX?UZYvG5|qyDS$h$E!9pJ=CE)pJjLx--IxoFo*soM@!h@^J-V!xCCY`0wCO9x?#&Wf>z3#PL=Te'
            '=8iNgn6$N$L!#NtAB-BA(FP!)oa}{)2Tl`YH$(wdM0F#F)>}A?caAv3`V|D*zNpzJke`8#C%|k$^5<23kX(I^^|igE(KTRag;cZ&'
            'KUq9Uqk?FL8Pg{GIDqL7_aHf70K(AntV)-Ix*38AhUz=`AHS-p@H;Ir%xLVhcr}A_V-dl*w;X`ba}6Y4(D{o81J8D`jutl{Do@pQ'
            '1kz)fn-5XcYJdUp1lt<wmgY6)<iVgGN|nw;@|^L@<zTT{4wV^Deewe<2mwNzER{4-cWV^H*ICM11|F9SK8=q+naDB_8)5dqr-011'
            'Ky%@>2mn`Cpl@xel#~ZJnKwX)Y6&xf4UQ_wHr|3*|Lb30zB(hQx<eQ*oZLTWNkUppoEIg*LtOx+MSL2rKqb5j^JPII!4e*WesBYW'
            'g%je%3pm@aRkT6HFiNxPdJV#Rc%5b-kSq!~^a&m)7@(ukFvcLQpq2yr$cOhYMt_8UNaiK|Doj8iK)nM+6V}3TWGFDa{@X7EY6B}r'
            'DhHmp5o%MQQo>xqYnUNX{7uZ$3DvKlMP+vd^-J>s-PVyK5D&-$jJ`1b8IYU~>J`StE6OFP%2WHRmp`A;^C-_7&ueZGeJrQ%^9sZg'
            'P)IYx=CogxtI;2UvLf7O#k4QeZ#ztx91wuJSsPr4w;v8t4+<uQV;WmzF+ep2=P?<d1dZnN9Mxg_iSR0GcJCPgtRLzk%^`rSRyYs$'
            'Zq0@;Q+M<%+knUnC(O_M!0e&Lm_Np4A9_;`Y?0v4AmyIXpD4Q643MB~f#L@N3aT}8Ftfl9faHOXA=ZL!=HH3o?}FafVUZvr{SJct'
            'Jq7Ne?<r5hZb_mT*gs~=I5*swpP)WG1*sW1B`U)W)dcxPoNlX9B$T_jT*LSk>!iHV(14Jlbpv=i8Y6bdS%fz|g-3eFUtKl&25xl+'
            'Y6i2DqT0fj(Q>TbLbv%H9!l7M8r?<0nm_{@MVoE8qmM!52GxQG2F%Wb53>dEu*yZ0M2iyPzIUPJXg`4u&^m-Gj(XHW4K||q<Cyf^'
            '0yYw9%?f$p7N$>TDC-P^;o!8?JX&QrOO&X`HuRSUB2F!u5p+#yMFx#e9UTM(eBELU3Nwy(5VRfdFW}9atbBndsh0NtW(%M5Dyd+q'
            '!1qDg*D&w<PIsm&IGoEt0RPO;mUoEK<M);o9o+~=(65+|v<wmFt$2aw=Ab6+(R_|bfq!hA9Egt_P))PEa`93F#v!v@q3xgr4DF&<'
            '^FyEmG<DG*K%VxPfZk|O4sU|*>LoO9knHFc8G1gBlk5(K4utri9j66|yKbOg(l?hj^6Cj$0<+m}AWNgU=N}nq2l#04;5dTdLF7Bc'
            'v+CQ3CPir(xvEmVDr5B=6_Oj+sIXzHPhkpjLzv@Y=|SK34R~2JAu<#;>*)^FO|+YWXggnqgsxLH@UTX_VV1(D^a5FfUJ;mN?;CZF'
            'j4;|{c=U|!vV5u5QI#`i0SO=Z8OV3nG1`_>%fhaW%Xm<ZduOlTz6Zq3etrAyxn+$j)*yQ!`sok&9RHq2WghP8zu2sZ=ZOB7ZGj-W'
            'm23IK*b+gsBa2~ty(SACsOEqg{Hx3>{+tZV_(wR;=vaiA>E=mxOXj>apif&KLER+NBVvWRL$eV5HBYjPhmwca2n2dr=WqsVaPc}!'
            'x%VQYd1gtq;eSE)r|02qSl}Pm@u!3gj{tiS(Z{H+Xh>zWPU0IH$W2H+MgIUfVNLeYGQ`QEt&$E-5Y+N;QO?NR0K_mO7r|h)Lg{uk'
            'kJD4MaDx&O-dSvT$b>k9(@=g>v-<Zw3!*Z{pHn5aLEAcF9%^}sY?7Wn1INj;3Iyc~18Z1={njzIC_J*Wl{W)mA!VjQIaqVl8$ZDY'
            '|LgeVhwmRhefo^-QF0Cnj?i~-{KL&(KdBAqDeBj!zp&I@0s+wHEbs%@gU{1qk;f#1J!YqiIS$o3<N}2PH3;_;aR3r6pjnlojI->C'
            'Sc~E;$Q8U8O6Y5>k5YFD^wHoxW!~d1j{eep8nnP1Z^Im&XPX;j*!-i=ay?pJXW0$?Z0qI=8uyH08N^-ZAgSpPNOwdVeYA!#&gNm7'
            '^2Qn^*>!?`1R`|AX1^xCZ`4Z^;6~&_BeN=qJU~-0mKc~2E*9FG<$Quu8Afluzq%R~#nqJs>@KT7X=UY$7k@H2olI1<MI-M3!9^cK'
            '6yXCg>Z+6PwLm0p9}65ZL{dG~d{=^g_$%saBhr}Au?V~sMOLk90viQgQY77?_O4j~je&|hH?Sc#Th?M}?xa~~q6SBch?HF5`l}o{'
            'xdGFMS@&@}<pZ;&0z~u*^=@K7-(E-QwxmBtv3%?9uV84wE9DbsU)mpR!Y#s@oZg;J2&j1wo=*OI-~RM=MXV5n(P&B=W)R;!J_#P3'
            'oXW2?{c_qYPRVbR#Pg4^y{Q90K*8J=h4xigl>9jUI9qM#?_G8K{mEqO5YzMQF5Sc*5sT<9j<T_Os-BFGjZweKZ)08@q%&T~5M8(*'
            ')nbh#n-n%Vtuv4ZUOoN$+Yj$+^*%;7WKq_h*%TO%LY^c5LHZ#B$r`O3qmc#BI4-NQB<F69j?^5zwZzHtXjLKa0STKp&e4ao7H(vK'
            '7PurtPr2&<U<Thm!Wjd~N^-`WuhH{|UJtch19Jv0w;av$?5==Lfpp@POP^u7y9;+j)QmrDOLiKz_~byApjL$WbyS+^lk--FLMoO8'
            'a0v?nlof>=K~9bkud5I^a2c`hC0w8)YSVYPzy<bcgpS9M-;KiCEMCso<Og~s)A}a7iDsXoJYyQoZtD$!6eO-b%PyGv?09HNoOwik'
            ')2TV=L`j#w3zA3i$Rks%n!Tx#<Y#$3FcN=TgHsDbsP+kU-jPv=-CHFnhAZGPAO)s7?SLX)I9BIX4(k>P-#M}=b`ynYQeO(6BsQbi'
            'as?coO`OXt;-Mrc-T{~V1e6Cy<mB-W^&$}S94q6%MlzjMz~PGhRtG?r{qY#>j(kWA8Zw7{c<9sR;>qcyC0g(rG{u!NE5pQfYO<l*'
            'VPOqphjXTP+qCV^K=aZnS3Hju2z>=e=T%ZGql<grqV1aI1ug9+Sef=vsLduD+oYua(Be<|l3@}&7FjMv-%a3HyfjK8%LTvn(O#cC'
            '{vN|qB5fQaPE(frkAHxH@^+_Pe@uT1C)h4Fv*P#%a@k`qp15T1<b5k(CjXl)0Xnaci7(=tFpnQx0TT1IL`p$VDB$z3SazqlB+`jV'
            '*hek50D%~>zJ&8+Q9oX6tA3Ykk8w}HL}uk~TQ>kH8%@4E%UPVRvQyyqSpx4~gh0hzz+Bni$o%ikGzwMEfX}iOMR1MoIM@orZ0fTB'
            'sSNn$3@y^Ij?p#w{&@}VD0nFf7i;|jnds*uBlR2&)e$fo{U^e}f5Y4B8BivSael-TI7+h_^cpSJGk&U$Q2i~APSu42uLegSz(4qX'
            'h)&OPcXWSw?^?@f*Ym_?GBM3EC8+E*t{rbP4lkQJZwB?+>cyP$KSO81ruGlUj>gpt4K_oCYll8Agg36qY2ci8^_svgME_Y8UPlsY'
            'GKWK&1~PpD>SMS>TMs~Hjle*`7kH9v&@5R@KlkBif>sHf?v@jMQ}j>OlgZ?Ms1S`aBkdL*I1K!VsY!-g6K0-O1rfuP>CbVsSdD|<'
            '>WJ28MBaC4G(<yhsf*`&|F?M%YzAV(9*0|um>bA%aj2lw2wTwDtD@^MJ_8;Bn&}|W<FLm-=2`DMXBsn<?HW_DL0C>4Tp9O!-G*#K'
            '7`lR;(N009*2Se~yoJ%=%P~7x#b=?U{TL&2TKyQZ@E`$Kk@LcgNF(fXqg~%@R-iEUyz;?c4}{&a&6cvY+uFgau2%RSn$Ty`2cf;?'
            'Q;Su!bz;4y5TEe``uyGlbl;$!JqYPU1}<9{E;{glP1@1WM{0wweZup;vbb&wXzOjutots1ZfWm+&<29YR91>`R=>ZuB30^x&Nu2i'
            'M-W3$9=<ip?9%}{(7t*EH?-bTuotc|{A0E={A>pAnKrXF;%Q3EgpZDH9S}pJuXZ}S@9mwnh$I}AJfj%pVqn`Abx?FIcEDza*6gZF'
            ';-b80i7dRd0}KlF^X?oWKPOl!ybdv<Pu1Owt!|E_gCm^J5oqCCRQitiEY@;FNIjz^abOu8h1dfg1V=m;W9R&)8FFX(@TQ@?4pYD?'
            '9IfbhXVXb*qp=AR*Ug{G2%pP~A%-sE1oQ(I2z0yIB0(Pf+2HZi{~i@zfe9~t@b_b&h=GP?Xx#zdV8r|5>xytqk?G6m1|U=*zG`TH'
            '$nVu)E*_0jB|{!6J-$F|6|9a-E$j_yWq?EwIk^y%vkfB~zPE8dX!&FuE|)Vsnt?sZK^w$z3mw~^5z$P&Hh|Ar{dy4S^;)T!#J7@1'
            '9o2$hPYjl9G2XSh0RtB(O%=5zHlP7YGXM>eeIp1^_Py&!x)_-L^r2uLAj=m~gCnEp)J`qrTnazX5~#?KQFv20T1uI1XFw3>cb=-_'
            '$q<bevn2pP@yK!<;M?(&JsyZP00Qc|Mi&=Ddvbh2tYsN5O2b87QBHxSWq|lqkZqT&O5_O$BwmO(1U3A8M6xId{^@p{F2g)Wd)HCE'
            'X*{ACvVuT<HP#C7Ly&66<MHLaIDg4FH{U7t%)_41xe`mXyrh{b?=nTuDBA#)AwaN?9mA7^hkzQNXGO>hi6%MR8}e|B`4oeOKwsoj'
            '(wIdy-V|1gRTNSZ1Uwz=rU0+T^0fXlxR|gYNJz4qqr?w_c}AlA01y>&_&c;40L*~9FY+$HJP*VI?%gm>JSjC^enhe(z}JXVf_ia!'
            'Iy%0zOexr&Y+6%mR}~k~`qJ@NgLXU!=f$9IFj64DZ!Y?JE&G=N#y0!Qz=MG=aWwXDJ<->GWrVXDxIV#6{~RjV&q*_~g&TOn!yl?2'
            'f_rBYh+Ag}7%a-$tk)hP<&283XgQ5>R;NuY*qnlAfJ3Bz>j7`@Z=zxNW%r`}lo(Ws4(nSyP&Bgm2dot?Ye5@JK7f5fYAxY_hd|zx'
            'b$q=xYs;UGF>Dj*%Y6ua1?H%uW+Mcl{-lnh(eX(`x`QF0p@9fi28q1#i8FtB!=Ny3cThtPj7N5heppTnn$2-3!#s|PLDOw2!iWC7'
            '*XgRV+bB<fl0em+8D)*7Ddc;X4>V_nT@Oc^vg|c9(LP2;KWtEdW;qB!OpMaP^#Q|5{V<+9G%yj{;n?P>S*<7I@18u+d<9#19xiUW'
            '0Q}pNj#QQ*9T&CKc}D97Vj|`3Ew!~>2aQpt8i?wf4QI2Zq@R<l7nc$+c1}}ahq5de`}qVG2w8EuNGdo3*l);DU928&!W}z#Hqey7'
            'k>dpChB2(<2~lYj=9o%IiGi}1z^pmml01qq1%!k^2!Q6G2xF{%#STico00WH1FS*8{Lyx3@Tvf>_-$20OBSdpfzO1xl7}Z*5}>Fk'
            'M;elvN9z#HH8~?mTw44+utV)h1$MefvV!t0*IB8({b8yy6G(TRWlPMqQ<4no*x2l#e;t0!te`>D!Vq)}%gh9R>vqUWM5nYor(Mkv'
            '*WB2JXv5L^g#)E$(LJ?rqp!Sa$Y#H=w0St2M*3}I`&crgXR2yrAx?-rKO=R)>MgJ>WtQ1ox`ruw<A>ST;qKlfv~>Zd@a^wC1>INI'
            '1Y{S_>;!-de=Q$53nk~FZdsGSg-sT9D@4wvta#0QDSLy$&py*a*VuYAXp%uqO?q=V<U}Yk+s?@5b#H7C71_ZUQJv9TnD#y}>oY*F'
            'dGn{Ki#;u=C2$hr+Wd3Sf127DzHA-%P7vQRp@7Z2&54qgSEsc!2D=jyXq!dANkT2tBbX-h#$;SLq%~?8{hzcL({YAc0%e%Q*J-pg'
            'blL#c^Fp&q7)^AI^XZ#!9je&%TO5&oH--TH$?&+_PTnBcGZeo;zgK5pHI`!J?6I#v7H`ca<XWLd?qCfsr70eaCE5B8ud|2z?O|y0'
            'N6~?#Z`AW>h4a0|_>o&6y^!QT8t-FfHas-qlqdp__`uRAow_X72sD;R#_DCsDL0&nPf|l^L=q7j@Ql{7g%(Hn3~o>iV8eMf^bXsg'
            '*MdSIGwaFFIbZ`fb!KW=T>IM$^k}q(_IdszLH}qO=Ug!%Oh$yD(e1JAK6c0CC}ckOw~gI>dtmLYX#7qaIR~6A<g|6(>~HLMYd!ZH'
            'I-8Ho(e?xDOPl`SJNqYpb>B>L^Om?ya&wjuAM1q^-H1qVX4Op?)S;H`N{v=DNyFT)!*uBhovMwO3vV7fma1<3gSU{YZ+s`&e>nz1'
            'xHhfH9!=<<Qlq|FfqGbV+B;wo#b$(5yWPp2c5yl*#bE}sMXz|f0r0yg<4Nr-aP_oj7>jvWu@h5UMGa!CM?x`g96`x$e*!ZiFbrjM'
            'dvb(Cy9R~Bj&Qf?vS?R?UF${c#Z>Pl7^`R5W*(;?8wg)s;*yp-A>a|HLiHoE(80Ega=qc40DSXZy;1O#S>b#z2mqOT5`CnZE#Yv;'
            '@oPIc`Yyuocu+VeqvOeh2dOX)p-5s1AZ0R^Q0ePDik2~)8=QAkaAYi-{BT!DAJ8#oGg-p8o||w0fvU?qG<`63L7*5r8o>N+hVF@i'
            'MDbXd-q22^X};x&VWdh};7MGgivuM<@K5n{hmT-|2-b6qb9?jly}Cy1V=W-JZVVRSi6ZlMCT}U!z8D0q7D|*flUdiMOU)y+l2;o>'
            'S+d8Y&u32{`qZpN{B+;qKa*pdwHV_TTCALY$PQ?QBDD?zZEY6-R4XUO#7|aE5Y3%dr>?vbEwrfYD$6j=dgwumRjHPH+7vlQixd~P'
            '9iz~4SorY<@Z8cd)lb%(aI4U3OOIB6(i1X(E}bk`F9PF&o!yuAs9I;Dhr*v5FRv3^QBW}Ccm)_kdvxBvcslwY;po%<`Cp^|`@fGa'
            'zwHlc2qs6zjm$uAZWCkp>cWXMxOB6V_K&;qt&a3SA3N#P>_#7Aut#gG2YqDs-&DR`EVGMZ6BajECV?Fbp#Rkd81>)uW3yO+PuuLw'
            'KwEhY?y^xLMH4RUI!(qSWjTY7$5xRPXGec&e1&B_IxxPXHWW4ED-7AA;T`?)?)$cdgfH73x5Pw%bXk#4nfM5<uHU~17;+RjU#iCr'
            'a{Y!R($wvWYGd(=KcP*qUQ;4Be9~gUc%UC%b3Q}+B1NDNYK!Srm{AuEYC0LJf1d<_$2ifMWUK@JdhvPe%|mYtH`~qLG6q9PZx(XZ'
            'arXkx>a*RR$R-N8Iq!NNzQqMPMlOQE%m@YmlSWcRrHrP1d|3Vi*}JpQCv;@t9&sXPdOCQNkJy`N=W-=?b%s7w3>)-iD<>Z6hh4ZD'
            'woTR!@wIZqK!=1uo*I|b#oYz`W|n@33+L?oOSZ4U7x>DZ=e;Fi6|QdTktE3BlVNj1kgVLY!6xJ5CwtH!A9NEAyq*l(*B=(~iLKa?'
            'KQmoMABV<Ek;0a&HYD^699&pd7GI_^Shaatjda5B1@&`@#08@0P@O!8+M2^(^!*;;5S})M(P%}F=~EsUZCze4UiC)}7&SG>$mII~'
            'M_fmGIO-4Vlod#tIeutiDMlTqkaCxvp8;D~qKY(j4CEu(nk(1%Z$7mh0+!(olX{{;+he?b&{^3D(Eh5W58r3%Opp19?tF-D)_=Fq'
            'C5)EUz5#lfgN6@wz|Ln|KKoW2(iT>@T2hZgL5wBA4Hu~Eds6Z`nc5vvcG9J7&$kc^Y7(UBaEha~q=jrJj4?p5t=HnD6{0Ys038HV'
            '#nJ$dyV{QFeC$e7<cXXY0~c!sUQp`@`W3@(mPh%?SWfq;-a$j~V}OW03&~a{x+VA3C~@NsJTsAF9MTq6ZQBB5b~?_R!ND329u-^9'
            '>wVSTdc4iHgLdhO<lCmfsR^v1;S9TsIn$2kO_)uC&b0Zc*{sc+Gi`h5Hg%WYLLn;T#5qw4u!sP)(c`X04gKBXz2cR%zaVk<Ay_q6'
            'YKNr5lDpveMgt3wSIqER)i7;70JJ%_PLN_0?E>iU2m<LpbQ*u7p4vHqb2N)pOD=~CLR*444x<s;_hV3Q=3!YTXfaY;!C<VOqKx7}'
            'WQhgf<>cH#xdqKN#(%)MpA|YAbl*rPg~Eyjm>=rFBPYJaj;TsASM$A%!lcldjGW~h>V#J~lc9GcTn`?l3Z*!Nc^Q`&tE<jS;`3M~'
            'nA1>euFDxh_>hW}da$^lCyWJn9Wc9M+_g08u#o6-S(GfPd2y{4izXFx?<g`(5sa4DZI&6QpX&xkzG!lEd2wV*mPeQOAm+7zB0lpV'
            'C`TOahD9@f7?>l(c0v<b_Cau-%nR_E(7siJexJOp9cx8;QD6O+`rF$#AI@JtReyi`HwAp7+=ZWFRe{(6y5T1J6!e2VJkhT6EOvnU'
            'B=!UH97g$2Jf6rs_6f_5k`AL|qpXJ=c5gv(T__p`meAT4Oywl)vFByBilby{0{MlN^*gG0gp}*1{_SM4b!@3*Z?5Ba*gS2I3V0OO'
            '#FmgC%ILF<!xU<XrRF5*UAM$(AE^Px{Od=QWO@{s;)3-YuC9zt`)(a$loq^7;uOn7I5oZELLqa+z${pauu4j)TrG2#dh%Q>EYkIh'
            'xww|9Qp~i!jsRuZY6WU`Ar-{KCFT=PxQH>wGh_v2wqZ;dm8(EDMFD15<ye=_NA!)=iz-P*k_B#Ft+7@&^KT5FN+P!vacba-Sldp@'
            'P2}O7RSHeX8P?aC`s|yX7g>UZFHA+UM|h3_V(bZFG?=cj+$a!MBVE_<9T@BlO|QYBxdt{Jey)Fbi3ZYvt`VnqG#|tu>vOavYIIX9'
            '(Nx2^w*-Ts|4JY4zg0G&MfAFii}k4S&7qZBDBZn9QnQ9Vro(A48S=+NAl0~>_HTev`a@lZa=PkI9)G3}?qL_ipO8BWCQ&I^5@RsC'
            '*s>E?y-FpY>2YA^xYt`&2e1el9s|zn*0az2@5K>{6-Sq+lau8=6>_iRqWFWp4U^7P$G~j5u)+hmq0j;9xbnmh$)#OBvte8RXEWZt'
            ';w+8MgJB~n-Ydo8mSgd!=zaK&dL3>}DY|4gR=<kygn9KtU6uwfl3@10u1G1zfgHjEhu^$@d;Z>36CoB8>hcsQGwzTVQbCYVR|w&t'
            '+9%^<A6kylrCgEcI+nN-eQGS@)hf!Vuqq9zgdS`51|U6PA~phB8mpuKek44a@{V;uy#j?|tf4dZ(0Oq2nq~o;(a9+sc2ad<PmLNl'
            'MDmb})OZ6wXC$p8#D*@~U@=iwTyvQEDF((?=}n4Pnfx&FOTIW|U$EK<gQ#oI<Wz-fC`TY8pRt5ua8~#gTHLZUa!!D%L3MPsx1Ekv'
            'pg$k6%`z>mrCPe`usAgCZG_9is%Q9B8#*dCOdAy-J9$iFOtoV)@$DZz5atJ$y27ol^>9qJ9(uM`2jX9aq6zVA)W23(;B$>deza?v'
            'E9*oe7{%7^V5KK?%1Ys-DC`UYF3%^3rS3Q<1oQa)hs0<6=)Ucw==Huib;OkdRnxM<cyBB{q^~_-!bI>Spr{k8#6_V+7m#oe90-e~'
            '+X8EmLd|%iL`Ij-pc=ZyOCN+7>1GEk>C7SWqBe7!1lcALpoW5R`leg12MZf&xvCP;c{g~J?F!g5Xb_7+lw*+<(WvZ(93EH3ITi#<'
            'Hsjul53gR$&YwMfBR3b!o_~1vlpICpXU{Mk=#LnN3SiC=Kbwq%#A^}<iPy0rOETqMNF}AUWY2f~R7<q*3O04c9)!pDR~Yo**K_<-'
            '@ROyuCK4`)I3YA~@C%D}#9LY>@cP56_b-2b_44c;4FAM>|N7~FSU08|JFnipK6~^2eD?F%JNX2IdvKV)l1*Kk7Os<Q&YCz~zC?FO'
            'i6ug!d#K1R$QsuyY@4d3xP}Ptt}X?_=^M3B4KfCDOUBh$okt+s&Ja{2HTJdOa%Io@FR$MI`RS|K*_*R>fBF0D*Rz*@`RjX%Zu;J<'
            'yh!Cq4y?N9RatzfMB*8DK`8q5uWw(S$x(JBd5w5PSQQO_e)gLD`p6YvPl*!rGj6YA%BfH*e`BUmv@EC`S`_(2%$tTg;{94@rO2w#'
            'o^UoL<$#hI92CAPFR)+2Kx^GZiIK5uC+fP1cao{1r3=<}lw2@oiz=@Ead@r7uP%4kCLjKG)sXx;eE;98A^j$5&KDfYUsp;=3hEYk'
            'J;;|-QbMwy*)}vBAeAk)la(o-)lW*EG?R><Wg&Wk-n~B8{0fhVE7SZ+g^N5ZP-N9x>({5h4A~$Miy)<U>hlPsM^dL?hb?t<8pyOd'
            '&GuAbVlN+5<y|(y9VQHbrRflD7rae**c9hQf?c83Rl3Y^yW`z$(RGM5E9Kt0t?&rk+RPF{+3V4WbEZcyuMsDO!$j!TuLx^2@o-ok'
            '#5^+kgQah|WesFIsaOtuid8z(Rm2si|9U0_&Tzg9Z=*UxfMe-3UV{fOkTIWjUl&6`MNSNH?L?&2j5}Ww&%yzY>DGlts&zk~_vuy?'
            'W>NOV8>~99D2d$bIx18B)D94fm&Cs6uW%vgsZ&G0jGT{X+PPb3A7e!b8f>eR2P>UxlL`>lXbwUcoGL@9Se>fR4Cs;i<2}A~3#`*%'
            '4!J_=I$B^n!Y7sGDfR|B&pQ5OUlT=iG~=vtahxAx4XbiMojhvLgJpEbi$|&r1$s;#ypaM;Vg`S>E4j~sJ%C1_8!|v>czVEH6uf&R'
            '>Jk$n=M`$U$mMm8GrWL_9sdxFLHB_N81O^W`|bpt-QS)3+Yf)VZsIXlx(Fftq>h^dkz*3?9pV9usbu=&$EF=dVKhpbHb*UgU7Rw+'
            'OTbaF203^zIv&=rI6x_cV{&@jz?L(DrmEGBH`<%p+HSHPcwHaM>LV$T<In9<Z(ZHmL{Sz0nb*-&^t*y%USlW-0^f6^JN$O)Ov1^C'
            'ua9A!8gC`DOY(x9%Wj?9oI7uQu1DV4!qB@34y$*7^_d+i=Y<(^J%rIQMpjsD0g2jj>qnE-8T^1YIWq@xW}eb%I8+a%w$~K^Ujq<8'
            'ES;KiVv?|k<{fQ$mZO2<q=&jL5(|W4`Y^>X>g*7fS8!15kmJ<|hVD~Dd0^YA3%e{Cg0X(0HI=^c1FB{1jlHoEyy{1RIx(znyr*;Z'
            '19=t^+Bi&igCOYc(y8~NX&U2SDe^Ti(hk{XW=3HWji>Mzu>F?G!)gGC64lZWt3>Y8gAW5tW(l2+^L<xZ$j>nd9(>U&tI=XsxWl5`'
            'wQXT&bZ1JyZ{#tEDE0!hQT9KhTBAW0Taw!_iUtHFk?gn#c(O4Xi^;8LmRXAvJ57tgmDw_hRc5KWJV{03xnSRu$-f6!@6Py&3Z<`h'
            '<BMuo`3E86%9!A0oD+97l!QKG+uLBU&kfy-h;DY)-(0cl3vNKvb;G{)I>9pB=o>LgEEb8^v(b%$`>ztT3Br>KtT%!4v3Dn+oE%M('
            '4H`SZDhlU9dT4g@EV+*KBwD4<f0B2hk6S3pa2;DOqoPKe<}-_-{iG(nwp6*L6+K;UxpWVT7<Dk8u&R0t)<r8xMuB*UXnRG5_EPz@'
            '0y2q*U@A*@Y)3S1!?fkX!^hlV&*c{_<ti!e6xA(Gy`fe?`*nyobpHGib-VW}K!4mpugSjl)vBb9`lm9!Z`JYekb0JFmkc)BwaNh2'
            '!qo9<&3WwwOkx0wF71~M_kC`J8M;*(<JIFc&DQ#zWi554DPvD+Wl+|WY<Yh>Qaq@YywOO)@~`e>T|nO_=}q6sbh0mng!Op{b?$Xj'
            '&|FX{g`^%tJ<8;h@src0IEx#-O#mBOW!V~nx3p|6EbqoiTP3P&dK1`I#tOx-*(AVy+&5<a)3h70O^II+Je7rZ7>$h3PiLd`b%)!T'
            'VQWp`pk?TQVoke!EofKzOyT}gQY0v#){e}1IAr;uU}ZC+?s#QWhiHn?d4t}=akQ`}&?g)%?Rapq63J8d+qR(FMl>-ur7(X&Lwg{x'
            '9_wjuZz*5~@o*@Xj=7#rT~V@y5Umy3TNkYn4X;Zm8kk;&*aIxSGlQ(zT}xuv9zD;Cnm6U`9)M_(coSMpZB&fTpGgbXcT<170<&KB'
            'i7u*F$CSwWiD}wsp-sol!2Rh*zd4#to!sB0NxTcw8{fs$^e?@9bN=%AS<}BHk!Ouc2Uo~?MsR-BsYvXoG{KK`YB~9KQIl}aGaZJD'
            'Ap$1AD^&k}!UeWM^`BK78J{4BUFetEd4r0|QPK1t5~s!Wt-HV~l-HEYz76%Iz8l4*HKeSV7eBBHZ_yx4u?3`k2>3PaR4%)TH<n<S'
            '0i}+l=<}0GfO)Jhxug|lMT+iS0v!q~OcC<UvcMz+I%l-A6Un%!v`dp0@8x-IT(Gn@b{v!LHX6xV*oFzjTu)yQtB&y_E*H9><XsAL'
            'q5DM(9eS)IA~FfLOX<!g@*x|SQ;AAp9g8EDI!{5Y6<L{|RA5tKBEmeA89Pq3Cxbq{5}!3m{jiVL;<E<S<6z<xb3h_5YEN69+#{(D'
            '^zt@XuglifelF@jh-M&u<6WKS!~32iU0_sHd67oNe-WpUK?A>+hqb7qOO%nu4q6gQjs%R-sz3_hA7u98PMqRUSSoUAeb@4tho+&C'
            'w^H>f-g-P~Xhs{H_F82$ZFQ0%gOceLs<cB&TGqW80@u>OPluD#AiW0smJ{8&<67)q5w$W0QxsEXKv{mC;rgNRyBX2-OLs^IOspZn'
            '4M0DDA?|nZ2gbADqogJF{DD}3N9v(z=stu}zF=*n4&gG&$;~0sqEB%hGK`BcPjD1{OKu%LP+FCde7VrsOw6fePoE()^5eV@JEhXS'
            'fL!5J&-<SePtBFh4+^OIOA=AhHbTNl&?%teV7W|6TL`X())s$jNKNN-;UBgDYi2I@apSdI7}z4n)4^pAhq&bIrsuH!M-SD0{aTx|'
            'T&6Q?x%w!<+vi5N-;H2i^Q?a$ptg&W%*#DjBOOw7xAQ)PgG){Q&W3i(OcZ+ykLADkG@5o;c)Y9;6U!}$Bh9ELCuZUW$g@^9NMVIY'
            'oV|Jf^4*zzrRR|9zF>iLN@LLCHPR<jou<qyy1A38E@cw<7hhLdRp>Oe!O8cN0%H^;tAh-_+M;g7Q;<9u(@oM+LJvh*n9~+7d(>mr'
            'Qngw|cjo#%Eq_Yjo-OxoGsG)^4YBC2A3A7BqLNf0vh|z+l0q=2I*#zwM`C{>>x3)d3QLqBPxZ}FAkWiX$9Tz>2_8nFTZ@epyo@P('
            'ShN>rNW!xwD_zSt(_$}1%oK9LBRWA#?eB>7y^j<HCIa_BWa!W&7fNj_S{4@5sQeVgk_^T5jyzEF^ulrD+`-v~(d1YpSQH|>mvGE+'
            'Z70ZYxWU<fygYyZ^37im`bgG%8*e$s2Sj&DM3SIAwE7SzjWLerz``B%RaBDxEB9n@YOh^zOjj8L*g{1m-3O{ij>H4{Q>{pboQueD'
            'jCcnlD-E9C)A$B(;6tNJ5Sxy$xN{n>P<I0bPHs|jXcDjJWFJ)OXnl5#=#N{eB#9C6qApOTDPL4@W$}Y6uY#9R5ACG5e_*&e?l_88'
            'q?9f6O0{fK#ySP*!uGl{+0%eaWUYPc%;{fAG?7e{YL$^Hj$t(mj6Cp^nBO39%{skw=%Dqe@y(&pg8!nfwaA1*nTv;Kmf^$ku(4?3'
            'xYp*?67P9s(Z!+}oC0c2W|Xb5!t7Z&8MgQRNbLzg4s;FeZ(#F%?SI1F>=da2qvHocgbTiU{e5NdNwJ2A)YS4xK6KAp9UiO$Qf(T0'
            '?2!EUF1+pQ07|k0Osy*ytUwVePSA`W!N>k#?Zk{6qk-Bt8v?c6>tlh!HQ_AT*t(>1Z_VDY=cYL5O}}w<YhWjG$dtQ7G(BkCST4pH'
            '*a4zLYflW-+jIIOXk6)vxfbSfUAD)d>gt3TWBdTHpeVnk^VA%vKjLZrkE*cmM@FuMVH}aVu43V)Xlnzh1TT)4?SVcp6I+161i(8%'
            '#=y)BJF|%st_;5~kV_X?oweaD7iKPE^cYMq7|&)YuKMg=-$3i<dSG$O!KzQ0q2WAClUUe=+J##QrNt^8$#sOH3{XBW_V79ebc74o'
            '3xm&g_xC#8aut(J&%eN!;?6nL6XRcz7XwcZ%f<TC2}^zbXExZu=+OFTT@USMvn&lZB_W1#&n#msE#A2M<|U7i#uXqSy%}uc0#j*S'
            '=55iK5R!M*3-FO%&vig|#@+bj-Wqo^^_~rTdsKjF%d?^5@R&OMx;-hB?&;w%20HDBR=-Wfu@ZsocK3M6*G@_JL$rvKr3rTU=Q;I>'
            '6V*ZIv&Ttg9DVP@gG2hWWS`Z;s<jo~hOiAdz>=;Z^*_*8?zvpF@!GzT&;C@#&TCXOhvVhr>YOUu7$CT4vjrS-GpTzZ%6K&A>(Y8x'
            '{Xf&v>+CR>He&26qNB5=FoXaYe6FK;F06}pFELEyhe>ch1}=0H--LF>2CBHC57(MLFtJ#Ee%27l*VLL->>kly0L7fytm~9?jQ}o|'
            'Oe5&#Li#MF31ID`htE?p*P%7y2Q+k>b@+9^<e?)HT<$rgbzRi@E@r#d?o4hOZd_O0teez&fAzF!*D==~yOx@kvQGAO$ZeJz=NfBa'
            '1AFI*A)?lOrp@4q4witR8{fvORtC{|0d6-(@zBk|Vdiep;7~{&^Vtj4xjNMC;Sg?OOksF5y7w(w^sLQfQ`c(N0>-;t1mjJR8_wL@'
            '35yH79PW~oDDIrpz6;NiOoS+OyZ;Wupc$dA!}n?^7b34;4xNYOHt+4N9M%${=x@y6pq;L~P;XQ0RAT?M2dmAYLc&X<N`Es!px*p@'
            '>hrf0fNoBpl`oODIo+2?+N@Yt+r%4gO5Y4_hQ*zeMf`Jn(2ow_(8J{49k&g*;T$a!Yj5SpakMk#M%#PO4<X{qn99p72Tl<X=YSel'
            'TNFw$+KX1qt^mwi1=Y)1c(vbasQW0<sY|*J^FrD>*OCwULTu0;B$DvHA#Q(-O2@+H+Z2<_jV>n=65n)^VSq)R-Kp@7uUxY;uS>Ft'
            'Y0J_<SevxsYBFp^!`|EQEQIXW=I;GSQ(LEF`P&y(i~tsGBFd7aIb9q{mq(N$X||KE#+lOLthG^1pH@+%^_1MTA8!KeLFMZZCCpLA'
            'JZ(wjXzh=qi3j_NAAFHA<S9h0vM(=*EI_RkNrYpqc<f7g^-4xJ&3wRg)jD3x<J`Vo0adbC^7l}w_hfGrx!cA{X0Bx%{3iMv*}Gc|'
            'ZDK)3Guu-bW)h0`1*@MWNVlqp^Dy}$dz)Jlwmn?TAbeP>;cZq(b4{{6Xz_OyTrA43xKtOzP&Y|D*G1;OU;+=8erqZPcJZa&VXiu~'
            '(GEQQ*{Z^1CK()7F^(r-*NmH*I$u4>krF%B6o@fSmbQE`(=Tfxcj`&md{i$E>@}<vgT5Y?^+%tZ%EP_2Ve5-2rv5EGQ@Wf-*Lj{k'
            'KYQ`?!>jia$Z>92UmUzo8`T}ORAD&d)b*dbr9r2Bw;Dck?Kxbi+A{$;v6{ABiO6>9g`FB|CUK%k*<S1>tmCp;(APde`(K*aPOFy0'
            '2%V?MdEI-qxNM#;0Xo5|duPd46qsd$M&C7S+}Ep!_U-0-?Xzr|8V3gNL(V<_88{S-fv|W%5J`=lXz-f2p02v7c6NE6+p!nqiEt&<'
            'ev8rYFr4<;Jv+l;LJR*s3-WB%k-H$(&yIb2k0qtpHel}~b~|Wk2(3DFT{W{?qa+1#Rbw;xsp+qqG}c8(y<z_Juh4GI_HwY|SV)Zs'
            'F25V!6*KWa3Q-wZq4^=Y6)e({%Oj4RYs6hKg(_;p8^(q&SOzVf)Z-Np_Hvu9XLAat<r7f`ZVo~RDu=Dk+P6A;WJVN}>s_m-8=@SF'
            'gBtj^fp}`=o=9j!o0PKQsgWaG#Q1&-T=uTBG-XN%NnnXi#8NL%tmSAmu3ABX#a67XFB`eB#jRzE+>l|46sO8YKB&+8!v*hubmEDr'
            '5EBZ!FetkmDrh@>pVQaDgA_v15?aa8##j?B+!Jry3b-r$xUC!7vOp^18j?*{CbFW7Ky;z(M|@oq0MsQ%sbH+NP87>Q+L}b;Tfx`f'
            'GzM7N$kX?OAGtAsseMs8tOlfq2rCguO=@X7ZqS`o9+gIG0y&&vyHWzj<s(ty0HkbT-#;>!rsN)0G;E;r-a9SY?2vR~w7uwVYix*K'
            'Q~mtmmX4jU1B(5Ptp&A>)e2xV-2w@f6|SVz(*71s(jV2YPv5<H`Q|UD_C0wRC=zY9<xUN-Vz|DGuVjwbx5FPb_+GEKNYLBl1c#W='
            'wKv%)+p26%-r(E>=a`!UG4M;_G8Yt81FB=0Rf~1Bq=E(JE)3Qru<khp>DHC?`c`FSH6D*m3)W7#g_dZRrb^!<t)%u307X^;h9uuo'
            'Tw=%9Ey@bE^}d$eoMIiGJo#Zpu6i;C;8y^f@wbexS^i&##Ts;@F@HV?>WVs4OoxT(&I&qwIYpJ;;L#c5-4`6=d}>#585~bezNZNE'
            'lkefIpZEKoVS)#vO*#b6N$8UI>*yo@HP96mY_&z=IAz|pY&pYAPdVktMFP;lIc0zl78Iy+QqPGB)SIf;(IUqCAuvFI8Lou!0X0@h'
            'Ij++-uv`{~fMW>Ocv)?>MXL@)<3aN{#{bO1ViCtv@>~unc$}`>VV8&%#*9Z?ndQZF&>y0Xeu^pS5<NM^;9nMw$2^KbzbaRwKlU9o'
            'dP^<i4O;NZN};HqCYNl`YnzDVayg;ux+w}FBySF79LPrD^?b(x%kuOejYfr~tcjjX=ah-s?GH_;RGb_2{FI7zh}XyKGc38W7f#z!'
            'Mhs|H0!Wz<MjD2?@=-$3c2{tQl<_KDloFqZQ5O0Uo?&E!r-Z9agclUzEW=PyYvlTZ=;$Lfc93e0A*vOI^*gsoNY}P!2_}nC{mmrA'
            'V-lHzm9)i?FdSRE1&)6kaphV#jZ?~UL1P%qQd>KgLi%1w3gDfakPD}xIp-Ec1QLdAir|kLyRVL*vw`d_I^KwBEFy}Z)Ag!AUvhxm'
            '8)B!$%B4Q6MRPb+;62XukGg=AG};%;$8L|=yR%<jp8b0EeD>$R1L_NvVSLV~AL|0Vi+B@b<S?N41~x5d#IZIvIo4ORS&-l*aIb9x'
            'xq(&hciW~3zG-=H4*&e@)!DQ6Fm64Mr*HnIw%Ml`=v1Md60Mj8+9!2o431$1uszEq1$fj0rtknanISmUcyOISK0RF6%iF)aH{{{q'
            'L<-h>J@1*MC+4?JdHUJu<KD8y{@BnHHqk9!f`}g`VU85X$vxhP0xP5|aClms55%z)__4_xA;Wzu_d`m(xD3RzAa#~z7SIc7Zv!$f'
            'srFkZ@-8)|eV~V4000915^p8m-~njx;zPTy3p=?>YK?Rsu~NWNWRBo)m&V8$L#KP*d3`TU%G8!ObG~qfaM^3Afp5};<i=F1(6oCG'
            '0H2+9e2{qqs`HFkv;CF~Dpps>-kqBUNHiORHbjD2OO<{HZc9@X!3X%y%#=g(I<_DjP!MqCBj1#P&x#!~U|@Pw*Rvjk?X7Ac$`wQ#'
            ')w-jEiu$<6_`y5e8!E+Cqs2dskYw}v$TXYW8~Pvedw<*aUxc63F^PHcr#Out8}$#p2OD^zU?br5j;Lbbl_R;m{W}WM$yr5rKZ2@s'
            'Y8M~qYhi&xW6;Rs%Ne$1hjlKp3-V#UY`)bR*{RaREGH7WlIV`(wDy3h+sHn>Fi_A2*m5MH<;>nwLodX%z8oMy1z@M%zI%T5?%Yb`'
            'XA=@~gkzMDLQ3S`KnF`Q3tI5*))A#m1FxeLn{-DR<r09a-cbTLB~zC=DO@*c-Zk|NNMWEF@~2#rzfV~dy8`WhMRk2m253yUFZy+g'
            'xXA>{Ru-&y!`C?swXwn^WKdZ~^)pD^k(!|0pfI-zphLvy6c(_-l|ca^dl_A+$Lea8C3xjD;MqaYRWP<qk9I?Qm5zrkU$Nm8SWdXS'
            'jp2Agv!bhR^gDX>*By=a$KABjv}x}r9@~=Rl)iDebK`@c9YS4arnS9(?YC}Tn&$&(e$fS^YY6`P#tbMOxG`{g>VIo?y8E#_-g|wu'
            '+roO)a5UKm;%W2Vg~l;QK7wYxV&6R-0F?Wn>uYm>*)7b&E>}d0U`5t>e(ro~bOic<b%C}I!m|DVe<4>weun1MF-SlZ)%6MHJ+8_H'
            '9_twv+QI|60nHd*Yb}Kzd1{KTxoH9DHhTcLFR+OO9Wdn#2?1#|E)Oa;6^i7e$c7^>G`0#1I+}?y8mIR8-WdgQ>$=y^?ew4*puxAi'
            '+Bb9!bH3^NFs#Z9)+Sz9=j<uRkT*z57hW|=Z@Y2C{Isni`z$8=4H$J*93Td(D30X+@gB$1bBkgH9b9Gvcpga<4+WklT)q{w{%{r5'
            'Mk}q&UtHSyFdR-PUr0MJ0EcWrYm`B~Xh<t6jfxcWvx@9fQeo2Ib)57s?FlcPYZra#Xy}VXLnHPKJ0y}lr_j*d;X`*nAGWWC9&Osb'
            'Xgt~ZhBeY|cz#Y{5$FE3nnv8PHd+Z&C%K7K6mCIuIQq=jp&hB{rigE9MXO;`;pO4St~Q32D>+geya!m7+^S&hj;T2$w>f5Jl~l0Y'
            't`pkq)LmNYvnlX)4#H`z8be}Q>*`uFEb7yU&ks!4+Ed(KThW$&Z@9w9&b?l3>I*#f-r2WpxB`_+R~xm6(CDqEBZowVz3Q>q?_4$4'
            'uv6pbcT2N-1M{2F*FGkiy|uCB!#Zqfd0?My9NzRvjJ})9noYTd>GgLvipq#2g4p>e*t5$qqnz2F*pz4F-I#unWYDpH_HanrQ|*0s'
            'BDrDXMcjfy?ORi|zAY-Nl`H-GTO|*)+14zk?Q{wbG2(W2<5ZGr-5qX!#z@{}hI`1edPpB)9JPyw$Pu;6Zu-F?=a6L;Z1T)D@jVdB'
            '=$?_XL5RR=V_}qH+APXZ5k}@lzR7S_RGUpmxnlbl)90x^02w;1IBo}LmWN_@Ok;0FIYx&>MG_VE0o!59bjQ?U()@bUB(!|c5ZPjC'
            '>{zN6`(od&c-Dz$Pnjf)cUv33YAnof;J}Hn3~ZW*%%MQ8?jK^lJaop!I_WxWWy{z;+15ba&da9TVs)idR2|BBSdk}F_}RHxLZWna'
            'J<o@Oa3Ai)jmtGXH9~U_9(E%>mq+V5f}3vOKO?@y3%s|y3SW9(QBf1V@Q*+o88L|!f+M~tGUo5<QH{jw%Hn0TmD>WOL}i(cw6OV@'
            '1A|MBEew}Cqr`QdYHpy0(NWq=Si`2(YS?0o8#c^q-p1mhLfQvI8<qNPy|>3wGpXHB4KP?h?kQ<WUupLf>&YXJoPM@?=6j1V-x(bP'
            'wdRCw&LT`$lsi?Qsq;VN_wE%+I6@;y_{h0bKPms6UC;MHFU#TM7Vx@UjP}{u(r?>{{Zf0@fSXnNe#`OdI}O{1Yu_rC73jF0WxQi*'
            '?!kAWVs--F!Qx%QM+X(O8B?(c{?z}b=OlF!?{Hx@Ia52_mhT!V6qc7qg3(V^KgkwhGNKw)9LPPoJ!$viczj~7Gl!t6byVdjCNHeY'
            'j}{lfT30`E+L)!f3j9L7di5Hm7>;q${xJ>|*;aEnKGyaS+K!O^Y3s>iwQ>>#>L1S3tW<&ehYf?z76ib5wOJV({so&1v~ks=AM%hp'
            'aiCr1YM&y}`0G9D%CIo>B7<lnI2cItapz##$hm*oO67jqg>$4r8}~um*IopF<C*VPFyX`%1~GO76}S3#9=m+%;NaaKTMAb+Iq_qU'
            'Zxi^*XdYq!wFA9nC0aTd@MYbWYZFL!v*Ege7am?cJb&B3;$>SGQd%4@!=u>-Ke}={inbh-u7`kXn8al3fC*q_cy}ZgQ7WS7X6*J8'
            'w7X1f?f0~q$Y#`@rbN{e8r=AZVvxxz(0CPeg#UWUUxwGZwd}EK)^`)S2cBycqy>gN<KvSj_?Uu?o#)3=-m@QVGFlAprOWrw>I5GU'
            '!-3b>RRS9gSn+HblkrnpP5|t^RkhVExc7ziT3ZESzB#2^=<O1d@tWxGIkB$0T5GWhlVndB)!(f?>z9^hZE7!gmGxf4G?W*-MV1iq'
            'K+V){kl`}imazU-Whs3>igx3W#jvSc{1wi#5nVq##Nf;8BpOL5@2Eb}xkjJ~r7-0)2Pv2G<ocJijBtLZi#2G*nDP^x>RL9sP$-b^'
            'b(PD7^+Wq|cNL2Q#kiN7+KZtszM=27rs^9*CD}ce$!BO%WDDCE!Ii*<Qlh`ER{4M?rNd`gN#a1+l*O0HWVXwyvZ5+BY?{R+V#+!l'
            '$J<>xr-1*$y%Yh}o_8PK%zk<K{Os-Q>6@po{(k=Q{6{r8_Vy+!)*{f_!1#XmyC@%Vu~S?!GbTXJ9Ajw*jby5!TUaHK46_2up;>EJ'
            'ueH5?SW<P1;V~2~tkW3^j48n^9E?CuuCqKY*WPX0->551xmR>Tfhf4efnuJOq)BjSzobtGk0sMyuDLUP2{<bCrOUtJQqH4wcq_5V'
            'c<Ejtg5!^x&<ZS_%hiFgj4$4UjOAssUR^`)!%M2>3o002PQeKnI9}j>Dc_2qV{N$RZ=dV_VJ!$uP#^ym^+kWr0;85-*(i%9!VYea'
            'y5!Zqij;N(s&3Vu1yAV#;yV2smm;{qgetj>`wLRuzoHJ7WAZty5LT<C<W|jVEeGXJoI1lPpCi59fTpF^;U@~WZuweHD5@23K)O0d'
            '_gR?xXG(1yUSi8MX)m>PKxr+{^1i3UmhHv<Mm<d>L(59riuvC2Rak}z=o(nHC0>kZTRw&@Hj+dxsb@?3bRBB`A?b~LtENuM1U5ok'
            'afDJUtHyFrn9Qxo@exMB#Mh|83M^gVnV2Iq0dW_24QDIxsHMy_)wNMC>7p6F9mZ657EnfYAGD>pP36p{`+Vw}mx{l$taX}6Qjj09'
            'v?$_KM&$H>=?$j6F-C5Yxx&#DCfRirp|DQXo-x3Fysw}k*(#h$J%?xMZFBTa9ew!w(|0dLVZWg@#jJ&8mM$wbxXIEAq&`4ev5iYb'
            '&Jrw10|&xi>#`-qO_nEIz4*x6)JH|QjL_JF6}<p6|G+)?EpYdBib=Bt#zMyFj%u*SJD3w!G+l%@KT0(YXniBdZ9-=({?8*UVg=~i'
            'A^y}RDpSr<*dd_tQR{B!qnDV29!y-~DA?Ry9nKLus-@XnZ*RG@gW&%bWz?MNIZpj3=Le@J(h?m|P*j@T%bF7MmQ}w4=cb0txlIsO'
            'fH9F$9v_JMFD?l8wT0mJ7l3P=PQmc-x^BHXF4}N7BkuuOMq81ML689SEkKbe9v@E%!}9dCkO)G6QH~mr$1@SRw8#{F#ETIMtJDBz'
            'tM*Y=!+dj04qIn*kQk{fE(>$78CB&X#YS5cKo+Pd&%WkMB#fxH7-M~hcH&T-Of=6o`U)rNQ|Y)T#2z<z{S}2&aN0hl*|CI&Vj9d^'
            'Mlt@1D(FTJ2f2wo3S{j8t?-H5dW6QdPKAv#!5cL2lkLK+bxxrTu3Xb>^|+&ck!7fFo!n1$=#{IpVzgYdqmeZxyz6Pu3U;K9_gx_6'
            'bb{Em%(czQ1y;oR3)}%npnRXkGLKhyr3NcaAy#+WjzhZTR|#|5ff7SkY0&9~#-7lrB`u>@D}c0it~CN;T!AD^d!Uf3D+&=f*v4%E'
            '@~#m##Eql@!dXT21nR5j%uO0Z*B$H4_xwK5``$vlbv3e&O1nk<(rVF3w$l38u8c$%(*98xid$N1PnF28uMXL1CgxpD|8JXOrZ(kR'
            '1)bWg`>=mzSZy8j47>p^cjiBBM~f}3>Wi=2e4Q=z+WH67V&elHr~#OAT!e6F?Lw4mKq|*nGfF55`v0nB$o>^eko^l*8+({Ue~-#z'
            '*6XhNV-G6C(>*?ze}NKYc-cvQ9hD=JH%9uX#i~U6kKQhNA_1zuia$n6xj|@T*aAWgzNoet^Wf%j_E3>CyS2shTbelzO}m-J_*<I&'
            '9wpI!=MprXg|5H49L+&RuK&eK(>U26Ez!PS(UI;fj*#A=wNqD{kV3f(NT+BFc!;s|D*1osXcVTaO~Q0%n=9BXF_0<NR*Qg56hN8F'
            '!WSgiB#)vnsR4Pk0)>GtOv|%H#AcKbaFAe96Nza_cP0sHuQ$<-<BcRP!b#(>y^U^gAvTi;Rl?%lJP90W0jJXr?c$(jP*%jLI{FMy'
            '?vLCsVbtMG7LwC{NAim~hj57~_|q=Cj2m>)8SQsub#O#R7m_F_4AlwVab<EH2cnN&idiDOa$xCD9Ui?!34=qEBj~s_tqSPmomhTs'
            'zsXWhwpo9;k@MmX`LwO<zLN>VpSRZz_Ltd0=EY|i?EPgkA%?2cI%zR(Am#r9Q4JHV'
        ),
    },
    'script_generation': {
        'sha256': '34da854594b20b29f749271f77e57cdfab1d7fe379b116d4f86e4ae7edfac9a0',
        'encoding': 'zlib+b85',
        'data': (
            'c-q}PYjYd9k>B|%aP$X~8<LiH*SRV=Oci5uy}FepN1`{irKl*71Bw$3XP9|N=IZ$0uO9#gm>E*Cv&rSoRhGyZpaC=*{X%!E-ELo~'
            'B9rq-iPO^$B3q>4CLi3YSfw%#lUPK_ZMYEGN~S6h^Q}1gcE*inWusxZ*~ashj016WG}|UcUd+`@7J0I!_p`Oka+PwkVXJkP<Z7NI'
            'cK}S>DYeN&m}eq+h{d{y@^G+8llw4G=}Z2)P68E)xr!I7wM_5A_*Q&65#6+iWBA<4)IH2266YHgk568X<9wASn{e^+eUjY5x~O|5'
            'GJuE+px-#vNaZ=gl44b?WejUiQq{^=YTbJy;Fo}DtT(w>BypCf#R6B-7yos6el|!|wt=V6M5Y<Q=o7$=O6}CeLPk;RA(b0=it8)q'
            'p9y#+qVQgcl}eS!5;0Hmm54P~GRC$@V$}-Qn@FuyobzUcFcT#Mkt{_}Y@%=>X$8wv-iEMJnwy>yZyX{U(IcRBA_G8jDmKMD3bU1h'
            'dBod%1#O3d8*S1s&a;=$Gkb~C4znB^iSfr1*r)BV)o!<2OTdbl&6cpqshZ6MOfgAwXa@7AS!S)4e)hL4iOuIcTr2Jy$Xww!)6;zK'
            'Q=4CL*{#dzc>3vb_Vf7k)8tZ&#A8dqzqa}e(+UY6QM)hN0ZcneHuzw+iR419k_eEGuTpg%s)t#s{$7MYy6rwgfc=0S!tbID*esUI'
            'a1mm6n1#fLO9_~y*U&KFT?Z&K*aC=MtJMnBQp}4m3T8#7(isy;H&c;X<SLl06P(Sl2*O3)N7~5|I?~{=05XELw1IYDjC8O}l>$;)'
            '$c>&|!@nHRVLRTc^frtI5Y8GX{!SEIAiFSy6~KIRncgZ`N5N-umZ`-t5KImoe}A+WzL{Vo<J&^s>aqS?oWZ!z6MyZW+e#(rb_PA8'
            'a*i+)*a0AJ^BhPzf)_u?C{sN#_#-wv=FMog|AdIl=(MTNa*+$fsX2HVObie}Z-F`W1(6RN*Cf|y&W1!{2#g;P+JQC5`0izv6zM|E'
            'GC)j@*f0-N9MD(TROtLNhsMm&dPAgd1eO(RIH_BhcGoU`*&fFXSv*#D`>KPFJJ&m)8mE1=*%0=p+o?{g(-)2HWE9@U@T_du>+J^5'
            'hObxo6m-)JU}wGKXH1{524Fj#-fk$S7JJ8{xkHIx-HFHO(NEgZp+yOhuDabC#_;z1Y&toc&Q8Z??>~*-PcHl7-Q<Vyr_<@I`l?0H'
            'fNqw^2S@d;G?yx|BRAX!u_yj&aRJS00I&9Syryn1#1yF34wM!!tq?dnut<=jKb%}#PDy6KaYpjy!&|lod8nf0v5*=VU?z-^tGAOC'
            '^?OLe9GfA42^=(F=xrgFAZ6izObd{b*^0!47F=Q;C5t<lf(A&ZE!xuHAW=!AkSu`iEN@pbS`L;;6eSN?UobP|_Xhk5gp~#d14ug&'
            'twjtSL=Z1P1Tb$1#L3y^$-4;-nUSazu=OAf;Q_z_DJ_9)VY>o>3|wXTu|dR23v>{o#Fb&5%s)yxL!1C%0XKj}0g6z{vV=ycARl25'
            'G`RM|O6EAP+)T2~(|JgWC=eN_6hS$cEJ_Hb6yF{V^u$1Xh0#`n`+EU05Ls4=j2#3u=s&PvP)=YP4=_<V2jz$xz*~^N4|2<R3YR|L'
            'w8TeWXGB0QJ0b>p9xp(rDMTh$aF|sr7U7*t!${8I7%bGCn1`vDPR=h*UjA}^@#%w*i!cj86^X(Wd2`zbr@@F@=o?3<n*<U>j0CkO'
            '2Vje09YN2#z6gT=78MG)mLP;c<iPI=tRz}vUz)*O0=*2yWfG@S6Gte{aWY#eZb9o3Tlwp-3gJ&Y%OFIVhiyaAaYarCNI-%H7)TMo'
            '?i49N=vC0R3>9qSNC4|;yM5cR_Tg~Y?ofLeXz5F{X6!H9mwNJ90T@Zr@Z~m1i#4M)F4qFQ&;_@1*XB=53mi~T_YK;9qHC%^5sN)R'
            '^{5%)Rfi=hfdcX5vG;KXN@D;!!_*^^UqK6tfr$Dr`MNH#gJyBK#r>x}{Us@;pyi9p6|m~e?rLQtVJ_ujcJA|O5+jyCzX0rfNCNoc'
            'DogP2lGP+ocT!XiJ(Z^~Zi_`!&`gjGF>Pky7Em^qStR{QybbSQZUL}R7^aGxSgr{<-SW)>W&n?R9+b}I_}%2~gtTm=H~{t7=5-r#'
            '0xHAVlwUhe_cm~g5E(+?Ar>0{0o1fVK03R*+wJ}u|21}cw=c_0I{s_?s5KNTOj$40mYMnt%rNVsC7=RUMM?9*4_5U;wj3w%K=Vu*'
            'H*@VCnSm*?OaRw$0tXy8y9$OZXs2Z>%?2}ZgDbxwHha^&(wjE|&5W(EM$!hDJ^=)^FF>{!faQjBSffP2xh$guX8<;m6s*!ix0tMz'
            'Og>N(+J7q-2pqE7983WONs@9X#&c39L6jG<HhAx41XD+u4DyCw4+dz4ksL&+!JIk1M^%#)PZ)_*7936^E(dH64O00q6Z+9=+0l3E'
            'HbT8uZwV<QOesB_QM=t6!rO4u?HO5sFH45N5U;q)HNe6TSI6HU9T{!oJdO~ridONkQj9&YiDKMGKrU=#DdZ3@aCWsY9*PT)Z)pIn'
            'vc!?I7T+kWm^2oV487GOrqLS8g>RmQ@lvHJn&%7;<X;7GERi!-??J^4m#F_|a+|^Y!^KL^lobz{TBxzikL&34*!w=TR=B&ttJfFt'
            'OP?GB?`u3`)s_0RH9BXwP^|(pLFDPy8Pl8%=mO_nui<JRcew6pnp!A8clRf?ounxqiRpHu^v9o(km+-;c^ow33<}EUi2w52AH`7}'
            'G5aP3FzqW>3UNQpjc8@YTZD!hSWgdRD1u>(Y#c3rV!x&JX!(@3K`p!bh`{g=0t|9hSsx4_F%gd)ZDVNoz<zf~*Ep+;t=TRN+0u3^'
            'DS604R;1@e3F@^Sw*5N@x`=>fx;iN+;N(^Ty2_wcoNJ|TDd(y2!$^B7%mE}>0qP~NPdI>UKr{4ThZ%t~Q?_QOM<-<W0Zw4O!zBpj'
            'Zu`+uZ%{AgcPYQz0qTAyUfBk)(;C7^3|`@|*yRr+etPVwTDUglP*Jx{^9GaQ7DQ|j6|hG#2%X`XXkiAptwVAx;pSd3a2V;j=g!24'
            'No205o$Kx}DSS|#<SWzg+Me8}W@?bg%Ainp87L_C3KYwEk~uTh1j^&MM`}X9&_n`O2G7a?$sN{Rwyz$2YTaG)zRGPH!XZHA_s8*c'
            'e0c&I0RRBH`%$h@6$hOojPAf#I1-?i%R8_Ipb6sX7@l4KN4l#|kdA7r^+Jr%Q}vmS7<m?7eHi4)9ncj_YoW3?nuIe$!#Qy29ElDP'
            '1S^?bPES6Jr;|$ng$I4HQj0qy-bwm)GIl~DJS4{r8lEU77Dbw(7-9K1Y`tia5eIN1_6UNKi-6vqfA~1QI6+jrh4r7Fzh^lP=388P'
            'cspqHMd4hL+4aKr<AQYG3`w0}7qQrc8)yutbzOjMbvF5Vasiy5`m$aC5;{n*D~PHWaQfhMqe+DaTpvDTm=Ryd!q2d<Fd%GY{!nnb'
            'L1R}a+0vxAT>-os_Uvh%Ww$-+_HM*Wag!Et-TMvi`dVadLcp;$ty?c>QrcKW3mZkW4DKZwZ9r}Sp}>_Q*C|ppY@L6coQ+Rrmv1jl'
            'K2B#J&fiT=(f4`PZQpM4!Rz7g2Y3=et9&&)I_itx!_Qt{ly9>HouaSl<#$K;_PRAboBnul{_*5(6IhVuMW6;N8Qv9x?+33f=+_3c'
            'ZUKipRyj5JjvBmr&ER2?@6XTQpH7ZhTSJd#9s+VNU{jPBYv+3(4}FS{Qh|yFm}(!1iiwtRsz4SC3Dnbip-C7}ZFN9^DhMiUJ}V|*'
            'QO}D2T_(U>FFt;FIfefhAK&SXz@5A$@Bd7O!#|@{eyX5(dV_9)#a-4X#uFq_1pI(0{N`4z!#Es#JA92^{2NkjE+!w&f1bQMW=920'
            '1m68kfqX*0!;!Z5@zjJlsi`4e7{GZ1h^(frB%HCVSS*wZw9qBm=C^Rp&}9S6N8gq#7*Ei>TdsgMZFVtve{wm!_{H9t03vQfq-B7+'
            '0TKfG()tvzp{U5{${<4qPy}I)I&W{-;{B?V12G~XCdR^dKu^m^W-Hk8BieEzsD8E8>n~|Fh~Ow!n*tH(xqQ4h|M}$I<YH#HiS_#@'
            '8x_kCETyI+Vm$li-UpiPp?ckt@e0kFP(e<9c3Pp3j!;WpN>B&b#Heto0(BCS_LVgtU6O25ptiyx;j1Wm$X3hS@!9#=$=mU%g-@43'
            'UF<}h;y#q1P*_Hq+@n4nD8SeYb6CpL@t!cvhlIRY@Iw=72S0`6&NiHX*fP$`>a0f=JBWmL3KoliGo(v8xOkA6*PTy)gboIzmI`SI'
            'q)7N!0>7P0ayi0!hGvg{7@wa0Y5ex5a<k7bc;K!Xx$o{}-)_FKarI)zOAyWA<aNy)NcAHHGA}luzLO#d2BggOuWPGK8pRM1*d<7P'
            'maDTcHlN@a_HacTw3n=#kP)i|D1rNU0fH7zWfw%O+<@d6EW<Q|z65uOHy?}%+E%`eL$WcuqVFdM1A@kELAp{J%(``0fJOlZw*-{~'
            'PIVT}$sG&~5l%edhrb%66Ql)Q$ysd`6n2n-C`T7bmRp?@R9A`W??~)9g=Afs*;qpgA?vQC!c^1FJl$>Co}&+Yt+pyQvO1Huk=%yp'
            'K4Z@ejRK01=+4z0OkAOxB}-DUGrD^w*xU6=FHO(2M)U-)z=&ho0mVLv?v<t;Kufyysi)7o;ogEhm!JqTU6-oQcr^>`*}~Rfd#H+}'
            'Xnf;+H&Iz&H-Kg{)OT6Xctl3aTA_y1snDHIJDvaEn+X$SB`VAuldyT4X=-Ha05XwR)s}@8FusJ=Xd3Z$`!<F)WwBN3FeaAmj$P-j'
            '<01gP(w#?w4ln3;D7MMnP<&ALivn-Ut`yFJX6wxKy5iE?r;vJNt4Xzidixk`kUY!`XW;wTy;r*0&wMZO4gquz%iie}p!$F19EJLM'
            'xNf@HB@RD%hqrfwcb`6<g4CN%-jyx@f=h^~Yw_CQ_%X>sveAx^3S{(vZX#B6_^hGoqi-MU$8IS80wRgMM;<m1*fcEwOG;XiW>>%W'
            'z?o|_4DN_gsSB<A-y`3626SJS@l`m@o)!*@q_#)-N)C}g#HZb!S`9H~4WNxUxB!W?#^2c9nm7huRxCe+`3emriC6v0@U}o@aS>t6'
            '1Bq<X6k!cCSe!V!f&<OjZ6xzttv85U1cz6I+J&)5@!Ba5W#qd+OB{j@x1qh=AeI^^i+!Z_>e%{*N2QzCxu&hK4Z|Sn-L<E|4p^S?'
            '7vj=TifxU`JT9K_@`mK7AOg|h2PDy*RT+Ppc5bqr5;UkwT@v~*Lzm<>*bGn_^oA~9!YNlyYZGI-DqWIISc}!?yxVnf{)FZQ1N&P8'
            'h9Aj2&y`dgsK$(&XUOM;J&>henJ5|XZz8**B);Vf04aiSxm4tTFk`ihI-_~t@pBjvnl#s?G$qGPqmGAev<9`HwqnCfDw7#96X@>k'
            'uz%!UuVwwxd$ygJA~_+S+(>^X04DbX8CTyt3w8e~aRY?HxKQ2;>O9kWGk?LA@h8S<T}`LAJr7O;8mU$P$q@|hGKCkGMH~W2;iUkm'
            'H$cBvr6y4FGm^2EhU`4<6vg@2S0I!f_w>~(-;;^00T2%B<En-yY%-KV(1k7~2%f1~tsTZwTe*#v|H#y5YyxR5pvzp!B`>YJ8X7w`'
            'XTg%1ed5HdmFALnFZdQ6Wh7KMpi(Tr_8Fk<5h)7q)2Egs0r^3#CmvngQeV>;3+u6C7uo3>XKlZO5^Ppv0enQ8BwlPeTB&1YZok9I'
            'Ez1YkNgt9zO{}v+9})R<Ut*A=aixhH#?cLF3oQ8=heKWztw-h|@$xzObGiwdT{(qxBn7^p71D(jpjc!@x|9pOQ{-krk5CbFcnzCv'
            'HdM5XXw{<jr4DLht>yCFPXs0iQ-v}g_EM5R6chxoIKlLfBGfe&cX%ir_1@4j#v<^729G5f>>92S@*`-Bft5r2(RKaJ*#=(m{D!#6'
            'LuZqR3v?$dU+HJN#N}#hm4VW$veDIJJ54yQ0!a+*c72@>j~t@gUHiHx>J{img<TsR4ga%miD|~fG<t03+YQ0RtA-hGR(sd47zy}K'
            '6OzAPF#r_Q?=2ecreI(|LxOHe-~mIuw4q*x!xKD)pdlU-hmURB71m)t1B!@<ac$N61lSFkVN_?^IUUeYf7_M(1n0;iY%`=Ut&$^U'
            '$cVf2bf|+J=n5eNa(p#AIy%DtJ@JiY9O8GTc}WTfy8vrxJ$tn_R-$^@L&U=T0K34)eG9J}E^Wgw5RKYZ1igt!;gL(M-+a>)AJN-A'
            '?>6Y!o4ITwE8-p0H$wG|JQ^PDTn=wNUjG6n@3Rp*@^8dR3`;N^q=9M3&zh)4uaAzNutgp2^X0tex8)5y2J!}s`S>?w2<Ft63}G0?'
            'o8mbq-{}I>^i-dJv!n01jQ_8letpyPPdNJfPgg`2Vk7N0l5w~M)klBT^P!t4KX<|STlQxxh%8cSGh4<Z>pk`KOMMlh-8@dF(O@9P'
            'r+*p$a%t{kb)ZI^#35iw-yfC{XsNzW(OQ(z`G)LIA8N>Irq6A6I2`tFIJ%cQ%NSLxR3A~k$dijjQlKJ#i<$_qzn5r_CCg=B&<0}w'
            'RF;KHiuk~EAXfL#sWRX*L%{B3gmgiKgh5V#-8}pixB>xj5`f^~-9!n@X=bjJhW6I)hW$STP$SLwfYvfl98cw6wA;INP-@jS^P;>_'
            'iRCEy1a@$X6iyCi3##FMOC$THMh~NoA%xBiwAME%`i2+G`00-)m)cH12T0#2%p9G^Sy6zy5b%I!KJ%PktoLW)0rce+#apxcDdRu}'
            'X|(O7(d^N^wMdgJGuO_(Gm4W8Ojd!#Z>vEV^2-KE$RR<{ro5EJU;yvnx-kKcf@Tk7UyI0*d%#CE+?FTH-1%iA6KLuT0c_-|oF?X;'
            '*h6eSl#ujc=U;ukFf#rQ<8>I3Uy{W$zcF|py(%Hu(``QVg-#Wld4re>&@zL_dh;2VAOmvH`?EIr;oBHBTNlNqDW9!&=~tpWuFojA'
            'MzFZJ>!7&@^6L9)Lv~7F%WB-9mPCCFKI{2?jVvt(J0Lxsf&R7fvMv{WVPkOUKE3Yy($ieM)?pb@YM}6U?C%mTG?;mWtmBirI+U&s'
            '_@Fp8__4kAAgX+C#4nGhcb4dD#u-aGB-Aj{1m!b^isOZ9_^Y$>M(i;__YK7_IP3ZfbnI6FxL05`ftpko-}Da&@TXL2Po!hy2`r~%'
            'HqQdl$U<zkA6=?EIg1*jF$s&`_kAtmS)eEZh;lhma)-)7`?juxd>}bDuyGvwloAeW`r*^*sf}n_tYjE#!T2eT!aD_;M_q{nRijHc'
            'GhrMXPj{a@lAg#WFjj2$`s`MEkkK8#s)pm>N~&QsP0uLQM!#6g!S{pjUd;!gW)@kVY&I0Inme}}6r+@3o({=nP^>_}4oJC_q=cvC'
            'JuK=}a$$mhIkJtC&p5zP-G@oR`h|{ig9aR3`vs!;pC@xk%sLcrKmFrEwDD;Pqd~aVAN3ogZ0jJYj~btroxVs!m)Wlenb`Bid84>1'
            'l`RcG`+Kj+Ua|tPssT8nur90|(FokR|Gy*&HJ+Z9dDXR5q@RZ_gDoDU|65ADFPD1zmTV;2A-dU@k2KaPAcIUUffIvqp*Yu4TNRoc'
            'bb&8n3V`twV<e|G-+>2|Dcp4xf+u4yA1kY@zI8=Azr$pjcuzm`&h2HW${y=V&-#0}x4pE}Z>u|9eeL!&YzHe8ROjozYwV@TXHP#v'
            'Tux8&8O_!zmsCZ;?^t>1_A9eX$oihm705?b>A$DZAvh$8o&dvbJd88{YC2K=Xn62um^57-qI^-&@N^K5*)z=rOwv<Ve`l{^<@&fU'
            '3{&e1=OotB`gO&LXE{^3$Bgs4OsNW;gpXq<v|Z+fQ*J@0JIye3{t5Sz`|t0C1a$=FjTKgAuV$~c(&`tT0I89u?dL(%w;3BCNb2uo'
            '3_Vq~!uM6V40C4GfG;V6;RjAxJ(ePSz5)W}{gt&ep4F#0NsG1Lub9F!N`ln6w-fGFA#*CxK|oVX@WWn1abj~-k?4)HoWsU}pW8r%'
            'GUDc?sqjHjGP;TjNFu3eVL(c#CI}Otm72UpCo|Dc*jmavW$P7a&!T&;`*wVGc0M&(h?-tT9W#LrdM(4;$#6k$8^A=(s%8BT_c9VP'
            'y)CFJhWG)P#tPaQ4`pCb?!>lrD)nnReQhal;CGzN6J7g&Dha?8wkpT0ie#P8#PdWrvs_WYLB^Uo27Kt?2`+}v*XljsXhU)5RiC6;'
            'g#MsP)@5^FG`?xJ`jY#{@!;Fnzvs#l%C3=8cB$YJ^e^b5TPK<ZW&IVG7ljmIwhSq{sZ{c7{7TDX`JkU6lVTlcPg5DoXbWob3~Ymq'
            ')4W~8cX9F%yM;<B+;&##$gYMVW6b@oR)^u|*RH6k)PSjX($c5euDO~^yH+cv-Rsqd)3sin7|gVKVkpzhA85hyyBYqXDz~ko(b(Qs'
            'G*~qT^;P*ecXW3vIa@WIokgiTQOkUAct`#^VSP3$8pOgJ7{<EH>tsVXD>j=5Gai6<z0po-Q{cq7)E#&x?-{-Nu|Pqw<=pTU^v}@y'
            '&C%ZNas*KSiQ(PiWdXEmmjQ#O(g6B135uWb%*o66(C7U7s}MB5XhaCCU^>0*6Dsa8VuI5PN$63Xm3XUJ3UVjp<F}ZlqDwU~Fsi&_'
            '3*M01L(77GV<Ndt0N3UUcclA4{^+c+D#FNB>OX2ywRO(+@xIgU5tgGu5rH}1>qoE3$#9J3H+%Y4J0KV9yR^`#qgnE3eCv5HjVuC#'
            '0(zO6Y(wCKT~n5ASmcY|Fw7DT&FQ-RYbqW9sXg;{h?Uxzp*ro&#%D;hXuj`r$vWS>raN70U_wPE%9!P{oS@Cc#ON#?EGC%GpzJm3'
            'f^+fqq72+6#~+DklTGLNyr_6VH;gfu8i|#2q{?_h4eS9g9+^>au9Okz$Pi>2p^&2>c#|)vBZQIfT{T#Y80Lznu4}Kz03la8?twdz'
            'teqJNPCXp<Y3tz~b69%6{%SVg!rKZLu`h{SDlgH)nH}h3&SJKbpdfe+cQ&HC<jJKYiC0uQn(Af#8k-uM^_t2?G->>Zwn}X!?F$A9'
            '$w9RZ-(bnA@frMlJ2{<vm|UKSKTUo*J^AJQ0v)|rLS&nURFFmn;X080YCWU9N|eBwWiXL23e#Hli(eV&Ux@(F!!-^}4URXa=!;Ra'
            ')#nXm^Kcf+XE+R?QoRIvgOl%g(c2BddX}OcJC5e(KXQFJ*u^ArH6~)ad=8xrPgt-u4QGam+S3;fIvQ6ymUWhmjo<PfYd@I#to01j'
            '>&bGF&#w!4Xt={ABaKWm$JQ_9co1-<CvXtAKBPGaGvX7^w08AwH>Eup>(;cXB|$yM{N)II%VBFbT2C)p@3nU=r=LClc5Z*<4PVne'
            'qi-&=JAI+>1$TP--oi}yE3ftN8o~onzu9|c_x-ndhsosG*M@q*6gTn@?K@7oCYjA_zdL;1{lVe<!D4}<0g&_=+`{io7(;&id_MCv'
            'RbLNOem&Gh7dP~K)@s8ZyrgZooi$-Y=Vuenfy2r`6yl|8N1)?_vpA9CYZuR3jO(x-sV><bx(4LKD#66RB;@Lmq}pj;kS#?s5Zc=%'
            't2vD1JK0}CWGMa;>VjtnSkB1g2cmtX69m15I+?;a8BW+J2NIXli<7t0(_h5p<j<cbXVa7MDSSb-_hK^rba8fR3v1B5F9oZ)!My0u'
            '7GrI|Ir%frwJ$r8$~t4tjjx4Nz4;YgjEq!VvWk)|Y~l#5l}m1-wM(%ey>f+Iom!WH$^7JaCn;WM(31AvP!DDyOY_qPkWuhhW2Kn0'
            'Jtb_s0ygDhojNjv5{?P@#t8?o#lTJ0hcR<CRgR6P&`{>F&Xz;v2P=nIbulx6k~J?OXR)+O7rEya?G3%S$(n2OjV*pxx+ZNIJWdC-'
            'u2)Wa#FDm{9OI;(-h4AN$v2o^3ZyZxX))04^4(ARN<O2>Ite#f&yN$VB-d~iIx5BrI6(@U+hfQ{He5B=k1#NL-+W8j9a=wY<B<ci'
            'oix`qE~ctHUb#kf`+8SPHFb2xdbD??6%4x$>k4XRj6SB9jUu_82U4tIc4neYrog_b=#;+7x}PYcRpZ>tcB5|m*z|Pg;!a>ODX=Ff'
            '*S*~Um-F8Rc8p|N76hnFxVQ5=_YR@f?|RiV$?I0DcJ<|(!cOA@PF&AquIRaYjpw>PY#no@T{tslv;UyMg=hTbU+$~fwJPy3fPyEB'
            '+52Ve+4g9xH~)3_OQZd+Iq^&#WaNH%UU#?H@~A;M>9ra!8L0NAe~4Z0IbR^?x}Ux!>l2%UZa-<IOlBc$JDy8MejBk5PjjCn8g>I4'
            '?c1Xrp9YH6(9=Xg(Pghd-(F;L41X<>=ZX7lZv&Xh-JcYiLqBSS=DSf3xjgx@bdO!{V}9g+^<gx-Lq;`3*w`$`m*EsR*Sv@&5EuBG'
            '>wB(&6`35mAjW6El$r)thYZxvEyhx{ysP;4_N?7-F)F$h5Uf=8P{LOfWh`@s6KL!87~u@$TGrk2TpoSD*jCl`(RV6rdH#=`)+4xV'
            'p?4hYK=pKIm#I3VHfGJ$!f{PQFJ{nXg>IpWD-5LP5KYP$GFpX6;?0Jjg04Z4$89n%%MiM2cLz(jsCVO&C+y9`TMWBK8^(i{l{~)M'
            'BiE}IRj`7fZn#ujFTS6pzt4>6I~67TLPK`Q-vTwQs@Af=dRJvB9?^ZQ%>O_+k^e|{JyKu<mepG?sALcN)he!B3(8&&+{q1?QE$X5'
            'T%ZehA#YPGhQ0|Ss+?M78rEGua%&aXrRzM&Ckj{WDQJK>h%O+C$27;mTZgSnr_G6)5mQaRGIcJKhuG%YR3LV}P-QPh`KjP!ihfoV'
            'c$}A|16Gk>sOk?YtuHO|Iim?|*0T0i1<H*}vW*!_WP;l4qoH5!!no03yU;_?J>Y3$`QNMM&6i=!rEjlVv7_8@P8)OGnKh-xj(F9O'
            '8${(#oa~@hQwD&S&sWJ))sffxQxociLYPNh8B)9M`)ZB$i+Wztx~gV9r4`TVlY!f<XC)|%4!Jp-X44q0-RR*w7_|L#@b*5pXMc|s'
            '<Qs#vO#=yg<#B=Q{X6Ejdy?<?`?_a(Zqa9~{mP-gt)^f(Qw-&?DV9S|qY`}0DmOoK=i7Tq_oujYUGmFDP(LR$t{U*{&cl1vtDOC>'
            '(3s23bo0K<6N@D0nGHoZ4{RWOLzMiUDENO+7eA<vAJC%Rz{in`RSIvRth|P^-59DeYL55yY@!BWUuci^oEVn(ehFt3X&gg<0eQCO'
            '^Yq8b8I^`1l>x;L*61o_cPVUYN20Ya-+kZtaL+Cid&31rW9|SygI-Qy$5ka_`^!#3*>7iUrAxU~IY#^~iwq{lRb#1Y8u}qQiPx5f'
            'Loc*dwX%mW2i1dXCnvdvH&^y=hs?JdnPukhS9Q=cNn-^H3o>F2TC(@(;fy9gDW)=Zl*q0Lq-a`?7VY(5Z_=<_z%p_QyQ5~Op;G9j'
            '-eW%}_OJ?UHD!PHPMt6>kE*<l3giZ`o3+Y@2I8pRin6P!mr6jXb{amL^0TZ!|3$*dx1;tdyj{&q@DGL9RZMR0n(NRVSfy5wz2Mq+'
            '^X0d1yqFs`KJ`2DW(1@9b*E13Pg3KhrN3y=_=hggk%QB;e-hL^5WV`5M`P{akw@dx&m3v@oReR^<|ox-OU$5bM9h_)0*)vw-}93y'
            'knAo5&J=hp%`*p_zduIemh3<mOjqQ1H`KK+D+bHC^*qhO?_p9j>Un2gD!J*@kWB9V(VoV%<ljQ1*HgL6DY-(ud}_tI^_bOw{yRu*'
            'M%7nKZMXYBQEIoI^dKzTIsPrTKyFL-K_t^MXHFzhP>Aj$mJu*G>Fq$k#A=PcHS3GHQ}pM?GAZO;j(?CZa3gW{;1w(W{tr_uoPP'
        ),
    },
    'tts': {
        'sha256': '7df5642ff7d3b8594b5b8eaf3a9970a364b6bd4206eb43cacd645254fe56d94b',
        'encoding': 'zlib+b85',
        'data': (
            'c-qxlTT|mmvVQlks8K&)ov;Nxy|XbvhtCP7+p|k711!+JXM!MvY?W;-vgFYP+QZ?$pD(kjq|(I=eR(*C2e758tjw&;FE3R%j&mW)'
            'Jeb8&6!KFsF4A$nk#fEgak`A=B3}iW48?3K4&L}hk^A>)G?!kq*(Ng)B;nxTz~822QO=|vltsD@iYQI|b&wY_Q^UN$;504dEKTn~'
            'MBGWa$wgG;B7I22x{Qlxyvfr0D3qC)$z;A-2iaYeEX9Z8x1v}{Vcw4CK^zYrvS723Sx&3->s-ciUdT|fmfmCCf9Gi;=4n!7!MqT&'
            'GKz_K9NkN?k{PC?VwM)GL8AG}m7F3?q=?p=SgvJKuq4F11W8eD;xq`sbS5_CEROP(Bo?;C3aolU4ccT;QsnzkUA|8|MtMPl#L@XN'
            '<lcG%$8iRWEL{uVUqES@^nF?|%?b<yH@s#(80crKAYaAN%zP!W%;$Vn7ExS%D`(KsT;{oXQbcR1mI{MH(l@h^`99(?e?n$8ZW9!='
            'm>G7CuY<wm)zQ_5OaJ51$%pBsn25)Lz>gz8LA;2y6gwl~gb+VYH}t^Y#KByyQc^j+%H(|{AN)-IQARK&XQU`#6S1xIT{uyamy1O-'
            'k7zpB3+Uk@h+_2`1H*F8c9qE>+=9f@U~qMH>Hl{2etP0xO#gO#d3Eu-#a;}4k~o;<&Y^g8)@dkXKML_hzw+7cVr*R*#Qyz{v<#b%'
            '9`)nv^})f_eEpZJ`2PJLKSy`}I{M9d(yXT&nFLXXGQ4B_db|oNTJ#VWn{Jg5L>*f5U*Gw2SX=IwIn6up4&EFX(whh)I+Xe8I8I^z'
            '!~}#NgmNK*GK^9`EHmYExttFd3n&Y&IE#`IZ>THA|0xzQ-9r42IK^Hc@<P!<z}kd0Ts^0sOu{e|9?Z|ku%O!7#JsFbhSl)v#+uIf'
            'o}`somh$e8^HuubV_&mK=De+G28@^(u6azlpg&~Y_?s8mkU75@Rcp+H4Lb{fW>XfEtE?pJ1zxF-JkB+O5tp74Vh$yWA12~pw{%6e'
            '?Yzv=G6{#QwIOlkvFi_+a9)Jq!oAjlI5*c$CCiOF5^oeA^3z;y3NhtBc-PN>+DC#lzcyKd+2K4UXMi^5S+ps9lJ$^OGZI+cC;g0s'
            'F%kX3*e`$OeGj8~p%w=2+$xf{B4S9Y*Yng1Z^}<j@MJts*H|K=sF2S74RjSl;9eo~f~i3Rc;$+Rs91?xkC^j|)K`JS1MUI~i=p9X'
            'Qgb@dyXM{wYE=-_gP{lkgX2)TK)5NSn6Jv@F82sBX|RzSrby5<G~eCsHWmjMMBwJpuc2GLV2{B&E%9@Vn-7)%=7Sc*+Maq9Rttf|'
            'u-+@@Vdl6Z$i<>&lYU@DN>C9cioNE09^*Y^(9V#@)yu@d@npVjiYmsCzV@3mFYtaCgBP1su`ga992~gfKL9^(4%)W)=$MdOBPN3>'
            '<X{Ehajgg&I4g~Hgm4sEkv1H*(0T(wf*2({^kHHV+={T1p5c`ili&>x^|z;9CnNph*<n{6g-3^C;nY?RzguQe9*J3y2lU}j;4Y>='
            'IZw7@DzyJ5r_`UjA>dF!^Rv#Xc<|2f=jgb{JIUgB>aE9W_KZa%Uczvgr-^0;f1`v@)V;MG5TEl6at~<}1KToKlZz}6j1?^3epipA'
            'RcnXb&3tWcN(laKB<b}+qTCjsVRql!le*jcdB4(W#LIl@|E@zRqZCNTUg%Jq&9FbIPsker*b0fDa}*QHls=SS?e@8@_xzek&{&jl'
            'ycL`^F?aV-!1cm8tt#G~93yK_O=><uJ|q%ZJfztjvx&r?eZb~Xxg+$rUQ_Xr1W2lfu(LEqBFps8kFNgiQjHLB_9cpxmQ-mJFDpOE'
            'd09}Ldn<0+Qfm5&WiL&VPf8KWFa(*f$Fz>bI?8kMTg4cysK%%bVlHDD*gNHU-V^)<by*N1#;O1uL_}po1rZ5NWmR5g3ncJjualMU'
            'VLYTkBCrDGd4}SISQw9)R!-WeYOWyfOKbL9h>C(sP@a$6?uDqL6W502?6yuv?KM_KB^G5-9IS&dI&4fA+hBtiNl5S40Tk;z>Fh*p'
            'ERCc5mkTcMVHM3+4a=<<YfQjSk|}eQ%5pTOc=8j~hC|n8zOkG={BsL@);D*EuLYl#ahdQVF?_Vwdvdi3w6@!JO6Eq-S+4fWe%W5i'
            'T8^rkX+=%;+N)G7zAVf<Q^Rl_V{4bHlCgqiRgrY0<PoYK;n<?bRn~gbMC?wW8oj8fsSBIW6gOr&%-TI0@XRK(V4FbAtT1_OE&1dE'
            ')lDe`QrCw+9UR;|>Fs!7IPDFHs`$0y;ST!(gsqz)l7q%~h>)nozkh(-q2%7!xTLW7x%3i1wJd|BN-3dXryz~dG#{v4q}g^t%IL<#'
            '%cJ+xcT=&WQ3DVcOA3-4fEI|K#3;NF2nMt-#EKer0TcUMnl6!gKFFDnNw`U)L?xgUL91wrfD%;kR71S#y+yg!<c7O;`-mlBERpjh'
            '+(KI*(8Qj&wM7~#hauUKv_e@y8H1f7RUw*&dc^(hTdq+l^Z*|dPK_Z=a3_VPL;*Q~;e^z>!)77QY9<G&&w#-HJ=0rwtSv|vw7I^c'
            '?i&p=tvB8YSXN`Icm)Oj5y04ge*H!aVX<>+L6#ep<09S%dCoaWy*$yJh>j|oI7Dil7`t+bWl{sBC_{UuD`br7z<y+}<K@Bqw<OrT'
            'h(L?kfWQyN+5nvYM;O36YT;|J+|Olp4|9NX1%QPV3XkF_b8T371-JZeKcIGQ+U%-jPP|k9fs$g?u;;9UoZcI}!uWrd2+zfKyofSX'
            'KU(dF!YZNryQ!YgMS*FFlGDGULiX(Z>iF#R=mgm@d~g9fn+5YbPv_(n`0R5EJR&$515U;aNYx<$0TRezG<GeHInf_4fFVm{;dQeZ'
            'wx)oM&l)($(wx0kLwb*oF~A)+q{-|@qD<llph3IlRk*0|X%oOf(}g*hFl(XSLr%tvX<?ycjD_LE+wl^ALK}Nq=t_8XI{i4k5Hl%E'
            'XB<R^xisGaiaf$UIzPWS`*`%r$<(J0$M2^Xm#+7nZDjz`??1ew&GM<yUadqmaH}I!*Rd*qyMy=BUynYVTs3A=tg_|}FzBQI>(R-{'
            'FGuhG;a|SHI6lXO>3?6*3My#*Qlh9>B@$^4l5TMqt#g?`%QJaLCp2NSL^zGeno$77<u=Vcag+q&4r;%HHIERVklo*5N(e_UlQI#G'
            'K7Gic{4k(xxC@vFf?+~#7?w*66zMX<T1@g>-cd4&;R(W8l2KeEf1L++|0Ui@m`p^^4e1<ow#4%i8w>*CUMkj!MgV{R6RU-CiCV%L'
            'xH|ar5fz`W&8ZThBO2~Ev%xn@V>(fI@Qv#rnNL%AN^p-DOUCP)SvltblwMzB`c0JxoMm`P@ivs>u-wGaoQ<>I@Y>OoQ4>T^B|~mN'
            'P6Mpe#qqx=>NokB{2EY{C{}QosunGf(gQ%wPNx-Zogq;58B1MmF=_S%?zWg|v^K*9N9W2NsYaMd<zNCPMP60;9}(1vh2@7_jHL-a'
            'Jgi_ENI@WdRNy--7`h8Rqa^-Y>68wZHmrvYw`se&=G2_t?eZHHI?Pce)pth8pxWmzrv9p?P=^1R1+4IcqF#!RL0rly1)ye`vv4TR'
            'Bx|{a29T5??(xp%!7>iEP)Cj`?9r_HL-r(w>~D!o@SMzPxQhiBMVVE#sEmqDXd<p}Is)06thwh29v$%};n0k&O6->Lw`L)op;^{e'
            '?SwJ2U-cf7ZETE|nVuFwhjS%KhX<7v@yqMO*EbD+?w(Bhh#nDTPs7U34>?b-+`F6n>nC{;6lJcSb<&(2KV^=^5q2tWeMQWmhIMo|'
            '>Cc~b{QRLZ1CUEm!M^r?Qq%j(la~?=3KVvm0y2`BX^ntLe?jvLsW^Yx4FJ|Kr`bgDAxgOC4U5oOzupRJuppB`dS<{&U47{3WQqOu'
            '>44a;N(e6FTn&;}1ci$-QF+3<qtnx~E1fxjbKca+o{9N;8o~<HEIAcaCip8r%9bT%4+wL1?qQ&0FQ?=wUs=gc5I+Rlo}7ot4U=@1'
            'hFiL2z#5k%E(#>9zm<ji7WlkWpwC*V$noU89hD@fQw-uNU$PP!LUe|3<e}?SrD&>zv*cf~>*orlsik_$xu9QVrd2wm=hQCc-YQe$'
            '7M;+tDhs*%`_cHvpZ;nRH+2q3Z5#f&D%ao_Bw*E871C_mu^>;&43&oVMY@*(!_?O^ynGda_|Pug3DzCQ^(6O9hE7>5#y>l*>#gJ`'
            'BrXVaLl=1QR|Shm42+J<9fsy5i9^A0byPp{XIoSBSPz4uQwceA$tCX@OWr}7r?_v-dm0$Pt$o8ZG)XX3*bq?GI7TlXpL&{p4fl7H'
            '#ZxWUbL*pTDBby>n*oO;i9Tj9OU~0W4#~u{KZU4^bz@w2BJz=~T(v<XZqTD}Fn}Y}T@$y>fK`fW<(WcMe{Gkat#Y#qsZQ69s4VR~'
            '9JNPQYHNIN5#I&(n6~Rxi-LWbq!wy-nOwb5&Qnim*3UG!xlkv7*i%&u;ksLWY)xJQsVJk<_HHgTc}e;(G<TZ3vY5MGl&9*R#?ald'
            'x4tm~M}?hXWfU-y>znp8h6>+ckfVb4ur@Jkym6wO!vP&~8oTwDvKZFZ<%-Ef_d1N*=}EOSsr{yiy6dV{Yp8Y`n!zAmm(^A6`Bp=8'
            'UHS6uw$^Pc3V@pwiSkx0LHSM}U+sHdJNX&5JVNHkF2g8WOO=ntVf0oUpI#oF;`h7h$@I7B<+1o>`j3<2-_I_nCe9<MJ%PtYDDzBF'
            'mL|bgI8EBYHd7^H5Ju}T7>S+1Lr^S$GPnoR$znTDR>N4^5j&{W_->7}z@;*b>-wp6eo&!~HBLRU2U%FuGPLTAWt>f*vItwxZR#(@'
            'tz9#o!>@0{t~PExcQbB>ytaON573)>?G70c()Chl;$uCaiIc4b?3W$VeF)ll)>!E?u==f#`W0~cQYba&BUzd7rGYBhH>FIYe4_3y'
            'u%xH7v2MKi>bgl!$)nSG`@Pt8OF>sAW<dL#s)u^#m}W#BQopayV~98AtID%db+ZSA89GN)!^mOuf_=lf1=Vld0v?-0n^LVqAwPj#'
            '<iy;&F-;gcw&%PVTeVde9fnm0mjaEdR!|3oDVH$?Woiks|7Aco5LDB1R)(;0QzpHmR*xyzOnezrsn9NSP=;j4+p`D>-R0HA@w=;&'
            '-^JzhpC6{DSI0*u_=2*b(%WrsYYnABb$>ig(~Y>8UVXSYy)=@kj(1+p`K-7irrX*P7ZsU=A4nZT`St`VoCS5!tp+)hkRj4gS<Jmk'
            'EUR<Y*-qU;Yjw@qS}<KMbvY?10`iG^0ks`f4OAb^?JL`Us~Tm91EvAuKz3B5d@sH_Hn;CdaD$}R)OkZRW1&~yPO0c93j6AcN<`;#'
            '<#I)hmK-zMw}M<=Nnb<B>2#&uO_ivd#2po^N?_Dw6nCUczXkM~gF0&2o?9&T;T-gm@|BiKt!Be&RhoulrDbdhj7+r<vHZ4odmg|y'
            'u&in?rCD2uqto}hT$OJCc;dLI&m9^D7T6aX%+i&$sw+YoS8JcL$3EX7c&p@A19c)>4c=Azd*Vu;@!GwMx^JOa>-Xs}jOY<p;F=5d'
            'uM^a?o4ywY?xRJcm8!$Q^`6Qo_HO7(MBPu|dtEhffw6TEO5KSN!ywW1w-Uc95yiTM>k)9FxYUgxwkIaVIfL755a_^T9~R@QYrpf>'
            'clbTrz*~|c*B3tcu7;ngE!XL0NzT*ActDnzX$L>$Da~gpb+B3yj$5@)yM(PwzI`Ul8A3B(0~@=JZASVk!sqLgR`Fo<OD3HiZOCwA'
            'oh2*Qx-(4Dr1Kobk=tIalB8d6Do^eCzZFvoAPuofBvsiBX>+>D)kxM{cVe}526o$1b;+x{UY&KUZZ|_XO?Yhnt4k-v8Cn0W(#UP<'
            '+dB3s$2DJRr0zL=80m$$A+DD6BJfHOO|2pQQU!R*R&E+Mm0lJ*npYtzX3Pmdb*9+p^n=yjux@8lqh0aUIxN$BYA)heLs<>x7@*0a'
            'x`Sm6n=}tLyVv*UizWE&{=3q(J?*A5O?O=<z5gE4+fvti)TBFVI<|qSeVb`-y?L{vO%LX;>;FXpHGWW4lsDa3ZsQ43$v%Y{qx=G8'
            '=_$`C%DS%qc_kToI~BChL&lFJ^PW+{-p$k3>8tv-<6psEpvzo-iN=)cJg2Za@#+=GcH>>8&Tbpy@!msl?>r|Q%*3>n^%<OEvn#N{'
            'O9K&&D?<%avl_OFSWn+%_m7j_VWNuC)m0vXxhJ<X`;{&rM6{-h34GdoNU6u7%uIgS&UsV?f-!u}q-U47=k)wZVy`Gl-qyx6doMi!'
            't6wd4DcxwJm{2>Z(*?6R6Z7G@egCT6gw3mPU3r@O+>WZ<G^Z$LJB?YW?>Bm9n~v7u@Cy$Ktc9C14HMlRbgYB<py12trgO%9PCVh8'
            '>aBjKI^Q2Qwt4c{`1(Wk<h~W<9Q5NzT}o9K;OU-qlbYSi)`9YsI7>J^4**+~GH?+@8Fo(vq$?>?l~yO`-NwYA(I9EUQ~O42177&%'
            '=WZ}={PY-WqaV|_>p0jAOBRoZFSq39uQI+6ZN45BEXKbb7rrAVxP$Ma!NHgMzNHTv00v*D<<9MmFH-AwP^UZiU#3|4ss'
        ),
    },
    'render_utils': {
        'sha256': '2d7d4e0ffff2d5d14c61aea86a1ca4f8781b882486ff03335b99b26ff8f21f85',
        'encoding': 'zlib+b85',
        'data': (
            'c-rl~>w4QpwkY_YPf@MR*&q{;mgGy^bTj=aw%qZ=u{@TN>}*Ac0!ffW1OjYaB-1+lzVin2=RD3l&3S=&g;|%XLRA5vWXIh-y?1Bt'
            'PAH&ip)RXdt^3;B+j|$5c`%7X@$%(r9WF!`CbKZ>ucLJsM@cA_VZ08rT;$6j3uj`o5&KWZ+-R&C4WjiXnTYPqGMX+sQG(BM5hNn~'
            'YZ+Y!ahMdoc%H_=q#r@YHTF&lF-_MSk<LZ2jB*jDi)h+|PBU>GX3=~zj?-xn57wIw!xhnLon}SO%~$EHgh9`uESwhcM$FR;UW9@='
            '$w`QOm0pFS2=k&h=ycws&@)-UjIvOa>o^T&xqxo#auP@Rl199tCIj)R5NQHnvk)6G(0LS7o0+)DA^_SkAD8sOK){n2mM<^Y>o|h-'
            'O3Pwh7GfO~%Ry&vZ?7}Y(v=vG=Vei5;dm@(1z?l32#P38@=ixSTL$?ujwb5Q-|{q3zq3&N%9mvk#rm&uvd+?JnCI$AvBWvRr1;CT'
            'pa_d-6{_#5zkPxK{*Wdix5EL$w^g&Z@MouUdUka7<LUUPqt`zkpNf&V?Fjg1FMJOe0=<(Wi1&Q4Hw&-BI9=m|@j4Es;WCY3b?{Xd'
            'UPs{#;13`Mpwphu5P~&~`WpWhdr^{?^LaFlusaMdz=!hyR+(Ny!&y1Sg&O0;HUMG^Bm42&vsY)YkH<fM_v-ZR@w@R4Cohg)(-^%X'
            'N;aM^ynF?FgFjZ`EGk#{D-N<n*uQ?*hs~Iz9sqiN^!Du4$(!-ncgH^*kKevKdHL$KF;{OE&Vw>8yrFzR4`ED*1!F+m90mq25B%!w'
            'JcWrQ!77C3-iz?J;HOfY21)L@ZSv>?w)y(dcoD!<ufy>)jnix#<T=9lx9|4hpO^5D31SlD4eegObla_h_v4#1n_-hj&hsf8a(w=9'
            '-+GZPqGWsx&kqmH7l7koHigf^o0mbHhsL`&EDE3y`8t>a?m+|Dam)Yl7O)=UbUp_31f0w7TF=9__52~orePeHxj22^x}f|2@C7}5'
            '`mqaomX*8Lb6|ojGC-(l7Om-E!1}G$h@)NIf?B-@!d04xGx&axZNkJC@0LNBY_h<7_EVIH*(Nt1{jkY{EDGSn*Z{vv%RDq2{tUD>'
            '%+lN!FS1|(#H@hb^~Luv=3l*ByawtCKWlJaWjHKC1^9jtZ{TA%k`NB)1`t5E?zvlxcv8cK)93wn;i81jgQA$%(P9}KK7N8d_8;y)'
            'dK65aJU%>pI@y2x_|f!_!Q<&c_~hYie)z}Pqww+5;Om2@kG~EN4-TL19|Xbt>nBfV^Vz}vlgVtpUjtocF~VII#X288c#s7*gGE#<'
            '%L%MgHiZ^I)CSXZ^<a^vi#U9Mo$?2(AW9yj^B8`}8RXLk_n1u_O~V9`)Lev<m#+uK`=Y7YXB=F^S0-=;0*wZ$90zp7@Cw22!Qlx3'
            'j)EqE^zS!C1y#-ABcR)3l!WIC@qM_4>z3J*_i#y1BjiL@Yd~<gU>{^@98ZGjl}PC<4a8Adq$~L5RQ%=SOk{AOBNLINHxeE!UQVpX'
            'Dh0j*F5n_6;;@gG6Y*6hg%XmoiC_ZP-Jp>aj^TiR2;)z_VV-mM7Fzh^=|7$pe*L*=;lbc=7hT{L>HmZz40hxWnlOJfd-`zp@X_pG'
            '{$%>~ql1G#J`5iP)A^$b5QX_4A3r<@9|upqo;(Qxq8qdQ{ow1_<m<!WQMf;U_~>)f1V)DkJc<AI_}iPsviNV0U;JZffgIE)lL2c|'
            'tvmB^hre-5TVfCYas<rW$xGns-krQTK9$_v_s75deDdxEa(CV8D6&MtTnSuMOS<Cd8HgKnso~>7-Ff)OjlA{bH@yh2Glqe$XVq0m'
            'f6^_CzcP9I5{wAeJe!BeqIP<nP6s$mVDs}d#uFOw#SUXwg<a&H@ryJ5dnocEgE@EsP$~dLyZ62r<G=JfuP6H7fVea*hRkxK<VV0k'
            'XqkoZ=~a|OIg&Z~F-@yT9?+zcK8a$JLJ|~e5TYIxd=5V9juN}@2zvu~9j&{)L7d)%S+_^sqqza$4{A0;jSSX%kd;J~h-PMeXz><V'
            '0PGC%6lHjvWoc%!MEK8q@7o9@)>R0*sV6U2_EvxM583va*lXy#3a_Uq;n8Y=><6845ShYRw|RKItqrB8|B|~s^HJ=Cy9gFie;zPI'
            'Z2_+*^F$$V-PL@PGy@q22>om<kwET+i4wFCp1t_^58QFnCGB1Ut?Xs$UWFSZ>m+8Y->?C9oGaKfqj7z71~-TQyg~M_Q(qSVBM0Yf'
            'ZGF9Ja)mGEo*cvQRzlfU{G`2pZ9tXK2b++jy!>McbWAWkc>+L;Ow4QBIRiklstq+;@h6Q&atuzJQi;7eu>kzTuW<?R3J-b<{+%o6'
            '7jpTxlGw;QkZzFiyo}>6l3Ah~fJ3}WvuFXMhc%J9i2?VKXmkTN2=Gl|LOqedO@QtIueA)2kXH+hXf$1x$ra*71jMHs2dl{}7|NCd'
            '<es`u9zTBgr00tXpsT$}Jd8oPM)gIPAoa{Gknfk_d;Zl`1Qo6kRJ&O8bd1+Wmw&?Az((`>0VNFpUB<I0s}s*3f-9t)nnnT$5nKmR'
            'OgtZHhT@H(&4O9xVjU$3oUqgHj{2}@gAM`vmM=NMcNwf9@SLv_kppW3Y$QP^)fuY1Lb`>*_rSQpm4r%?AQpe28=7Ff#EXzLQX<O5'
            'EWJrk%Y|yFWD1XxA|(JXGPp$h$mPN-sVk$o0=LVQZzflhUZL^@m0;xzHDa){A?n%!93!gs;9(9sKPBZFd;#v0Xg&`!RPo`u4Ah#@'
            'f7ObOgt_A98h0<KmEvNcnS3!=ghkik?exuo_knskH0UXxNA4wg1|l-dndP{()uTKufm27eBHzFXT@hZOP6%OXRDcnoGOSzShW&o9'
            'x`Kbh0l@j^BVt`)$KYxmr&shRcbEZMMhR*B0AvOJfu0Bx`1k8@iUfoYTD2bM7T<c|-lYckL6p;yby>SwKjubTeJzO5$o=*m3yb%R'
            '3CUSxx`y<cS<6i)B*JR}->VgedHx2UA7^P=s6`+1sK>p5Bx1ls@Jh@7P*24+;=p*pDnJja^=sx-FVqT}Q0`npv+@1bkoC3$)PWyb'
            '-F<WKs!Pz}g<I#v=VUkM>>~qUUawB`YPvSrZ5*@g<6Agv4W=fV_?2kVtH!|yf%mNftr3H0V-v?C)zcu&;AJE)*N%P-rx@M?pY9$p'
            'Z*Z$dvmGcpS+NC0wy|eUH=F&H0sP5#opm5lu)1JvdRi8vM~~?O1aw`42;lG<)({6|RTxh;MVNP0d%g&bMUop<$3P9ILcc%kKtG#3'
            'epe<qU}liLD1lYN)!}PRURQ9PEV2^r=^&59BtV5=9)0lap;fcVLQ?&A8<DwjjqknArVMK=_2K><OHG{YCt~UScPD&5MGez2{RgQ#'
            'j7h<UclBYdGq{+$bMMDrj^4c*@|wp{9wfQAf|G{Jt#$-07a(64;I_^NREJ3M2V;fa`ZyWvZ^Kw?V+3@owv}$O+$9Y5)|Y4iyz2Hk'
            'cOY7WFm)EL0$PC*IKdB576-5u!7LKpTl<*N4qa?}uo}1ug9g!T?`4o(#SwuEL>}gVu3`~Rfiv0!>nKLJ14kB9!tS)wX+3F;*X6FW'
            ')-d(CXUr6(40Xt^;7JCYK3Yg|uxiVVkE$*Iytgp_*gA@b`5cYH;|YMvqEPQUX%F!0x#(6+K-QYWE=@xGqc<@B0s(F#CLys}nqf;}'
            'mg0Z^&;P|sy$Tj#l*%u%^)+8cYf{w8hct|h&%GJA4+9s7xbOw!Ktm!S9hQa#GjghXl;j1l<<__W8K&uKMW4gL4a9S@;O1ggf}m68'
            'p<+g$1MWotn=K8}b7+|hQWgW-3k(SI?BY^>tsh<XEN&Stv2ulmSVDDs3~Ynfx)gb>*=y2~9o2H9Y|j@m4F{NGdIb^}NLO}Oq;P64'
            '!}p;)n&uelu;`tQYAsu>WUCy+D&{bNb<Xg_pk!AUB4L)t)E0J)tmnK-#IB)a7_KJa4DCd?V-W~F6$UWOMHZ}=2v4nM7ijoE>)!Lx'
            'pAgnJfP&A{1gKuYyOhTPnfYO};4?QtHWTU*a&<iZoB#&mXP~+14G2*rE3wL1pawh<-RDD`&+}h@9rpB=8QK<R0@VfR{rW3jB@ZfT'
            '1@4zFsXxRw;($Jlf1Mv(Aj9Yl4W*X^O0V}X#8(1<4L$fTJvkg+n7f0|<W@||C?<MeL?DQcjaBQS6iIZzhm$fZmO~T^eX$Iq#j=op'
            'OyV*d=h0#n&;uBOshV&3=R=yV#;}AS=ZiVqH`wi9un(A$PRo4!zIwVpczh=`?zFKxi7$N^&lrT^Fjkt)z5U_!-ksopKXAIuGq^!!'
            'clV@TV{%D&uRHJ65c*w600q*wYrxqQ--f}8g>FEn_uolhkb&SW@Zb(dGsZMa*B%^MRvqKgXK;rgzAiGPke<{ND1sY^=O;hBeSLg('
            '44(+#;E;C>l0>-VP(v8@Ne2PrS>QX^!b0G6nUf9zm;uJa1pQJF`J{^qO*9_>;=?f{WjAZR@~Z#kIV|?YCH1;I=M}uT<Rz3oIywA+'
            '{R#p1HZ1nyIcBdIc|%Gc8f-?cJo7Rke-6|s@!^Yaf=w>i(<7XTeqSKpMYI8b$OXAnsBr0vOTEEzbmq2Q@<8DJnFgrLpQgza77lMn'
            '7~30`-($RLfDqyaGJQ$HaF&ZV=%xf$pLSH5E(4^0L@{WV^eI+k=_$aT!);e45wNfWaarG07}E;3<K?TLj^)zxCXy~9Pqix%gTuv='
            '4oSEbGuoiq7T%;~3^WbMc#NbfM%oGMdO2B)^K2?N>T)js!sl|e=8MaLyJj%eU`?)CGt%nd#R)uut<Awh4+jFSweBIF^;Ph`%ZDD1'
            'ZMUKccqhS&o@ADTWV(ba;=Sy9p!)7teI?m&dTj3Ou~9wf<4pRrg@?IJpN2FlGQhlUbo1pcHXRNg&9@H@dtV+7<%^~JV$YPBY3kqL'
            'r1tj@NGug{QKUu)&NX~@zK1_Xw?u5W!&@e_+aY|npWgJoyj4p<pMl5Ae!8q4Z|E_@&F8pY<}Rrlg;axhd6+b%rmwLX?)l#NtvW&5'
            '3to}hDB;LW!*QBL3t+uski*xcaX{X36eUyCK!^UG1NYWg`9aLa9?u891nw{2Q){QThE~q=wzoLw?fI=f$=i!P#~5JFw1H4o6aGSc'
            'n*tS^p;H>#%K??MP)x8D;5pCr=VAYFI170WmwdXWXr=>pJt2#}0gl$iuZY|-$X38;<V2GKf!rE&1A>MQWraX(GWWs8HmFHNJ}@Gj'
            'Es(R>0Ilo6g$?H$s0kqQ4U%|}duGfc0sQ>0B@Q1b!%zeUB>hz>^wGTNu=1x5{&={LL@!0!{qWJjLqY7XVyeW!{?mN}9MSWE;<RQd'
            'bV7m@qQD}^BnCtXXfb|JDSmOxF%s(_F(9Snrtl#odr`={BLcjjl5jzKDAZo(VPUK&tyr1?nZAmUqRq{b1rh9as0lc>RW6=H?Dq0<'
            'Iz@85H~1)|hwHK!9iB6NyoK|%y*Qs6e>`(h+iLnb&rBnc@E3^XBjUtHw-r~mg<-S}aQ^!Ss1C%gGrfh6w6Bj|M*bl&WYqn+-$3oY'
            '>$lK4bHjE}gqn4CzBj3ixe`a1-@3*B{2|+l?j*bd^bM1tFvulyy*5SZ?e#2%Iw)r-QW@Md-1o*p;_D%hnKbqVur7eeAZ{hdjm_la'
            '-P|Vj;eC*NuP?~)<~ohbm13*!fB_ORK}eil$U9F^UF$H<!QQ<A+FcLs?po)%19((to_~nm18soOiLP2qV#oxoD3U-B#?d>R)Ps6J'
            '%v&SFCOK>73(uAz!HQ59NOq_xS-^l?-7EnsF6)~H+|O&`!4%UAM3k8&;CO=z<dx13@P}l5aT@p(_%w&d%>~kb<Wl<)I=Q3#)sbrf'
            'LwpAFL@f}x*552sKwe`yxn$|yfGnKn6~k`xTvC0~fO;<N!^7ij-}ZvhVsz^wO4HSgh9n1%@20OB%c^e~Sd6p_el?Jej~OaFX$B%+'
            '1d6mLsFj(87x-fpy>Fyg*g-eMe-Vq1K7kor@gwx0&(Vc_1ia?sA=Am*>a1>uK~VtB7{R+RM&dboxW9jn4M04qe^9<qBMRVOBii?T'
            '1Pn>ZGabcgf|l#y3}_SX3wiiP^y(0x?<9)u&V16Y{O%pG)|NKqGtE?cN0o_GY8gsMsoMZPEFYp8y*P)zFNpksK)IkDwH-~At|cu{'
            'SuRI4&cmrvKKu_>TaM^IOuTfM#o#%hNgNj8>@7d2Gy)}Xk&>Ary4Bx^{t~bYE|?^NYpEZj-ToS6M`p3UqCpSDJG|3!OJ7`hS0L+y'
            '-U}rL<KzWgZy9i#D6fD-JL@5zmZVAlLzoe1QZ(^rBz)hVoP8%Q)hb+yZUEx$Bo8yVq8a*N3fErI?Xj8jvc4pj$Vd~!1#oes1|*#$'
            'ktuQ6uYfD{M@}&`EH1;D0%@`ja8h7z`+3@*2N^VxSi2^-DG@A?0Vhv0ag`ukDc-7q41mPLwxECt)HW%RmagF3!J3dVL~4Hva`6S?'
            'yn>|}s;wZM2{7>K4O}X~dP>+VyHjLO5lZnM2S7_;5O7r9Etsw6u}wncapk_mv086fP3QoZK8g+aQeSM+u>&XyhXt|v7v>FnxD4PZ'
            '%=lhG-GZ7)2+YG3!H4bzUT@3bI?UM<1pQ2MIgZ**p*94+LRbulhD%f5T71*+^iLzPUlIN33LV7GO+~QRr|23Jwdrq#-S`I~IQ;k$'
            'G5n%aS-#RK>P)NC1c+>3DDXjFA^e!PK>Admx^rZE29rq(kqnl2<dwy|{}et-AJ0dgRLI(vZ~X9p2__;Z+Hv-WcNwnco-#-%GpOj|'
            'VH39wII>UCGd`;4eYhF+hZm;RPEu=ZC0$K^*BYB+_w;$h1{ke3JfMC_ipL<gcZ4<{H5Q~ysiOi9d5yMMd(+;MdA4J18O#@LvSzDS'
            'Z;yRRvTDyQjkj`|xpOChqb=tsXpRng(i>lrMD%__Ii46l#YV|kxS=G+4*89UqOg-<ypiC-)w<aDB%TK(h9aj2ND3&KkOmJ%nU;$s'
            '*%VL`#iNc3O$1~HUyD;T4@nKc7jj*QCIZogOD(+_NCzS&>SGv<ad9CgTy&(z&|HjZR;Ky2W=1yw4OoNMrA?DKD2>+A4Ybu1JjBr^'
            '>0g7P`1bYj{{DgW5#&$hsA!op^5>F~hJ!Qu)n#SKh{p^P5&1ovE>PSR#ua3Rv`JahLFEc@%26Wp6kz$B2`@unsBi@EHDJZEuEKis'
            'K*ZBKj^<<7Fbu;8XMi<ykGnQ*38rq?mM<K8_EiXYG4{nZ%ATl}!#G&w*t~`b5eL)tMqcep&{cUbcRk{GfU=BBI;8r~Rqizy+0k(}'
            'O3LtVPl{CpQf;n}GYzxRQK1r^+do%5`9j^JtiJZ%JUH^qJ8&MM3Wrol7PK!|MK7mD+GQRg^8&aQO7RrtYrNwS8tS^A(Ix5tgC)!X'
            ')3a<t?Q%4|lu1-bb1>=nWvmSR#9!oYIIz7)nX5x}GEqBUE!qQdFa|n}{~PSvi>6RZa!RBo=*~!Kf6(&<|1F{6102-I%M)*>sJsG*'
            '!6F>ZJsE|4iy()C{rUF6EkUE-B2UCsI*Z|y)h-aay+-nw+ys$2P;!^Q1zx{8+tm$B&Dr}OU1@vmbj5(IJZ>J=$cI7K&$!X?`!z=T'
            'Ho>|Ag6W*9+VJ%aEZssAB+l;mnJNsXF`$#tPtlj(QruxV=ErJB<xMfN2qs!tkro(_l!Cg7QfOfoK8jIrc!Xy5pb^;&<oZZ#t7BP('
            'TD#Fq@+dpRo-W<`)vVCPP~G-QUQ;duEE?op!-YUfZQ57B2~=UrCwf?fXszOY0(s~g1x@}n$NRgu6#wVx$s4*qIP#thsCBst_tWvj'
            'm<Msb5i*+qaop_w{J;O_|I#WR)P%E07NFa=4iH3#5akZS8{bD6d!~IuV2Eg;Lmjr|Q6ep}d4Zx*I;RhjH#Hg@RgQy!qX06OZx<3O'
            'Uk>16;*#B|1{H$sV!+RhPaxZPZSwM-wfxls-U0eMo#THnZvo+g^o+Qukn;}4{yr>V$1)G?X6IW@s9YU&L|vBs&^nXyDBz?9ltQAb'
            '8p9+Mrt_*@M}cy>QJ7IzR5YXfw;;Fn!}%P&fCxu~9OS0K&q%Pl%cVSSKH<G|hsrDaF^Y`pl4XY(7@*@Q#Fi-ZoC@JP0UF)a@{sS9'
            'GQHQo&8pc{iOJH2kHjvoG!l4-3MsoQcR2z`IXRGJMlto|OdwK`E+ALNpMf|*4X&2>;dhznAX^G4=q9YR^<GG+W#}>ch=3FAks-5_'
            ';27f`<3wKbvB2HHoF{7-L>y792k>?)G`*`8ArfPIo8EfrI1I)4t%oQ$9T%H5ySbxV2#5O3$X72)ivJ;F57;zgJlpzWFc_$#;>kk='
            'sOc$@jJ^4|@m4CnBa-vUFuBgq(=lY%GZ>S0dy{)lgLuH-Rg@L+Aqy8F+m*=`a#^l|qzYzcw<W|8_)e9lL#JO^26><6`2c^BR{B9l'
            'hgDJh(gQUCW~`<10}vZn)Cl8wWOf-~qnp&;#sRv6=<&s?Y2LGnJo7rt0`>z9R?#NS?oSd;vD_8NqZUH<g-WkO&b~P@)+{%zKxV-#'
            'CwON_0Zshy{ZBHj%$%H{$<#{Rt^rVay9xGr6Xax>zzK=h%U}`~`7;!Kl^fgTr6#MFmm)>4#2dv`IdSY96)uk6oW2r7MAtweM3+4C'
            '-UoQ?i|P8{@bM$o65;`8an=CFvLH#xvnXT-O;&u-sYS&)ri($8LR*OsI2&Bk4ar;?$=wD#{Lm^uLj(X?0yY=Rjr6mMQ8mWL4iSH#'
            'R$4DCVCs)&FZ)lQ8Sv!wf$BoS<X~=Fm+3%cY&2p&prh00uU^3zG7tsx!Qbcx(>$qh0kNPig>9vdp{PTO4;|!TkWH7}tfG3q=3jPy'
            'oqg3CR{!q)dHCx9|N3*!V9O9BEn*|QQzn;RK(`_M@BF|2np}MO=by0skpFN%v3d0VppM9BRRdL7o~+3#1t!JR8G|;Tn<J3V@5i6J'
            'q#?~W!6p*vRk<PJVX){2j}<#0w~X0O($mP18^bg0WqIS;bpQOPKm0oD!7kk%`j5B0_xe}&&-emX#(+0K%k;XtZv-8%v&RUpj;{WQ'
            'Y<JM>vCN{rO~MSWi|#J3fad;(clVbIpj1tF-g?rNrn>T^TY~1{j7=Qg_bb%L+xDDe!d7QJ+!|6<@PRSUa#R2$Wf8&;rI#2vJ$PMm'
            'Q_8eIhSVQdk^uI^m*U}*{eAHjt*(TQPoLm3R?Wyy_2l5STYtq$$X|XNA7Bn)yR~>w7BFgD=MK@UD1pMCa{w%AAT>|=7JI-=f-NuM'
            'x<F#AcnUJC>-bl{x?pT4dA?!mj`9P|dBNJv(=mUY0#uV(522&?5C5s`McPH32&BVjdyQr^rsseKWC>GBSv7~Czz~)Kz5C6{^t3HV'
            '6A5HViI)ukVe;JZ>6ffBT*OG0D2yOMYmOv9I2Rwa&GZJC%EcngEcm9N*C^!GbTisNzg57N8|+uh)~@-jX?ocQi?rLB(st8;O&M3O'
            '4k@)ewrS%|X~-O(y^|81UGDInDD78aIi|b2erWo=H*4tN&^B6yrAs<ywrPHi2Y;~Npw1{s!pK96;Re<tiHeQlg_Z3ykiJW}t{9=_'
            'N+u+2u`i%Uz9KVE`W*$rV^|)ygioNuA&}t`y=w3kjz~5J%=Tu$C`ED32m$AB8kgjt$gX)xIOel=6EMF%yyN)wv@Z_YR0#XE&mbFm'
            'QGA#pf6cUmg<L%Rz;mLG1bKbDiDhaMDJSBg!+2Hn?$qa29Z%;fr&9`;N}MR`rjayNhE+Z$4_ZYNDWy`<4H!42xZ@>LOAZa_sV%AS'
            '`g0Ni%pACt<{Mh(>ai#Bm6uk=p>HLLq#K=Lp6Oh448Z3e#iDT}o`)79J4CtO6y$GWs2dv7eCQEc1iP4<g;GU{Z^VAbl)D<gZ5W2>'
            'waUnFYo~WZS0(0Qc*%9XKz_v+zVph4V5Gm8rWxwP3>e1oOY&$EhsHZzV<o@Vp>KQjsCK!#6%cu#hH{pEb!LWYTV>F?MVS$s^)1Fs'
            'eZ!4xQ7%aaon<|Dr-A#3@<qjj@F>n5oWgeUcjsR5o;$uri&1M>+uGett*vfv+OXa2jqBTAZ|kfN>o6}Qy9O)pd9k392@nN1ytvYn'
            'o?n=fmmxo}H#tf2>M0z)ockO6E+rO_jP->fd3`b{=AJJ)cWx|3u35wG5uLbmm$9EU#-2rAERYp4-uMEkp!N2A|AHsM;zZAHc`c=j'
            'UL6A}+@hh)z1y4Z``e|F6zC2t?A}<39a#KAGP<;$^9XtkNek|U<(4303NiClVIh{*x8GJG6tv-&T1eRCF<^W}2GXpG>R5RfH7j5D'
            '3~+}yYTqy)_gbOqvFlARIVPI4deFK1WNW|?2PkOi-{sWMcS$#f-!7{r^Ni3r2e<*|@c_1{d)=ecbB(7*-fLt&&sC^@oksAFdlQMy'
            '7oz|9{sCyGN*TBalTgmMdykaeig#KCn8B{|g?2XcKyv!=&G_B%n-|CL#2QX@2t3{A9X1;U34O<T#?Ar2g{;;}B$m9i?Jr(Q=1%xm'
            'B}W?qeHLQQ!Wp@T02jPUXR4h~3_3S9#s0i%sXO~1ud9}E9OX(|LiX1#b?Va-{MkeK{$PK9-^LaoG?iXpxPLg?4sT7Rfboc+(BJ+o'
            'e{i_}5Yv{K!3a}8$T8~Z0hKAxGw-S&(tijU%?p-LDEFoc=6!TXD_XVDqDckd9Q6-D26V+kM#A?{r|C(ngy_x*U-}>-<ZFt;*?^!t'
            '5U1#MErU3djga63dHBjGcs&~S-d~k*5v@s!y$aD?4P^tsUX)c}f?5>DKPnv?U2YW7FZ7VzplNt<&ZE9iemSP&A2~r{hQ!WJ{~9Y?'
            'Thb6-OuHYBRqLjB<%T>2{^J5Ir^MycLt@S_jU<k5NCVmSc8`S8s>)#S0lBvu69S-PgW%%HsJ;WZX^#X1SChpolPjP9Gmgo(?;P9M'
            '4aU@2Evtmov3;j`TlwpH(a<rWV@xV9SD5KZHr=;;y;P<^nyTyM1%ps4jPHB(*EIUCnA@Nx;wy3BE~@$coo&e_OhNsob?4bq7NE_@'
            'f&Rc;mIkjfgw2A1xu!4<Ycszj==JW62xn}S#y9<;3f(;6u~Q6Z&T}xYzULZ5bGu#0qgL@g$E9iXO{b-K_>#;bXh|nIWe?meU3_R0'
            'MXm0vYy@l<wrET=d+#GeM9MIMz0D}!U`ba6j&pQia6JS-O^fuvv-gIc-g-UXyKTJr0JcVj-WIOgZ^Vv<v@pHxUdu<Q4U@o@7Rm#('
            '(lp3KN7$?D{hG*?rcS-OIZHV_sQmMjkWzo>OdQ=RP=xt^*2yUyjgzC+xU0r%jLR#->dv|=Q-i#V#u(-olMX`zR+S&(AX${bLcXxV'
            'I_`oVB~gLNlsL7%Ipk$pFukf$?dtpVOfO3=npT4Kc+O=?^r)|QTBbyZ77508U}6Eh{wwn@oh5OS(2c7iWxG-@EmXCY>4s;Bf<}o6'
            '&~Xf~pT%_0a}wFm-9jO4);7_7txV+lmhKxWGdPU72+)Hvm)@Il#$L|4k6Fcz1N%p@4V7XXj+As7$&@WX9yB@AF^4%DDOW6CVf0qu'
            'l^m;;lxcK+=0ATx#Hez^5uU()nQyywB*Juj*U>Z_dDC@C*_Bu85)TN7KqC*<<f&&}!^Dh?F{i07#-z_2@uvpW8L>kgkWw@u2OKxb'
            'Fsm^^Ws|%PW-<u1A^rh*TXH=fQy~Z#ds^nSfg`ENP^Vy|eqdl|u%g+HoXjjX+K;;S->CW}UuxfR&wb0O3B@7?TtK%WATT6l1-ZCX'
            '2j<esjx@&_F|!KEDqzkxm6<*u?_hE<NT<^>qgZ+}6h#R;?@<J)b7_5{&nu+_$ijY<{6+%1SW^@fx=TEBzsZuO?LDmEjJg_PeL3o='
            'fftOyAr74DZeU&1`G7b>Lz!bD*oDi_)_LJp?--lpOwyc5_!X8ALC0t~sI#&jrE;g}(Lln<sAGMvPvi{Sd-qdIx+(JP(J!M9ft#05'
            'bSW=f@0m@lv~l37i~;?lIG6@k17n^odRemBx1>aCSFBX0npMH$kJJS>QiYZ;)RZ{($LqQxS3yJ>S}cF}-G;%&Pa}Yu#L~6z5t)Bn'
            'iZJ-+x!2|5HR!aHL}HU(^$h7k@`Y%X7>Oqq__4fo;84-m&ZA)j9CX#=g84W+I~fMob@zSE^CLr3wd@&jX^6O#o_f?Y%?Db-F^1p>'
            'I#OJ<R;)Vq5wP|8Bl_@nap(s<=UofaYeyC19Q`V#cEzijBh+0k++)obt=D)|JSQa}F9@bNRQEFWzBi+HXTmL+jJ}BvykB2yTw@IW'
            '$f8+DLBADe_M>$4%7bel{p96{6c8u_w4cC4R?iG4hFOSN;CNSw(V#h+cKYUznkq_%RT+q?WE=McT2QzfW4#>c4Ens}&EG_F%U+w<'
            'Cj?G1qg^yJ#kFzko%-Q!-OVOSkDAS)IATHmXT^nKbvz8t#i}>#Im-7mIW=R8BrOwB<|XA!Utr>yn=DNh2&=oNN>uHkw6^yyOd|M='
            'x~4Ll2UE^5Be<?yb8_+tOqn=~0RTjpry*Q%VGPqpM1~g2Xt5mV@p6fu(6ChR4<7gT2Tv)uS|(D1VHOCB1}YPFp_XY{(D!A=0GW{s'
            'of|_PtV=1}$qA7C`+4jLjLoXZgHOrXs!Zl_uvmn%0VTD-KiGw3mIXJtOoM}+!wgvLa1+Y(JPQnQMu3Sdc?gd&E($QDPY@|zMT*}F'
            'fJei~I251AIdj&~20@`fv8Dhz=DYN&N?#B%!wfpFPktUhKY8==)r;dd&yUAg2Lh`@9G`x7^7;k&Onm*Akc3r*=vJM=mB?^_rcwZ*'
            'SYt}Kk`NF%ZgiNzOh>Wvu1D*R(4_r@<Ph?ACf`bxRA@$W7UEvWIp3m!(|!hn4iE9|(Yv!#cFaQtA5G(#_usp}{`_UHJNWX?J^J;3'
            'c&JYv{P^b8^OG0HAV0z%FHX*mUcXji@K}C$%w!h^-lgeDhm;ja-SByK0jQj<G*LQQugS&Tn2k(uLQpV+L5Sh{Y6`|uI1cPx*+GtH'
            '2Iu96K&I%&9CksDj4zp0I2}_`trZUYnG72UT~AqZ)>b6aH<)qbD0}#UbyNhH#aLQul;>BWO6iz$M8|Qaz#vr3Y-bAPGHLA7fS!?t'
            'g{IlF>HyT-E}7)<dQ?_nL^TVyNO4*7@H3}BQX5FD6LuZoLZal-?HN^UtOYZJ?y<|=BdhEMIL7U*0dRXuL)`)*lKG#pE<m&yS0w<)'
            ')!OD=shdFE7pbzU-&LaqRD2n|$0NTCF-s~lazDR1`|jk&GZ=6wYdtK(C_}|P{E?duKPsIUkibxtQ87M8k{2mCIR4ra1G6yfS9<x%'
            'AH7ewiIh4ff7M8)W=$w22BiUfPYl)rWU)zV7U_iC>}OR#10}zj2`z!3YZa=z<b*`Ej3|YyF+x>RaddvWfq$oRpMsl!D|}$$5}lX>'
            '5n87NHq9A}G9;A@eh@d5T&!TsN-e`K{7kp5!f>6-^paZqH?uv^x{yvNtMX*(Y#*8m$VEphHEyBa^pIsZpHL?XwTtDY{QFW}eR>m?'
            'R3#5&ND`Bc!wXl7O|US$N7WKi_m7+6gVbIOZom1`4yGt{xWr%`M46+ah32qG7)oto7zz~^hH6%)MMt91<v!3xi$RDQUQdfAj=4x?'
            '^=s{oMXlQUB5%kaCtE8)ay=|zQc5A)<;K2jX_Gc;StB5LoV!{90n_&R5pk>PiAo|MDH!^n@s|t|HdGTOb%QkyLQVYoR-$Xks*ENe'
            '`&R$4(N$|84V<)hQ(1tl`4}Emm?wI~)s1W;kaOSRg;BqYvAf(c6so8UhQ5`1cBwIj<5~x^u=Og!&$z&QJ?%$tERh2fm+~AzUyuz6'
            '|I`<ZFZLTOQ!ovs*GGdOPQ9UlUg)~Jn6n28`LJKl-KDzaRT@N01Z&=+shM(shM6l5`G%e7)vTJ%M70Rq%w|<PiBa4Xk(QD)%wO&`'
            '%-*y^8IzY(kX3a=8ipF5@vy6dg8m-3Uue6k_N{g#a1*zkT7v7=GiGJ=zAp(^9H_wt$l6z!ivlF2!Z1tV?Lj5CLC!~ELDlsF-YhU~'
            'KwoJz<-PK1CFyq@e28RuD`|1)7jV}@M-cJ@op+I3-+^vn2$<#+rRrYavP)Wj&Ra?$s!6H_hv8wW%wSV2V3boze-K~vBz@+sYeZ7J'
            'Ak~cqS`t@Ff*0&q{ow-AF9vdCV=RS+a&*_I!t7xNGP&%Xd!<(qee}t`!keEeB(r4%BaHhKV8T?4fjcGBBV?6Q$#nw*)EL^iz%PA?'
            'g&(1u5u&mj$Zx_Yuk;^z81p?WD`DIhYKsNWceL?ic0j*JK271Acw`MY`5SC;CV!y>omX$}36XUxO8-5MW*^sG8Aenk#F0)dXN3x!'
            'X$haBZ>v)SC-vmbv0QOXzoycQv1W?T>kCuKfnn8@z+i@|3W9<P0%-@k$x_O>f~-3_n21+}PUfxxWfUJ1qhIhw<iUn)c4V2<At96n'
            'A5#RQB`?CP<yA5@(p`|G*HLSTO_<jsv0)y)=NM071X%u@ri%nil52Uy@wUGwfkY3#U4oLWw)Uk?)RdBB3MZ-F3QU=mqH`mPj>^(W'
            'fH`Q34SV!oB8yzGVG6Z@#*8FDz=1g?3q^$m&^|c#7;sDXmzbsHq`*2BVMY?e8`cJh{lPv*L11y&1t)};;1mF?`y&Hqvn!c0Uat+$'
            'Ax>}l)z?Xlg#{S<ybTH-D|}XgEI(*qA;sPTH*|VDkfmM7a1)nk$|K^6V;txbD!eceGZbv5xM4QHST^7Um=~-ug3X5uE(7BtXoBZs'
            'SW6-O1bZ!DI7vWpH!LJP(pUiNH#T<9RQTU2#r=)J=TOBR6j90VRlj4ifN4dwzNy<;1KHZHQ|*iO>Fe`xOJa4~wYh&1JGF$J%0wYu'
            '`84B4e0Gg*tQNqb1-3NS==9h0g=zB_ruG+*KG+}lLY!FyOJ>F7&~U#>+j#<W=Cx(-hfrpn)U}5sGzz_=Xmn}D*E|J4zJwd>QU~OL'
            ';LX~CER^mcDV0P@O3A@|!PFL&qmxk9yEk%eq7<KCEl{i(!_*ue5%tHAuK?AIsGK+GZx-AjNzF3{!QhEx!Z=viD2J;m!=JMN*LKW}'
            'ya^tzvhFc8Ho}xX%nq)(>@JkCzFZgIbt>!Z{y1U|O+z$muuJX<hN&4TW07iH5E6j8UrN=(so`a=p2}lIF~b;|h1#S+lHO20tPjZA'
            'm&LR<i1L(e!`46%m9Zc~Nm8U0JH^xoMpdL#U6C{nnw2t1P`xU0K3#^ZU@Y^q0bIkk%Tp+>L%VBSt!Jjyctb}BQ!w?JF-O+Wlwk~`'
            '$|U4iK$)(N6F#yXqhq?8P{AH)&!`4n1VxFc=rKb!#>^a;*~~0h&yA`I&0P=pvfnuED^`6@OfL)Am;#P!gx)53fF_V>h$a5eje#=M'
            'V_Fh#2SW+xP9ra&Txf<o)&KD(rDPa^)w@cQGGD<;QMx^(4l#NoJOhTV*noH^hX>>NVBSm~kkE0t=CZ;tL)!sq2P*DHdP9tW2BuI^'
            'YvW26Rr_|i;}Z^5gUsQv(9vypWv-A^O0Zr4y#(A_$tG_tP3d|uT4G!kj5eGu)i7K|8lJgn+f7A|xF#oZVFH&dBkNOa)<Xt4<mIYs'
            'ad18HfJSUT^u?pz{X`&6)@5*nI<tgDDlG<UtS%7y@%ixa{zZEybA9bs)=pH6KD3oGgI~Y9TydjzRXO5E?Pem|xv}&))@6jt)S|nR'
            'j$(EUCb11~adYI`^ZvobdG&&F7%JLK>$-1FwJy8TfyIlzn*)Z_->b{aq`W`ZnS($Ti?RhbX&yIdGP5BDFbuQibiFZLQrS=jpJVJd'
            'km!gK3}ALot{jH1Zcv~?%gdTw01RL<MYohXGp6>lXoe!NLD5|q92UWSL|T435*df2QEK1k2f3mT=NI~k92-BZ;*HHCjq;<`Cs9%l'
            'b%z;Yr|QvovZ-p7>xXo)->G>(s!ho0c2_zqO-ecUn7rZ}eY<BH5C@~_SQ@+X?tL7)e}~i0y_~=A>jK{S812?4xQYKBXK}XLNzI4$'
            '`4#K9#Pc`fq$2Es1ZXL@LE9Wz=Zf~lstO9r_*nXmOr?3VDV1!JC)NkpHMXc&qku9`$XpCq@We=VN%<RCC8CU~?DL~jsMrdd|3q%v'
            '|8my;&+n{#QS)at9?3vN#(Bp*+qNC5SM{(b6}?LD*&n>8eM0S*m<{O;5~N+nWtWD~Ld(#q+BF+o-Iz*c=cMi}7Vm~2OCi6=voqad'
            '=xHbue_CFWccwk(LrC89>{BguPWn4N-+Y8?*EB}#`==2@YW}(m(xah&g__r=^}6+@8TGaiir#O@Q1i$iR$I-_$^ZXh%Pw0yq^l(4'
            '{>F@r*SOT^+l~Br9rU*McZ?*E_T*2BByfk>5GX3u4akT|uVyFqK*Bfto1Lzf{@?9QZcD#2d;xFTRdgk;!Wh=^j_`$9`9Yk%I}>+C'
            'F34ah<>+#^k8hPY<bK}0Hr~{)+|e$1+T^Q@yUe+=>_01ZX{3HM5$oEH^BO3FINW9T%q$?ob+KxjlVrd*!u{=J103q#yEExP9X975'
            'sAV8%=xe1RP#rKwfG?cE);k=i_LWy54RSlYm7kP(2M`|p9Nd~f>5*ryC6`I>Y6jn4L*=)QzL^;Q?Uq6RfB&EV+cO^C;<K*#1brL5'
            'p84cg%pUFu9cxM+HA?(oDcrPgn`sZ^V<|<n-<-7+2@m&oWp+CpJZ*<H$->}jya?8AvbZOYtF#N%K&#{nf5CZadn(_@q>Rj$X-3Hy'
            'Ik)0^3HM}`l357=(*Um^#bmP+E1rlfW8YOg9}S*X4hcmy5qEK`$^fTI`S)Cl$+_jFLy;)x0)!eU281E)6rH0QIflGi<Px{#*k;j}'
            '4<)ek{fjC-+5!VB?m$q@YA{G!!tkBq9+CrxcR41VDWqA?>L_zcWUAN;Hb#fWZr2+irv3r{^=JIo5dZbeb1aTr1Yk^fVil&=bP^ST'
            '{KcNkJkUL05~Pv3*Jz@kp4Lubq9yGAwCYcY-rQer#Wym!%45Ysxrd>-KkmUdDlyA}>T8{d=8nxPZ?&wI(J|Mt&baXbs3PHcIUDjQ'
            'ROh4PJgldiXgVD`J|u}Dx3H2DuprL*Rv@ZP^ouWLEL#`{n=xkKeXpn!7H-}p2Iw9e-Q$mkm?sC==rIanTK(`M{4MyY6sN!|>{O0W'
            'PgbZfz^@OHKxT_58DFbshle{y)j*IN3GWUNa@nS73@i<-o1VwNeTP@+OZaE!G^pZbbMKcg@9bT|7H8<z#|n6rs%ux5I4n@Jk*~?$'
            '+q?}L5`+GXs3DF{Ptlf((kDyE<TXPwGUFEfm9HsihG}>%HepP4#0b!FfN7c$0D7^80oFbPS&J?a>#`sPDVH2%Gjo(swFF!rkkDlX'
            'Zsl~2Fi5Ik7ysZoMZGX~+tcQ0zO#VrZYu3kINKB9d=64OS5cf-x~!`p3t%3i2o^aFX2D7-@jO^RFN;3aU09_Q?jn=Y(+DhW5+5!s'
            '{?*177_7iZ|1wZ-R1rnfPk;ft>BEgu)7}fo$K#Ggu)m6-{}7;G&@5Ojkf-0IS1MQdlB>R%ZX(|KDXYX^CG+&cv?cO$%(ONX*N+DK'
            '`(GJv=iXa5<L|=!--m)yMpZ<jw{2?oi`#BvwS%u=?f7qDX~OI%nVl|!S$Z=B%rfbM_j4AkPchfiP#l`i&EMxgJ^D&equiR$OR_Hy'
            '#T%eMzIX`)CDFfFWA`?TfTV1m6aM+)lsHi1!AV&lf~r3&!Oi;?#1{4V+cX9V`V~eWO?~krvccF3;K6-;6`r6g2F>>0<$r&H|ML0t'
            '#SxY$utC$3(0ud=paZQBv{#a#FMc3u_*(sVr+)n8i({EJ#o8FTjz@&pKf_o2TTP$0K5dq!x;=Pc(=K@662q<U8Q_^B;y?bsefRZK'
            '`LloE@6-Q*TtNT3<ty5zQ@!D0i*5YJ@V`R~EL=SB|D3o?l7AT7K`SyG6D4azgxA3aC6rTEgdC&ZLoPF;$gTw?g5rxaa=$Pfh9u#%'
            '+|`*dpgi>6=RfQaX+G#ZyX>HBZrsxiog77(N|(e-YIJSkSfe?A1|9sxpNM&D0?T<)-WOw<+}QpKbvV~x+L&V?OmlM~+)w5o{muN`'
            '0e8{Pn%n6wcfhv_3n*IbHfxNjc0ED>p+#|%+R_nRZR8D7#rn#P1>C=etGof)3hpwDn~j09&FMmmH_rs}`y2!J8~V5#^$&5hjD-!r'
            'KrT<O<z+h$kl|5RXP@g7O{e9mjF|<`Lkf!(K-YrFpwq0B#M*@o=q6<+R#y7;CG6vigHKe4HY#JAx7gInwhz$J-*Mjaaf7zqlwH+<'
            'zTeT*`63)ewoZ>VAm*-1)h;xj$F}oY+qxDym`fTlvUDW+i;XK?%>whr(vRi^x0ggo8m(OC+`GN~_19_re)zXvf1Mm`x3_w@TklRF'
            '&aERsH215|vKcGd=cGsG&<{dgSMgwjXz0<tRP((TQ4psKG%>?%=IRK^c(AQKM+?H%-)n$oT;lS6^>3>|Z}%Dm*}F2XNnY_QITbuB'
            'tO={9U~BfWll-vZ{30nm8XRbB<~#Wn<C@l|K=~W#2MEDt7__0v;UjEE0(LUs#Pd4;p4J><kT-_X#L;BHEu~F~f<>vsG>gPuHt~8?'
            '8P2}4b@mf9zIEeZHJJr4F0Oa{<niN&Pta4@bIa;3RnmG+tYWU_a<B~F%P6Ye`S9S$g_8?KCe>wyGny7<3{F1A%2OQYk4a@?b_n8^'
            '`Z4vANw#rnI)z7*cfi{S>qD^b3st*A#b6j-v5HXDU&G@{w$!1(0@bp3flz>bosAkjj*L(H{4hv@c#}u@7(tjXC`Yi4HMvPTT3r_2'
            'P&=c7%M>G|6WQCyalN9lpg3lg<B@(Vt&%S@M6-(!W_QpvTwcO_-=xJ$)W7Sxoe5bGWikP!^UB%XQ^-Wpvnapf@C&Rr%vUvrE5~7w'
            'F>{wDq(lulnKe6P2Qm!VtmnzxWH8C1DHSHH?EG5gD)$rXjv*>L$RIua**~(!;ls+0+SmxZn<_Zm$!;MJe0{5$y4G7&b*p3;0_7jI'
            '5Y!y%xC&R0fo#d_9=&8MHCEPXkMI0ut)T<pj*mOdWM`@(9!$bm73&5PQ3>D;6ZNab!O#eIPQF?LgIYEX-LKD^z`3LJ-RViV7em+F'
            'X?WSwOq{rlCS$FkubA(FL><Kus#*f!-O`$DJ&e?f@*swzw}|4QxFyhk$hHF$hX+{j6z+i!<&{8&6yRioJxgKIRlD<zv3@)BC-Xgf'
            'VbSKvzqUP5+Z1%3?%wLZw^)U81B0PayCZS&<(r1B*f}y6K=;;p0^hIiIFGURygt^Bl^LMv3Lt$L@d)j0dQ}-0v$#Mz^{psG12nu?'
            'gBxklMsd+0OnN}}^G_Lsd-pgWZXj*?<Mnr+GJvLg&Zd>da2iRCdG~oeW4G+a=XS%3PujBGAjuDe9(WD4CK$JC7pTn=dvEIeK+?<&'
            'B$x~hd5biOK45f!xB`YPqMNo*C8j4B)-VvZl;G_^c03sQRhUFcBrwO62sQzT+yl=r_gDFKoCO*&{us$V+E@(b#e;`ZQ*q3Wt{xiI'
            'LNpsfH9~Y4$5pUNf-6~i)IAD7JdM5FigW3WVUVr9rg8a&dDa>xW#i-hrdOjqwaW#jTWi!ZZ5v5|C?=SEPHuR&2|GRnNCIzbYzQ?m'
            'F7<8h&7k2`l&zz973HvUwu}Vgr@fNOEZ#s}`!(9EmbQ|%>_f3@1*xFKr?1)EtDlXod0^Ed0q=+htXcy=$SdaPe>wTyMJ3Ow4f&}Y'
            'wmDWSca+?)3DSoulVG=kB;`8?pn7_@Q?xTk8P{*;W~?Wn>b6b=DDJ!<R-IhBVoa_DIM(MGP^_cj2Te6=#W6f28#$((OHiT&SXe0?'
            'hEe&T0*4jJK<CqwAKyJcM)_e+MOQNkW`e;*{Ln^0K?{2nF)8!RWnn)nlSCzzQfGzki8$32Bi)2k@K*jLp;u4hm6Nb=hB*>3pnpx-'
            '(Jl=}?b5dj8LMoaLKS}Dn0d_6-Ip~7FZBh(-@XJ9&+tE2A)=Jni6l=7!a&=*#L2n(7>mQ4tgu?CO_fHVT8KGr)H4>$a9yNN6BnKL'
            'wdCqo`&)5FGM+yu__$C=Bd-&ch`$7?d}&}@m0zG3h)auXUD8}G<@vlMFJqbBK&=zTaBu(%0GbfYP<h!^skv4mY$7}cl1FOcSu~%A'
            'ne+;ViLVG8reTB;ORUILVlksKA~}9h?ljCRDYUB?%l#3VSY?c#EbKm~LadGHQM5XtmQ}RvSas@V3<c!8E+<`RiLkx35^^pygWjxQ'
            'uJPc`-E)O!a}pIG0<oNcOm&c>Ct)-Nkx0n~n%7QaOqIIVYhxS!P-7c*4}nJj=m{4=gkG6pB7-$IK@9YM8i+<FVG{@L-L^Ny%!pC;'
            'Oynh?)m0>>AXlyE#(=Sv;?@}~31i<gP#e}Bf1|`$y5BGXD13Sb)Nd_{@G1*3AR9?I*mG_}?WCujrzH>6kVL4=Q9;tJp7d1t&?;@E'
            'NyE@!LeVBw9w2gH;G()RXNHPD5@+8XopHhuKEOu_#*L`AsVoF<!bf_sNKy=5HzN{?tjr6H{iS&MG)lC)L6s#H<`}LyIbl%f+;ni-'
            'fe^A$hJ%6o=({HEMv|aQ1<xv+q%#b3yOupU>E7<tL}osv=Hx_G?nfoSR()hSJ3CaYXQiLhL1Vihs|4<-$#tP~Y$E}6Ur|tFPq|*g'
            'omS5>1k2Pl>S#CXK-~~s^*m!|^6@M(k`L*`#`mb{q1)O~TU6DsNWBK2yTZ)WzK@Y+w}ZNe=GNEoG&0W+?9l?wb>5chNBe6t?{WP4'
            '`1#rKi?$3uq}AfX#K_%{f0NK%=3~6V+%&Mru=>of=ON~+xir!X4aASI7cn5<jm{=x9K0J&hbdD{R$P)i<am+FoWMCIDWx2pGC6Qn'
            '?ox(=>dc~M0_e6>&+>#;DQ@N%f`$)H$)|}}r6lE3k6Zy?mWtMJIblwy+{#c*%OW?F+zgZU;pm5B$!1h@Em0}0f|UxR#*5vJL)ECj'
            'uJmVz$!k_fWo}l)FjrYJjTZE^n<$Pc<1_LP+>3K0RZNC;7$rfJi$(c@5aI&)UR_n1IS)p*pvV*h^_(ifB$T=oH!Q0aTgb3_b(~@*'
            'sf|tfIJ+m;HXuxwG2ZgGQLK?n3gb6r9KV&XrK!ix$HwPNF>F$8tz-OU3hj~8Y@iE;yCkP;#1PtV{2JLpv5L|=j3sYK!F!sSa7VFA'
            'm+1W<IpJ^`ts>ZlDdxTS00N#1Wvg}biXq1g2o#8Dk5{VNZM_gDPv{w9N<N-!6o^};Jfc<?DrW+b5&08tJ$D(-J>!XYv9;%~21xf^'
            ';(m1Jnvf<sq?!y@qgxBywyt(Y=Q^M^n^Xt9ICIx)q`!V{!Ia5Qjmt(=*ESyY+%|*N75ux4nQ{QKjOQf#(m1^`5P}gOW6l;KX$DD('
            '!9aXpRJ#wH;xhOUWf;7Ji9IC7e`386O9gPZ;4g11TUXI+#caLGYlh5SCQL)e-F}%X%sKt>+p||^uaCz+fA{M2?eV+u4<|2<U!UGv'
            'I`C$gTm@66tbhluswQYRANpj3N3MM^z+`J%`d>=~IMY0Z^cbn}8WaLa%z+Mr9K`7jI+9d!M;(~rD28TrVvcL&4>cPP4X#{gQ*rYS'
            'Tc6jOmuq`2`fD-<*3?4|=0KJ?t8M0;2mKEhw})HLZ<vS0ZPq_h@6f8AHa2#zzQ(*(;yT;}BD%s;l+9z*ID?>t(vp_+luBh4>ii96'
            'oC>9Fs*X+1=RK{RjJqOixm%M_z{pXg_C?CQaIi^rQXM*r(d=4(`Sv7wRJ9zt%bGaWZLgx98F6GocQ{qMdZKD3Cx#K(q>>MWiFMd&'
            'uv$OzOtOcmCAylROL`3KW)vG_4h07$WLC8!!w<Tj=bqdVO~%mCq#w!0_SSj*!fdYff`$W2e-*5e0`Cv6hx_=WdJqiv|7^OX_NT*Z'
            'd;=We`@<)Xs4>-ApyXbdSPi)#dOXeM_}{~)+$efKp0DW5ro4W1xWBf*22*Ow9ftw_HwdO`LfV2Q?Y)MKz-#NvkVG@^W@OK|({KYx'
            '+6JGQyWy+52;f$*UZUEyFT`J@3?nnKEK(%b-yXj_d3UT5ybwbop3C$(QsK!t?^sEDgRYk<)g@`ixwJ|E_@#MW$t%m$q6>`eMGe?9'
            'O8%mt&O$6*pD9N93+|p5>00H<tE7o(ie(9KAX1km6AjFxEH60C{-A;9R6-m27wK#;sFaDN=2R0F=j|ZKGj(FS>YdM(kyUtW(}Ao4'
            'i@5K(nPKL_B(c?LoA2E=w%)>p{2?x{aJP`xq(w!iphX<4&}^RtRNn~Ab5b*A1v|F|57$z@RHn<y_H%ENSkH*pE}B<u8sd$2;BwVZ'
            '2OZb$s9N-K)AsnCS_%oh^y-OsA!pWj7}2>85B^yd6_5GVOzNm|-s!OT9hsb))70|LjZ{MM9g}v2AmYF+X4!9*lVqBpkt1=L+=`u0'
            'sOv)L{r(MY$i6mmv|7gLs`}MZF5lX@oElHmE>^(K{A6{PPw&&cN)zQkP%m}^Q8+lx%QYuwmr40#P6sR5jy|n}5<|VNRFzm;oRR4a'
            'PELs<GqM;2xmm_zKue?(ub-QF?=GtJ=kD_owr}Kdq*PP1-3qQh#9SnD%`mny&dL{iyCN3N?u{AY0MM+=fm$xg>k!ssZ+EK23Yw9M'
            '?}Bw1p30_3!Csb<U3GIjN6CwqE(Ugol@?+)j#6rKVK-^Dks`ywg`t37c>eAs5Ce>A68O<H<Furxbdkn6rcpnmqUy5Y3zw=U89Q?x'
            '|NGJNv)8}K5-^xhE&||Gra-2(W-Pl&W%=x=(6uvguQ|MMCJT^pidvGOsk#_tyY*2D3^Dh8rBcIM1U%kcOg2T>he`DD2jBQnQ7N)G'
            '_QEjsqVT=KOjRoLr!t4WV|ChTLHbkFa;B(1%PBR{O#HPBC{qeAW!_Plj4#c-xg=wyz(a!<E9_GkBtvD*3rKZ_Wr2zmO_-$kk_8Wz'
            'Q8x3XGEMBwGGLANs<LDeSu)}cDfXyJ9<I0=5+)gvHbUkW&;X)YEosNbs$0u6nwo3DD=d&U#F4adB;{%nVwQ?YSuig~POn_nP+)m6'
            'w78OXmt13IrRGbiB~TK^|Mlb1>sM#LjNiUHK0Q7=y~G6IY*69ae$wWOUdRb^Z9%puS1K?@er}SbH#uUZV?wbpvu4(y6ufpDj46?S'
            '6W@N(AjV<Tn%fKc`KGpP&vaYU(Sr8Y5$}3s4401-5ioVc{@`(swVxI0Fs0~vIS?agcJLJbht*aq%Zk$Q%bJV&HdLnjPq(s|?TK4!'
            'zv7is32io7Wz=u&w}C?SB*s|e85Oi$P+k;+mUB6i@l7-<mR88Rdb14CJYdD9cdQ%?7A2I$YZ?h+%_ifxut?jBUdp))Hq{~P`;A<v'
            '9oPQHegA&eqfaql)%)CK$Wpq3oC8QzBO{{#$AdR?X|ihRmrizomb`~n>uM?QXx$vRe@CaLX#G37I5`IF)>bPc>%g5bO>}TT);M)O'
            'a&0j0XE~%I8siUXx~eh;0Ie{hhpE9lotBgpqRJ}p*tgzo?03c?+i8n@SNWkR$w0}5rhN=cP@cSd^_N#~j$TWvBzZ8YR8tKmNs#~3'
            'gjfj7P}EnHjgnn3r=pSr%w&KTF|9e)BjJb+^n9XPig0@Hc6t5!2fQE8&Q7(>iXD*Lk+XMKj;20V0{BW_n9xk9S_Vxb$_)XSQt4}j'
            'vgsE^83{31^EwoUol|orq%EVNIHp_J=n&*_&e9~KL2Iu+t8ML{!MMiT@ku7Oz5AQk_Wp~Q*c^|ePqw$!e3yO*uki$7&p60%i8*n!'
            'Gqm2;bvpV)lbvQnw8c(8={YSvi@okIz^6yZG|%Allz#(b-N*Rr{@u-X<{{kaXTHmA_-;Szrd4Q!)pH_coY!uFPiv*Vs(G+`;@tA$'
            'qfi6s%9qT@K3I7?4<639eb4nuuNsK29A{|kkY7ktX*WZ4Of%9)+M_<k@^p`0vQCBFLt3mXmJu_?fQ<8L<|_+aWe98ffNSac=Q1BZ'
            '6E3=N9S^Mkt;l4{SbJNq=A)_1r*(_y=deaMKy5cmyPy7B*r%JOS^vIiv2K)_#@dMC#5ifk@1|km+SEVBx;?emxHe;Mgm*>n<E-Yj'
            'fi{`Xo7P|li2JO9r(Eg1`<UNt=i+<Y;wNxDtHa{h<!cN?H_ejpYKU2n@xCqKa>b%EaAPK;+u9f5Sh<YE4)eD|Wj@9XZta5>g$+7Y'
            'J;PiB_J__r{noo^07cYlMEa+i{M-(X%G6vqLo+35V|1>7-3Zlf<c_rxxDCj*6y6BeZsJ1KGP$z?)$=Bp`fY6mwDjHpkDs@}qqhz4'
            'SiY};laHIB@yi;pDtF9CR_SR%$tt&o&o0_M2aKIZLx)#10&`Mv_>^4GzI6$mFsVNtlrYK5%(=m-MYrinHg$pM4ni6<W$Lt4u3AAi'
            '&F1<y+agA7c8^B2pR4J!_<YhT%Dl{noYd6RH;ON<FZaZQ2jUPO9$G+H0oe#ezG?^4sS-(=PBYdb>DT}AmSJz1{ky$oz7VH?#$k%('
            '+a3=Nl-ec>`@9oWSm@~Jxxf<1xyYmUl)N&58;wjEuTM^oPt7`RMK0oCgSHe5L}gJ(RidRdT?mX}>naq>fPW=a*E%=Ak;cpWG0}Eb'
            '^>y`;y(fA+d)`1Cp`%4XsxEd@*HLb=xFcq97FF45)H2chwW^FgYE*G>tA;Lr(yX6EmNl&SDwRvh=#l}05=^jGHydOIP2Ne?);eBF'
            'Hm3hBS0#OLu)c~f_+~*10!NUsJdH)@f^LhX?K=lgW?c_*$96&?{+q7Uq;Vh%EXS0K3ic$c3k^N4;K6GKTd#+w3bW!1JZk64<|89?'
            '>fdy^D!WXAI`J|z48>54Bg8m@tBA9NKGOkYeR?kI@KHn<d1eKx;KM+XUMpKaa=Rbve$$`|(VAOCffJ{4p{iOS$E97wjuLV7_U*fq'
            'pN?Nx;UZB&T3b7AgmjRkS4KvP;wn_kgak(~NUZX82!{-`;=;1gds)kvd54x7(Y`8POjS+$TA!;8luqa+7E;BqmoyK}n$IFYYpyhb'
            'GPE^3K%~*Zzl!nYa;;dhSqFvbI$OrVjOsQ6M5>0EyIQ+mRU}V&YEY@=u`VNsihlc5<vF3>Mf0YCRrm{-r5QG=>)Ft{7C2^B97X`L'
            '%*XEj99`yiZyq~VAhbjYVy`G6Fu~HjxSWVaWmW7|<_Xl3YxthIjLn~e7Wewvl+z5Uh3M652+?>N2i6fSm2~bg*9*7I$~lgtVMJ}!'
            'dhVX!MP2HwMDF>X+Uvf7S+}hSVCKW}%H%468_e78^;+B>Pm*8*BBfDaMpZ*W?n!iSF_j(f?p<C&^F3<Zj+#b2NQI$WQWnO=D(vl9'
            'wP@-X<QAo|oHy&uw?OxbD3g_7F)?-)VdStRwlj=li{P>0xoJ!Cwl`pHWA5$HS!Zr*g<<aFt@(Q#{F=QsYnp+bwa<3Da;l_aafR39'
            'KZpDb#bD4`Zy*b*AbNGCt7DVvw{3uTa|rHztL}e1xVjeaEE?#G_dmicwG5R^TR0T~XS`$aVj5&<j6mNcc3Wesq?Hi@F5Uv;43*ba'
            '#5KVQW=w8ObQZAq9J8{5Oo5WOg{BksS?Z{?xk)sKdHC#?9V9Xk(g?<;>=RU+kP`EuIX8!^HBYSs7OE6F>>w@~#dFilkZI4wo?}{l'
            '`})UIGSV7m=so%}@y*L1o=b=XNmejL?d~<&Z28tsQt93|Ho|grxhcKPHV1as=67trkRVG~u-BJ4VUqjdVS`JF?&>Hwzr9a&XWp-)'
            '*Jr;R{l}}*SO1}_P}kBmodU*O2f*h5m^=ltC!%SX0RsrEs_3rD57E_w;0nV964<z9f&p$SGhD9*?TE2i>I#)aR3O<#iYE=Y$M4s1'
            'nuXb~-Yn2_z|X>YZ~!705b8F_g_&irOj7HvS<l)8*qy9smftHE257<`<|M7~XV<2&6_j4xIhHHAP+er>C%eEZK6<AQtaq#XZ8b%p'
            's{+$tICOs7yQ5k-;%6PlcJ!$^w^Ak~vpzUt73Bw@5N|2*ju%<5UTSZ4{g~@u(#vkg&IBwKKc<q$x3x9CK!+IC$BZfu8N(@t-C;(S'
            'bBr$^Z>+}~7ZWhi?`}*`cjf>!oa~BXU!p4Ur+J|7z6taeswLYyr-9Bc65wpkuR+Ye*sHgmQx1Jp?_(O&YTMPAh;!C8I8>#Q%3@*p'
            'kg+&i_aBpqeysXXEt77pYSXEeNc0=l+&%zRA&Zfz*E7oCY(|!<&hOL;)u0}JR9EKvRT-O6T|aNbQ3~EsO-*c;6RGRCT_~!?NL_nv'
            'w{%tWqeg|l-)XOAGip%d+wHa1c~n<dJM5RGQXaLar5h`!mJ{ApnFAfKPX5JDk^5va(5+)auVZdwF1<eLN^IuPbsMjt-#Bf{VmG8V'
            '&}8J@wv0?e>219|Jjc~=m?C=3Gq+FjTpwlJ9q+cbG+Xc19q)FSFM~C%IU3sTar`wk+IjlX_oXQh-luGqecfp=l9JS1F0j1IDW167'
            'niKBUw<^AjsxnNzRqADA)G~1`>_&N((L>d3bm+@cA;wqL`Q3ix=INO?Zy}%)XT`!1$QJa+*1N~{Y4+lt#G7|xjMc1TtQ&yk<)mMU'
            'efj#%lrEIokQA02p4oikaSJLJRkL15Jt1pnDZcbYbQNMHr&$*KCce_lK@nb|8y;|YA*Jgw%tXMYNYRNM5s_QRXiG}8UnL-;+V)sa'
            'fzrF-zq<D}jzlfGYj|ZZAWp%0&|nbZU94F|?rjipY-GwNVni^&gWksk6St}BVSwrNJPsD5zx@i0FkFs>zgFA&->uA-h=bV=lj%lg'
            '<~BD><rSmNY>JMQ{-o0P?s;@o5bh+=l?qpFaKn?It@@&$s&Y)_mlYLiwItM*V>kC%`L_ZQu@(kBvunjdv8b4BY(wjzFAg4K73PlV'
            '=i^M~TrCPMns<S2Fz^ShYLeb`@y7>X`UYh&#dN|cQ<H8hI9TeU%n6Vut*p}jcIL59P_4_F3c%~=HnfGopSwRWmfmC+)+OM{-2@q;'
            '$Wn?%TA*6(nM|RJdEz9^VBJ@gOhcJi551>o+f7y5U8C(-5+XUO%z|YA()x!uP`E|ket=dMIj+(uy`09g((4#Whi#NqstrU~=_s!r'
            'HD-8GN!!pLZWggsVPqESsD<F@d~U-ww`AT?X~$BneRI*LOLDU})@dx4X}D_%RlyFnFyXG-!#*bm7}O^-L!DB=jfN=>un26<^>io~'
            '<<*68|L_Dp4?@yyS(Q4ESnb`h<G3o-xmh#aF%8(-cdX7;CGl$k;AG%8Dv<?nAEP6dGrWy;I%FyMa@DQRq6e3;QfE7Fu!?)ftt4bG'
            'fo&+_)=m^G^&;BTGA47>f5HHg$FyALW4UV1_#BPG=bT4Nu^V|Jh8BML?t4X|h7On7t@3*_O#mKaRz4F&xw4fma3w`G9sL56Jp<nR'
            'j*3cnjT9|RY?>zS&=uHJZO3>)T%W>)IWg3L-=Dns@$`oy%qXo?e{w!Z=v-})r*wh7xc|tT^_C5T7R`gME&y0unKBjDG+mW3Zo#gB'
            '^lIhw?tM8}<FU-`3_^1ulKof(XoA7O5>j|o5`=+cK;}T620}2?LHNeoC;>hLG4!_M2Jl;<nclGGVQcVe?U#npxAC0|pfR*%C|WW>'
            'm<jwH*L}KBY-j$)ZiYuYey8@Z>a{hNO{YAqEnIciz`KSU$QecLHbY|JK5HHW(ac4C1j>M-mceTY;B`wYdxpxhx{e!GuF`Vq0$bIi'
            'USnga_H^3Ir@Ojr8oO-VE~cLz^swCa?3SkX4q95uyvCNyXje^})O()EqE~Cz1l3~StEP=mci1HnSMO`J^y+@t+IyQu(CW#34>+uX'
            'Q>X3vlb}d*`>@qvybVG_(iK$}?vAu8e1c4MGp0H*|88lkoKUOvYmOMHR+XN+`BG}|^g6bIldF4gXiWGtYq0~Y)p9pv>H3PclRUEE'
            'Y9C~`NgKe*HGSCN;l8Vr#Q(c=<#s1|5$)32%Bnd#Kdmse4UY00X#rAmbsvg*Xw<!Cr+2e+n{wT&xwV_UTI>T}&3E1H)L<&`Tn}}-'
            'kF-)WcQK1P-em;Poocln@3aRf_R72sNdcB!H}FP$2^2b&5~(|#*LHG`p`%r6Y>Rrfq3M&}c)3@XbnSkxQ_ChI`GxrTyI0S@GkqWN'
            'Qp`CMbCgR(2l^SHIkX@j$N^>GLAPOaYwWmXf;pqMN`o)U3kNzWJ(f3-{u*(bUgfO)8q#xBTBHK!6d8H``uOPGDY-{0uF1-zA0>)?'
            '#-w@VncM(ScV(C9Je2+=1-ko}Yw1y|TNSAw4J)*DfN{2)$~T0juH7=(*IS0?4;+j!5>r}P$7=E&$xL;xOEZVQ4PhJ<QjzG@i`qgr'
            '(^*s22dz76x}?eIqYwl~x_ex@WsT4&rzBaxm3<5BHb&O0p{QKr+~ZR7bHZa&d)U^?`bfi5r(c_xi%fjWE0-zB3RBu?uuQq1%=6|B'
            '+U{runpTgiT5XjY!sVbz_ez9|tYs2uhOVv0>DFxNfo{z&-1!)0cDjjB_E0u^3}kG!Bel(;$)s-##eD1zq(#Std)e~JfzYawLWFI5'
            'SCj0NA6Dh;K6IQG9inqoxyGSO8+@vqaqbc#sz?lws%eKgbZ}dhc{IfUgrbORS3rOX`WKi86+P?d!s~}gm@PIy%N7Zd*aQf5UQ%U('
            'n<$y3H-h$NL$M!GQM&}VEQyxliPR%aj-x9qq_hz!5p7-j)_16AooXk_5g+~cjN%DzPK=~2Fazb{7l#YDPyl^HZtXzqVkH;DEiehg'
            '8QlWPt+Yp8fe}H&YE?4ZtMuP<piSX&Sg`{=yW`1q!Op0@<Ej${h61E>7}~thRgbxlf;-<7ixN&TRe*DG)YVmhFZGA5G9o*ZS%M)y'
            'sxbysh=SY$o2$!ANt>Qwo*x<O&{wy<I~${j3<V#|fi^D99K0|CDo%sJ5(8JxPgmmxPiG~<HN2|PP%gSPHr=A(@@nep+^cmr>Sjia'
            'Epc@&j5WGO{o~<IYi46Vd8;2FYhYF(uLB`(5BKRkE&{`l*<fI4R!taoQ|v@~_xdYe9JZ{OQxvGSa)up=S?cQ1c!ETlQ&?3GgF-!Y'
            '4@^P)vD?1JcG-O`9U7ozp+0dxd!}7H9?kwL1)<7^cW1$MGbNYt_!`qBB4Lnk+C3m^iSu08?o$oA!L`k)i1_gt_S!S+wST8Q<=&1|'
            'd+p$wa`5fUz1I#3E+V|o#vzqppWSx^j#x?Z_;nhWn76J<cJ}bGFKZtn@8Z6<lBqQgj5&>}$J5<O1a~G6Z2!{zYEs=RX<Yk9+Oo;*'
            '$|83cw8r#u)pSr8xi=o%k&Ny;K-_*DUSmKY=Uv6pno3fHlbLN~=_epVlo6Jbr0zd)>BS&gZ;}adN&EX_jG!$iA*J<L1yZ4cA-LQO'
            'Wf{2BrZN+(l?zYT<T6sr`C@?#A5gZuVy3EiM<9PO2Nx)$WG)d8KP@S7K00LhD`FN*NMDIn^d1>EPEnWW;!f<~LeYmnAe7-BWM}0@'
            'k{cQMeHjqR*?sMlbz|kmz)8T6xa0h3H)JZmH<T)tja-ywVO>sQjJU+8hz+NI?V5Hl%DM%D-A%B`d#c8tRvpMM44EgKmkBeI&yU_v'
            'ZByn|1>0^%)pds4c!Yt`S2B(bSWc~JP+4X%0-JawT~`w}j#V+p+tP^Y?gs>}i@|8p#enM;>SYy%(-B38_o`?W$$u?_T-;zL7-U$P'
            'W!{7^oVS1qDc9#pjk_7zuh$`L1_h{81ruPbtQWeZVp|mwtC`9lVU-%bcc}`6%K7llpMH1LhwB-*W{t_b4wQ(AGCOlo#%WO!;-IHB'
            '4F2fN3#_jj<XnHDVF`>(!*!L<ItvZgk*+FnE9dPCeN$G?IEX+!@54$i1tryyGf`2nn0k>VwGh5kl5D_uJw<oWnJBgVbE%iB8qB45'
            'U_7rG0WV4L-~-R~Xe~b;E~|&ZQBvCC)q@8pOy*$-gaek6N?x)c1xF9PNM^M<tzU=}k^=x}I^oRAERacGquv7<N7AG0%9JEK=Qv|k'
            'y%Glj_qznAzRbli0JqZ;^Iv1VN}yEFWL|+JPvellkRXU>@Cub~sS$9c6mh_}g$d1!Jb$>XAZ7wGWHG|re3{C!bon|;5+!tfj>MJ!'
            '^f>_8-4z`GG`FrqMsilVs3m>0Vq}%K*B?x8uc1*8p%L(a=*p}+-*cf7c~?~RcIP*@p3ov5$mNSD?(Owzqt!?Da}T%z+%o2tFE<a{'
            'zJHZjf_<RdAXbe$E8w#c47$a^J3`T>2E9s<bYnqZ!c#cnFOh{-SxryFoE(cXIC!%(-vsFXQpRP1?kJd!uIU?CE4Y^ARO*;9v4uHT'
            'Ysc-js>Ey~9GUR63B1%=S9wxRJTbeYN(heq&Qf4acq?<0y5k+tCn;s=UVDO8BR4CjW&Na#tmZU+Z(`QQIXYQdjqm-flC=(gN48dD'
            'crs}#)sS9AIVZ%MM`1iemvW$}=u0`;AM76(2Ot0zY8N(9|6VPPsqxPAAq*5|A^uM4Xx3O9*{s(w$-C?a6QKJ#|D5!C;+tv)9RtDi'
            'JlXGbPIxs(oaojoru)LJi%tqBW377nE~tUunE|;8kGxrM6;V>;IKouf9|B}lX2EI^kRxLqmC$y#Df3kc_uWt>G&Y7%k(Pg@wEZuY'
            '()M53-hM~6x9_W*Pb!PYzmmQEE7{w>lD&Q2WN-J#^LBTNH@+rVwU357vg%_{r<G#OVw|hS%rt-QL~KV17MWO~1G=4i!5|`Sd4Cik'
            'W=X;H8O0tIL2i;H{Kj>-U;dvKO>pz$f3~z7ADxxspEo_m?+p)c2@U@nhK2u*kZ>Ch|7M}!e8OxT;AbWR)ajh>TFiXIQswWki`6Y*'
            'z26s1?wfP?J4cp(MbSgQE55wg|NL>~>Q$rK$GmetM*#YHz)${(BG9d36#tk=^kV;C5$OMgBhcm2Kj7`?EB`<vGCktZbkXIHj|#LF'
            'Ji~y>S8q;_-iVVoN58xhFOL86>h+6P@62#dkJPi6UM>(64GNN8in%I-1ZyB*j+)ON!+ytn{)Q3lt_uv==|;$zFM<9HWa;O7a?8jF'
            'GZpH3WijtR^?JR*GJG%Hs(a_dgC`f#DD|T<GTOF9&c^%t=<N8Nw$cr>mzDNbLb~Ktwd2vta6$%q6qqI;Ko=;DO2ZdfyiyUT$)d7Q'
            'qL>-RbGUYO^;I;rca(!=P*j<Yqk?=v^H3Iwla@)xvglY9yHfCz5UcFR>T_yedfM2ijErm#t(h9{%s!y%i&ws4yNyt7DF-UYrzTKs'
            '7NWbMAy$p0`f;q?7HuuqJuuC%G}?K$(CN2rlu~_LVL6Yu3I7`-g8%=ShQTyr?HEI)h;(x{n8et9FKI9~b0erLg7l1tBk*=>Onp8J'
            ';fS%JOXd@7NL6)L-iSK}(`|CV6a|cG72|QiF!b7ZWug)${;%FVKYo4u!|~}W@$K>VuV4Lg^3F|3+2!c<lCFRma1vNcDowJ0ryFcC'
            '&VC8U8Kik%w0d%oA|B9?QfW4&t3oGL7cFcPN7FKV1+t7c5ST<6Ml{7}dXQ)7j@n>GgEYrv%Bc9S0S5``Gi6?A7}!55hQ7&_Q^klh'
            'ylU-k`+T|(?6)5qhHuxD{09Z(SBqb_+Izz`d;ULPK!1B&{>KEq-x0O{_lnft6@UNvV(vdK&fZaApewke(%2b7YUnhe>kfs7<G$`_'
            'KQP+EdGt}yrhkVB)BDBueM}_Wf1Xf_-w{3WQK1_DB||NyF?ttTZg|=g`%T(U*V<E2dr2nJGm%Sj<0ewT)<yvGrX~8p4$$FIdxo&o'
            'iGq|buELG4_39uBqW3rXYly*MpnI^^o_nyK48yZnHROvgmFnKR2Pn=9PLO0<!f~hd{|A;Z2VD'
        ),
    },
    'youtube_data': {
        'sha256': '048809136491091e9650d1783a6708cf1bc59c4a719983c19ccb1fb85b4a9f69',
        'encoding': 'zlib+b85',
        'data': (
            'c-q~4*>>Ybvfw+vB8T_(6;P%qmF@EMuuS`^CH1w&wRK75ZnvyPfkaY53k29$q++S{5p&MG&o9m=Of0zqpr~@YUdHFRL;$%)M#h#A'
            'k;CEet!%1zmdTMQWt{95dA1fm7tL8Slj10@V{!22MBIEO=0%ptd7T!y$coE!J{}BCui{cBVzw6hUq^K*^CT)(tD>x%JgwJJH809E'
            'zZ|Elbv_%3{b$ilTE<zlf=Vg1q(8-ZGT7gb)<si8w^7{G#WKbM(K4=TS#mjXlg1(zv!bZ0x{Oz1F0)LexsdVvO5{Z?;nTGUDk%r='
            'vto{2r7M}Gx!luDo61p@r>m8$$E)>wu_#JWUr8Z9#Pizepq3x%L46h1BCUj6&Sa9neAB!xpm<S9(`glost_>@r@kts6!Vxy2IXGA'
            'Iv%9URVJ4*uW3&39_AD$qFCtC#VVDRppNE6lO-Z4(?u;{%vh*e=LBszZ%TkG6v<(9Fwh_=a0CFCB8zJ&Zm%Q&BS~w3I<y93UZj}}'
            'Cj%jN#Th;?i=@e5*z!h}wB8i}w372u)?!f>OK}J-%|uhlGQqV6xD?e`=CE)ua2$JGL2E)3w>S^z38r)-pVNe8S&3C#RRl`JnrpeH'
            '<@vD6(s^0~VgW$j&zdw#qIs4AK;DmOY_AF!EsXg}maqsowiRGbU5yac;9nTV3R+LAnj0_UTipRS@V<(1Fdt<k%d#jVT)6jQ&w1aJ'
            '*?aNw-RYSCypC5eB3MZnPMoh{fOEORrJ<=|UEB=Ss>or9XGOB65xjf*0_p;)NPq$&#4W5w4G_mx#Q!{f{c7*<^d|y26sM+W7BU$@'
            'y?G2k6_-VEnaOyS^7L)b6+jRnI?tL!(i|5Jpin9Q(a5T%#<2h3_9_KjM+}Goji?|(ImUH_qU9P^pp2IiHcdG0tULTNn|y+^U_`*h'
            'g9c!w@?=%OkX=-Ss_9}qq9HV^tcVj{k7kyo)fK{z*A9?Fk&W?SI2;ZLouX&~`>vExBw!N&Wq|U4v9u4W!9cycimR(Eo$1HFSFjT5'
            'H;{oV{ky32uTtt?b-I+?72vDHC*6^L9#NS@W_8RzKNh*FvVz8;UtQx3_BA*?J2-oH8vS(e;@$D7n2I}^`4Es;mMLxGY&a6b1R$3c'
            'D|`{HvUo19iY$>Oek$cnDsLlPyA(*>kYP6jq$`UXd=^7kyJoR~O@Y;Mr{T**45&k&p<vPwg^X~$*3iWLVDReo+m{C~PX2m)6#aB^'
            'bo@FxeEs6x%U3wabME3MO=J<J)J6>(PVz{rd6^QHQQN$5GKu0EpXXPwBr;RA^pmQsn}z8BHsn+xORuGUniorgfq9WMC6n&7Dt`aX'
            '{!{NAkQ&`+9bZ<|0IbtxQLfbh61j+*td6oczii@5$wg@NQG5dsLSCf`5&^<laf`6qZsIxc9l)=K+D@yex`L&R%m84QB`Kl?kDXYW'
            '0>$8BFgQ9nJ$(Js@!K8{Z-o9A8v1~%bA&7Gy(M+8U(m+15UONc&aa}P1bm@#ylgPp=&ES&l?8hh0ou`PEGY9!C`Irt)f^BD=bGEf'
            'Rw4C{n=9p9=5wc@Uh7h>WL(?#%*oATlETj$oIx4q*Bv+={&4W>)$xliGCqh{7C1b41DqBR6|Mlu$RZyNm+^<F3!$Zq^T?ZkUYifm'
            'Bj)vQ;uL5w6JX%BCDLdu4E>{lQ3%}J6=2dx@G_2JGa;#j4ZQ>&4zz`tKN1*d4)`VUN`=<q>HhvMSCrLW60gOqnO}nx0%RJv26W48'
            '*-+w2H;pYrdnj2kr^X>(%3Oj(1+*>&=E?)q4gM(Q3Yb@Y#p5BkRRRY;Lr#!GV}HU5XCO*k%ksGq3P3!oB=G=X7bHdrU_t2;B@E;<'
            '4TwI#`H1n42Y-6^`s@G}9~djpO1;b#fn_)mPm$te@fsgbYPs}2(9(Nv9GK#p`@zxi9}nKWIEzk?58fXB5FH-8coDrgd3kacy*Yjx'
            '9Uc4(y#h=PLjD{<iPwJyynOxk==d$7V=&bF9ns}b<4-9wc@x88(?g6h(=bE~_(6i01rweNl7)A74F;SdEdfC=OE1$|nxQO~z~0Fd'
            'Tz7!Y3W$jyLW2@5t`H&=R|1NgE%p!;vK*1rRu!3q_MxVxPGe@)C=^!6P6Tg;O0`6kdoI46ekGP^-qdD5x;_ZRFe>21GY}jPVOzdF'
            'L*X275c(FxeLzegs3U5k%s?%OK#niR@T8g+w<$3V3BnDtd@PJ(Ruw?BIQ`-E+q3BC-P?n+lh?1JACBL>Jvlu)IgDN&`~_C)5ZC4R'
            '`-6cd4<|>_n}f5n<F~KiODV^|H&(#zf^zt`cZcV{-0oi7eYrpS^6UG5hgFD9PG2MMu<iHOrL7GOya~Ra1mC~;7WPy1-7m=(;ZgYg'
            '+4t%tkyLv7L#W>|`1tKh^A?y>0c(kG;mt2&{Q7-(8h-!p!vSChfNb}-`wv<>1mOTSS_&HsS(e1T2f)Pz40to*9>0mRMyfRof8#L>'
            'fGi1?4?s7XoAD3}l>v1firw#EeC31*DJqO0(Shu_1JGKemmu9GJ17zEEYqrj!SCRfd36k%k;$l?CABw3;gA-N+CeU?n;D99q2>!D'
            'L;}_37Kfh!_{XRxfQEolwNesz55eG3Jy>g5pJaFvaE%gzPg?GUrbt)@l6aZs0JVh4Gf2~@p%?)lt81NN{97+$^yBf*(VyNS`FnHl'
            '_Tc5FtrmQrhQG`L&=~$%7uPcXXFQ+Fs*3o5^}ui%od4}VeyM&L?p%BkB1t4H=~1u()|5b-iJ-*^B8m7X=yJdUr}U?TI;S-Vq7?#W'
            '7i#QQSMdtq<>)PyCY%DOE3r1zZWtwkbV2P@4Qp?q0W=M1&w?5laRnlfudRV5c?q3q&5T!4CgYAy_CsB-7&}xutecVgzfm(A3D(S%'
            '23LvqF!1TO<MH^r_lQ!&OqWenTiqK<g9vhsh*8xE==x@7AY1{2067A2NH1Y)Xh_quQxN}*fq0-A5~xDJ)Sf(f0s`8>;o0#K{2YdE'
            '+nX?ETs5;Gi<h$`7R!lP4DXiX%d%)z!PD@5+G=qm?C{iXCW}c6V?gx|;D17p9%`}#wq!KfX<Lg`i_c5l)xi;%uBwgD<dtsIncrx}'
            'xF|50@Ut)kMnIJK>XP-p&q45A15DuV-q5Vdp&kOt2<;7`#UV5b(wnh(8>ba4;!n6%ym!=QRn6m4E#j<Ni$Gh7K*B-u0?0S>>&-k!'
            'EYec#GNcUEb-IF1LC6OJ4MNWqF-h2+bGE&qRVJ;`7>LLe8XSPlL*h$&1C=;#XV}5;=hyGf-u>YiL`x8BflwjZh;?!O+yfi*cUh*{'
            'GG4}6I#yeYaI^)ivC%TLm4TWWxR^<Af}t`|lA!t({HrVsviuRYh1u0}pc^tWtxo?4P?efU&$Ad9=3*UTaiWR=P{q!k`vVZJUAb7`'
            'a8#9a&fKq9^enigh-A`?*-jP{f}%Puv<{EG08bpD0Wl$~D&TU?R^wd|xN4wy@nv41bX{3;S2`=w3q+bqr6FjBGcdJaeL=ASoJD~i'
            '=RC5dLfbg853&vb3A~i>t0oAIku#(Ep)Ci*y}W8buq%MiRH#8466_AXQy5KIV-%n(>IUt0gcKlYCNdZ{^<ww;!!R7fSb`zV5<0)T'
            'cVFOe&hPFo!t?zLe$A~To(%mlG6xjLnh|yfhUQuCB>+wXWGcSi-|w3Xi)m>NlsHWy#rTE~HS=_PGay5NDh+Z+f_VL4YhQI+gAxkH'
            'psfT$(tL5+Ks?xrgbMMCbeY!fTeO+@FT-xr?qI3iNb1tthQZR*tD-)%A}!kWigK1FK-RW(hr1TYR3vFv5sZ|829?vk0qB~xi-!RD'
            '_2y;4ZWo{$Cz~1|>dQaBn~G--VM--SAWckvxVg=UX%Mn!2M33Sc+f^3iL-PmflBhWc7SPSA?w?sypGV43eflUj3&R%i`!g}3slqB'
            'pqU<$dU<wydKMl3<;{zOSEO$`hF_(F5SC6vKY!p;lT(tPtDQPVh9Pd`I<}`$vV`e@-U))sM?e+;5U>xKq+@YfWa9t+pa1>802VnO'
            'R>3L)wBxHfE>m%hXFX6r)=&jFg>5jMLo++C^dlq6j;Wca;^1(s&0iaK9n1%n6w}+KoTFU)k;k5{YP9_%FwnFbi~k~`3_?oLBmxD5'
            '0-k`gKns=l514&<lfoGAmDOx3(R_rdZkp8C8ppSwK7zjIR1zSvDvQ5EchK5(nrCRLBGAH^1ujdPrR66L{zW<~W8h`*bEl`v9sF=i'
            '!!)CB(Fqvqe>Hj5T*FMEUR)6bozeU#0j2n3(v;1z;nn<vL6YeXgs6DEfadd9%;IYi%$hPz;_DP-hf7(4m<0uZ*GN&vSAcs-T83uK'
            'cE`hTPe2vQVTrC`jWPgg*5n}CAa?#pZ%&SOpcfe8yF*ZE1RinC;wpm@p!6nj&Ikj7Sdz&dD3%nLWh2({N04lE5s8+iBC`W%ZB|6a'
            '%UCoS!i#!`3B{LL42UE?!V5I9!4}#(dUd)7i~a$`jiPMw#sD@PixXh5CA2JbO&qpGxFp3x-D_B108uVG`$ID~1(IvKBLbz7faU?}'
            'J`5z(u(?djBxUFT8OC$#ycfkSjW(^9bfAnd62ThrcaO=0nwVWIkEA+V0Z|gRRJ=eC<6r@g#4Lq%hwAf24HW<cnrHJIzr-JLE&5>K'
            '(?{s!5Ar&b`BJU{F|f@mIuTY)#h@9dglnLqF4K=#GOa=`1d4O~cZHQ>KD;9N6ebl}i7-NmFwiISEb95@1XM7qI!g0ZQ(N^Bod=98'
            '`$m=s^$HK#LG!!aofl}2cg@gjQ6~)nxmc;}S7^(xnv9hTY;!DLk#25uc=sF|WZ9U`ofV3ZDhf-EWncy?M}6mJ4F}-`WXW=3Px<UO'
            'oFkAtnkAI5Q+CSu$0rjKP<uY7N%z}TTt#ffts-rThf<_CIK|kmqb2!nGkA&*_)Z6I?#KIw6)b|qdAs!s6r^3w+O6Pphov84*5_VR'
            '7j3b?S!PdKFF^_~6S9#3rM&?ZWGHq}(peEbLm-`H<k}6jh8DUy34(-sIUlM{lHtXqrC4l+&<3gO*TKj(b|JnHi=hbacx<Fq>`?zZ'
            ';eFWds|`bU1PZNbu*G$E6@9e5AXqav)SHbp(!?w7zV6iN#c-?*l2I1uCfX&at?&V4X-lnm`UUUo@)@;vx=HhX%s$W$|I!wqHPkJQ'
            'wqSx!=`XsY)TX;c7eQ-ej{%s(^bX?tko*j0GR6f#JCZkVL+cYLECF!^>GglIJc2A^9_K}#;!&n9!dUTH4ph=Avgpc(D3*C4N=`iM'
            '58`O|lx$3Ae_VTvW|dVChT6^pJ@DMNw(ix3zj0YtXdw%R|7lG7+sGeqJ%<HvCV`ty06c#EmOEp9J@-~Jh{X=cmpj5=NDx?d)MD<0'
            'qAmu!`Pxeg(pgMiYv?EFO91LT%{g8M@l73Mu`!EA`5aGUl{_lmYl{6It(U|uNq%@w8{j>;wUHCn?bVu1{89}J4V+}0{;(qV67&#L'
            '%!W*q*b170(!7yuujG?|J!W1&$GREM)VUx^8@E>liQj-8M&w{4`P43zUh;v*=p2DWOB<#J9f>OzrKN*D!9sf$bp(Vy`fBP_&5SL)'
            'rSjk~UtIr>uF#=m(YCnZ@$s~G%Y_{_y8;OKxINYohrNMAMj^kR4%1|$yhBIIVs`|nm1fll`5pNNpj9qSrrgt5evtE~mVu>#pvZT&'
            '>E|C2?8qOe!-ob|$@VyyKUz}17#g*QlpC$rh=A3<YpWBo;-@L?j5LOHjYd~-XHe_EmG>Kj3hddj;-Z5H%VR4xb<!I&fL?UuD*Cj2'
            'CT)%O*NzJsqaM%qF9z-YtY`<N&dNr1O0=XJZ0UU1l4U%(NEKozsHJyJ!(1%vRPPmX&<Vyj>zSr~Y;t=jCHfR6Z&X9xH`R;Nzx{;>'
            'S^2|vQ54$f47x)cNZ!D9R0HNad+&5xaEVB(GJSIEZ58XfXtYiC>99iYOjM&&AG57CbK7bn5b)<f$uB;g>DI7D@|+q#&IT>m-xpnO'
            '9Mfk!{_%KnWw@jWG?qs^pNi=*tF!tYaMdH3H0bXQ$Wx*Bk*+4}9uNVY>WN~`K6Z27Ft@-=Lvip&5Pw{$4q5AM0sMKfkmQiZBhyYx'
            'ISL8LQI)G5HO2HOWY-Ke0zqE;5*BkQ%`DLcG+tHo5g`cem+P@O&{6<9yXW+cr4%Je)WwPo1jM>4KDU}lat)~g^;flb1kErQDR80l'
            'pv)bXg&XGlVnqK}fVJvC5q(B9I-s$^!HrMXVjWKZFxq~C^2*a0$1C)L3$#q_me6v}z&kVPp^7lQHc<MhyFst0SVVC~4${$P65Kn8'
            'c2DCtNrJv%dAbqx&^KAD@p#Mp8FZhM)7QJ`8E1X|KtQP259v2oWUy(e{kV+}bzr@@mDaHUZcr;4WDRpznW751W9DOGwS#xwoPGWB'
            '=~t&9+SBo}a>E-_Lz?;!c}o|(fw?w1)UeqHb%Z9T(b91=Pz>c|Lv9IeF93J^oiilth~AO@g`P5-d=Zs~=C)=OHMct|wz(e^@NoN6'
            'G4?z`BCSXOx-_C3gAgWJu!0Q$?*PwX()(P9TpZh_*p=KZro0~81=VY>s9J+|QT5s@s^&ujeIkDU&DZ<;;tMsTC*rGb=uvn5M0|sf'
            'svWD%Zt~I(+{gi=;kHrDIG=)2`Qm4!-UidYor<lVswpd6xy17(Jo!QQS@O^|F&adQmFq0B3XCE0-GV`TCI4ToyICJ#?WN1UzkB2e'
            'EI_>~aLH@*2G#}5DN|BN^0!s;$_hixP(v-z!FUG{pPJl(#fdMiB5O8@nLU(LwjOs6xG=>c7M6YZsn^%7^a!Ya2HJOo-g4ys0>G||'
            'Wjd!2ELn}j+1ZQT_%<#<p#-X2HmiCU<FIzCcp-mt8xPs~#32sEpx}Y1D9R5V3wKQz6$XW|lOGS8bbS~r`7m-cRKlNjQDRl9Z9Kmw'
            '>Ufpbdr0B-O1VIf@KE{HRA^1WkPY}<Hn8qVk?$a{-mQ=`aU1~0YvmfLF+Sk-3eW985GVIVY7+ag;yH^;mj%Ac<d_v8J`+h~CUFBJ'
            'R4eqoLlnhc0k@uOn5$W$42AKGupA7PwVD_YnPSy&UhGo<D}-L~(20(x^A>l)e>lhv<tmnpQi@@)@*M9CvVGPV1zA+uAu{wVDaa8p'
            'CRMK+&k?Cm6k#|~6b0yLZT+mu3B`pux;uU!o6kTZ@q1uigc}Num)GzwQc{3VHANX^1bZ8GwBni`TQ#*JbvFb=z@wx65ucez{h$`n'
            'Jga4h)XdTRvVZT)5aZp{^fZbplOB)K_yHoJ#aY=&|C@N1r~hbV4%ozDlpOhhva#`r4RotcgCQ_DfD(o7I<3vi^{Or|%XoDKE1jg5'
            'c%)_|eRQ#&*TRVpVSASjlC3cC*lEq60qYe{f4^^3HqtvVS{8+L7C5kvXZyRLHn@#C%~9M3>l?roVig;lComTb|0$X(Ta)IHVmNT&'
            'pdn4$X$kQZBzpKa$M9FCr5r7(On5sLS4wQesg6b_&0_4dhA~3KDedg`YM=mf+E*~PMfw2=`Zeme$O@^ct7aMJyC^23aMT)H4x_G~'
            'D`#5mnoO~=&8naSZ45b53vb7w7Hq0RH^;CdUq1ULSPbtp4&49pVSoS5;KBWH42qub^>!sca6^uL3h4oPjFRg*`sS9y^S>SJ{x#nH'
            'xWD^9qVetp`cy?j7st*g-|Tme{aX8UPq>4N?sP}+lLv7>MsGcbBB1%;5xWH_PMvY&-oz7y9h%{UO;Ah(ulOonQ*aH+d&rn|<Q3Az'
            'fP%SA%UVdm|1Q%TnTI5yP|PV)HdFLDBhD>)ww8`$Ft$aehP#toC}Rxg;ME6d14wDLBLeHgsVxHJ(^Ibjl*$0KtIbSKTWo{Z{_t2A'
            'J?Kl=u$kkK%aU)60yeD^)22T9(N;;Z;21<k#NMFo+iB?Lxu%pC;)|(x+GpkSV9;^nWV1YwxBxpuVmm}U01SrhJJmRh%xuy1gp)bD'
            '?z$eJ!J?%Y6AnZ%?jjwlJVq-5=v!=W4RkZ0lIp^)M*6V3rwfY?W9Wg(uZJC~We%RwD)}-cov`TeR<uT%34?4J=%DOzQ=^P;Arp={'
            '@KdP^K?s<K3dMN8?WAZM|IXQsopo{NXqjivG&fDvZGgCPtFEeJ7mi}PFDy-|9!pcOBJp7#lApL|97McxkZ2+{FTmL1Ur3&`Faatb'
            '-`m}?Tt#Xmv>kD?M5HZ!pzSi|-{fX6hQfCR8Dm#z5`_1T`PObS-6HgO`^b$(J1c3HE3;2Vg5X+V!hyJ&r|DE3W0C)1J^j-5Y~@3z'
            'Yy3Av&Zq>?BIf|wH?L1m{z6pw&(KnFTYU!Qh8+5g@c(&K7B}A4XfE_znGj#4K7gh{5mako@oSYeN>~wy`!*ScqW>RdS?C~SmG2@i'
            'NS`I&t6NyYWt_;6f}QbWvz(z+hI+y==#DQB&W=y1rCD6DV3ic}1|!*#S%PAvBAGD_NL&*qrHl`?li~t<QHsQKYOBwwa;MS`lq;NN'
            '%3RrtYz8f>Dd&JYxd}0%ZfXYKc(d-%RTVJ125i7<5XXt3UZd(_1E=r%GzNn3gOyD0Y>AAppFY~sLq+Vk&UMiXO;zYM9i7;f-w$Cc'
            'KkeyJjr7yH)zj<iTYuX!0E4X&aR`Pxc8I-q1i@w*I5WP!5KkOHQcu?-@ieqxwxvVcXI(^Om@|D@zXqNhWp&W2-GZeT**?;?GiQ)@'
            '9&JNsEj(fmI$UWuYpi$g*})GlPhK6JynXIvBZ${!D$rdu-laIDDkWdziu`ba!gd=)3AmMc0NPI}!9Yfp(Q7nmswQi+|MK1?XddvD'
            'Jeb*mxu@lS(WlvOF&z0&BKStLb)*N4oJy~eTEJEncGX52LU|*wjILyHqFu?N8&}6H3{`{zGSsEM+6dWhL1Bl=1VY0$)h*sUR7odB'
            'q0p8n!!Q=8y){3=_HmSQ8Kd{|Wuzwo?JnaFUEJFrZSN1{H+M!`mQQc@2i8y*aVp<`4yQ5_jz-MCcA3g24)ZGU*r)s)rz}b=Bl1nZ'
            'Cp;K6mWju<x5<s&_@WKU=}0>=M~he)&K?NlHqyciOm+t9-v(Pxn)4GmQjzVF^ht_a)Y#DK7o_?iCfQ*(E-$GZ9pYG9NhcuDZ6-t`'
            '8acV3S#qRUQ{_rAYJ@6v7<naf$~jAZvF^A%GL#F*6qM!^GDAr|lmLWtGs>(|=coLm#kvf196NzIA5t}Yl_r|OxWj67%N-8FZMm=^'
            '8%hh<`aGIVO{PH0Q>fK9V5Z+KPqf-JQQdy~bd+9xhRyqq%n`dFpV7M6RE3{ozrylHJ#IFg^Y%yVWqZ=RY4>^-167&<y>VY}-Nw%9'
            'yR>KEStitotqWkKWfwc>(WW8UiEv>3<Z)hZ9cQF8ncChQ2B4#%2iEn-^&O)e8@>Kx{k~%xj=Z$L*Lizj7w28?s0-tJh3dXR6oVi-'
            'HCy#-@Wes0M<5Dk#zNEiMDWgh6yiMC=9VNM#!-oZY9{NrFp-CgI`O`goN`1pVcx<sn|Ao9DwBC&nhM3Y?#vvE2-Ne8FoqD31K^zl'
            'y!&W%hvo0RNQDPCL(hij$!FzJzpb0XZVa)irxqNYZQ}K$_exJtckkvgm6NFhf)N7oEiu7r`fPth@H@lar>lDOkhtb;I^XBdEE2Gh'
            '8l7X<!Lhq-sNf0OA5qsLzsfY<yk|pil?~4L<#Xusjh@7?$6jo}>rH0{+RnA-Dz7JTKCyVF_i{wX1+beuSKd}Q5mOCQ#`QJ~&ekib'
            'qL^*EiLjprahEw7;WG3(R3ihp5MWt89tk&nA$sX-h>!!N>qepj0`6IiU~`Yv)DqC?>c$L*WpT?#KkQswi7Sd@6qpIJ+EW=ID-qa}'
            '<na^oxKzi#BA(H?1|Dg!!wS_8$)tt}SlRtud-E((nI&W!#00qd#8wMi${BXEgDm={+_6n%yDzF!SYfMIyw5$h)qd0Mpy`%$vtlBt'
            'QHda^ivFf>*YmWPZgs+fFP@m?`1qT2pc7|+@EE$mhsv2>r$ctVi=AR_646#+_mf-2OElW5W4;~KUL7yoz^kFYoFu*FduO3Oo^>YU'
            'B$AvsJKJ*_)|moBx1y8!bVEffVJf02S#e6WW6+C;_5!B>M6Jh}jF(DvLkFtR=05h4l=68e&f>LGvQ9jF9QBmMM6DAK6<wUc)U9ol'
            ';dHJ*wB6-#a=W${M}2B7Xea$^2ifTR3;mh)SYL^QCP@oh%o|iYo9x8jkE#H)J)L4U)J6{d5$e3Nt$NNEZy)$+X(x(msExoMPx+f$'
            '?V;SYlgInaxf6HO-A+2}L&w)x;guOhlrZ)CHG*rOAh=o3<kE!f9VvD6C=rz0+YRUYqh$9Em*J_&pxB|kKQQM}EtVKJ{%l{I904a='
            '0Xr(l`7h3_LX0_+$RBXN7MBeGJD<yN#OVTcbQY}&>2@jy3>h=f=L_w9#;x0Nx<a=>O2Cu>rwowt>>(A=d{t*|)oxk#486U8?m_Z{'
            '$9l{zid(%Z1T`z4<M;?R5ycC#<9U851CSPdk$eOIb7YvAa+$S~l+(<_ZSo7Kc+o4c52NAu??svi6~ZwIs8$&MEAqOVX4Vtbd-1JT'
            'OrK3@9326joA$L>9jIlOxG!d_QTpw0q~<#dA1`k|N_cL0&%<(S*Y%B(Wml#QeG|^-Is5!ku5XJn+2vzDn=vi0H&U=onPgi6BL2EE'
            'iFz@CGi<=m63rV~XxWU>l$|=De;vx!rwW_eiQQlreLzP<?LZG~klY%vVdQM*7jCl05~I+DLT`||40NS%T&=PcnhU+aPyCO!6&4!L'
            '0lteOO9D?xqIOXd$C6|n;YbW0f9yOo7E!+~7ZRCMB!%%oP<Uf2YjujHT1~DZpioXIv-Y2&<<@EaU7v5I+HsAIW*kpDqYjhJzBUL='
            '6*Q?G1BQP^PqQ!f_b2=N!_b2nw>GXnrum}i`cQivs<Ou2fa**=;@qBssuEYaRM0K{>lMprndW%DL(|pxOH9_mV1Dj`yK6R6NSzh8'
            'BM?C^kpVyls`ZfU4f;X1M#a9JVx4YtPsjVF%y(j+QvvUf`4l`t?Bv!iTnszsF2pXjiGD%>)mM*LQ{_>%wN;i<KCH6lIcl{f>R6k%'
            '7I&;r&=PqDdaF8yI?D9#JUg1vqV~R^;ek&ml<m&;+?9yb{4YuRD-ziw8R;@A)wvO?Hx?z8gJHsDFi#7#&DRYK9ySGz27MF|eIL<F'
            'oi5j*gy+&bttJDWo0x~AEI<xZ9_Vjk%B-vc8d74Vn0YcYmW2uBfyIkl>N4gAO-BAWC+ot%p3&186KC94vHEk)*2E`7bbyOV*oy`*'
            '2v@{C(6pZ83YTedHQn5lhW>$!ql@7Y=Kry&mULK&8mvN|JNm1gr|9^Xf`&c>Pe-)h(3Xd~qJu)AB3df7vKRl4vI?dnGNu#z=^vDY'
            's9d@l9}pQVO-tjJrBT5b`rD`cA0UQ9ffmj?MXfz@IN^yo9}yo<2@t3_S>$Zx9&&eqOY&jbXsgGMhwx3Mlrd#(_09(AX+@PWN+mbe'
            't~qE+w8l@Z;v8aCzmFQqyI7-gVgVj^QsbW(y1;XNPP)=|`_<!*Z@6JC_C9&S^LqGz#!y&a(f9;0mi><xsVqH<8{IGBhL!7{u#~Cd'
            '1cRf=p_OmamK%}rG-R&Rc5~G&`<&idzm;U9UxdBpWzBf-9ugx`>Wfem|Ix=cPW-kFN|fX-K}FaUe9j)DwisxCleXmKlX(hBi})TW'
            '3aR6$S(k$Qy7XHRu_+Nc8yQrdWgxeC#UU*tbyrB0UgnthF`#^4xp?v7C1yuTK}KbAeu@(pw-Wd>aye9Zg}%i`JbU&#I>RR&`^){`'
            ';a<_#*@&n7kwc#pFM@{>mr%s>vZyfjfpm#ICd=sOf}u!H_kTAQFO_{+-J@akGj0k7<%B3Txv>u^A3(8)XC(z5Cq5GZXroUv{sDyE'
            'tb&IR3v?@(8QrJn?GJWC#2u2@z<P}<{3rBMp*Ty+0anz2+A0s&ltS%=fOs%@IDERK8MBuTHSZkMXgZIfvprHio+bDuV6;X#(zWX+'
            'PV8+AjE#W1Ya8F>8BY7!4uLlDQ^#GLE(}7ltw-UQg6$CKT*EEp*nMp&bME03^i8;--><o-NX=}2q{c(Rv@ioRb*sE_o-VrursU12'
            'a^kyEZ9sz_nNfxIE)B=K#PtmKu9AX=JALh`;Su4(ABax;%azM*?1wV{w22gf#(}sxln8yQ%{uQ?D7%jehbGgqx=#yj%XCKoMp|oK'
            ')^X{k`?PvnFjsADX#xk?tjLJk^2hYArDFN0?XK5%-Pc6PWM(M-U10m+21h|PVkKC$dum@8Hi3~hz<A7?k@za?)bZ}h(pCJ|J@wjm'
            'j_Hc6_apI*>!Ifm1Vz?N-ULv{m=TzBayD>I^;sENLtwuHR_n@6&OMt&#NBuH+{n|@9S561uPY68eQt|ULO$;!XV!CkUR+;fbNd=v'
            'IFRBg)%qOoEgWaX<<k{T_`D7Mi?DB&%1On6Y`a-%tGyx@5445afHxmSds=g3ofrc?)bYvmsjcsztg6j(>cjWmv2>dy$uCdjhmkHT'
            'bI{gsI)SM*>E#Iuoc;#z@+gt)Fcn{lCr?DMYwPZb;LESY6Y<|2ZsXs9*T49^ST<sk-GQ$ojMLR}y4?pLw%3NTGZ|GoXpQp=uX!BE'
            'm(~%T<SV)s4JPYb#`nk1p215GZO>^TF1&JIK*=w$x&mkC<5O&=g{YevdddyqNPBGAn`v(AYC}V@dIk?2z;qv`UVbb6W8c`0e|e&A'
            'L?hd#TD><c+RC()o9eo^73wPV<d?ly#MF>k<bHu`{hg{aj1rXs-nnku7bDS2p+=&IWuvGvbdM=Zl#DCM;p~pK#}@7U`hoq8Ar})D'
            '4bIzFGLnJ~KU_6R*VgH@;Ipel8OOp0)u_5yMLDr%+9xBO9wqWFrL0Ve4Qdf3YiThqHeRK<X#1&ojLXVJ8(9~MY#Z5ETOg;A;GHHk'
            'B6Q4E1lQwScm3E&hx1Ok{!-e-qG#wGcM&V3plot}(MD}0M9~3Auh2mAIM+>Fc#3tW6gPLFsfaT`Uup~aL`7{FJj?P`Ee0Yior0of'
            'iW6VLMJ!Xi5Klu}L-p3sFaNDJ*O5rr!L~5mvdFYwY0R9I$C;?Rp+lulc-Yt)<;9@yT<{_MPz|`bv9=&etA<K!xIt$2vw(VV8$B$E'
            'D{0@yg&rZljf3qgWhbOrk@ud3tE(P90<2+Yterb*H7Q3_=qzKa?BlxZvX0IrxQ2gVBphK8iVTRo>Zz}h){VLDz<X=mTfc&BOsIV_'
            'ectI!bEVEyS_tHcRbCcI)j&VOK7xZO`hb7PAv93-b>baic(IMmVZ~p^;?yZS-OZ^=GYBw7OSuyKX2(=r&z!6%u4hmx!jxWy1%AEZ'
            'xM?be?@osgpEFHS4jVdWVbCn+rzFGWcs#z?a8Ga8IToYcLBg*d+LzC4A@`)F&EIk1e7^q7`Jx4fKB++C&_?OtVUz0SBRnYb&2Mt<'
            'w$<p{bqhXHs8z9yUu(<hu9~G1U{s1KWhJy}wV8?1eZ_Q<uyw{LD76)aNUj+uh+iXheK1*qIf;)x=emm7-j&5OnPuvZO|{x+iM`B<'
            '8J+d%Junt0It`}^xj;I(Z1BFo*7+QH8+Ed5Y<G^*#;9%1AK}2WJP}d{be9Nv7vO15pgzI~HG=D7IeRISs(#K!Ya}ZC3op#!kB>cQ'
            'yfBw(d5!-D=Z}7a`XI8s>8xoJ&m^};r<x;6w-F7X-^#G=A3f?OcPV*5BC|RUkkm~b>CB^nFdPp<tb>U14FQt%R|OBeu>n_-&|}2F'
            'V+QW~_Vldr1;-t@fVk*A^sx$ux);$%u_}l!F>y4y<B{zQM>yG>_l6JX`n@9D|9YK4jnAFa=V<fMuN$z`A@8Hksiez%Mm_0T+RSI<'
            '9T<20>3lnc2ft227$btA{svI&WT$=50Pg5xgUybwq@uNyBQZ3VVEw{3FYn)XS`7OKW)&rL(dv3{IR#apT#|8kQ&HcP<-u)hqHT-9'
            '{ThhAIf>lBTNxB+GTKpLqaQ-&B7v>9nD}+Ib;UirrZeyM606V+<y<`NbDthOZtIw?dNzO>BulC8fpo>EF^UW*&OP0D#;k@k>R>9e'
            '>D!Sj%Y=T5{G%ffYYld*RLliGQ7+a(hkDm*@6iprvfSta>CO0}xnho%0SBGSVuzUc4|G8cmrhIfDo!pEnJZuY9ljXe8JFoQycaPr'
            '6s61ENI~CpullQVV?ok9Z{^frO<z>kMcDvF_;?B2JWJ9NmjZAb?=Ay6xRe#<?kIt)r^A-bm2Rf6c!TpM=f0M?%FY}bHhFL(N(Z%Z'
            '?n6pC0xCIOKBuN27V$O6UyC%Wkw%vo2ZQ%ZC==Np&-Xgc@Op-!9iw7y-T?iq;C77aQnk$ktDB_}Z1f<@%O=C?C}%MyNX6SOiY#gL'
            '2;6HjekCyYzDp3SGY~ZBPMEni3_3Sb3^zn?wc<ra?Jue)Q$RhtPFU#*FBFwgS&&5r)3`HZuV4)GxO6?a7E&fvgsDIxU9absI4e+d'
            'zv{t@GQm8!^(Q78^BmLo=69U>UBBUwPhZ8*6VudLt1r5B%=kuvK!b)OPl#bxfS@8Bslsp3uxDZm{DfS(SEez?!9Cyl>@Th?$Vpke'
            'SI7ko@l|a*)VO2uvXs#&Y{*Z$|Ba+m&tfxeZJ%jdb~0q$IB+@4h~50u)cXAVh!uAn+f@D<6lilwrGwNIpTbQ=wR{I<^-rfLI@7D-'
            'ygHs`aIjVQLAJ*tAnfvv45+QieKx2e=U8o7L0hplgRW&=F}+SaWMvKdRr@w2{Kh~0fG-vKVU|>>*8fUc3(g;+!j&J+ZDAv>IV}^n'
            '@TmL+TU!2?g37wLsMvBBPJ}gqj`HHv#fG`Kks!<h$ihg9pz<<&dwlx(#ZSjatxM`KwY=gFP}g3mqFZiVWu`2Tw8kOkRHk1#NCA`B'
            'R1ICN{9MH5va>4?7EIcCHWDbqRG~7~vZFMJ7!TcCW@6L)*bAF}z&T-eE}!hZwx}D)?_|O7LoK}aH|1yfL>1N_X3LPgF~`F;Epyvc'
            'Oiabd++9UNgEn;OivEpOJs{n&<J}3Jd>5`Jvmrsr;PF6eCzj~J`%i(D=M}I)je4~9)F0iq`MOHlHOAxj5w}E{mD+warn2UDJviG@'
            'n48>h+qWuNi$VJ;CAvn*W=m*?nqf)>@6=PNXfzNPOMEX$?Euj7hVG9Q>%DG=Yza51>#*0<o>Q0uRrvH7;tXHYJ<TBh*D2=1;+qA^'
            '>T^U5Qq$IYtR<spUfocJT)tIc)Z?Mh%A1LIOJuirxeDKP0W=#>dWLsf0N#Thb9nlb2(Dy|<~>|Xl|($mkeJ-2+<4DD(Os5!JJlK^'
            '4uIm{<`sH9FDdyWTAJCgewEfLbR<vWYOJg@$5;_>DIwFby;EdKmsxOG+$bw`d0w)>__t(?RsM)#y;0La+Gv(}HQkhRBy?1BGHrOR'
            'Y^^=exsu21oc`F$7m~%eF3civ;@}S)p`kZ%#7oPI&{@y+el|00hlx7DjE{#N(<nWA^_M_J^l~$xbkWZU7Y0km*7U0|{?Cl+HWIiU'
            'cc3?c;7u@6cebN<KOr~Wi(>Gzf64SvM&>Z3i$)u!rp2Ak<m@TpBc_M8EPF5BwmEK{jkS6?^U)Uz_s$3-lf2>gBpv@1$kIuj|Co&4'
            'Hk&tJAFwr(H&0~Nq#62&=!AcuiRS4->NHtw8BwV+Ju``Yr%V_+pQdBJZ~`+`nnb(ytfeUh#axZp_fg%D*z?ITExQ#E2O`S2U@|?C'
            '<YWU@Yz8R2;D_c-chtL<qJo$mWi~hE&4%I^TMmiis*-ru2phkE)1RIloV`1ZemZ#Z?)Vf1)IX$Is!k0xr0@90{r}%qE%(v2AoODi'
            '*^zp`cy;@5h^qE|aOc>AU{J5`b=!zw3Oi43t{a1i%96(qUNRjYGZ8keR~{7m@RjrEY(!+0pju%rWd~Nj-8fVS+tF@#O?x-EQAxdl'
            '*3&_&Q)--YQoZ}O8Dt4XIB1Dy8*JcMcB8m4Pv<v%;Z1ygXx&0dy6G!;;Rf8BPaUn}j#1X$q76c{dbl6-mc!$C8$M-f;5m+c{W%gV'
            'BWH21a}795P~D^;_QYY4#j{;x8oNg&X5M#S!HV(Q&yw3i8&Xzf@prU+5n=>*nK%xBO_g-%TErQMqXdVd-|JdsQPhfqK4glbx{+T='
            'dHU=rudY2Rv-fgMQ{FrH>Fv8W2jb_09}iAX_I_r?_jO$?DR%E>f7tcv(YFloefF5zh~Qt$u+wUy{c0jQGSaFva$I`jxGGTr4(>A='
            'IKJXpERj$Ez6S`2a4cSO2#-qkPNS!`=n^Wr$w&j0&U4xjrQl1tY6GQ+vAS~(_zy?$Q&SpY=mxrVg<->V1qlft#GVLi_%_AdYe0in'
            'DfU#JurOO2smh<%lv%{Uxn6$b;2&_Npi6<d?3f!+!oHeyCFm`;Gid41=EImq6|}H~uX~5pDJ6Z^(s&o;?4-PlaUXR&zbetxzZkyz'
            '`QYuz#9LrN?FyNsc;J|%BDjO<_u+G~1|d1yyOi^Eo*Li-wxPvJT_UhX^-&$J<CT=|K;48=6#GsW>~*(~bDn9oY;RQqIei1Tcev-4'
            '^qV&Va4UBL*1&^|*b^e0t))H!CDU(RnWVk#dP?JOx7e`5a{YSr?!j7}t?bnHzInOs++{(|>pr_F7>LQrH3g!2fEe{mcjV4t)Hk%y'
            'zwoF!hqf;stA3aqI#5wD^##z54aOLwbz#Sf$(#ER+ZWApCJO($bmJPSub{+u^FVub5$A6uf>Pa&p?fdy3EiVe);u~qLjNhgxJ}b2'
            'gmS_;_Ps&3t;^e@MeLaM-a>cqVebfsUO1($xak5mlZsxI8?-YJ{XZ!unkx'
        ),
    },
    'viral_prediction': {
        'sha256': '18870ba7aca2895774557ef7e2c3542fb89fd10476f857a2b858de31367800cd',
        'encoding': 'zlib+b85',
        'data': (
            'c-rM!+in|25`E`a^v;Wh2}#jj_+jal0VS3b0kRZG@*0L944RxK+2P!FdPXv><-hN#zRX>=vx{IKmWU!vPjyvw-A+~0p66YQQt@RZ'
            'm@0T7RUubAVh=Ls(KO3NC|89{6Sm|^L^2U9N;h)F)RyBNTkc2mSAixDYT}v9_Q{g*BxLh>uusdPT#A5~MH+K@AH-Z0A~$5S(daTQ'
            '#4=5HZ2pSzFqA~YqiE04^;+Z#j98kPMRsxVe)^!Ml{%|5&Uh{rRP=WuD@KWBHjPp`h?2=lFj+9E`H0V%$Ym^&0`DMenkEy5Ev(oI'
            'mgQ+!n&W1?*p6S1XqB&u62z6nSQOhdRQh<^GzxJ%CLVZH>a&3nFIXs6G8CFAB&K;G!coskzmw9Dd=P9aa+p75%e2@sXiaJtLK!Qw'
            '6hdX3MMA5sx58%PLFBB=qLf3IG-p}4jHKEMQfyyrWwOD+G*QfVw2G(VG0k^cz-;7s-e{etF$;n<9F&V7ASuul3}6v0NR13l$O}=x'
            'Y>vV`PZ<6G3p5;!?5m`VvpwUAC7D~wU@9!&zbqV$Zf;L+KimZWIKB9AcEe8C(}>~YH9g}^m={9#D9vag$RfTHWNVSrt6V%t@fhUd'
            'pQVI3y@}z#GdS>po|z{TRj$`^C22df<+QlwGBU4gPlF2&_h4}tjozNUJN<BR8@xZi41PVoxxKpn6kK0@Bqd+YtGB;iUA(>ea2ve4'
            'KK=Rj{OS^K=d+jX&6dP&epS7^zPh+Lzx*Zmcz*fz>LXsgn2$!IP^_!)3RYW~UqnHSIN^z3<Xk2}xL&|^v$r_NyPT8V9hnTrZ2E?+'
            '5eEhP7rTTj7TO$+AWt6^1UWu;)gNBX!n<cDCk>YOnuA=on9M|+6??;+KDj{DBu27dbJea{rDalh_cMEl@tBzDG_!Pu855BbTIW%Z'
            'V^j2VT)Vc*(+Iye@aYPE{|?(t*rQCs^l`ytQou^Xe)=;kJX6SwfXaqYzUPuR40z#<|5w&0qp_2=*aA7@8zYVTPBMFXVh5cgk`3SJ'
            '8W!;I_Yw#Y{$EA6p$>=xsr(0`oOVZ&ig?>OcOI=)8`d(467J8<1}En$5^|vnt3L-)WV3`PW@p1uIqH0+t8a8THmt|a&vbabGJ^>q'
            'e>?*K`D1hX`D|{Z21qUd#f>n~g>(K_{eBIU1&E;_&gSe#K-OnJmWl6ob(s!w8nYkRs|mQh8c*1iel<h2W%wVgC`9K^r#ENF^0VOm'
            '+3l}aZyQ*&6=kkFp{cdr)bQ{Y&(-NkccU#uQ<EXhjqM`U+gP7!>e8gSdmxK6cIjOEBJwm)d@TYM^Gx`rbx{*WU59!ml`IiXk09WR'
            '=Q-ckJHuk1i4(oc7*(@s6YHe`(oiJm9SrGGIi6$q*RB|~Om=X!1F6~ahP@EezqvG}%%ayOu6ijt3?Go7<MSmv-jsGnobC}vRvYs?'
            'MpSFj?(qj8^W2p%+ue!4c|iqe9Bl9BZ{fEz-w#wVOA)Mdu6vDzwSJ8=-Ckl+9qub1B!&~Nv$_jj-3~<iotjQ#Iy8PaStJX^YR#F?'
            'Bi#rwrJGeW_N<9XV-k^?`gUw(o8Unc?98JWwVQj^%PRycmLDW%?@!N;O~Op^mwJEt+u2O>xz@f@?CHqW(4z@Ea;G#;XuNZ|Qb!B+'
            'WP0!*ftm3%o6Qb~gJER)V(?JVX9_plejPxM-a1Jcl$9Z?+i*pbfujBz^F}NSIz9Egnp-7vM(r}KC^8*D<TDIT6$*CoBat+sfP%+='
            'fe7ome~jYPC7rStCjQ;A!M1)slNe&|7L`ncT(wmOh%4H~{m8<OJ^>ZCgDJJBvkm)A+`~G7lY7paVGxN`Gwp1)*Jb1GeuAIsJ@F4F'
            '43SCvYGsL2PVw4g<`|MSof;L~Y9?(YAGda*IOxsrU|r9epH-R^GATt*o|<7=5J{+uLYt{Qm?O|CG-0!*wzWy~Ys>D*jd_iut~A-Q'
            'M8E1XN+D6LxQaO**Fr6tb@}OD;XmW3u>#z_fgTMFbVS`Xg6O`&PWSO}V&GN|n}t69o*>XMcvafh@7#)}X!Fo|1lg*nsP)Q(z0_TY'
            'Q5?=EFHiu~9aonTr}^1>aG|r%9IB=}G%@7Kb1?|&Ivl+^nz3F#=!0paVA=lVFnL+ZDC}L41UxS!bs}Gb5Br8j{D3ZJN|bt4<efLD'
            '+lsJ--h2;yP;^F-FlrdK6N4$1>$8~B7>U1A(w~HYe;fn9lWi_k8a)8=T+PuDbD|@vX*Rm`fa&5)zv=_*Pvmr^?|+7`eW*Z?P4qSZ'
            'WH>ESw0_}&Z8Vami3!K_ajI27&s2+N&TDy@7$N<8sELN=Os;mCLUZd}VHrrI?NR!O;}AN}9DrIpK5~~TM+`l)hQ#jO$&rRxv}h?<'
            '9ChwyYT#@{m$53m)j`A2U~mR0HHN)qo%8@0zj3Y8zxA<=DE<uUS77J&)p>K$%Yv~s;*dro#~1xPPc~FOpFF(Hsl1qQ@!(GTaB!_`'
            'Un&V`kN+AqYgC8yGEVGuW3_zO-gDocy=pM3HN3Ox-M5;C9v|vy#xto>z<3OB84t3eqkW4JMz}U3JIA8OGIxu*b?92AQS8`XPMDYO'
            'y!tea%?_T|-bSoqU;v_4vVFNjObNo2<Gnz?q2w1dhy(TVP+zl9Aax?-n<Dd-*s+#mD~<V2JtrU7-_f#mV(Y*Z*ydx;8m5}?EPl`>'
            'NUB5<m}H%}mKFBZPM~^}tYxT2;fO9vb&EH#(Pv{Y!7s1hH8z39dB$2|DB9c2pKPV!$xzzWy@g?6Y)&1E*We%XM6OYIOi47PCr3{`'
            '4Gy}*6loUmn3JLC!e4%=JRC|R5cd0qH~(Ap{T7T22EucWW<tr0Opx3>LT?!I;Ycw-ftJndh#CrFy7xcQR*0mLu8o9_jEC~*P_wWX'
            'JLryTTQt72IQi}ow?EWQdRSbSj8@Fm^6r{_R6RI!GoZ7_by)Y!{#zqR#cYg3>vaY#I}WC^Lr*0N$#&(c+`%^;&dxKhO@D_nle0~k'
            'mr+qhB}*f|l#$H6)^N~0P-jMHrnj9Y`$b?U_9qDYCE~?QyIqny-GATde!DXs83|zB+tZSL-`<2Q5P3CRG(AxD#P?sls_Z12D_9yL'
            '(P=cMg3rE{HnqCiV@%k5Y<|40+-$2Imios&^`Sd{bt|KuJj&P`M&UQ~4bR++5589NS-Mx2%%i)GfJg^V-7YcRZ>(B7`ZZFzuVQ!b'
            'Tiv$#Pc6S4##P@}Vs=AK5iAraNeJCg=IB7>YcvyzzCKO(Ub3Yibd>uvuN7$5gl7+6d#tbO?bSdS$$cmpRqL%>Em8ccGdPV4zD2-G'
            'R-)6?c<uH21DBwcQ14S(-j++jZnkM&pe2enX)cSHE2^hENv8)e>>{0>bg}WEWDLPrKzU@lG$|SS+LEn#4l1Gpl%R7BMFiXNt&A8{'
            'iDD6x^{tu4-&Sl{<{TNiD<f22i62(y$8mj0Pha%zgre`1p^-ZcAu&7s?VQzahx!sOeBD5S>hPxtkM;#!N&!pKF9?vIZ?BKZj0tIX'
            'yn&(fx=P7Cj&}O#@|@kAeZqY7%lXCI^J{mq*3etaGOk75^@3s2C9?ka7VUUw&=I7fv<5(G8vA=$)h4w)CJ5_?21OTLgJO%JjiQQ1'
            '@6OA{zJdO-p-W@+xjoe}pTXhwt%KKVVz9G6cMsB6!)1F*wcOY>Jg-u{9DLEVqb?yNq19#91W^<RH2R^A=7G)-H`%n9*h`Nm&t<r`'
            '7-(gQo129i`#T6x$6M9pHVxFS>DoB$rKa|Ew$go1{nVh@U{atjmGr=}sQyg@MyW^N{j-KyS*%)9jM;w}{ToA|miq'
        ),
    },
    'approval': {
        'sha256': '119794218b882b74fe3deb51c7ea08faa1cc706fe6145e48bc4a40116a202d9f',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;OTXWmSl78o}n1ehBtfH9O-Ad{dT{=~YL?_%RlGl=x^{P}P2o6bvAi%}N3a!$A-><u8fB^_f%4YZY!IprTo}RvZeVKth&$|ec'
            'G>O6>R=YGWPFFh7Sx`i2qSoa$NKS*@E=#}Q^(rXzu-E(Tr_bu{r^&7Q<>vC!)mV-Hb8>rk+dJ+2(>u4bw`E?axmJO?9e+6gcz!ol'
            'd7*czSO<k#V-cO?>iqrP_#+-^RpvUw<6H${Q3i3mSGm?nk6sxL^+E+%5iK!bs6GUF9wjR^%fl?%6;okaFV60=QqNSB6e*KkmPyEa'
            'qh))RQqJ#Ws@m!xkq4H%igNH8Y<F1Fvy?p$m1)6ND~K=d@m9i2GjCC$PEWDnWtxFfvKMKpFfHQ+Hugm;45VR0bw~5CMv@j7sjR?z'
            'B=7+d!n;6a<$MmAFe6aIt!&6qmrjao4<%x=IWcEtQ4bmJ_OpSS6-7=zVXZ}Pmg!`nvuRmGu{>SqIGVFqAoL&%HDq#qQ*i~UtjaCc'
            'qUqRoqS6Ed^%tG>^cKv5;+IH2PBZ=AWt8c~O!;Z1X3HRo>6dTSEK2fnxs1XHRHlod2=KHwTYyBI?uhhs7YCtUr!hF2^`YG?-KL_p'
            'RL!ggW|i6KKf<`oqc6nqT4#FDBi@*=$F&wUk%mZo<!%>8M&q|XouB^eci*dY$-nA9NZDbU6tq2TMRuYW5cdz0>&waYkLvuxhmSYE'
            'o?ofE8+CDWeK)?oJ7W4`|6|ByUkfkOVVW)SdJ|K#+%#VYU<K+SP8QKh!(9wtmW|$gPqykCLoM_l1;iPYizrn~NEg3+i56OZf!Jo@'
            'T8EpdUEJ@_K<X4Sg`{$^R<qC&0&bZ6Nv^^&%SaA}!GX$COt;1aUki-K40@oNl}SPm@9u5~3i_r;Y+3Y>C<|vnS)^M`580)o1Z+i%'
            '$^&<_)3nTcmAqgo1-S_%kz-RyfmvaYWqTqE4Z*t}gNzh(ofd0)NcvahE>43*K2W=I9!L3F=gbss1)C#pN_6p<W*eYN_9Y5|Yo6!z'
            'SPawYvMkC>Pp2x{?$V459iGV!l=ph(8K7wmKe505o~MaSr`;40Ai=z`zeBP3`)itL8Mg}xB4CGoz^}cYv92Gk#?zl+aJTAA`5r_O'
            'YKAxRi^Ll!PXtdteM!MR3@M{uiVV<Iy%PC*{a)`IKnzLO(osZ5P~~!bF}a=GTvwVO2n+!R7g10>9l?<Tddw&Wf?|_75n+SXVU&}D'
            'Mn$gUrSiAwLX*jqS%`nPDY<pd#+{NsB&!&Y)3Vr=1*icP$-;zVQqwY8_1WS;ukuKeIWHJ&;MBBfHKU!}dv<y6VKxLgak^!s0ks0k'
            'jJn|(>uT8vn_IL4)+OLRg#mJais2dp>=-kT({KY62Sot5m={>iVaW!S7h1$kvj}pS5XrB$Y!}goJ*f9$e0vKdnSQuBzn)t0Pk$bN'
            'hO+C8kkVFgCehL|(-2pd&VhoVa*$HZgPivOkWAw!+>eMeLgj=I8S#ghH&{b1fS(p6hrlVtpFOEt!6OR{HYzqqxkE_R6i9}>>+!GS'
            'k5k7u>4(YX_#+wUlea=Z@t%p}bF0E_uowKLh;Nr?EGC5lG$$X&@Y1{h6I)VDB=%^Ak`g6z5TFt8n3qv(unHS2lZvnbMAAUuO6Syx'
            'BcNFMih5*wh=Y=uE!JtSU3yUhv&<!Hz)DWcue=209@kO0R&(rN)5;gfU^C83#1a!q5g3?k76hcrNqy(nmx@vpti?{nu^1i}Edk5-'
            'o;4ir;obw$MZ_TFJ-ov{f0?iy88c@Of&g>%K8SPR*~jsJe1ecKrys}H(9{&c>T+@iQ)5u+Dg5yY9j-a`n;L5+vtYe1ynzg9<x}=?'
            'CCV4xkh@|p1)@+swXoC!9Nm1J{5ZKjznY$Zx}4nnMFs_0rtktSX6-P9wR#J^L~;_uC46ye1WbvQUqjm>3X8rv{RjUr60)8*Hj{GB'
            'ZT6(HN(B^mjIoM=3SopHn*32CejqJ*Iw_N_E>1@3o-81L7^st4i+KDr91frHP;#Sb<J~>pJv<MY;5m}J?XK7>Ll7{P7Z!SNuCG2*'
            'R#-4_iBSfwA382*82kaT(qJ=824-o&icEi@cQltmE#fD;M~7*Pn3OuTsSMDq=6_~ho+=%H9h(ZOWgM(3AK-Kg5JC2DsJ-^b-$oL^'
            '&i)~GJ}@{}yJ)|Iu><+eh_x<XWHHwiBtQ*o;~R?c@qCgC$BW8JvC+JYIgp)l?u9Lr+rM#l$g9>YZ(vT5fjbU3grbrKQtD`gXpNNr'
            'Wuq9eQQQMjAK*Xt52ERsb8@PxURvqG_hfTkpTfe^b6+0T6YLA76D=K5GSawV-&QW9#!<CX=4r_J`NDtEkNd)@Rn}CZ?wM*bTOpy#'
            'Rjp7O-yk5f9L#j**}y-gEagxzl7F-W)0ji)Rf05>Tn3;L20LSS*3zd9*8EB=uR&im5CU&;iR>WFqE(b&i=6XM^KxE9MXYnLnx^3s'
            '>NxcCIN_8KjN=kOql4>_%Srv1WkJ&#YUjwy#3c7I7TGZShCwg|(ffgK9#Q4NI3sbe$BX7vDg;TW{ezJMc8z|`&o}CVb9%`%v#oJu'
            '!Bjz7L<7oA3W})Ax<#76%~Bj09WppIVCB74Dld?by^?n8`K|^JW;T~0A-$}gmsFH-JX25A1dm60BO{smxuYEktA72Y?~FTC=IC$>'
            '&iD=k^#t(zj#utVO}8C2ol3?%n=C;p@F|tmKK(U9RLwXqz`s>m@V<;=<s;cb#T%0U_p3I-TL0lx?g34M?odj9x_f{6cg5BF(1NxR'
            's;MJ~vbfI+6Yh9kf2flXO!{6~EKmRL_4`9CAoTryljqR(@TOFV`sw`l1HPSJ2lZ^`ziY1YFV3%Tt|u4gSL)wyQ3djn6n6X^RVjka'
            'M(=a35%L{U>AY_;h`MQ*r&25-<e`?10PkOI$<Lj!sxl1X$6%kUJ*BI8^`5C2p=El7S{`_o6PH+7QNYVc97t;*JYs<&Z%T3UG0Lf%'
            '$fazisovKMK@={-qkLn^J~HS45|#3glK5oND9|9MD2`A0fm5rwAy!ne05*|;hl_H%%WG7JGzS=xQ`aEK!zem4$q5N@N+RSwB!1Vs'
            '06`<o@-yEXP!t(?sB@vo0C~e1^d%;Fp!*k0#<2E+bx=FVr+&j<F=iqHdubZ68F1F-N%(z?kY=Vp6JDG;5_1Gq04NjzH8+tgT$jm)'
            'zzK@Z{5aUo7s1Gk98#(3fB)Tg|MoqkHHU9B!AsZ}mOIj?&y*SxGw%-tWSehXs(=fBgZ!8MexN$)16Oo5$?xbR+J1%UmT*w{V+VhO'
            '0TrofozSM4R<WkS*XGyEfZ1HD@xw6*<g1pAWJ)i$S_Zb8VrZIDx%y*hx-CU%;h6DtOQ|s)84o9wkKE1tZF2V$J&rPJa)IC6%&sf$'
            '#%!m9&&|vCldG{<mYHX&mo}G6N`6c;z?maOR}fLV;#U-HdYom98K(7H7cWTJBJR)_B?R-hR@Yt^%Gbk{cz!_QM=Eg23Ahmf0B94e'
            'G#Me1z7haba9J<85I2aZWgz)M6G0GyNw+oxw>y2cL1(lPXCMC3n@0#)TfN7Auor;E2C6Ev5)~xT{9E>{{JGx55Nm>GZQ9R$n`B!i'
            'D4A@x6>APE$ZjR!FEy|FZZ6bH$Sl2q@g&MyFv31=`trf{57Lx2=R<;I6@^r~suN4~gxq6Qs(re7haxK2V`LBp8#RZOdWS@#Jk|(T'
            '8{l&mDIhsw!d4^y8a4+XY}76&X)31~{lB^;!$z_Q)Co~MLF5PxKeQVJTKMXQx_@v9eu<E$S5dr(vZD+eH0;h9>^3?q#q@6KO_`<;'
            'qeZ(=bKZFevT|PjX%R_>2Z^2#X=vSL2nt8`f@~nv#)aXw8>qAYu1oK2(*R&)(dJDC(3B_VjSeFnf^_=xZZUFKo$)mA$n<wC)H>Ka'
            '5X7ZcDgm%V%E0f)X-;`nDkfmfr#{Q2=EEI2lgK^O4>h=@0*7F=dVLO+nMx=R9*628%`$1cNhb+TPwjCU&dYHV-z69a1-JO^H2Y)6'
            'aZocA<BWoL&^9@KD&_k9Lp3%m_2E!@crel!rxB0AXd3yaZPYX)Ydtrtbc4!ctG@+hL8W<VqBWAxO`4SXHt4cYN_?Y2>4mY!!2z5F'
            'aj_4+MheAP@;2<+fdDdQ>P_4~c~r_hih#s#>4>R*1{lxep{^eyLVHf3)nu6uWRa9&_&iNx-_9|Qr<9^Q?Tdc@nRCEL?qHG{9=ZsV'
            ')`Z!xgW0f)+3<B{yZ8;?#&1aclA%1j5oC?nJD{OH$E%G_x;&ISAGX7)SBD_$Of!*|4j29Uaeo-)QWU%vsFpxo#6=X=8I^KrzDK};'
            '-Nt1SY=cFFWTPgFG{ie52VTwLz;q`mld~IqOLvkJz-c|L*#mOOxf`z^cW%66(<c4w+;2^--fzcVokdNpA!mhiO?Adpp!Ah;qj4Nz'
            '>y7I4l5fZ)S}*vrhg8Z{buqv*vRzw>bFZCA@gXW6?WO;%!*z-@0(FFwxT<Rg_-$F#SZtM3<+I5$rPIH<)k2$r1JOTD1IA9A(h)^T'
            'j@rr><Z-{*MRKl0V8%5vo<=vw#!9@CO1pl^MG?7CdltW-1gp_Qx__`uAgOltPU90hyl*o@&GpWzvDRR}MZsa0W>?1@Ko`1O<5VmG'
            'Uu9`?V0T25VJbd`9Jm>cd<j(KY_uVG&JXy40NKy+Bi_-=+hO^?z;QE50#Jfg9PAMh)yY$Xp%cr{N&opo?a~-J+61T-Gvu`pO&nBl'
            'ZyS7V1xn99a($@pguwoRfXo$%b}}XvCY_yeXNV4oQ1E;nUoz^O`rU6@1|7lebEM+MLCRygK3aJb^~mR;bnnYECpvsuKd0<qM-Ud6'
            '$M&pJR&(6r8{iS&PGbKhl&jCMsG_$0Y0f1d*K{way1h`HxJS7|uRi&%Gc7tIh4x-NIc-E9dUJ0Y!w;?Q^XqogvYnQj(s*w6HWuk0'
            'o84)7Y$fmM>_y#NpMO^4>mSEgR}*R{vPmXTH(zFh);DQh#xV-8QAa+u^qx)z>SXwP8YRMzIhB3rKM&PM2^~ZRMw@VkMrj@@xgesd'
            'ecPSlZ4wk6$i4R^O26nd*dY6_*iZ#behVHKbp8WUaB1k3+5+9xUIP_4`WOvQt?pXK-vi}@B75JgbvI4k9!a9A`)u>Qp#sZ922DTH'
            '#TM@{qRFj3O_$*eFEa{45)jT#EX5PPM+uYXgYHNgQ?nyH&Xl=u@XBpA+Y)RZ*av{Y-<V0|(|o=MVh_3`8ZP?7j#d-3x0AgQ{j$u1'
            'SU|uWYSj1t+?zQH3A$Wm;@bo&FGK19EX$ZGe8U>S5r^#R+9lZ8!V;*9yq8wCZ1xd=r{((R$i4B%FhLhinO>ZxbSiCf2m@u3YbjqY'
            '59~6&L0aEK?2}s3dBK--ilI6u9mPTqoUf1o=XBC-u4iTXcP`Y;4&Gx38Q0l~V8{tm?!E%Vyao&Ld%hy*TwY*mey(3S;DmPGes8Gf'
            'G(=egN3ISzZA{)C6ik<?$|8$a>q4#q1d?4fFBbi&D@Fgzc40Gbc769aUC$Y&zNT53#GoiZx1qKJp%MX<+64Or1@3T*f_K<l>ln85'
            '-d1IMN9J3GhUz8>$#Fx??YwmvA~Hs(is21#w+PB^rw*$$FBnggs*%6YE4sN#segvBpio^43Js(`giT^Vs5fBYkX*JXS7lNrEQqMt'
            'ku#OgH|oON5BXvI{sxXvg=jv(mo$THNE#^Os|+lBkej!X`fubU);*|al&UVo#u@;SJFsx#oH>**%965@;p+*~|7BkAhK34<YIyqP'
            '_;PY}IT_iL0Wmx3t8jw2iK8l7R^faRk<PjTxEt{8G&n~im?M^V1ZorBC9rc(Hma|YBFeMcyQBHh-eX66w~4Mq_^zq9(%_7~BD~Nf'
            '@j6{>^+u7M@6#;jgVDoW((kJcZ8<qe9pqAtL`tr1-can2@nsSa#l_i*>txTG)Dg!W(jocI?+@ejQD<Q7Z)Xh-?DMw*#PZN&1PoZV'
            'YR(;U_I1<frA1C6<v$%^@P70!-@_P&!+uK*OV5HUjPUl;`Ss5e3l4h(V*knAhC_uKD*2WaG*lDeK{#AQU2u}8Nd^GY2|`(7dwVda'
            'G#2Z4*9CU4zzm*XF^^3V6zpaX)ZhPquO0ui*8WPJ*b<>zHXf@JNuC*edIOQQipHLs-@+sKUP}|}?Av?W&Y=3Ej`5vi=>Bz1SJp9f'
            '>B&iPwMww0F1bn15O1HH;QQRB$8)FkUEyY1rs876t%t$k8_@JdZWhorCZN8n{g`2xq>sM+o?%!PA$8VLpaZgR-)-e^5rB#OFC|&6'
            'rp%GgV5oEJc#qqnF{MqF!<mEp=9ex<i<_>#n*Ac?pk!#23i7#Ax<x5ZP**7PMVY)cteG2{K~ZP?EUxyw;HtJkA<1VR^n=*L$+LBu'
            '>KisLm7DmFU6|`=)?$3;0J=U%>9=XgJrsV;KMSpjThs4=xJ<IpESp}UX1D}3D>!rEM&*Q1)f(@J@Y`<0)}I08>h|vZYRpP-`yPBv'
            'maO1-rBRg_brDwIHKZe}^Q+q%y1ry{ngyj?;$rou8)}vE#}IN!h_4pVRRQ<%gv+g*a>0MO+B!XzhTj*ykdT(Ewev}oeAI>Cx4DkV'
            '3`Ma<c+ILn?ice%7miz-%SLi{*ZJK5%m~w~TfE)PSBkn0MW?$+>)m?|sH$oT#??s9ol%%c-QEIgd3w@E(b4zXT}vU~>$yu+Tr_&6'
            'm1`ZG6STmud0ub~URouI(0*~-tuo}WNrC91jw~m2Ky~KFEv8n8lEgu)xyelVm=6^c3V`S<V&NV^KcHf&ck~$A6=tK4ZGHHSjqBIj'
            '(w$b3mqD_LT3OXv=fO@wFCU(O%=3#@K$ne$T{j`L8tO;}U(=MyEPfncARU?f*T^zRN?5?ZCA}gO&bJ3i`qJ}l?vR{4SuUzv4miP^'
            'KC`J_o#hSHXOpp+>=R|DA@yhU_>KD2!H6=khzd@xZ|+q68Lep{3GR)WxW}y(LyobtYHHW%24J=RioyA4z!^{1_5axg;%nNiYs6Z?'
            'f~ulXEtzr-4F_)N=(JXer_D=MB4z6BF7ffA@z{a;E2t;X-GaP>&2uKJTu(EQ5mAQEa1lq1>&130PLeu{!FH<jV)M6tYumgp1oBrK'
            'z#q-W(1r79srv2w?&7EMrCf^?R^__O&8Iu(R!dY=4N|^di8?2@of2+p7@{VHmM_J{a|rX}6n0%*5VWo$^{@RHQOw>s98nX@$><_7'
            'b!+XR&a5HNnYooo-`2}rI_GP8{ydZ%m-{VhCQ)QYA#2u*WBh?pP@NXZ64GJhj};~5__l6LQX=Di5y^*92cO~}?RUSkHM|4+QS<9_'
            '>F-uwxN@JpO6A1q3=GEf?I+*BKImyST7WON8=YZwyV2IVRnKux_mhtG9A9f?SoCU#z?XZKsyYMwCv-C=*B9ff@h@Zb^ZDn?Nuxq`'
            'dKtVy|A|hAk8sw1p^sqzMRa(yZ$Sr%PVeHNLaywjvqH9S@4be*w2M_c?66pM%)uMEHq<rde^;y?5nX37`dW4SYyz&hIQ9K8-yUkT'
            '8m<*qoB!rP(3&m^O#vfQu5HZ~`eq?pl)G5p3vScj4;8dtqzQ@*WCo!gF(q<#{=%c``qrOGQ1llb<W$(N|1W@6T}_72my@kU`k0sm'
            'V+XNlKH8GA5iX78<0&~}q0=JgKM=_!!>F#39DXO@4XQ6u&g>@;T^|lqn=#*I&`y0B)Hnz|x}o|7Nw(knA1&#y3j'
        ),
    },
    'google_ai': {
        'sha256': '218f70e619fcc6e8f197aaf8138b7c74e17e7510b1ffa1b6f0bb84d3a479d8e5',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}dw<(Fb};<EKLx6Jo?4udWIM@ZXPwo~tJq4S#+R%uPj=Sh(v(EmtSC}TQg)QA``O=f0YHEkS;-`Qw*7T}?M5a65I8tEH=IkO'
            '(Kzs;I10vI*b<qS&f4C+m-ym!9N&b#=nX{kwLcG{K<o(kG4~vi#LLW2#57Jsb{nK(k;MPukF(C!*84ZVis75Vxi~&Oyg2HM{$B^@'
            '!}G0n<$qf*eHg;$G3N1P37_-t{A4AfIP*twJQJg35N0CCQW4)r;`+vqyr6Rs27Z(^JDrYmExc&5b?YUQaXj%S;u?nW!|wJC&A9!v'
            '^SnI`z4UhbS_DxTM1DsMZ@o;+y@?OQieSEo(=-^tm$O(jVQsT0zTdjP4aT>E#!6v&zL+Myua+ssUL?G6w)DbqB@*8o-?l_^1YLc9'
            'p<B1Jz)!Z$-W~4@;s4Lx9l}~BH+}{$<GC35L39I8k_aX}S_v9)986$cUMQws)CLf@aB-qJcHlt}!UBZ1wMhI)FwQ)fF}z+*0{SWL'
            'y%nx3hhxC=K0WD+Md*zY$X>R;wI#$}M~q?8nV$+zj0u#YnfmZ7YX{N9U-%JDqb1bO@d}>f!u^Q@{hxGjFL-!@5P;w&NHf^BAH+D0'
            'roqh;CYb|4{Cyc`9t=12mSL8{5KlW|5~Sl8_BsV<F1%3?2GBGWcY!Cac~Tt)e<y^oo4nBg0>C?%Yet`+i~AtEh4J7&0HFV25e8$}'
            'RM?mJ9=pM-XC1K!#xq>&#D~s^7HJlY8HdFJriVBWEr0CL24YKinnEjrLjQa?IPMMmhjJOA54)H^`{x~4c$VO-cf`oUZ^Y9n{Dx;V'
            '$~Z{IOPJ0Gpgp55f9QxiFAOFGe1O;zkO)BqQ&}uCk*;9(#~GePK&;-$p-BC?7s&_E|EG=!<C|0-5<tg0z?ejMS?13d8A2zfZSj&U'
            'n8HZV<&PbadJzEkqp$k;NdgCV#AgrB03Kk3EKDTPYoBb1z1<E1HBJdtQh(tkF!PoMG)yw`=Fn{%16hFWL1;>{p?7t)wRIv-Ga%<O'
            '&G2vo<l1obLVu3f=S3?39nX9ONP(E_ujp(hnMoah!2Hrb8@w98ZuDOF1}Eo3ar)}j;9$@@60c8BUjxMfdUbeuaB<u}8TN*Q(-XvP'
            'U!3)OhsS-<c<ZeqZzd-3O|lHVxz})NXw#Sfx78NMbSwskDMH5Q6Mvg!i*$cy2k6%gfv7X_?{*}e@)p6)Bp#<bjPo!=;Sd1f^AY$i'
            '95BSoCet|}fh{2)rTQ;8fn6ewEkJ<6K>mE>PxL67gw`H9BF{dI1JhgLZozi*{M$r~MY>D?)qFhG%?;NW+~8824t$g*dky4x1qg{J'
            'FZ6)SA%%nS02_fCWUGarcD_H5P2&-gq8Ap^Sa>Po=;7dCcrMZykvh%1RVq;kDdd)*WzOUAj0q-C4mfs%bmxa};p{+<5Ds?-fe(X6'
            'Goa9J2uCKoM%wHzfao-EC37!qh<}I%5E=RxPC&4n)6d)B=C&c4a1Po8NC$UdGpMLf!sic8&(8V>guHiYho-1l&j{$vbRQ-N2Obdu'
            '$j5aH{)`vyLj9OAQ^c)@e~H?_hA*RxxAWw52*+;@=WN25iR1U)4R!zw9dSCH!qEmEbhCt>5ikxQ6EKz(0iF_LiGd1wcYqoBEPx0R'
            'o0e1_xXiB2kl-r`ksKH-#k0Q>Cw(~0;=F%yc+TV;1q$_{&IiSR=>T85C<;tm^B^ET5QLds10N1xyO=W(pte)u7(imwb|azyaRlD$'
            '0aFcAAGuuOkG(92qt0fqmpk{Q^CQa^ZJ=`gH39}f8xrw^4*|0liW;8#UIH_`0j2{x0$*pwx#5A4=Pji{Xu*u*@{4d_aGE^0KWQgE'
            'w`3Ogd5a?QfHQ1AxrQU}^B2<P0?!em6f#Q}kL2|WVw`kvkdXoWQlJcKHZVT2q|E=2ZB7~V*Hedh0Dl4YXBvEf)xtoqRdIrXp8SmB'
            'bTkd$qtp9J3m&HQ6)j<K5I58jSS&}dRk!(2ID;4ZmsrGrI)Li%QD7<O8Hidjkr1F0AUK4<9MK&|Yj_Lbhd#4E3D*{JIeY*o3*?Gb'
            'gZiQn1_L&5LL^S40(UFJ(=!hy5s;5eyz|BZ972%=bHA}U@bFZ$d0GIyKC*hGDL1e_+&B=(`$z6s|1}8YXTM%c0g;a+9EO9_lUIY+'
            '7iYbfz(WuFuX-0pLs$e#3?!T`;rt}P&4C5evE3KTMHqXNw51`5KT(cgVFBgI7W|t4F(Kh2e+Z{Mi4ltF*2vH9eLqs?1n4Xhs&q$!'
            'k!Z~%28j#06TmuKjYeaO=$7kFmq>bCSBVN<6ydmtid$Rq*~m-%=g-t%w{Tv=V5I*1k2H?d?>JSz64<JEuK)7YFGh6r7j6$v1Su^3'
            'rKT|<DM>wTQA;@Pna4jr#*yr}@G_i@>h})*+}iq1d`)5GJMjjTBfw^m*j6G1Iyi=%qSKZ7guXDdU_>+(NHP=aHnBYvB>>rBI|!R$'
            'sOd7IgAV;}NEm=OX-rrxreUyHAiBZo5V<^%<yo;x<7EPTWPIyIH>`({Nbw!&4S@YA4w!<V?2}>z#J=;DVX*bxzBnVqK%L9RxZc6R'
            '>BY(LeCOTS>A&?+egKS<XR@Tiz_REE4oK4d{0@MCoI?zV9D@SDO4~Iv8aIkviPJ9ibQ6i7C|tqeTcX?oL<R;-mXQdiC=4S65uCui'
            'vW_@FxZ%v{1fL_N1F4_D9D?En7=`pm(eoy7YZxr_P(kPI#=fv9P=JEz6iz&lHCXH~gTwxrgOi?uTFly23Y?U8N2&P80y<m0qodRJ'
            't^}3)my6S3&pkVp@^x3dLPknZ&Pi{&9L)pT+f+$+uvXG_CFEgBfcK!okZceT>6|R_%#)zzt#Cl-2HF#KyIq2g0e{^E{yh@%$H^k9'
            'dzu#FXqsD?!s9)VH6PW`gmgK21?V0C%0j;p?AE-q_3EtOcZY-inS0oS#RCHw9Q0x7P0~IaQfNV1zeF0B1|P-D%V5I$z!QNhCqW1}'
            '2V*9|42BgzwlaFOLgg&<wBFR<0W(mwU|&E2gQm+Ugnr`2N7*7s=00%UF>tNfG7O=6NQzd1!vbzVkDK@-sO@T~8y`^B$039<YCKy;'
            'vS0I201n~HQ#IbLHwyzGq;mmOZRM?k;*veN<hj2JqE2JW;RER1i{?09tP*5${=`vq0d?4T$0PoV5)BE3%v=~>q)7P!P(Gw>3}e=i'
            '5v(Z1W<aphF=*W==K<-yM~yrML4j2Bxu1DJi#;NCDe5>9pU-;l-Q)h@pyv)xU1Nip`Q><h4v2nn?*7s{y6B(7<4+6+(1PLnbF$bp'
            'TB0#Qni((fflDTr+c<<7;wwg4H}U_zgsqq~T08>C4T$~|YyvyCB+(nY<8<*Mntk}SJR5MFTs#je7~%nDcJQWu+;ah7=X?_Pwzjs$'
            'Ad{vNoqB_JMELut@;A?b48ery4`-j}&}f`_0UpfzTR<64+?KcnZpvK{BSo2Xkoj(H!H$T>37lyES)2fw(qDQ7%SmDPmJ`@KBz4?z'
            '2gD<xx`}&_?GysE%Ms~g)ct^zGa;!Hs5xjcxb{EzW1v$mqpwRu3n(jbr6|sA5n)KY1*i!c{%An7rjdD(mX4?asyY(YjE!=7LZ}zW'
            '1#riySVs?ewfJ9xfUVpJ-Gs(nH(Ga?EPWt@L4qfG>U|77wJKoXaDZ7-N&|U1^`LOlQ#|QFOubt;_dv;`2~eE`hzCfYJ5n@eITpza'
            'kS{NtdowXZPFEN$xDC;6#YASbWNrgYXyv8e3`kN3_$%>?l@~_A$FV1Rzn%=Z8PTeVUmTLS864poqG>*nqE6lhh{ji!gw9F+1mWiw'
            'FI@U8yx%8q3SqlRi~&3xlMIQNB5h(V)Jo@QVw<4k3d5k#*w%tAn{TD<_FG6U{SU~o4&VTe<O;~@#%eC`RFKa@q6r|MBFO_OJY1$^'
            'k-Ue)+#Z7>2AGUC5M+U200J>I3Xqqkakyk82mDGpTj%|Qv;NTS9SwTtCasg_3wT0fUo<7Vdkb0(kS4afxxo9}xBXumjwEg?K#1kY'
            'Ct9XE{>l%r9rXP5^kR7Nvd=8F*dNr{zw^h=PXD6cw$$qg*hb({gA)*C2EB5VF;W`P&_P2`Br2FtkAtJri^Eq(y|cbPc7wy}j-dCr'
            'Sv>QjQpbzo8(<u7`zLuv?<1-s>B9HNw=ODu*r)fm3uu1+uHQd+V^7d(jz;u2D)&H*?UcU(6kjqY7q;oJe=;2Oj$DxCUJZ^=7->jO'
            '+1OGK-ILyNABKRJiQhp?0EW^`8vlOTYyZt_f81^V@hbo6cG_2;c3aP%KK!2soWFz9quxtbqQsm2(Yrc)(1*6cSPEDLs!ikr`<n%t'
            '05#?DbI|;A10)M+Z7>6Z27F@s2l1;9B<Lpi7@D-KA$xCQquyj+Y-`-x7R@_<<q1E<k4)f=g^1fuOC0()ezI>4m&=SPY4l3)0nqHg'
            'YLF9olLRM+^wtX}!3{DBo<zqSX<J#f1yBN5QLwF=ktBF<-1`-T!DtB*5s(BBjTUh9R;fov77s<|#?OFB8bdOGVBA+|$sty;qrTD`'
            'k3m(UXDF6xTDk7H757DMfCgoL8cVROI+_SkW402ws3gV+K|C}DIPoj+fgyZA9HRJz!tvcx4WvhbIR;5N4(9;1e5nswzBvFn3Ux0p'
            'JoBeo!sszgL!pq34Zz!DV8VAu@RbO_ddM<U)(w5{22x3S8{bD#<hV}!MFP^q&NV3h^e=M)R^3ocAc>Udu)1TB=Ao9f_W06V&4L5z'
            '72@B{Pfs-a;Cs9jwX?Bakzzuj@6Gs92Q4sCcWIGbQQ$?8zza<>r_k*eny8VM18+z@(zHPyy17N#9z<TUQj84r!Ca!DAFi8*ahAF+'
            'YBY@*AU~34VihhzOaJAkF5Y-4k#q{w1b?~&U!)m08cK(*3#TA+-Dc{Cpc3Pd`)EIDiF{OgApSuzVoO-#w|K`|mdHT=$**$XuBfMn'
            'PA-Vy96C4!=z#y&FY*b&>->+3MrwA@E&~O{Z(2_syN8@8v{hGrZob2=eA|@&7Wy(gxhwn&8;mew=>C7u1r(2LrD2`lxe~NV?uN4z'
            '8j>huD?V%*A8_hMq__EC`8MT40`8!JHUX`UPg+Nu;$gU!W_o52e5r&71`gH|L~T;$QN`{kaoU)lF|{0%=sa0-DXpuKL!g1;IgMl7'
            'MEhvxZ;&}6x@qJ?XR%AR%dv|WPWd<RFKeae>U6jCF>I&a6m4g$cajO{AO<1=dYm{pRn!MfGkDOeR{P18hw0Sq=eTO)euKcE-ZM;`'
            'LKAf$_bmi0G%^nLei8qxngVu9>p%6Nd%^OFFsAZg5&@*3k_hPC6XH8K#oo=_+eaOZyZ}TSZ_V!jQbi~=r;R^Zw4P3I_mhUx17nDM'
            'A|dlo&^GYDsREE}D*t-mGcO<Jz<BtxqGv!IvxI1)T9CO!CaRj!fiyS%!le?sAy7#b+m-U1%h=?CzyRpfMCqUCpCoaL7o{B^q~Zin'
            '5hV+3r3Tu70`Y*>2?Lz{MSClFY~~-MDHydMLT^RmK)CQj=BvaJ@g8sPMN431upXc67WxP%+J;P34FK&M4F`4)4pY-nYBut~q9ogy'
            'E<+Fsvhi(`X;@L;Cf%+|%{KU}`s`l&`7@Br63}5`V_4Wg4OeTiw_XB!3h%vnOKf+x;eXt1L88)jdz<Web3Y7Bwbj@WNy%snq+FuF'
            'F1NI4Fh+AD^k3;dQKJe*nEz*CM;FO7?PIlsbjI^YL0$kVrlc3142b3+S_L;~{s5r^jUl{?C>YT`;w2YPOv*@f!>g|aSfJwDN5mP}'
            'z+1i(BXFZk#7*S&kMO1lczLx$1Cnz}M_jA*qi?_>mB?o!T&gCS=3227hDe>2Je*iHT8k<)VkLEeSQy4^WzXb&#yQ|yzF5-$q(Si+'
            'tX*O=$hLH)GS;s_6UPm9aN>GkY8vG@pm*Q9^U&N&Vns?cxrxVU=FpKtVNvLTal}s#j#Ga=rVX^ZM00-yx=#6c?2AuoM*osLya2iK'
            'ZoJIksh-6{gGV=bZCB6wQgyzfHVV>Rv;wQ<Ee9cEK*e~)1c`VzOI!In3#Mvd+WA$p0A^2)18=FZqA>(8kF0>)e;py2HcUv!6$WgC'
            'M>X*y8tBB0599zIOLX)mXtGcI8PErwFrQG|I`hQh)T${xb)tFV6pgv&Q_YgqzC{}pb(Oq=I^-z8N`mMsU7@-!37hI+AjLtusibfg'
            'jUi$t1^HHPXktlTp+#2#(xQ}A<5+5e`m;QWO<v1n^~<Ou-LAzq=y|91Q>jyQ?N1vrz6EhlDT`$u?a(E=a-9@SR>H4M{U2R+Y`%R>'
            'Mn*a?0w`oACNYpc-(dq~erv^alrN4`ajo@k_{23%5rrAi69ys?RU_0+8L1+%j8Z?_3FGlBSD4W+fZRjLhoymsW{T#BhIIG!vivw}'
            '2LW<-OuY2grj+@aw+)LVpx_==9pwED36|QHWwHdCgrnivY>wc-`H>$X^<oNXf7^L*{`WY3$>CE+O-nFi&>AN${zhzXsvi~NOi8Zn'
            'O(x_JqOB&1vC}2k2XyE{w?4dwPwt2--_mWjH`dpI<UG;&ojf`3w$7ALQaII8={e2=K8wy98U_}8fqT$g)6AbgbTlV<kWXpF59(Tt'
            'HOi4Sdsb@l2mAJxm7u$A@Z+Yompa>GM{G+i*m&L&w&4_}0LNz%INOr6ozNK{EA1Ayf{>_mEtb#F(7wySYnapfWOOo=FQ;KcP*3|4'
            '%HW;<h=ZtE?5UN}2fYLZRRfr}ez<VaEiGpfHUQPBC|bWY(P|a#sbuSBi*XR%pk$zooC1kar|~T=vp5KXUHMV=ZdSPKzd)_?Hkjbd'
            '*c3x#YqUh;dzvzbbH4hem3af5W9l()Vbv`(`tQn}sxlwdVtz}m%0-JzoUE9=>vQlD$3i+|IV{VN2Xer|@CXBqq)y|_4E+ctfotol'
            '*J#g|jE@W=nhlCY9EUQ(g#GK7I7v@`4?`uq6iu{1z=F2Cm6Fay`ocNCg7iNi36yR#$o*+JDFLus>|@vP*NzfUY*015MSxdS>?m#t'
            'D(DXhl1C~p(P7FVx=8Otes&)xGa;uDM0aFqFOh+Q1fIjURiIl%PCr~NDmC{CoPZh5Na(h0L&W?O(Zsv)1!(Vg%a6+)1Vy?6zyfW}'
            'NIIDNADvE7X_qs9jEEVR`eoeSvJFnk$L!r8rwFTf))Aj736k+?5{@1!rfBb?x>K-ivyy_<Y`F0iLl&Upz#vxch<secck*w*jL`cS'
            'pe-Pl2om|EtRi-pq|=Jhs{gDsaw*YW9d`Ps%K}Kn@opJAWS(~on+;%6W?1BTWOvwnqZ=o_`Zf>?V^F-P3n#$~d9A5#LeckX?x7bJ'
            '`ijrTftW2h9OYxD#+0<4ES3Oe|M@N0&#@Kx1uO)fT_=nA_0Dxh|C=l(9B)wNS$s}%GzAJu$HncGW9-m*P`R5fqW~ksC?IZ(ejDc?'
            '4WjkPPZx1SQArUXB{VWzw54PAJ!od2|FEAug^7J9U`pLx@;+Y7;RoUgdm4qo4OxSvTI4OWc<$*Sv9Hb_d*q7m4*P#aFSAB5wx^*Q'
            'oDa_~4u%&Xk7BbtUd-iqG2QO<+3Cf}A-9q!!ZtWp2OQ|O!y&qLG_ow!kNU?ixvv!)q+j=5gEWk<y&J@~v8&<>lyHR}4CChaEsTjo'
            'f6o@ew<R+C9{_nv*wHj;x9PNCN9RCFx()U=Y?ua4)3L5gK6r!>e%fs`;3JEmTLzhJ3jbvI5Ad8C`|8S$|28fx3~|(ji9<`Ae&y?o'
            'UT1Z$VX@TPxJYRf)~@GKcy7*zK?5!Mhw;X8Io#*_JoA}33PG%~&yX{ovu^<WuknmU2KL8<mkoI)U1M#^wS;@&J-i)IpnXH=tcc#e'
            'z*wmE-<KZ707^0S&X)oZit>*g6Y0Ea$MM4K7~OJ{C~s)DX`HH-D4ihvNF5dfR!9zhB!ulg`El3X-Q6X`?LOJf+mvuvhk~^@60D^I'
            '!CEp7OnyN-G^wTh_}_ggyu~l&7K0SHocm@tD@lo)P{K;MP&m~$1I}I*bbj3ZHaordHCFUrzX0CfgME3E(^ipXLPwUVi~vtMKN>-8'
            'L`ED@y}wWX3Gsd{Fz`RgcIu+|=mJ2-5=V13&SF?!MbuuWQHdi@z7EzGab)*ffubPz9$x&de>6Pq?;~o=gHnJa1w)cEo%r?iZ3|t8'
            'S4$%}K@X%*%r$}+GxT8g&<Kgra$t(a73g}nFHp8=F4)-~O*(k@I1Xl6APNrnj@X1AsnZLxQd^F?!3GV_TMkJ;-KV?>d&VB@8GB%f'
            'Iq2qHBipzX8qp{rrCKOK`h{F5F8QOE2g>LrKZ<{HI9%=jm{l@n0CwhqrC%n=TQD<XWFLnV3D*i%d%N;rJ~g35Kl1?)I<_SK$RX4r'
            'xu9L05BqPA`X|Tzzey@&gw%o>8Uwrp#+N1NfUAOZ3ukfeXAwH*ZsWlFqSNV!f8hUaqb+O}tU`2<_*nGUuK+w{ixJIu0q0%Da>pEN'
            'o5xEzVl%+cz4o*A)4fqU$Tp;s3O^;t_nsRdf95{J--Z%0_sY1Z^h`G$-}-aUvLC%5KTZX~fHJPYW^sgsw@N6gfniiSc(j%$&qr<V'
            '*?;@>@J_!1jRJJTo5Z_AY2Q*xS|VOxW*Kq~s^4zu5}%KvDbntb?I*iWo<9yI(o_|ko+~)*T9nB+X$o4#Hyf=Af>i_~UM{Rs;BMZ3'
            'coMOL7jPsN0h%h9Lfv=?4hv_D_}-nDq~0n3DM7sLz3UwfDxvr{KZ)CH45o7uq*mUceQ9n%&lVl8f)<uMAB!9e|Lgzt;OGL40!`(L'
            '?0*=CONzF|@LZKT$Z2VxoFzthiFUpG1{3*a(wp2axly|%sDtRB;|x{!(Jlomy$~_mYi1-gtUS>>KkA7ajFkLX!}pN8!ziD@*XsnY'
            '^^>n)e_ybkN0-7Z9Ax|(|MRHHV^9Z<v5c}M$E$*X_w{B?;Q#M5<6KnwXP@;0#f=Sb{fd-Wow`gvG6xlqhQZ0Oe|FwKz?@fXIo&5>'
            '^g?)2tY@D=vc;<WP$SBv<7w+Vg;Z(_E8gjLq{WEB9gha*L+wUyTawhpz^#|MQ5<~)qR4kzob$*Hj7!_x%^kM_2DFEwD_hmey5+&<'
            'rON@1_ca~Tq19#4K6oYm3|m2&K+3Nt{8}y{NL_79=HOIU*_zibh9X%Q)EGODS(mB|)9h@E4s--|y~n^FFnQRm%t3=lMqpf%olQGZ'
            '$d#+v;bmaK(1$Cp-<xl7*D%B^Ftp-SG~dCueeu-kh*Na(REh4UuzGaxWY<qb)pSWAQ`2QAra_Ws6b6h@pu^L5=Xs6;PBb(3kgtkG'
            'u)v#q<z6-Qq7)CtSHA%O7C7v?J@hr{4F|taCV{f8vsGa<Y}FaeS|u^rk)L_#zPvbkYk#7mYlX?GRVl7I01Xt#+Pgos_kP0b>JU_k'
            'AdIu+z9<q)nUOK4Gcp?Z_37Eg$;GdyXD}FOaxltG5{PHJ&oEGVxr!6KWN$>VK)+nXe(m|1#jrc~6h@bJ#JkW(d{{2f<p;)E#Y^;6'
            'nNT}uBNdOG5^yoA!HnO+^zkYg$teh{{0xTj=jgk+V)2(E0TK5-;iWL)RUk6>2ROTk!^twiDU*V=jFw5boI!tDl1$DJ0=b^xfE4i@'
            'cEn3<3CXQ%lqJy<D<Jmp;f~zC1M_}oLo88Se<<&Jn(M6^tw-zuYPi0>D{59L!!6bBmA*7;u9fJa!`85O{??^#M!fNDoXum(q@d-q'
            ';*pWend5I%S}=~Ft!avpilo$xSC0-W?=8qGk<~XFEyI~h?k<?I$^JecJK69qNIW2qNXP*Q*_xIrT}9QBtcH2k3z&iLMaxBXyOL>5'
            'H_I_g4N0~i19jx{+=!rGOH^o;)VA_xBsM6y<ei_in`l~S@kw*EHByi@vLkvUOw%E{e@5Q|+f<3<o^yB*-VEOiPTmeqUb{#AUobb$'
            'It2Pub(v%sJS*;J^xOLIi`wf8vZy|+!P0LtpA9Eec^1CWj5eH7@yLCH*=#nE>O=jhzO!+?Qg`*E+dQl%2w(Xt)x_j0f25KSedTXd'
            '5wU6nkrx?N*daGrsz)ePg4B(IV4kd&|HPQ_ND1JWvQ}q%ma&kM{G9dwaxpmTAG)mZttCIEQdmzE%B?C-YPnhQ^psmvqjkAm4Q7@b'
            'RwGxrT_t{$n^kRb!DM8yDdQ@lf=gNE_q9yQG9DI?L;0Gsa@?jwAY$xe5W<a2Z%V1|K$h*j@BJ!X;ygL`qlY=UWoozdma?T{Z%z`8'
            'SQcU6Xh`nk`)KczHgQUCq!^-18IDYEp`2D(9mkXdb04H@?}Ky);xr}cHS2-sdMOhNWRz7BaGVo*lKPADna7d85~yMAqXwCQ=sKpT'
            'JRF^@!Si6!<^%>UN<p3~=X%&`PH`}fQ_A;v-aj}!Iixgp{@<5A+8HS8nhau4)_^?Yq?L7$5GY<^HbII8;562p#}02QGBeYivSQ14'
            '6qO_tbKV4$7u*&mC~;Jhs)TZB7K9-=99t>F%W&+<jN_D3ER_fTxc65VWSVyuLr(p2Zn(Wr1|-odx;u55C+*9SA`p}@Vi;qG7N~JH'
            '|8wuh-CZ#}L$8TRs+?=T6BKL&uz5O@h~w}Y(<LZm$!dr4*r8IT68=;mPN$X{?LBv&{`71OA+j1&(jcovk^Go16=+_TLBmuY4@jP0'
            'lgtwfZ$;Tf$;8oFN5Fn^_x|`~J?S!=R1+<;MJ2g18mQ$Q?B}W*^q`!Cn01;L!<l3OwKyFqdnpEy&PLa%CwlT*57LNK!b}V~Z}Q!o'
            'G96*6+JD%|*3hb9*n1?=l3|bNZY)>^m&qkK|C>HeXgi8x73NMfunCmbxK0FOLZsv-){$|wIhu>srRuXp);jz;+K^N<4z`l<Dbjp6'
            '9QKdjkyiUmLdeD{8$2yhHnbJ<Sy12~PK*zc0X2K%7Fju!(Tyxfs`5dK*~U!|eY_r0LW9a44z#4nccKj5{J8QbF}odf$ic-bPghjw'
            'IvQFnjPk9wbkZ0lQ>Tt8+rVzhfyCziNmXpBez2vKdX*d=TcYX6FkVh#^7SKOa_LP~mB3OIb1w>}$UCIT@X6N?bjgwQWbmqgK6Ei!'
            '3=<U?p18Se>8crZFoRs9+<Q1Ya^IX@oFQ&MdDc-@WOo|);iL(SxJ5DUEp;Rd;Vy^4IBT=1lX?KRiD%0N9eGS~&zw9}sfmKb3@}Y)'
            'm<SFcUJcI9hsu%<EFOqfFy6P)^XsH@h|1UO(w|=MTz5c#pVN<-ze=w;3^dD<V1(vdl1Q#^kThpWQ{P8ZiUYCDgKRt3zzZ>1Op4KT'
            '8P!-Jv}0rT2q`Ag(c5^U(tfM*5t=-SJ0D%&OR*^cg9=xF9$rFKH8nz52=o-DU+M&}bh-__F^(M#!K*FrpALawQ4mBQmk%AqQzqVh'
            'DYCC<Oid)&;)H-<F!RNBjlpwUQx(dS&{b#!p8<6VpEWoM0d5R@+PVkHWJGf(1$hY$`A);RZ0}v^gdMyNoRzB$wJKVH=VQlpe7|96'
            'hn<ZF7I4rzIXxL1^p3>A=`kFf!+lO!Op<NQ%@$u#u~3;8EBcx7T@n6I+wlxT3%T$1zPJn~=@mSngcRHAhl_zZ+YcQ$CyLT|A+k@L'
            '07pw=Zh6XTtZvJg*z*kx-RJWaf4i+O%<gZc;z_b9shl#zN|w8!dpJ&buP%8+vyf`3OlNfL*8!Ze<(y=Hk;^?ikNvWR9tC=xH5cb('
            'A0sQsS?62p)<p6YciP-!Mlu-%T@z}9!R64Rbnrek;u#1_GCrWeYk&uruxSn2s|IoWCUkbxBUffwGLpf#DlgPSd2x!J(DGz7`KRjB'
            'I_HcMbDdWcBjYT<J~%Z?V7GvL1^HKvmGJqnSAowSwyFZjYk_8adeuVEZl<9r!?hL4(Q|~awWn2`)r_U|zABAiCy=U$1TACt7g3b8'
            'c1q*Fd=Di3lEZ<Hj8bjHIgMOopd5#x(nW9jfx;jzN^<CjTh_%|)tZm9>6fDTesf|v;*eQ8rJ-RS(;|$L78Dg^CB&pw*M;l;Yx;f?'
            'FT}^(&q=xkw*M^i$jle0T2TQJKWS&i#V^rH^mfu>^)f{^qmwn3cZM8pu5lz(xsj?J4^9Thu-<eIniS`S_HHn$Jrk!VN5ATb-9-|h'
            'k}U~12p3r7P1QT0vPI?1Gs1##{`a?W$b-R#0gt#*X&!T6j&J?(Oo#j8`K0P2-fBC=9L=%>3d(3SteP1{abpl0l|tdY0$IzZ(Xi=&'
            'k@&yRQdOepz~4<(fugg_##&J*p?g_K%0=t$CXD`(ki<=q*E%x!Q7*9%IC<4mHEC@1H*><bIfXC5>!I#AZa~1dO00$$hF-;`n&Fir'
            'v#R8N9$j%cubLKa>b&f{aizFLxJ07^;GMhqXfi9++7WY)kiuZ*XQXCY;Y8x_aD1}Yk5JuPT4)AS0|_{bzPLvrO%_bLqq1|1z;qVb'
            'q^ea$<7~+haS$qV1-b|Nns9<`X{pq9fNic*sRK%}j0l{1Bw?iHdyyQD&5$FX6VC2{UKL^1wRk$^oUS0U-`sEto@In##2~E8NPE37'
            'S3L(nQ7E8+F@W?}mKmVjD?cxl;4V6Atp_<{s5`=3LU`&Pbg~@NlSrj31AWQZO-K9#0oCMbIA-G3D&xeZ8u3cEGryj{*0$L^iKfbA'
            '>E>afoY9gzxzxj}q6|}t5Uq9$tM8Rn_sZ7y;&{?(FSA0gLIfxGu^twClmk-@NEBa|AfjVbxxM_eBskjfs@&FkSnN^fE(g6gai%Ly'
            'Gbdp%fQgBQjBWQ&VY56}{_Cog-BijX7NWB)jvey?Y2#$S7Pg5rE#m>lt0>!E;Ue)8$GS3e0#jxVkRWCORV(O{nsb>qDV{W%MlnYe'
            'r5-^&je)1!!Cii%f!Na1Ye~}3svsYZHFz*p<Cg@7;wm~=a3Pu$td-hf+T!H8FRxp5{lpy6*i$O9g5SpgZ2DoUTeMpY)G|?EXvk};'
            'JjpFRF5(lnWuS2WT;7-hnc>-D1O8PUMS@pSp~ij=@N&}MC1NAeb1PpJr!<aYITw~f8YMi{49#41gTp^?CsY2sT}Q?U3|A#)aSANu'
            '8Q(BYm+NH|YJT4d(pY6lH>2RQ(7gpZglnlcdsYmc<C~!v=MO7;D2dBphy3WOzPXafihcOG*)bPAF~z=-)ufP3x|G?nHVsalC`^0&'
            'NhO9hYc90%e9(<$gcYl!hADwIG<Q|lT;S$T^)6PKAk6Fo)nX&GuX|P1I~O0!j`>rgkBQudfG!0EqoHnLN&+VSVTc@)0X%K=!_MJg'
            '>T^n3H?v4o<yJ!Q1Ei=-k$EoA$mgPOHCJQcVq$1%sVX97^vjRCU0bTy!qud%5DB>QGe1JFR(b3Lv4TAe4Wr=?6sHtoUDzZDF=V-_'
            '+>Z=bnul1-0)0}kN?Gzky80G!tx#>Uk#`e9k1^H=Dn|uMYJ^Op!hVvAadO(1r42!{ja`*B$bM}@VKe=+TAQ@#ubNL+{b&g`(^TsJ'
            't?JA@s;^FWv?=qI++i&buDZ=cT}?>>l`2~~f4;^KAYG}<X1$5*O<doat@>g?psq*)sBm=^&qVW6VHlPFuU=DB!b5p6l?c8on#Hgg'
            'jw}C|f2jyHhI{z~mQ|PI{kYWDN^<E64UPHr%un&POc>x{RJ(OAE-y&=)}PVF1+f?CNqXB<Ph>`LZPx>&RNQqR^v2jzt^hK1Xd$*l'
            'f>cWeGBL6??9nBz&DotbkuvaYjN?%=po{VRk316>L%{w@E)}%aNCnO#c6n{xLg%Pl{K$!ky_hR!Yklg~(h@EESSR}!MANw5KE`v('
            '+b2F118j1?ZrS|yIi@;Q6aibejjuT3Pu;xCY?FZ6C0u9%kR5$N7kd*Igc>%jSJ~77)0SbD0v$*c1zm4o-}-#x9ghdwG)A$Tko(JQ'
            '=o?t$^HZ|vnClPI7p&K{O4gK*rfrh<##oX%KkJE4C1S7T#_T-eRd6j+=LWju5ZdBV9UUG?19akJrP+82KO<FjY3zq#?j4w59qBZa'
            'SCVAB)|bbdm9^Qb2Loccx)L{KVxykWnxZ~iIe#_t?iy(8h@ILL#kKw#I$g14GqX^XEPs@-7b!9>(pi)DdCdFRrDA~{a%F2$XLOq2'
            'Y>$34j?q=2+gN7P_D>DRsTje{;oWTVBc`2(8X_HZ*at|1(Rl<@q0eJ*HkA8jaXMYIszc?c${dWx;4=yTZ!oT|{$-l~Y*2tyi2sr-'
            '*tiX{a)lR_<B^(gm}HwgLB)4s94E<gk#T1{mWfYPvZzKsdMmm0b>+~iYpsfBIZ%xgV#w1u<nX$iK}Us;T|`jei@-X^grP0nZJijq'
            'Mi~D#*Rom`mm*L7b*WRPs>keSsxwjW^&j1=6|eu+T>Ys#Pw5W{wDt7WE780@yEsuUiU+4BCw=AFey(c>UsFz9^7&?h>vTA)wt1x|'
            '@9|zx_1hH}LLtv@bq{Ob@G;4=4LgY!<^BX2nX_EmF~W2>kVLa^<4-nE)IoI-oYWr&93mM+@Jq%_`qF*OpKyxPY&pmJ&&Vt|{WkhN'
            'cXYnMe<sHf29xr8sE1RHFk%DlOPx(DyD{MhqV&FPv?Q``it3uon--1S1C+dHt#+2h(lN^jNK18<xmJDKybm(%*}3Z}`gR771iCv{'
            '=4VO3sIpEYOfNNTuX0;2ehKqt(g(HWSKg<t+o-y5G__%kLIHI?@~Fi<IF*ybTQyx5yx`4|1<}&CZ=_I{?DAqScdxdJquGdeRc{wB'
            'Nf$Z<qFeGcD(2N{mY_lnyScqVHpE<8h?!+D^3_xKQ={aGj?U^2b_Zv}T`ixg>TNa*{A(`q^VCKWpv8QVtyFqcSHB^lOa0|IMc*oG'
            '%!m3K9VN1|5rTYuK-3YZ-{Y+HWe!^gz;q=$T;(@#_2&AwwkMSML9JC5LST3AKYV?6@BZ1@>6v@h8}{9!!SP_|p1nIpk&4b6*+S_a'
            'xT@V@sog}idpS5dLbrY@{Ep3l7-6UeD(Y-+R(HQF;K*)Rf10u~FSA?q?Bo<w8Vo@rL62QmVql<Dzelz?@1OnBKcj@wl)*(_m&mun'
            '!EqmGJf(;Z=6;M>re(k0tN!p;>gP>;3Za_<TM5<s!O1VZqrssJ>A-eDL@J_Nk+_<>7<$L?;QSnbfZZy%rEGt3^7iEPJ%(B8TMw>~'
            'jw_HJVq8%ni?Ps~48cjp%ODdY(7a}TvYk@JVt>5Ebi{4t)<NB{0{ZZ*cXB>}S-Jepw}X>I%AjCEuO%#KIp!?5q#bDC<J&yR2=W#z'
            'D<$1UA`HKw{poDI>Kz@u>>a!<evi&m6~?uyo*t^F8U<R$0F|>=f7%<@+Je=nvL{D_w|(~=M%<j8<dd>0iT4H{z02M9AGwIQ`&;|!'
            ')820DLAfVdZJR#_&Tk{E2ylrFe+FsyABetjBG2wWM!uK$$v-}{k=(SQVH?^&pWo8&FZa7w@Xv4OKh%!hy!`oJubMya)3}Sk<quF6'
            '7-m>b*SY-nfBh}}t+9Re!<s3kv$jzc-CUES3s9eHtUe{A=E4<uv8p^S;|wbaOgO;G%Xzmnj?x*SaLZP1fuRDI?1f2pwG^f$gNxEt'
            'l%bV;Oiqy89pjSD`^P;vOb38Zzk+TTW3?P&g}Ju@dHlC9nzbEI+aLzt$RHrlenFVRQXn1Ru))m@2D7OOC{!c`)MHfgD1$c!Jjq0k'
            '6okr{Ox2_naEL^lF@<2rRFr*@QEGEpN?<Arhzce!GP*7WRjH2@D^v4Qm(sdoiu^Pc`l;nt#ZZc4EGuc%&2Gv<+QefpvLU*0@Wf)G'
            'L^#42PLX2nC2oKWQfh?n<7B2&TgvL&JA75QL#b%eWdceJ$KwJiQv3_kX2sd9uCS69z?4BcMtBa{@nTHSMa4K&O)ThDig7V6or&F-'
            'H=}IOH)(wBI8_*5H<V3P>V~XQ-RcG{GmI_iH$0I=*2t_#LpShYLzXTxjgI_Q-lUq1SVu+D3=BpzxX4uQgoEX3y(D8VQ2a&JHZ3X0'
            '{)EH?HGHdb2Kig9KIZp`2r~SVVbclYdwIuzGwslyO-J9a5+rb5&vy4(;@R#~T6$xF=$q1gBS1Q!I@ysP3tKEBq{LL<0c-zR?|<7W'
            '7!I+B0NN^Y0#+JNmVI%h5MaKq*<2K8tYvh&ln)9=%cYp&ckaDZ7&Qvl&RSC3yq@*rlhW+8G_xmvq%l$dLjO9~|8VOq)2srt(Dads'
            'u{T7qABu~7#a=y8n2n1eeyNLrU#=1x`GQoob}H)xRtTPTqfS6TT5Ue2EtG6M-sZT3xtPmd{J8rB|K}<G&ojnh3NMI&QnJ9h9-@HG'
            '%00aVSG+Pqi+sga!523`3Kpl-#I>MY9E|ScMAe&#?@iFp7u^7Sf3guKN~ukMm6>E|=n+@*EIkLNc^iy^%t$M}9)iH`W$^kY5Xl*b'
            '^;}Bq%J*SCUR7m6RFR+2syX+P8Rq-TC(UKX)zhZow&vqaDrWP=pYoUV@JrnVIp(Z5OAK2})H~JM@Q%WH%2S9PmQlEz(Qpgp^P{g<'
            'A3cq=xgM$2j-}@b_N7*l;t9}}d!SmkD>DnEj}GQ^U#!Y;!;@S<k{l?7A6~@z(Y+Ut@ZbrMXVA)rQV)t|A!hrCFja`7Bu!3ZlK6ne'
            'E0hbdgbT|ZbF_k>_B-}SXruC+C>&B6Kopu-N>n1E85Pgb$K)b{2}H=39+vA%UwxUzQEErF#runJ0h2D{N9B%CJQe>$xel|Mn2(gK'
            'Y~p|)k$=1~z`_Heo3dx-?IjtP<ypmUqgY^vvB-EYwxx?UJ!<SXb{pGyC-ssI2Bjt%>p(zA@s<+c?UM3|Rcm7qoLSDOjun3sM<iv+'
            'G_GxlG82r4O;+t5DX}}ei-&&btyq1bimZ`}{yIW*$D{?#K$cv0P=J0j9KI9R#z5EN76=~J_44o{8z(x!F|kUR{y-sAoSY#r&Ky36'
            'wttGd2qMm>fQMQ2Lp$!%y|f{wiFY%n`frhr+mHvKqr-gdR2MRbZR)Yp&n;&4u~3fZof@*GEyF2gE|=mOvP-?>S!hrp8p{@?kaMc>'
            '4pJ8)c*B4%Q#bt6Q-6Z?e)em-_X;%kMuA_$x8xx5JLL=4lEJw^^yl7(=57ZQ&XbFY9On{-y>iO;&eDR~)a4ypqNFAl|D%eay52GX'
            'j~8tqk6tumfmKZMzNie~FB9JC3Itlp3lL8Blkb7OXE~ve+g#Y0{Ik%$BU{ro2@X>A&2oi8_K~NPi+5t<*hjHaon;-8Zw#ec*A8cf'
            '@N-*vFO1d&ygTS{(wSx=A^S6d_P55*hQX!rbMv>!56z$Vf9t@%&d*NzbHmV9aFl_@Vj8;a?MG-r?>eZ}FPeLfWiVE$7bG98So?m('
            'Iib`AKzlNYh)!8=EMO{*<kXFXyxDX{)j*=30Hc2ga1>wxPf`Td{{fI#TU3U{M)1fHKqtwd{0#DWk!Y;NfQoCQDwxPIoYIxYMhCW-'
            'Pp0h2xY0#~yw=AXrVwklV?v`Y$o{(}yUmd3I0yl`<z|PJcIbk(8p^(84&A1l3vAEnsH6`QEGQ|Oo+B$GU@|Qu?~kl4&&kU$Z3uD='
            'FD2D6#JB_P_0k+6&s!1iU<qrEF4rq@bb9*soKtdmNN{&7OQ)p0JNY!F$I8Xo5pXM@5xHGT=WR(;Ug|W96UsWZy2pfSFE}R=Ww+32'
            '+zUAy*sKAAR+(W)DkTTaY2y=ClA)vBaVhZCbszS{CwTBsWnrohYvBBF>#OO?X^|;IIZ74_#RB$!|NZx8{lgw+q=%mk74us^6Izh{'
            '{Y^hw_c^xuWT=B8FRC<=m%br8bkgOh8G7^41jO!r(VXj3FMcS0j%WwBNSktBijCG@cH_sZty#2Y&f#JcV0>}8FOT2VH@LZb=Z*2k'
            'u8qnH>SmnA2OVv&PSFwtH<Ui#9kNn`fRRG8Kf(Op@>cKp*T2;xj<ewDx$$%gUuRderc9c*%*OlLF;}Hq)AZ=#T&+!93=aekurJzw'
            '5R)a69C3I)Y>D<yBF5y0_d%+3wF~s?ME73uCuW)SqoABCn1Mx^FItQMB(9>xkjg$MQ~(@C=8TYB(#;o70OQXQtR&tYQysOv-TmEN'
            'nwq?P+DkK|(Mf!VNs|(>`$yT%VahR>FG62#OI~0`)}0Zg7PqDTiDqh;vTY^Dn{x3!h$iuUD)Tmd&_?X*>?1Dpn({Yc(woVe6f;zs'
            'S_4WJr~VuKJ%GP;D+W9ud%kNOFI6toP1U~H=mCKa#@^0RoVvZ}#t;3p;W+4H%|kRTCN77q6|v;`0gPyJ(@q9h8*16b5j#8L>2qLY'
            'kGP#{iFXAenY|2rA_cdEWD-QiCO&P6r!BE(iVkkdTicb5e=yv_;d$C^J{N6%*SYuo89YNDp8qTU08Rg(JkljjSYO)P)1RUwWBHeK'
            '`|6LW*M;x#P!k*|Lu-2O$=|e=(&5kB&*pK2u8Q_U`2RI@5sM;X8J0Rfe4GhdB#e2O>y*>59?4l)&(=-ikLzb3aS2EVh8$?X{K-TH'
            '@h3e|q6_Vxv>AXY7<?ZM@0G->!u#A3t>9p9%k)^3h6J^FB+2r2T#S57V2an+EzGLIlt5E&Mp0=*YAd2hG(x+N8w=?QV~*=%A9D2~'
            'KNNLw7}ghz_K(J4)O9q7NV!uOkG;@pZyf@5Zhnlr*sPd_?9jn;%y3${&*(d#2PEKby4C7c&bojpu=tdz1bXb5P~1b!mj65Kqh<t;'
            'f_?KH<Pjuw#0{Wct^$-9g*1-mqX2Nl+{Q9t+;cuF1CAaY*lUMKlk+$oshViJc4>@Vbw1(9tqF;&tf@U}!kzX(4wx?lg#-kKbHF4>'
            'C>0~7<cw3beJUAJ;-$)0A04~mXuM1kG~!{5%Gisf!V~%)CDgn1!f6{nm-&&D40m+=iw;pS0z>e39xl>ZcnKv4oWZP`{6{LwviTn{'
            '{xP1>pHg~;^>n!-3NAq5r_<mA$8qp)Sktu8fp(u_GWRTkNz-{~=<XecC%GO{B*<hr2caIJ315s-(1u(u9S4D`bC0!gqOALb6ocO|'
            '`By1Sgn^aj2VuTw30|jizi?l{E*!<<S<wR?03N4U_a#QB7(q7-cY!BPxv<GUPF*Y;*y03U9txbH-=<dbl1s|2uj3f_PefUNRT^&I'
            'j%`qqVYbkXmY%dRkC%=!TB0?U6f>+5&hkz$;ag|4?1IoJMMi3L6hzBhXapc8=!=2(1^|`7H*{Q;saUu!tTc07kRie;!=<Gr-`yj#'
            '$082HF8unj#S3JsdG|RyGIP&B|Bi`D>;M2@#;xffPz--+YArw0l6~Pb@~80`(G|LxY>~gThjXVBNxseHQ86&NK4ZG-N>Pq4i3Iva'
            '8Z#>|?lVsWg^f^v89uWm;Yy>iu@FF6tOQGN)ZYg(2G1q>0neg85C_%o_h%TH;nbTo!|7Iqu^A6QALpNv(dsiy+!(-W0I+Sq3iU7G'
            '$pDl&vDiPCc6Yy)sOO_BBiN@|Q~EZR;alN;ZTprNmU?<#vqVd)ShK`GcQy7@5Aq!ctH>Kk3X)NE(lD1lz9MZ8U%wDUT>MGRNU(ak'
            'pcYL^l7}<`)Aj>&DAtA2Vln5Ddg!JbPn(LXN<-gYH1!d5ti$ucpbg&x09|CEj^$IKQMOj+1H>GMaq2f!Q)jCN=e1e%^H9JFLC(`m'
            'AN3Dk_s^=5O!G}4QCFE<8a+}EY}Z$1Wlv2P>4U-9!Np+czJvq#76u=_IqRRlIXyxb;-{sz2dD7n^!*81zo5b1kK%jrd{^_Gzu+{E'
            'h!1Ga^A5AjVC?`Pc>VL?;JAl%C6O!eeK1>jWP**{M4Q*vI6B1)m8z?%R_mb`T`(8k2w8))t)joeoxavnX?9uhrt4+S>xSco-Ux%3'
            'CgK|NWME;52?rNkcf=rL+LyAm11~s1_@TJji0;nS?RZu2ESv}|*>TU8?WxX_O$9vo9usZGhHJ3O3dbIhXr`HaWHqMzl=OY#uOz6d'
            '+3ImA*D!$!EhB(Xc}~nj@?DV1TMD0p;bH<_F5@fXRg14HlwYA?lVibuBTg%=_#pSY)2J@6uP!pWyDlDe4PR!SOrCP^Fi$jZA4eT!'
            '4>LniQ<`J<AJta$5EY@!>w}=nj|wA_t*<L5NUplJ?I~X>RU6ghqapxVkBN>=${!F7rFhb0?GFD-?WXyb^CU_1u!Ah?a|~~sY>Hhi'
            ')M2a)^`TS_c<d=7RXMb&@oWFAe<X&zH^+mM4q7>vWBC6_to&?-F^rT81gka9mt2T(lBcyP#}lLHp1{;M879KPLg=57`o>6qV>=VE'
            '((Wo$L;r;elO!nBg%j#Din;PWCn-@Oca_3OPGVMt+=Z@OU{fagDxd5;1JkF6;u|ko!mzaJVzB8$PEH7MHD)yoR?=8=rOAGLcAR~F'
            'P&N*>=V8oXdwx=kwRbm9Qc(Y&eQm`bk{$&)`6@@?*Y-i;py1CgQ6AT%sg}WtYwgxyfKwyROR~k|)HmtCeoKj?SwI3zUFd8WpkGTV'
            '&8U5ONksNcG?`N29wjP}kzuP!e$>c^^{{f}vn4)NF<N0&mHod=*DuWzbEW5wOvbkAX|)PfZmqZ*8j}6JvVlgoY~^1sFo3oSvEZz>'
            'U(Nt{1gfyZT)$jg$L0x-mzSa7Wn*_pK<k=-)B&>P8aFx#l@jZ!E@<S46_J+fxsA@%NM57TFy*q_By`xVXw5I(d$M7$`M#1|+Fu7j'
            '%Bb$N(T7UYnvf@4Ra1E(P(P2nP_|3nm^I^Y8s)W3KncjGK^>dOTLEib+h`oeA;2(l$v3sW9kW#JeE(Q;BlgdwLSHh&07U`TElY+F'
            'x;p0~X@8SO-P&YeC8=YsDx|Sy36fjqY5gh!lB;QAemJbO$BgrZLse18zH8KND!g>o_0HDy&K~Pc`HSnnJ*ntzXEL_>TCX^b5Z+(8'
            '#+Z@|70q=nDn#<ErcM4-`P2C+O+bzfPyy5{Ku;=bV|H7jS-WD;I?$a-k6d|%ueQpO^g61Vb~TsAt@$%Oma8apG1h+!CX`za=?=Ge'
            'eF$wTNUY2bCn<^}{sk{SR#gP|Qn$LI+mTgen`#h1huxYXs}B2B$<QM2lg&9-QD$4&!bh7}G_l>f#B<cZsWQ6BuC-TQ)t^zl9=-S~'
            '3XGg2PIh-nehGi>is!p^ZoUlfDxh7l8eJ*FY_!Faua$SX{^YIWkL7XJ%adPYeP6b=-?OfATI|nVgCP(metR17b>Q!{2#`sbNFt{('
            't&j>^qI$u~$#b_di>Wj>y~lVLHiR8toDZpd!^lT}Zqx$MIz{WCY+|yKr8Iuwf2WZy9+y%i3uP$R?*9dg8mMo-^v`u6E?Jl*rz$8W'
            'q2U?{rlvudKZPZg9hGSSZ#h~jyhc{=CF`GI(N=erWtrCyvCFb<Jr&1WI2qnercLsfZ`M#u9mvE6k!%V3F$o?mDby@VW#k!umtfwi'
            'G#Cd0c*v|#%;3hsMHV%tiIy~6YWNv9HGE@VPl#zer|y!6dv8RGt&hompGA$e8hJ1NBnPPflFpF(i8k^<G)F-ez;XFR3sR|ZKab*-'
            '?t+Rey&!{DdwFX<Txf0NHd;&yjJGJ;kr-r6V?nlx$7BLvSJ1NBoxZ_ot~2izPTjtcxXs~KlQ>*bf?*ncyK409tB0LWF!%#Hx)|UE'
            'Ie6V-=9P<?uV0L&?zEoyM(de)vtTB}$AU)}#9xqWn&X;|jWuQK)|6GQDU)leCgb$#x8Kd|eyAJ%&-ne%0TUN#w}ADZpE88R(1L^&'
            'sh|x(k+(>1<7`9o#%GZ3YU_9@yVGZx!|RO`kK$k5haRy-eCD=3Hea?{tUvj+Gqv@x{l_)7$Ey0&>B*1P^{La4AFJ$BrzsZu70eI1'
            'oAtc>wSoa9|C;sUs&q%8%ta+du4K1oaWYP`-ziTP?TchNsg@<b2RU!Xx!qbKPgPctT)9S=c|;g(3EOP5E<I@>=2qT*%G-7~yTzop'
            '0PAk;9ed5VY6QJgIQY7&_}@pF>V?Y3x!4a~vA6ZbG@mUfyDf5`uc7~BHQ8WFEG1OV$1+Q<)9?O?Y(6LtTZmY#{jIS#@<LT1L;GdZ'
            '{?pmALLO$--zwRO3*au8`_|c6%L=lVM@w`KZ>SpR`01iz)u`9zOZ~K=wq}Yp9V}XrYAkQ&5SB^201a4BPM$~`+dP?@vu3lU;d~t$'
            '+n)yTRei($#CPJ5Ya^2P6ls$(2|gxO5Jox`sn69riEM7Pt8isUv*QM9!hEz<iTd`x3>iykY+~bMok5T<k^Ij>QbOzc4rMeY%bw{{'
            'E{%HKD#qbd!;_*XtFT08JgTwk7F9o;YEq1r;gXA*VEU<N&I@>gmK2N!gRf6Kq4Mp{y^oE@tlQ{b<tk|D<6kS6+uHkQ$>p^hlM|h-'
            'O^Jt|zIowYHEAiUw4_l=N;m8pGFmB*s79lLwNvss99`yr0U3<4m$TD$q@(rZWa)I`Hw?p=_O6Qz&pVyuFOIZRH#^@cen_ry%G0f7'
            'c^I&<Os+s>6-k4Mtai_iZ!jmUZ7=WTM?gzbu5s#B6pU!6_uhj1#63y`SZzF08Kv2@CyUw0^<vETqFGAQ5Tw2l&tBn0gnI}0Otvx1'
            ')0c#R)dD~oi5t*3XJcP{j1x36OqPkz`MI}!G4oavOo$#NPSF)8KY5tYq+Euus0mUMredIPvB;rXVyP9Q5w+u=$~s!vk}hIv926?*'
            'V_Mna+2x9L@IZkpc_C7Az9Jx|_+~;Sa)A)|A3Ii|LT#Dw&~Y-QwB!0N(=5wGzA;>+w0!?*TS9p(&Z<tM?mu>a+F8VlCO0l73fb7B'
            'Mdfs+9WH79^i-8kV2UKYcUj$GAC`U7iFQ-%=aCu;Us2^jGa@?_>Owo@y1Y?sc}f>VxmMVGuZu5G4hA(zEAG3KCx4O@T|T!o#ZuaJ'
            '1XDMjSuo0f=4-2lOt<=R2~PH(X5|g^MwMXF-;xt2SEE9<Z039%-nUj_*KoG(N&rUZOmGvQGaE`H81-opffb9;qq?`yx0zR|KR<jc'
            '^Gi4#BZVl(>6DD(#bh70C!73QAvF%)%)`df>n$TyCbDOGiFU|}GVa_&rcIdnD?69fzRr*$ziYN2=ddXLySFd;e|WklgO>e}GOJLH'
            '96&$bR+rcxUS*82XdK=pq{TOUb!t7_Dkg2e#bBr`Ee}d{b7amv>#Yj5mK#n!N;FAYn!F^*FLscc@fsdX?D+)r;Eq#6wctk(86&E~'
            '!?;m*W<0}#Vh0lI6vA8hPT3+#`3??}B{22>eAXV{dPz#*U5spMj23fs5=nGy7~QlD-yMsarI$=hRO8$&HJG+*mSZwd4LcPR_Q+fp'
            'MwlJ9Du>w>3fp{W?zP}_MsmG7JK~x14KiSyQBe)9;LB<2{STyA=$!Sbcq;z&TO_V0%Qk&ju^b9Lpc)=IXF=kr1y4*e2>6TP0trZQ'
            'sZP*imf@-85lri3A5NdgbuL_$;Ll<o^#fr>67DE@hGtuNEw8_->N~Q??QqyXem6Y-Vt%+0Gb8=iOn3ogb~y<$QjultAk2tIM*w-A'
            'Q9%e_cefG3+dLuo9>s%@h=gG(xI36vAj~MaA&76f2+R+Pkr+cMiGQ1hqLX(mVM1kF7%->5fQC0X?4NTb?CTdQ=bg&;Sm}p?wbLl^'
            '$8rSA%$8h-Jhl{#@ogN8DL=_76}Pb3R0WE@N1f!l5(B!jG?I>lK`M}eSWO9tF$@M1k5eE1AZNLt;8G#&XcQvM(K47&A8X_Qs5z@N'
            'UE-ILV^g&qS>V$sX=7%OE9eJP%ncbtbAJUyP7Zy8VQ;~{_=Lm#OM-W_{w&0*NAOHmDK<uT9-M|_mKjwuXsUD8P_VQU*_Ebp<Bw6U'
            '_$cJ|()hH$@L?ReFd2zu{}4Hj;8Zw<Se<PQ&eXrDc#!qpe4SSrst^EFt0HAK2D((gueL0Bpp8$1z+?lmDmw}SDEaH5Foef(%4(o&'
            'UwL7;QLPoJg0-p&!&A8@dR&NO$bD&b>33?IS9FeISFO)gi>PuSU0>l@>o|?l{+m<))zx+9f1B|3Vf(5g=tY7h3g#gzZQ%={OXo6S'
            '!-8P*fGbJQy^o#pAWoI{sx95E-CE=9FtL&-TdX$Rl`Z>!xfq=F58WQsUgIuS2|QUU>oe+UnQx7<S~Mz`)1u1&YxvrdTe_W9(VXm4'
            'S<Qs(+UNB3OAQ9*4-(n%SY4q4s`Hd-l{tv2j>KnFh$fQDpUDeyRg@&s_*$$bU8tm03TCb^AzfE_y0WbFXV3HT0@NGKeB;{GpAW|B'
            'D%P7!ss`84{#M-eHBHMN9ZiY#Dq*ULXI91R-?eTvB@hP+)!B#^Z2|xp474U?7ugE6DWT}Yn9r+PP;|;y1!Tf4y9$5K%K68Gv-N2F'
            'Kb`(^wK!8%susCRwFS8}-TvOyR>f1L8R6Y*SqyiatF5g{-fHpGEz7>Cc<Yws5n>!Wf`j$MY+K^T`ggGvz9SmBRl%9%FY}#d)3Sm)'
            'gMu?J0_Qf5N2Am)XV#5kXPbLBZiDei*BPurZ_&haz9UJKPHiJ*KuReH)p=21*=o$OcavzAjD<J13cUD1s`_l#l|>pcheK1Vq^h!f'
            'DJ|>wl-$<Eik_ZbV7oGhn66k#uEx2RJMyAP+MVWJilOg1hMsGnN~PfdFdhwq5>SIfCP|AJ#G4gmLG(G}X}hhn0dU6FLc(~7&<%<x'
            '<qVj5u?DZ!+?8zZ&AO_P(s#@Vl;o*^OaYSdI0;5p7&yIb7HgJ19uT@Fqbeh$s(ogcgwO5YjUlP^pY&2MxA81)Up@S0++<A~WRQS%'
            '1cSo6{KL?oxTr$g9^WA8a!iOud`VdNo_8+@g~W1!!J{zpHWRAJw#<ypD8}pDR-<go#Iw50LQP4>A}aPVUl1nFCCd<Hkp!_=R(v>M'
            'gWQ>_#zn=S%!*67JyaT&Tr$4t&r#*r4_U6);a?>K7CvOGxrJ?`%Z*%2(*{RMC|epnw?`@jwNt^X8bq?E{73hq5=dO>!=aJ|Nmels'
            'S@Z*+EVOb2Y-z8~KvZg5QKFlcr<f8ove2_3j~}-y?<Cvt^;TNNKgy6Pc)IA}&>%C;T)RMF2|<ds^D`qrfO?by1?2s<%v{5!8sa<}'
            'N7TD+VK<CPOQZAD_b14WX;Y}1_PJJ=BnK;}Q7?|sG5dEv$$s*<{7wwxL0yUbn6#+^!KP^Zy44FH`q^+gOF)C9S%@FYheKj+SMf04'
            'T9rH=OOn@YaOoK|p}adgeKUAD81@hGua`%s2XBu+ci?ge2ZLb`b;DA4f`+ZNmhLBGN)L^i)aI_N25;2dod!3iS+#tBr%|30V{B0V'
            'JY4~=oV&;%umBrSSR-4yRgZUr!MLa%NV_&4Z*g|DZk{aCw12+DWpah`YhL3u-z4E)UDs3|zSi~9L*R_~1~@jC^1~}rN*9QavHa$a'
            '$l`^&kpIr$UyWsaS<(^jWO}rCIt|9aihz>Sjq_7+vxJ72-ozX+y^WXQgiMlX8w|2k-3^I7OaiBT7jR9KnOUIojxW54E(Ibmw>rug'
            '@jS@#$~QqOlN*v`b`RrcF&aJNL=Fg_(^K*&;hrQQQ*RCu1?KjExuZuxG!B;&jLoyemd9h}O<M%-2a9fagY=M>Vklroj`!T>vZA0f'
            'I<hE%-+Ggm94KKUXQ$BtxwJ|d9NM_Sg&GFE?*uPh7E6?S*n2lng^p4m*bfpc{thNB@?TI}^NWVT`a3|v5S?5o|34)sBO-}La}~-('
            'qy&cZ3?~u=gYtSfz_|37vxS1ViH8OegmgxFK(0k>L$n=?NE?L)Wzg#Z_yK+MQ^Ot>Gz$7xZ)t&cm?|!90-L{Fz?X}I;l)}15NT_^'
            'Gs<p4W!uYB(xcCVImlM4g>-}Ru;u~RTHfL0VzoVaUJZ^xx1Omb<n+hn4sznJ&(QG<rY-FWk50YDO8!ah!h<r{pHrQa{OdprV1H*J'
            ';NL1vI)<F1HwdUBpNGp&D$N>KOQ9q;4_9hML^zAk>~QvvCcU&IA%3qI2)SFjlwA}VIuN&})UaJl1^S_7I+?fFJJX#;EzqS>ze|*C'
            '>@B)K;g2M`DO7nfi#gq+LRGAMq{}`*w-<Sadc4_zCnWN-c*ApJ7qkDs^s6u#%HYqpQ`azwYx7pun02)wh6>G~1r>xsS(_#A0-_OR'
            '<E%0p!`^l2e}M!hfb=r`eQ`{f5CchkFPcxp_umT(u8xDxcRN3dA8@b^kA&IU#S?vWdAvUhBT8CrK$G4TYS_RNb)QZ;uU~uHO3vuG'
            'jUW(=L-Z(>ES>*wryixuRSygLcs+yHDxZ=NzTTDY*SpjGHkZ1OdpYS`ZS}e7%o*m+zKEH?w$)Hi@^dJmp-$eK5z8*Za<tD44yg+e'
            'sT<FX6InB)bXRhf7ux4KcO}ax)%V;eOHpoM6ua08LzDw4&vY5e5}wkaBgMpPc_x~C1X_lX6xjex0xblJuUzAl(U`^J7#x_TQAHUC'
            'dm*lkEpM8g4L*(IQVG4uHEjEe*$L06vf%MGmn18c;XXy77EY4-Tb8{=E(jtgOC2w<)-x9I2_kg3!h3Ws6Cb3UE)p-YQ<<r91lWfq'
            'zyYht>aLO?%iNPb7K5A__A<0|C^FR+J$bH#wCICGFgO?<{VLA;f4SgnW$*`z3}<~16i&{SKt`nugpBmag{7VdJ+5MpaxBajWqc3U'
            'z=k*`^elY&cR{>lTmY3|8AK@ebEHJe@CxH}^W>_;nH$-XH}xVaVx%Nz7U424jBh%6d02T8bMxWek>p({Tc|h_b1Gy}%)eS(d6JXd'
            '=LIHt$U{9Oh@n{LfKT}gxoA>JCl!$pDo<bK+0xlHYd<WjO75`Nqg?_NCqy=jcp-v0*~U<dA~$R)FOw;-CXI$F4%(97PGr$}S)3EM'
            'AO29f3DfX=V$XK>cAo7%)!r#N7ouV>(m1DGyzYUh`A7y{(0o&Kpl906r4E)Yea)fD)DW!nWg?3x`ItX`_@KvByG^TtfeKk~88=Ua'
            '5y&AlZ8xT=j%=z=y(*>!-=GR%c+OAy+D-n>mw=RK=e&2^w>?qiwwGZmE8Hu46MW!&c>3<VBNHu?CDoUwcSo0V1;~t67{xF9=i?F0'
            'V5L<D+Q@8p8Jxv!K7QLRxgug#>SI%Hlu*&{Ldh`rj-O<16uZ@lCZ&Bxu|m|D)K5eSor-xY7asJTTlF;^+kFlHXY~o43&%)Xy_MI%'
            'Qyz6CFT(s)x6=R3&i14Aw=O$?<bsm^Wk^fvM@tDt{Ey9Is~w*zpSCK*qHg(7Vf@eXUpB%FrQnLp$qGfNxx7;-h<98J!A9X~ubgwO'
            ';=WEf@0V_1j8^I{rZf;1@{QIHQ{Oqi{j2J%-d1#1E7u!P@ce1;P?YD98kFIncGyL~Sa{6_w~F^Q`QI9RXe$70NHpcQkEEY7f?WAr'
            'k9<t#{&+jCF<WYe0zx!ip(u)J^g6qh_p3~Z*4?2Nx5j!cEzIWZv{%11rH3u+2Jmr=F>lZ5jcEzTV{)lhK{gRkrk7-h<QQInZ%uPE'
            'PVVug74C}*H<8koLB}n?8_5^q@!Q|hyv&Wd;_Los(k#?GV2axkPb;#}rBtc4l$r-##Ni$Kx|0BpS0VC|e4-_LTgf%`TdPPKFE_u5'
            'Jmz%y7|sAKe)s+un1fw7`q9YddptNlADq12WM4O_aQaO*g02JOG$4-&%l!)^@HrXZn}pTQBdJ+)H!IyvxyO7BhnXPH7*@Q6VVIQ3'
            '?Ui<)DLYG6u_aOEWfRZJbhny>G$>RdT)FZxjeK*l0lZUFr<=1dOkFr@6$x7FT$IYusdg+=b$n_eQIj2}>X74XMO?fbxh9A<SANBJ'
            'q^JQPNBUB8%l@`JE!EuOsxIm7Bfa$6b;6y3BNLOCf(6FP8jy?O)vxsqS&x+!kS!2}kbmo?jEgqndOy4RjWLH(h&h|CG4qBpd=v<1'
            '$pTUlBF^eVxAx|{STtO3D!s2s{z&%pa~kQ48?s$p_9~Ca<`c6T0-IY*D+@*-<!@QiFFJA*XIo5bR-*jsKe<y>gd~`*N_4n9<+-0y'
            'h}&lqZ~F|}5u97~X494(nyBtv^XNo7F7eA{d}YY%+6KMy(8f74FvHFTz<29)IYPgHVt-xAT}{CWk~z_qJ6)vN`5)EeJ~(Sv%`>HZ'
            'FmkSIo13Bu8p!;6SNS<M)^;oiDfnZ951GPAvmR`(FMjEt91Y$I5PA*=``Wsf`8NTKI02%P1s(BvdFMxAFr#=R_z>RMdg<JoaY%fl'
            '_5)J}(@v@?WQao&GX`gaqvQVJ04)#`3Yn3c^iVz4`mLzePu8=Tb>n@hSXP-gCKu7F4qBcSrgt%X)2fw0o&P<8>(7)_EB_xJC05O5'
            '7&S8!llUfCQqqW957QrDJ`^&hI{T%ZD6(h$DQf8BlVU&)rd!3x3ITebH^T}xowY~M^xnMn!v*u&ZOOPmW88KgL~n)ux54i`i5d@e'
            '{h6!0Mp@&3>LJ$L^5x*@2o9_Jmy6S3?|;ULt?&I=Cs`&WEXwQHv-EJy2qk?04VA@b|JCgLpM~_~rK!+Rx+AZZ#GSwLICv<JD4HWl'
            'T0?N81mI&Os*wC+0GZ${c&a;=!lF70d3u*M!%dbS{V)p_m>4_pGElo#euPzDftp4BTnZE*N*IBLYlh4GS>h!)1(sKCe4>oG@oOzP'
            '{9j7<HN=(k{@E}6GbOML2gm)>iy>E{Dl6@Os&S{NoJzTSrRI$PRXI<U$pLGpZXs7E)j&YaaPu<Ux$hOad%*u!!?JAdsbv^;D!~7R'
            '-seGYaPmv<XmIEP=DoWZ67(ueLVpr_`!!!GwXKDfzMB@`ZFCb%h0p2ume*m~m|E2)CoM2natgZG-YTSTlr9@~=6z}6sLd%<iofvg'
            '3x)AEGILa#IAjg!FT0?$R!~T|w3&&+`7bvw*Ax&dUr^;@mi%*LS?bGWs4tP79;#vZh^(ZV$xF*Csx5@}!OB+F6(+GS@4m8OMRHp^'
            'p_C;*74Kx3y^@Kw4iAt|?p2Ov-`~=L<W&p$hL!@PjCW<qquryr;?k3zWI?nJKQH&&dspga(z`@lySmz{lLm@+$t0G%P{EHixg<V$'
            'm)o?L+gA@D7<qKlu$?0=Xo!t`u4)Noiy~dl#?jcH1&kZ$c(3m;Q8sIAJtj`W^0qb;i5rcrZy{fO4Y{gDp89Mm^!qhFoBWiSjX{iT'
            '#<aDOw)ce;>2s;kL;3Rb_iS^GT2_XR=_XUJ^=+cHO&i&AZe8HZ+QF%p1enjK`N^P9+a(OyzH%O<x@3ibH^#v%3&h2*hZkoTN5{Rx'
            'L8oElA^bhaTtMQlo)0beu5C*{J9$AwA;jcf`+@yYr5oLO|G0ND92}f~ll*8tM+<XlRZfXWF2fYoYa}$nlJ-E-Qo~*L-_B1@WERC8'
            'FG;)=Ik3i4iYQg_#nKZujptOWC>8gK$G&kv#$+{4DXE4huCK3g7GjD;yPb|W!yCC&RJtRpBDb>26?^W9n<cvM8u1w9)*@G<v*gOZ'
            '%XxC;6=2o(nvGGMa#3s<G$SKKi=(k$0+k)dV4x1i&NPz-tgiX<J}%Jt`M31@X6J{W9e4(Wjq~+E+jH%XLj#m@{L#}62J<YMdye%*'
            'Rr?QPK$-z@t$-9)n%fi(b+hpaV{eVWFB=YBPzD&X*4}gSM3eA`b~EIGvB8FbA|PcRqsJ(WZ<gq@BcbNApznR)i6uOf2R$?S;0mYq'
            'gA`0K*a*7)saq^oxON)G=V#u+i{{=uNY&oi4m@<=jY`@$lS}><eE7=p&Z)nZT{oL4Psdv&WC53A0SFy10xX`R^o<BxeP^t^>(eGn'
            'ldU{a2q(qtHUV_wTYo&mpN^3kl{*^LMf6?`F$$Fe({?1K$zMya`g6;{pM8UA^62q^net74e|o(yu0J&}5d?VwHX0AtEc&K0|EP@8'
            'gZ{AgFvL8WA)Zrl&GWvdl-w}z-<Nc5;wg1(MjrZbgWNSCKV1m{hkc*p(1EW29V(@|E9A^na4Y0zUB6o@lN_%+@>BS~pe3+GJ>J`4'
            'kf54Q$8~X{uKOU7%gS6@*exsij~mIDKOU_r#IJ6SG8eZ%^Q7=+X@JQuXv}{}9*oGoI}8!j>c}OvxH9L#4UGIoX3cea$(ik>=X}N;'
            '6!~$!c+Nmk%QHsk6*eBh!<b@F#G>CxJcA`7S~N=tQ(tnu(z9ay3cLu;E_$ELF#048#r7wh?!z|Sc?sZ2)#tj9KC4u0<nunN1t|Cw'
            'ecsNBA~jDLY)PzLo^$eNd7OZlG^j3us6;_4h*p(!QFedPPn^vxYlRi4>02OBllcY`TrbH2bmAoY7%ordu@!r}-zbOet@q@|=L{~L'
            'oLHxPc`TVQOzk!dMjaldIl@d8xBiDoa0ALj(^;EpxJhA~qgCdo&KKnz#u%I=!#Y|inZ~N}XRgU=Ofms51;DvxW0_6cKiP<0SQ`Py'
            'TAT&IDJU;Z47#!b$c^D^h0*v+6v0RCgV+;OmyO_H1L3#@av1t{ZW_w-g7hi?wXDfGy!U0x4roY|=JOwaeEJ+t9BzY60(crKf|^FM'
            '18w<zd37=yhrHRA$k|i@2B^lU3|HT`gcXguAP?(|>u#(&dTg$g87``dj>!lc)#TBP9I`n>x^h5PD>pB2CZrgg>!Ovz>TFAHx*TDG'
            '3t+l-N|3c+pv)ES?Ggx_>n%`U#Z*yd{I3j*H`CS2kFN_T^-44a(TQUsS$%G+L?d~0B}YMm>V;N(9b~GM=pDiU4zSxn)ZRf8Vq2yY'
            'lg_xv(m6Q^M_5uec`T2z%x>c(_{f6eEnqHMx;?`;5aw}lcGP0k4Lv%2%~hGuvk*sdl6ht{_CH0@A-YL9X^0o%!B~lV^q!Xyv>b?9'
            '>7#h<$c0lem+3O%NGC4DB7CfUy3IR-`L*C<XjPbSE(@2wwEco|WyHKRvOuw@N^?~)yw~Ir?Jk0akLlt%YPaPN<D@y)h5T#Ep;d;V'
            'byYkXidH2*IGs9Ao>enQLOD9m8jrI@axGmm5E?0^`NlC4YgjYo)XWli4Nl%<fo^S@X4}?PNtKMP*(vXf0OpbTbN}ND*a}Qw1L{&K'
            '?;jv?JM%^$5(NTusQJVb&Dy<jP$0t~l2T}ho{<#>W*8pJpwurJnDqf+<*xk346p*lauEknMutU2hmH67!@3sLA!{|!XXPn<X#iT?'
            'NUGVUv1oPUd}dHu%~<8872{RJuvHwOVk2iwj2kH+<uUALj<YrjZcTR;GgTF1RyfP+Le0#-DW6CU)u<0Y12*6>Od9)n9daz9WK{&O'
            'E~r$9nJa(YeBZ*DI%i#cU%7uVPu?ccf8|kXnmQZA2tGa-w=~DHL@X&7;bs}Kztx0Icjl9FOmSsEW%(PHFh->??FYqSZAx#>(EfsB'
            'jJ0--P9eIpAfV}kReoh1rNjrFGC=nv*mf5_6O3_dU!C>)?r_jQa}RsN9#;Ar9Q2KT)p6cv%F=F5YxZ1K1|w&)iGSlKbj88-!G^sz'
            '$Ago@!C6Zc23z^r3@CXjlgv+$HL2K5r5aUBnR&R5WL=jHy9FoHve~6(=IWX(f5e5-<Xyae=#>L*X&_u?TG76QOm^RgnsiZW4fP45'
            'REuw*z`yg8cFs~!bc`c3XoB!P2Y^met<bknk(*1!9(wi-f!yHAf-1OS)|b=wCuF{w$a;6g<2xX$`YX)pfQKdVQC5e+$X$j=MPjF_'
            'P`)-4q855%%rb$mw09CG%|O{*u?859Qmo7uq0hBRzlkM+PPvOBhs^uNN#{tjtj*WBQR>IFR;k*Yk9lUZZng6E{r3!DcXsb3H>ng~'
            'WsT)-O#vug6kb47zb~)Usm(rEHp`Tobj#0lYCR{ft%)Z0x->ynA~rmbI4ZBhG(B#f#jsnE#jr5^wDGAt-otOvZ_y|I@<TCY9)|2y'
            '=MJq&Nx>^DB2eLPd($h>u>7bBp-NBeadS-5zgm^y=A7=1wy9H-Bb0ANyZH(gRJ-a&OG2#6|41xQMoqk9f5fb78$}DwI?Z(%6;U9x'
            '#}Z`OsveC60$Huj3Vz(L3Urjlc<WGF^(Cua<nWf%h^bbkCF|17ai%#t^_lwJD$xX0oN^hdk#6H`9?#h)1GRNY5mYwY?`6kL5`;~I'
            '2p9_t*ya%hk0_d47p>PS<kak<fU}${BgdIP_Jcb=6*4{>2tfjWOOIK)bHfyellVRrJ0cweq5;|R$RR!*gS?umn{wniD)<&vqgXUk'
            'A0}{gbo$;s>%Dg|G_B_jPhBl96%w>o>{bC$Fi&NJ*9pJnURQKuWWv^4c}8Q|537oxp0$G460H{d)`gx@srfU&98d@3+bMJ=SzPLO'
            'R}Q+PNg2_xaL3?ejpBAU48Hb#Es`+w6(z&WK))AD<bLKOYAngKvTwKv@;df&_du>`9z=kR(M@)1Z303UGh)jfP~X7Z*V-Gw!NTCr'
            '9*21edYb`#v<zpi40x8wS$_xEZ-blL_FWt<=f0W$3UkaU(7!An&>#W|6)I68LEnTVz8Chapj(iI{tlN&AiF3TVKeW>*x;6NjyZu}'
            '=zGy}0aR3KuiA@Qwl}6pchkTh?$94Njy@Ab>bIr>#+z-<U66P{`GJEai^R`<@9w~ZgR{Yq9@>MVlMy+b<DV%xBMADqncqomphJ1%'
            'l)Dhg(`l(Urc5O>Z*Igb!KG?>n*oL;2C5z}{ef(#usD*ZrB_KUfBZ~{Rb~C<`vQI<L2{tD%S9?dVJ4iD$*J?+RCh-VP%xRu%y*tR'
            'KL*YrqUC%9V8!i-7cv8-T7<4ug5}S>6$>=U9CaB=JnmtiU`%Nhmr26q7K2D;!Mm5@*JzoMrx-h*ct$-&n#iy5a=0A%cwCZTOp=NO'
            '!VnK;N~zN@?Qfv+yq$Al-ix!NbY~p!E~cbB$;)b=Z$)P4DIlk#wbjS?p%x7RFY@b7op8*=rV>t6qLS&&s0VkselYuFsw?)=+ook('
            'H66~Kx^e%$BtX@SE8lKBu8q^|I&VF)S}QQX`#ayu3XY_W*9slI1OtrEdx!mlzStgNsr6_HXo0s@x88`%FB3qBU>mQVmuPm;S+5hz'
            'ht$HI8Ch*8Y|uuakMOyOFbD(oBC&#BSZuR@aNPe}!Ryra95vl5D|W~9d%(jT>ay9|*D^J5mDowDTr971C|+K5^3fIBP*n(Cv5%35'
            'o|n@YPdOp<jlNR!0&)FXbw5_U#SY)V={E;>9sS-PT?_PmN=5TJ|0WkVfoa_Pqs}r3b|mME=Uxz9OCeHK8@as7vvF%L(J|d&Qd-VD'
            'Rk>P9$-&EnT&>rGtT!W}4Sdu0BUv#Jiw`>MYlT=IFKg?CXg;k=pqPb1${W)t74KAU_SaW<v)l{hr*sW=^Gm!P)g^CWhm470L+A<*'
            '3k7dr6Z2;98f(uaLAfdpP9NLCm+&7<T@PU1f4r6(XQ}j~jyIV!p{qTz+_W_zN;sQ+{AN|i4`Ls@wWiMxE8L43jHbo*@~Kl98YEFn'
            'PP6u;YkIl%q+{i<Yxt<lvafnw8x?ps$un}tG+m-EV-;_b=#E~{id2FO7tE<v@Q8B1j%7+<%&1D4m+w8GG4xqKFHSBez*lRRAfQ}r'
            'P`+vN1A{~0CsylgI8~h-#?=C#su3NKlRqNeE7hV(`!8Q8T%%6bF3SeW3@wJkb0tR|I{PRI_22a{q*ZF&+5{^FcANn26L~a|V_qVp'
            'MfG+6cyKasVMO<r)4@UiT(i8!-~7exDxvZ|%kd2VJ-GD}k~bS~@fr&LedR|<Ky8lviHHB50>8t5d$F&}>^2Snb%G?hjj2;_nZ*(R'
            '?1MHPdK2z4@D_f?KQHDZSaC?L!<)b#d1|Ob*kaIBsWWgv(3xc;KMF=vaL7wN_zN^9nwAGGCVoI2dVpj}<RxmXUP#eV`05<-&!d&R'
            '_QuKb1Ft*89tr4LvU`w_Pr~1TXGcDMb2$psXun{2gL&WqC#T-$fHLFT5q}#02to1k=@aM6(3^sQ(b{shRN{TT1sLQ)@U>nY^-f-2'
            '^j_23ENKtV8e8A(i^F)lWLvwEnGrYGgt#xBJQK4wA6sJYd3zL)!4@-6d-vkq!Lew<?=hGA^Aq5Z_>uG8@jEPSjHuLw?(XjHuKD8q'
            ';Bfc`-_XN@H@%aS{t=?gUSTpg3$i1AP_`ldz`^Osu>V(B(dpa%2?t<9@25}bL(;yls2Jh0OOAhbA+c-;=zU2qT84j>JbwR7<=uh^'
            '6}nP=H-79Ec(LgONewn+Vm9mNuE;*hqhocMik5lNnbKs+LP6PglDWU_J6SP5RBxX@>x`a1oA~5RiO;LDyAjFbZKozJn)lYD(H4kv'
            '6!-!7<|)uKLA)c;S;l}cnk~k2H;dhS?@po^JlGcssXU~hF51hxg>!*MZS*n;)2?|2YFzf@K=;8UyX_Va5{D7s7d0n-pLmN@A5lO-'
            'Pn->2z1n%-`=y_sRw-B;Ctk)NNiR7JZNLFq%#5!)u*z%9AtsA=j$i{&8efFo3NM%08%Y)tpcn~%0orzck;fjWZ1+LJk%71XD$=!s'
            '9{|-DtyQROOAkI;6R6G8DOJD03Ij2f2%|aJ!ZfxeKvP0M4R-OphBN#$!I&5QbG<OQL5X@@ONI#IUjrf85I@LubQU0nG#h^!06~8G'
            '93GS5c08x3pHTxLKAmR*JLDCa3tiv58NmNHkixd`rdCbF!k|{Z#%e4v7Spm80PFe5vA#dzV_mrjJ%7tPnV0_q;=3;W7)+-uqPuDE'
            'kq@FqeSVHtC+u>KI2uH`tauJRIKYnYm=UN#it$sFm#`?&``*#p!O3ebUO);13UU%{-@o~l3@JVFYVg<op*Vf@>byUsS2g#(Tn8h+'
            'BY*a+jgwQ00}_ymYgQUPR86N?e&OkJDm8}%4oR28t8rW|GYJfeU^IQU+kW<pF}NdME;IHB4W#V|Z=*v020jXBB}S&ouHXsTYvfoy'
            'l}-SH=mX!l#>$7+JJ*AgSEtvi_$WK>$Vu|NrqlVtzY&udBtKA1X_C4w-Zc1tMG$x;6oNVTJ#=hmTiN@wQ-Cf*me;t5feFE;Ok@EI'
            '#(mfhppq#W$0+hKi(`?_(F*{?$P^AETD)3F5WHpT%TIHqab1!szxLx=zyOGpLaI9Ay%)}8bw1i*RiTeSCqt3d2H3}5E)67yCtG=P'
            'SNZe@@e_vHAwm-eFm9IFN(cS$#U;1j*WI4wK}ERLTs;$y$ucqyiqRGO<lTR|axQnRxFr3(ZhkNd=|2uYf?Q~I2E)jdsuZKcJi5Aj'
            'p!qDLF*!~Xc<0MBlbco;4WY+)8Bn2-TzIDiArechC(X(z6k{BEjKIX^WYcZ#wsJ^{wrJ+db&R+qSO}vknjzsdOxi?{P}M^w5{Y;+'
            ';W><@w>P<lcoUj#J`FW`7xwT6?(l=4*@%A?dlqflEk+u$#HT|f*eV@F%%Hn_yHB2>Yx{zIA=mKe3PZJGojZ7V$2&Nm9d%ZAfUdOh'
            'z7HAz<D^?DX<}yoVAwWb2a<ffqqsua5~6)_J78!0aN3d#G4k=)0oR*OIl(iY9OjZzXQs<k8O2L=e}b&Q8L2c+7rvL>a>R(NW{h`c'
            'RI<?j07niTqVR1eNEvb{lIr`*+~rCO5C4CA*V^2+k!8Q@S0Km-k+DcgnTa<e4Ku}z=vWn5a!Gc!yjm6s5)j4Va{)p&t>J&)`|3tF'
            'x<Nhc%*;-0DwT+2qtWlak8{sOk*A8IE~L@vp=^{Lz9bWddfpiBo<Vs4KOj23n)WVKqugV9jyqlHMo=ZttSNU0`$6E)lsl|yr}-zg'
            'Z=*U!ZhC<{W@8Pm7rC5^Z$K%^dP^i$qVn>?#tcW|;l5U5-F1w7*y=2^&~4(@Jd%g8(cDP{s`6hRBT$jb9+y|V$kxaex0vP`(Ravb'
            'u`Ep!j_0b#u61)LugyVI^ect1t$L)_h*c!`32s-boSNTKu~xA&i+r2LuW+a0ny0h&=#O1IL6roizyI;*@O$|M#VrHoqh4;W+O7n+'
            '6c9c>FbLjTDSh?mUz}^teV`>cYS8xwM}K(PEe=Hjamr2N6`S^yL}on0v!GCg_oc;XHO*X;y7krkd+vJ0Zw!i#V%Y3$OpXk5uyD{1'
            'X(D)B;L}UDn$n)9T1*+e)$$(4mU-DMPefr#R#z`CMWCq!TrFep_XEyTS*ah?s2BVDc%|9ZsV!k08wC%C`r6@<QA6OKUih$XrD7rC'
            'GXR-P5Fba+HYr3oD>=E1<AVQq&Us;+6%@N8Zzmj-oF2l0n~lvl-Kr3t7Xccjn5c+vJ`m>bikRmnB2b|x>QKC^Z&Xy{a*7!d7)SvR'
            '_ExK`CLc9@hq#t+;o?+Y<v$)A9JqGqE37RGF3!(Bu!lxr2AUxL3o?bI^BWT8S15=^0cIwdCQ1ik5Ub>8p3NvM3x-7!{28cLGEu>F'
            'X`{2mdz#HxS=CLN3pwaG2=>n{qi>G?2g`JIa4b!a7_nI`*fZo=^a_PV`uGg>Ze${$0RAS8b1e;PfBAFky7|pty7m9gUI=eT-HPbP'
            'wn?pZQEF9PL}vj@R=1cpBo{8ZeyDxQA}vKUzQ!)9yP)2)V*>MzDheY#GJn_v_v*<_SRU%qtIZqY!4@hUB$eE8ArY`v`8;Pgl(}F*'
            ')9Yyw^lmX8>RhglxprIL)@VM;F%D#CB!l%Cb+^<pM_L;?f#Pw>IgE5#+0X?NJ_e$ZEt6=%EQ-pRhaae-nD>MVZ$!dtwBpcGB&E4;'
            '4`5%Ze4K=ku;vVnF(M;bu63TAE*8s-a=l5EDL3cz{Oa_iLwmIEgID%-no+TDVMrk+o)-$p)khG-L3HNY(xgHCHCclaL{Ty9Se+FN'
            'XP3ly9GJ=)j*QCR76X;YoVo|^AnM6zs&co07XJ!$23EZj%u9*>UXYT|Pot+%n->O{lN7Jdd4?>O*qD3{(n%tN|2komzRKcCmO*{m'
            '$xDwEAYZd!Qurw1S@<}NgvF9n4(=UAnsDbNK%huRoyxe{*-@27%SHv-atQAC&njl9Os3-jp^c$6)8uA_vFePVX*4)t<1N`MRAeg?'
            'n+YY;9%;FQ<TgUGyOeXD-f=R~bwGkR*&J}p*_0Bwm&152;k0_F1b%fJ#XLZ$mravd$t)rJUg=%xbKUy2xs2$vnN46%*7!rFXg9Ha'
            'Jz?SG#|DcSb7LE)OvSx|cPke5C>auI8Ln2D1jTyT>561;ZW1n!#!(K+BYr99%`iHa+i{dy{La2gDzfxdMuEO-U;Wy<ym<evb9GI<'
            '>H0k4<*-kx(N4y<HSb*!d_+R8D3&U4h`uXWhqT_O=7A1;<+gP^q+7y|Z$b>lna`3n?0myK4>B1tvQWL>q#KY|C!d>c&Gk?~_RBQ6'
            '8Gphx<2nA-szI?0BOOAL%X}qE%S;5sP|nXKxlqK%PKum(*`FHiAvHA-II0qd3ap9&GOdSV5o|A2RbYaQ$_v^fdu6B!Mk?I<)M~;1'
            'LxnKoD`q^r4%_cfPA|f)Z(A&r1TkP#C6x821qiLn9ap#c*?*yjv;d5nVfQ2$jI!!Ezwj4mV~3CQVY6{-Eq%4|g@YJ=RM3Q#4{)oO'
            'wTurk3O$}Ph#Ycwq@YG39=Lch2i+;o4SiWJ9}(q@cli?UQ3H3L!4RYjI`TS2#iu@BA9SDCHKSY5%g~L#DQez%Zp4W{_zTy6y@c!Z'
            '2BOGy@w7;kQu;Z~z4;B(K)Bf-jWf1e&g*8EG0t31Nr$=h+n~K#%*I2spkoTX#>W&RZs%x?<-0!1(>sbyL0XU~yQSRRQNYIUh}d*k'
            'LXi4crU$8!1x`+_&IaF)i+qYPElEUZU#dssJCp{s2Sa;|(GFL3B9|$c6;I=Asl*B`LlFHt-O``5n9=0Vc(S96QSdy<^GzH8G40j#'
            'TSYj@ngT7a>GFjlV@GeXfyHNcw*{Vx8Ab!`wt}T;E<Rdi=}_YW_*aQF)jV9u^k{x#mGxYF^hCvls281H;Dn)LT(pD`Qzwm>a@ue^'
            'F2}Zvocok#EWZu<o&oP8vmeN-;@+c}p#Y(CX<-FA);>(7t$d5P^?+hQv=^`NRZs|B!QI1pkTO{{t!_AGv6?}~Na$+S2*S7AZ|d@`'
            '`J6p6OJSEO$jY_JR<9C?7Vj|;kb|=>1gC<p_w8lpx6{sl9<zWN5cw{gyolmwcyx)>#)3sFyOr+SwKBtXw<ohLr<kcG95(A#UmMP~'
            'Haq@UL7_^`y-5B*lKvCcmpIs(NNh1Mh|VUXwi$?Q(FEOr;-|WS0n2Vtd&kK(ym9Ja$q^1Ssm(Z%udSav<}5Fr&aNfJWA*z|Bg&VV'
            'PaZIwXd@UI)M?mE<D|FB=NMaz#jGKljEO-tb@6fT#cV1AOrB0)UQzEASs|$Ptzl{H5*To|UsvI}uroog6&*&Y|AqOXP$Pm%1v6qs'
            '<Yj_%p)0hgu(uy?Sz5JXD4AOUz%eMa$~%Maj+3Esyaeps;9Iybv0H9d75~$I`}XqUxAxhQVj3&M7_oW%(&p}K!~$7Spzw~?9N2>`'
            '#|0uQ@J__T6E+GI7~iZFVnGIeu6hJd%E83EVSUZ%L=u7~x#*)&iGyIX0y2TJMqSw>G$#U$7*54L<ERk{&4!5a9QOE_efv5$0To4K'
            '$8OOWv6<MEQOZOWA^aqREtby}A4br(IXU_aFbbB_JX4WPJ6bWIcK-v6AaVq6v9A9!tsdN}|D$FPF<c1LsP&*z#(VhH#qfa9U%hA8'
            'AGQ@V?G6v{UKq`+|LfRehKngPaD4@9iK@ymmsrs?Jef{v@8{T49<ZsH6aGTX#y;S&-@fuZ8_N^yEYG#I{66*;kIlvNmF*z=68p<z'
            'Y%pKR4r4FQCm3T;Ae}}8XtQGJGRcCaf@i+OXrtPttTuyC+|~M*`d7Bzc;2w{b$CPmjSV)>u#FV<kZRM$|646wtA&JEDpVGKtosZb'
            'a-~+7HmnQx%PbuKC<_25-l^97gDY5oMvyhK+;Pf_ahfIjDg`Psz4BLqWX>YSSDgrTlnjy8;xo(xraUz&Tqkid!4pJ;dgF0SwEo*='
            '`|M5ozy4(`5Wb*rroci3gys;&4>nh~tI~g`zcjhATY*C<TzM&b2o#w&&DEwg?Wxv^8bgp)po^&BYE_6WAZ9R3uJq^j^dn+~T=lUg'
            'Ym}E4*|z#T(*6Cuqek=#KYGM9k^&zgt>e|}hgC0)Q>a!7S-545Fe>yxt%2GK8YY@3rfUmbvq3TJr)V$O?sauc(B9HlkzyEZg^rx*'
            '5&T-$yCnAQcI(C>ut-8%rY^GMrDJcEd*?fi3=N7q-|{kB!)P%>SmlVBywv47Wrwv&)Y8UslXHoe()bw91vrs0TXQ03OpL4!h-+z|'
            '!+-zUIqSUXT%86#cYZxP{cv$vPMaKFXy)nXu}ho1_IKJt<Lld)p6~W{Tmoy2b=~p%-OFt-el*EbSp*-)o+tR$wF!LrW`)Ae#M<{T'
            '&9BdFSQl7b*a$K1!&AA^Hy(I_@ucrscc1r_vKW<pH@91jx0q4Sw;1`0dcF~V!igKu1h5+3k|k`FHTi6C&s)9B%sSyLLs}B}HHWpG'
            '?FPF=c{C>VsOlt>cI^fu#OreK1GM56#x609+1B~3t7=w_2EM{K>HfhG8`JgC;mfX)qpAm5S(2>J7V$HTPMqBJG#$3iIaLaND90(('
            'YW_`N2%RMl%Om_QCL8u5TlMlR?kOdtjEE^pFq+GWoe2LY$0K~*BhJ3wVebdGhzB|IaJS4hu+Zv0r3gbTrgc+_EGPMOBuGz7N+oQj'
            'Yr1N)p1NrZ24ia~H(U9DgU)P_#4+WO5aT|^9zBcNRvl2CH$QhyPEOB%>7AUu>6~AkUYx__{U0xX^ptpWand;pm?af9Pa{#yX?!2<'
            'U_`$AJvqqdi=dBpJOzFCj~O)ypM%jN(E&<q8RGOeJH<Q6kO@~F#%^9|vgc6cTk2)w*?1bIoHQVmR4*zKT=8S@_khJp#hx&n=Gh8)'
            'jYh?D#dl6|hCSl!#=<&azK)p2MAQCzpi20jCbEs{N9kvJZsX#I^{{g8p-d;Y5px<PGSw2YZ-llbdV!lCp`qxtVz<<m6|y}a4Q^+='
            '$w>$+Wtw@GPpjy7a5s)<wNRrPMDK9^E>M{zI96PVjSa=gF48$H>Y|sK<>ZR=q!D)<BKev54Sh=WX3%$6c3*rNt3^X+UC*qGPrZr6'
            'ao7VeA35PWEPg`@;aq}Xki^FtI_vmd=v3M(D&>h+z&(nS1$Bi+Jh*z@Zop+EBvjEymWFl4ESjL0p}Q3I&@W9Hc~MFUUMM014bZgm'
            'x}J1anPki(DUf^+^sSw23v1cpcPS?N_jA9D7PB1W4p68uJ?orE8ult3ELA>}?G6{CAKm(%*U#cs{0P5`h1IM$Bk3|Kp9(}+svbw*'
            '8O9D>Yb!jOs&H=VoJAG+-#MYzEPJxU!OI3s1M+nECAHL#yi^*R-Vu=`j)7yKroXIP2*qa@gpH%8Rl)pZOtu4LEPi`}x!4-4Kquk#'
            '8PV@RfL^3o!B(XlJB+l4NvvuCM~fAmq6eTi{JozhhJSmaH$XQ48asi;<|X^L5l6)J*U^w{&*IZw>Ajqc1gC?=8jizzUHwL~oCfzs'
            'BQ|dEn%!m@dgZye=8(ZV-a-`2#x-^8<!~u^lKM#`8nV!z;8NIeyLWF@wFyS!G#&?VI;wujTQ@Qfyqv-cN0Le@$qzl(6R|yoSC-Fe'
            'ey{9`M^Un6OP3zy&#-QgC%H8&n3`G|WazZlqE)bb&QkgEmsL+tVgA!9!b5?2rvy}ATN!bx&S|#_ioDS#dn#rmxwgeL$KXp}^ZV(^'
            'O4BYVddMzSh?zAEQzpTV_Qa7X0YuuPhJRqQBE1g=+cK?%&TE2b4GMhz8MEt7RuLVPi|H(a!%~*`G)V*6^{JdU@q*$8VcmTSz6;hg'
            'il6oZIw(Jnm)vkJiZWU+GD?5q(nOL}hvfq|@()GHfs|b9da2TZRg%Vlt8E1bJQDmYd~|M5xbdxAn9%lp6mY46@8nTMFEwA>B~ztL'
            '7t~=r-oSg>!rRk_k3Bsg=N$$O1Zv29nz~aoicr+kl_2;tA4~}WxL@RGlp9N=?Zf5)x|bF!vieAE%t^}ZG?eBFLzw3rCmX~$NWmyP'
            'H-nP|wkAs4$W=|%jmCFEp-MSq)amQXgkA2dGqQ&=vIEY6ilim(B6i1+7;1Ry(VucU+2d+$6N>pJPL$>WQH#_L@edrD2EDxw(&0-r'
            'hnNRmClc=I(a5$;Co;7o9&0fL#gC>F`g73y)Aw$$^NFetW4y)D7Y)-zhA~*k2tt-6MY^k^?bM6hgyon4V}Lpdx2fLLe7D5n$u>FS'
            'M@oZe0Lzmr)a~t~yoK+Tl|iAB_I#>JEG|y#t>zRI(@dKHQ7ey_RB63MAwZA+)H24_mRPx2nyWRWCJ{`8nig6%30?$LC(d=5Pc?;|'
            'Q<EBz<#gSW<O8##pfS5P>*}h7B6f=2U~Tq|-abWhDlf%~wejN*?%MbzO_CU-zu4S%!sgG6<%Vgb#v!2%1o#^7XdvNUTx4yKClskF'
            '>iFFUX0qIaV%RCpAXZic8^XlzArO;#Wm&j%7lN98dE`1Aq5!-Y$CN;$a_WtaU57IiHd44E)k>iDyA*>$=s|9C=h=F7{Y8^-+`O_>'
            'ZHR0X^nylkh!8mtK==8LG&sVmfR~%6UU&Jcf4a^$dI~H?6ipNd3ihO|{qQ5Dh<ur&V)v+_0JWF~5HX0liK1OgGwscDZ9Y3R#&pAd'
            'H<v_LwvIkkd+_!MZp@n8oU%n2t0=#j+vX2Hc&^O1Tu#|<0tPc<&V&Y>O1R|ll;!gTn>0_Nw4s}zFHE&%q$zSG*>?u37gC~L_uMlz'
            'zE)HbSQtiE<eFhKavLs~Q5xOh-7a?ISM?(b4o}B~35RWY@#>XnBCAf2h8D`Gs8N&q_ssg7><`c^Of&fSMAXhd*t^vN>K3hm?Us+u'
            'oBdVL_GTBYVb%F3m~~CA(-TRqXanpo-=Fu|=kHE`xorP-`tC#TvUAmGU;g#A8kIh$^Pr#&Br=2nSMlbIVb@{ptP^pvuUv<qi>zjS'
            '$8IqjfG+MQvl!);j!+D+^A`8Dg_05IlB;6HIZ}ft+bP&9QFm<!ww}EEQxNdlDD>ezfcpF8nd}FHJRT)zt)zNztlRG{E<Xs{ncx?6'
            'Bogc+3R^+JBaYU)nIPa~)K6ELf$bMWMc<sdJ{+75Xu~U8s%7igFl~>zGHCktMYU;Dxt=_^C=08n+3wI=sxWre!m72!7#Iw{O0#GU'
            'x8l5x<UjLdogw>r8X<sb76lJw;SNE^7O)dS*Y|?k@o+Fs8Pr}t+M8us`7Z;sdxy1*aQ`Yn{b6}+LZR|kp46X|<e*e6AR%W}D}PU{'
            'J_nCZGXALJmhC>#a6AccHk_sm^3^cOoJ`}N7b$Afah_tPv<$=u5cLQ7I-l^43Rd|9^B&{OX3?bXq!^cd+wyl*Cq~_xg7T-9RP#^V'
            'WP;9lXnWm+%4-?j5}bfX33n6>Zn^)hd+yif;aw&312sn3c)duE|E)I^4~w~tzJ|}?*{lzgkjU?Ja1v(naZo$lQ%k?GeRX>r5??J$'
            'U%lcHc2Wb^;NLuCI^8jx1=@uA=)9MXjn&E<ROCyt7DG2syWfTuYe_;4zep1FK%o@+VCk(YSh@*&wQki%fL}|8@=52s^Wmy<7DT2D'
            '7=h@QAG>(r_lVM)p$G*tjo4z97kjUKAQM6uj^ljBIh`0-JWD?7i#AKva4M|FbEMz<>>S9|SX^`BH9XVA<CX;A$3r`9SZCz|+k*47'
            'uCi!^KK#%@ZYp%A3){pr2@0WEAuicq%Lf6AHvaJ1!F^hw^P*T;Y%Z<VqM&yln5}XOlw0)J9rF998h@20Ndt4zGkYwfp@LB!rLjsf'
            '{-^p^Pt{Gy8xigB2Jxfsm!K}kKO-1AiEEen9O=kOnvz=s!SJ#j5Eh@VdIk2BG6TY6^#9^}R#nkRrqf$$u+1b|&!Y(^nGI><6dAfe'
            'gSbp?aw2hFAZcMt2^WTo<vJzSwm6{r@mGrOC4P4tKaj<cAR51f$5Yn$L~BA;$>oYds71{=Lx$(z)hnWT-w$pgG<-BZ-i|XQT-y)c'
            'uCFjx?g?3c;q!DnNEaE#{A1pyg07Ai);yZ7GeV2N{yS#0NOdCKJ?48~MYAR1K=3{Lgv{H~_E=$m)R|Ir7O~L+d(L=Fug+DThmxf1'
            'pYt8tJGw3BFY7`_!M!oFn;;*;Rsjh(m_(}?3`hBB{_y_Sv(7mt#G{-Iy0#vus_h<V!<bs<1^o6QpJ1f@B*3#9WVsBJQ$_1E8aeVe'
            '3hcuO<aWewkCHT*Ovk}(G?{`z22$nico{6FaX!M!V3n*v6;4*xkuMe*y_DhJosZ|E8ZAj!t4HP;U`PS)KeR7TkG5t;CR9)}cpc4n'
            'neXF*78I=>D<f!yhiyE^2}bnpm_RoV4c5;&s4h?##)comfir3)y=Lpk#OP0gR>5Z-5%a%-R9E}'
        ),
    },
    'provider_adapters': {
        'sha256': '932a871e8fe32d3e6e21d2135e5436bf3c27db96b45a148d7e0044346046f8a3',
        'encoding': 'zlib+b85',
        'data': (
            'c-rlKYgZ#jlIVB+ik$NCL7G+x==OT&VB@oxR(G>QLocXp&$65%C6!Q(N>y?_paqxze&dmumGzJi-Pj)AyBxbgrOL?6$jEqQM5N<5'
            'r;99zBN=wZDDlUBlm#+$uXCA<{V_a>)6`ug@l7!H6CuZPk@-m~+8?Ge^~K*hy;kezcmEKBcSq;q{pstE$9>WN<>-8H-f|m%T0c+y'
            'NMuuAyp7}Q&=*OZXTAth;eVRSJk6lf_SuKmB3-~xx?&!W{ZM?f^9>9yNJe>(i5R}=Ac)o?SomQO`68Xhc{mnBU&tZA%wjQ$VIooP'
            'i#QTeygqvKrhnEy83;2K*A=(ZU^K<9Wh?ZrgDjX!7>apgG?me{-xX1uiC^<LlRSd?4dMvEwo~74U0u^OJQ?&3j*m}&_6`mYPd}av'
            'y#M-mIymso;Ge<K>B$u$>LkvlL3AyqeggB(qg3Qk=%=X&veZI8q9RDMo;VQmAWgBw%wOU-PfiCyM#4{$IN56<64io;#9zb-K)I4P'
            'G6)d~`)>e}e<f~%Yzi1iWdswv1MDWgOc5#3n3iEA!_ZG!X)s@eOF;LHp90Q$;v_b}{Tuivh;ZI1g33oD81*C%0aluauu2FlieVw*'
            '2?CwB7<W7ooIn^uKd?Sic@xBmXip@pbmGDaz&a-)&Eztr;VcqZg=8tRL`LZ-35I^sZGC`sPPY3Y3?w?1!<62q*G&h5!MRzv6p`JB'
            'K`%w*XSZ=OqgjgU*vC$Q;;yHN#~}KrKSF}LqTUdLSM(<0uPf4+y11Rjq2B^J%tJ+|eliMDXck1`3VI19OONK)TPzvhK+;5jazta3'
            '(Gsy93O^bz;vmY976_^NB8=rY?TSS{41;v)<DW^O@azLk0CE9eDfA*kSia(UujM#SYm&rs;dzrh%M;)81gs*=0(u0Zl}M#&tEHYz'
            'vw5ih{wGiXw;cn?1L6C+rT*L%`0pL8DZjUn*)$A>`n3=6vBlr?%ns0Bwp#*z4h=z{`I9a^0&<DN8{bniqi32hJq3n{yW8rtT3_vn'
            '!x+{=oYU(fl5<)=KlKYn6vpvPOyUG~3#@Rgcv-LY;q3IMqu2d2@4SC_)(7I)7x&bkbDhM$I(wqce{?%M3?~B82_$`?{xn~W!Z;sK'
            'LYesVb@9=YqfwkknHP+mt|)zxXHzeWXMW^pDDn=M2*}+ZO$m<uz=Iu*AWF{?U`jCocpz+?`J;F~2O_F7X5yl$z`Zn*3potJAY1k{'
            'kCovq?k{K!?@WRu&Avl;3Y7`cxXg&y2~Hv=unAIN>Sw1P-@XHO+C^HAK_Vo**5TRd`MGEC3$G0N7J8?__0;<{mpA|~^FL)cSNaNn'
            '0P8d)nCl79mcotthSozk$lqnKeZ9zBoa64xouN~DQLm41=6)u(VZY9$`>OZK{r8=r8$@ZA<fE+I#9BgV_8+6k_L^SwiqTYV;xM24'
            'pEr|VZ~bT+|9RE>)_wW!n@wZa{oCfVD0*;T!Q|zup*vaptvV5ZKJ>?9U`6SwDLB{uJct6<yy?Dttm~l!^bN0lcmAsUD)B;mD~0Xv'
            'M<d@Q_6nPHPO8HkL<Z1j1Ud#N9ZL{~VW&D`&HiaWR|ak1SA<CW$D^~Olf$Fqt{B74N$37J=s1lFqnfxdADF<+D;L=3Fy4@4);BgY'
            '$R0yB$t>D%cE?|d{um@@<on|kWU8eXkVFFgCrFE}3X;%Yz(?4=AP0d1qO?mbV7I7l`6LU0#Q@_lECaxhj@1)K&=^&ap$}{1iv#3A'
            '@L>e%7_bKm*d*~mFfcc9jiy1|Vkk#5OAX7Cz#nUX5~N2k2Cc&HS?KU#njT`Qi_7f}&fJ43Y=jd(PhfdVOHpE->s~jk@+TC_Fa%d-'
            '?rjoeK!Ygccxv264Dw_KGFRNh0q8Tm*1>;&JnMUdk7s{BI)CT=bb541e68Kkof$)a?uU8^)bRjT=Fm`%uxVW&4^axOT0>NsV3>fj'
            'd7Ne<3}!wkVW7T_Vff6XdBLSPJUw}H^!DS~!H=L~zwW;|_;@_vg`b1gj5tq#7}d0eyq3s95WpB#KLpZE(q2K*J&*<4NGtHz1)8Q&'
            'rQ3SNhCb>ourU!9XoIc8<I|6?-y9zRf<VjfAA3Lb&yL<49g=eEfnM52OfltK)Bl-!5C=zU-m_p1lLUn`0Fs;WJ`IxZ#4HE)7tKMe'
            'gPZ}DAsmMFq`~KR02icVB*yV|l83VApdQg0`xB#jX!7?geY>q6?!hKXx=O+FuKf(y%RX^;;r<}T!6@6~VL8qj!%NlP;wDJxdVA3Z'
            '?izKF)II;xA|`!aiW?c`upJ28nGYNUtbzM2-FmAlw#@#*53ID~U)H#{_Qd@RCcIbHwH_XLo7HUrv!}H|0lG9aV*-nSKO>U?J>H$j'
            '4CocK>`ZN0)HU%pc9+6ZSJkT?Aj_b!L$3xCvYe>Jp4rR^%~jkx`ek5}`cKq<^pN2(4Z2_h`p02hzzY)x^_*x9d@TY!@;a2uU=#pS'
            'G{C<k52TkBD1%n_4-W*>Hn{dRBFh?T8y{Bap@Z5c%?foQnjOQ_w%We981+E?0sA=hh!0-uTvDSEwy?VG5fP;Aj>gppC_9=2*TAI4'
            'ml}x*ZXJas%zPd`ZCqXlcVPgQgPdkE5lc|3+xJLty?=tq(pIDBJaoP*Pz@Yy9^46;!QLGQP6c)u?bfH_y@%ElX=0G%Ac2ZuJN5Oo'
            'K0T<-Kla&_EOtMUcG})#r=J_kd8=5RZH135F_#NkT;>DX05Rk5>77$swSwtXR+7esb1S-YkPtRvVO46qu+Vo|+e`EVyc77XnoxBS'
            '85-jCqPjP#T{RQ^%xsg(>bm0(zEX^x*$XtN@fQ%`bJ!^@Zq3W;rL$nM@W)mD&D5bj6y^{;sH|^oRaVeo0R<_{d_dhDg%3PL;z8qO'
            'R^Mrk8fQVjxWqXk)qI6b^K6l48dTAmEMjL*O{=ROc!(>a5<>-EeGTzjpr)}W5wJGydr^Gb*7kib&qkeIkjAK>%dD+bDTCw*kWU@Q'
            '6|^cxT^B^$$aW+)?WyO(x$5&Vr1Lj6)_G2wAidE&(XPz0Q(F|XM;XquE=c14ng^f{8>dy<{MFghQo}q#;f}He=U0;<<}Fr42t52F'
            'L2GS;wmXeM`sE3rc<M`ZtnD+Eqx;Ak#^a^>4AYS6Pn-ncbpP9(PSKn43*o^16bub8CHN6A)ll9AArMhU-zZN7d*(nXDF#@K96m?D'
            'vGxtTp8BJC$m%zbdfyDa+sq8BKQuM`r<j`h-7x^NWVu&@wvbEo>(LN-v_0&%T{Hm#@)w!t(=U)1tJ>c?ULtS3;S$h8EI@%n?;_CN'
            'Lv=yT3ry?Y2zE$U!)w{wPp>mj)mOrOz8`@07Od=N28xEv2}p#agRN9dV>%`XqKTiNe?hc|c@U0!>`+1LKx~VXqxT&&aOI3mty8dd'
            '!3-IJ#e~3MgJ$t44wai1HDKlKR;C$Fg6uQpKpzJa>Mo=1$>Xgt_?6;BB9YKCL8~lcKWz+e0YRf%d>o^4h>%kWmMEPB^jbV8@9^Eh'
            '!25W13@h$Tvuu&>ZEpv#!q*9E;BpaIbKAbz-NspL!{3Y0Cnv9iV|dS&<(M6Pf}FEe#PsM4!I+7|C5$r+Zul4g1uvokf#HDyo{>m#'
            'o5V11u;0@>naB~kpwV0yg+9#Em^;W((YDUYo<s!2&P3&R2m69dA%c+t7Y2lK53td1(Z8rrqD)*iHxV!Ee0{Cf+q2XEYD6O<58Xb5'
            'I_1G>#2`T!oE@B;AD$ik*xwM2%*R2zU2<0eEaudRQxDsa_YfuMe32<}`Bx<|=_B^j<d1AJMH~5+VUvmrSEMt|qvn@wgfy3**l(Tg'
            '0XbyQa(CzDH+8L;x08z)lv#jYAjGIBFp39nxQBY;90YO7^b#(4J4G>Cphh<)jhDIQY7JLA@5)N2ZE6yjlEPaPct&M3(725<C@yW('
            'bt-0J#mt8Bs>Zhcae2XzS=R>Z8oBsdk=?3^5a!)oOAH{Kc3Cn1aC$y~@t5enyrNnF-a7|5$i*bMlggVqaenk=0=w|O7&;rH56%j?'
            '=p3@daR<u<87FeF2!jzp+(zN%Jd|4&I>LQTUn__VADRy<MKbLD6pT(1ffx5q5|b_H<SD3(U9yvKXskWL7JIm~#!|>!pc2Nc_{+X{'
            'xwBK2xrq!wsW{6c6z_e~@+Qu^!Qg|q=T;AU_*c?atJ1mH0~nVNj->~ne{Wwe8_>m+Jc~dRJ3tm{4oD$-z?W;?BF0<&L`9fvjJn%o'
            'ojwfhY+V>9(k*o0X0`+9hv|68mQg=K?DNFQ7sTJ%Zrm@?fLYI}DorRv;8}f>E0aYd&_-FanSx%zuWZ~Rze6d8&M<w5rj8uDEGB(E'
            '4-<{YVeEGup|g^-iMct7Q`$I&0BGN?|H&Wa8B1ey!^gLglAc`-Q9x+$ko6LdL<rG)o)m}pDl}pUsGT(&(hd5*47|gG<6}Cg)0Tve'
            'ksxv4QI~CH=J3J(9`XFHs7S7rn)Bz29p-y=VON!GE8FTb2(Pfi(*1U)F1QY5lAzY}=HTeK|5`5#ZCyT50i^K3f8~f3<XaFakFAbf'
            '!59uYPo<Rriqz`|6PTx{Kapz5D$gSET_5m>@g806i)HR#35Ovil+I+QaX7B8d$(!PaR>$)>qY1~a(P#y!#NleyY7io6vX6aCv~kM'
            '1ZN!ksRD-jFklR2IrSS^=Nk$g2ZbJ(f_&$o8lE((cs)87;q3{~{fNE#pal4!8t~M&xWg+W{3Vb9qU(x~3-ZQC=Mbvo01-yC@C(-w'
            'A2^|MkSQ>RAOOi@24MmSj4MTjc*rCRK($2+bp*RQRgpo&D+n5V2PBNz3z!W)wLyw8(`O74GMV>~UrQFTHZd6n1uap429VAt*0EDL'
            '@w25e&nCb+fkjE<QMQi&)oK=zJJ3Kyu}HL=X_*mdkXqIrd<%jt&PP*!+{5#RWXaY@3^EV>f43BqbW6u~q0A!+_R?cPMKESOL`F@w'
            'PX(A;z;?%-R%Rg>y>kCx)HHs`5<-cej@aL4$<<)YqIspy9aup$%`JzE0pG+0XzAHsUwA@_hxh~-dQ;0)17cJt>V#2+L7KJgq18wj'
            'CJA`BxKt`7auPfzXd^``nZ)M`)&m$-ncKA+MGf_6DXrKM@;%gv(a;24N1r^ix>4_j2UN_KY{7l8Qif2M2%@B0u?3k{98CmI$r<xd'
            'p#}QDM#?Z!<Jngdx-1MzpH@nNLRgr;Z0WyW)$Q$d>@kCBmsBvulTowOyS&K!u(QUG#Ol~Ej>C4TB|Pq|ZC9vj^~U<U^$@h`R%*%*'
            '*Y~Vd!g8N#r#_Sxyf!D=RH!1CzhSMC2-9&oJ$NBlv^y2;n|DvXKz*=%UsoEdi&B5%X^T`SyN!e0MBObltbE%9psa4M`rb2VTkXbm'
            'a*U#o$utmI4wln6_)b9Au#@k19z^?o_n{*)naBxnF<{xC24q1v4xC1SJ*8wbtv6w>iPW5ud|hpInAaYg7W1e*n1G{4MEp5UnuP{Z'
            'lXBxXj<3#x&5EUsMhPf;o<(K#`B>DyLMHIv7Y%)10<Y5_O0QP>#rLph<rF|49q?~pL3|2J9KB%Ro?+gMkM=U2b(|9XGErTcC=C)Q'
            'b;Rly^=>uqz7P{f{2yediM+E;fbW6`^w0pT9U?OygRX!AY3L6fv)!Ss0#-p=;V5G#<p!>@CI{gPp7z-m#g|wz&+0<s$)Y!Vog<CO'
            '?;NSa=+;ulrm*g*-P@?>k>Y#RVE$qFb0ltMIAe_}Gw~}ku+LFtWX-q;BI9s0Xh8a8T-FOW^Xr*7$Y=8G74V+lx2ItLUf#XR@xUMI'
            '-&@u2V1(|N|8)ueE-t|$W_VMlV5?ax)qx04bPBdsYMLtk;a}e=Xi&JBTd;DFV;}Ui>F|dSXF*lRUkwL4E5e1>pFeLPxWoiqE5Pzi'
            'Pc)p~t-NuXgpCSJD$T%{y*<&Gk@Ml~=>5UjKa3+Q2EoHp96evRI+aU#BK%Ld|AP#KlT2r+@`0Ex{uNJqDuL#Io+F>~LFID@h6=N$'
            'z=t}YMxyhO>JXYDK>b)pDOlU$kKwwTpQcpS|F=6*DX~G)9y^bpu>9^PB{g)cIWBqTppnL?@#l^rp#Uc<O)F2rc9FwnN3s>cg7CW('
            'w>Bw-1XQBZM_~Xc+P?W_TY*it?~4u|wq>vltaJOG`gkY>3RuJ1AY5Qo3ey^~@+HSBn;AqEv9lSYFXh=h6T2_Ne9R0`3$&8EHz)it'
            'yC*euqn1TR;O`|UR!}-KeC>9Ay6B)utdJTd75`76p_{Xd5{Kre>;}1isH*d}-$X~;IplH=QjAHG(a3M>+-Ten9S*X_J!a1g=^8;k'
            'Iw?D+W=RpLVi>27NxX6f+rMZO0d?weL>k+U9-swD<?)=O-&tfTLNP+Kgx^)x1nriNl|+Xpp_=0a=1@wfb<?RP;&qs3%Dch`sXmyF'
            'c*qwmZ{@OA8P$ctGWz$LBbF=gb@dQBtIoI@1yhko+Q!vQR|qCMHFV<`w$<@$e-g1`r`i4~1PkR<;5j6%<z<C`sRbrAh`1LWbJEj4'
            'w_H-G+%)??M&{i3qbyDk_3ic=vD4v)e4wOGkgMWDfqIezF?9Movc#tQ-%gsSi9Rk+#xUn7W#Rz)w5=Lm>|A#K+~TKfv6UC3v;+;}'
            '=lSimaw_%U{I@XGNc?c}cb27XHcZLXP-YQOhliC(NDfgRw~<2)N!zW@GK%WHwCdE01bFm#>9q1lCJDz|@#)NQF!R-k3>B+^Ve5c<'
            'RDm&wr^^Y%<9vh>L;;5jUM2ob;NNO?&aq4c@uUwV2{5KWy;Lz~fv?8Ew+Iz6^tF{@;1uj*tchYD%>etLmvsC~Q}X+2;;SX(QyM=3'
            'Tq)asLdLB5<=9U%e+Ju=lXnqkFqRbksP_csFUf;&Hb~K#IPD<;@*`+UBC2UB9XyNdJVAY^%<!2Mh$_^&J2KOp7&Ek2N&i~qm`mlL'
            'z<M#8wNkZZo~SWkoSMhm5Z=d%LOnkPVc^?)laG$O6tSpM&sFPm9+nQECtKfBh7KmAAjjT!c6&RH_%sjqLK$7}^CotF*s@zFvXygx'
            '-(gyLaDKSAM)vUiHa-5~`)z`*ntz$n(kc_3b^+Zp9+lS7gXQZnf_w7-kM`q9#(tsr8YS(G57zGuo&as%>^dt{oVrvw9WC60MKE{I'
            '^P%z1xqtUj`r`2~D^9H^o$29XyZ}r$MWX!TzSpC7Pe}m3Az_6uKu}KHm*32$ckb@D!yt3#aTL3Y(Yykv$(@XS_WTHw(%Md*P29iP'
            'XF<=WfLY26`J$`Vy_Hi}wc6EtC9_?p&s9_}0{yt+#nJ*<KB!s9;>0+P#(6T%tzoXJv4v`YN5!`EoAjxz1z)GPkeHO#o6p7ps+{;C'
            '-RILiDvsvGGy2$B34!XPFgo}|>k(D3%KR%1^{k`?vP}4VEwCc|N-7~L*=8CIaQPepzpcIes1~WEs;6bDr{(CH$`rq!_P0&|v*HJQ'
            ';kQuxs5lW4bK9^8eg;z?yAqXBQ|7K)u3yyo4RS(P)cMgm7>!KT_u&szf(7>AG>^s!mRO3FB1Bn_MNz~IrgE)HK&CLcImohbS!9cH'
            'O`eZu2ZR1=Q;z4nKj3naSez2TQRzSpY)@QKM_1?;@G){2>jK{TVdzq6tg*QI^vV6@7xx528!MMrA{h4&(3>E{pmHu8X(yR-Bsl2G'
            '0g4wOoHW3qP#o4i2a=`uagH{pnnILIl4m$=Djh{pM2&&PdrDcw6c~0Tskr+<m*S)IT`DApQr}9Rp^8LGzrt!p(o|IxLqCjf*W{;^'
            'LqK)0zH%mIwPIg2e5|Q#U(*pwxAlNqr2qQu7nOa|foKGW+Be%DM6hQ6B&z7o`w#uMEJ}bvt#~bCJ~UbiDuo#>g;c!!M$F#bZSQ{T'
            'f;`?vc`QC0z860q{KS<xuYfyvNghRhf{EimkKV1k@kX!*X6Y4Y`%ZG)6|hzqXGpdOp6_FN0COsp{|ws*lnNMTW(t|MErMiXkfW*2'
            'b>{Av)J#S+L+q&R4~oVoDUC-0i5z7;UJ{_f4<85K`N8`S$9)fX8sPQiH{Q<9PU(ficLygYeM)WG)in|*4Z@@S<^|(+VH%P(hf&qW'
            'n!Bp4)Rmh~L8l!T)th9Rsws2!2yz3xuT5M72wzGEKuN<&s*ZRDtqZgcw;Tc4M&|^z>4+cp#VcAW^9<b;m>BVr9_vRSIb^t)%62&v'
            'wcK*I-g1{)M)|y<CB>NYdoH#un!(45J!+eSfKJ*S9T3B{qR`2#%?aGIq=ptYvpG0LlUcJou80N*!Ge9ez1>ci^1!b#MFqMTVLo_!'
            '>&5QQ*IOMbeR@J=;|LBG8a@#3VXqy1I6eZ?_vl<u(cS)Q`hK0Tpo*jvrjqKc9H4Ut`caWDPFI`4xJAKkz6E_lk!nws=u!kg!02lX'
            'ZiQ#0y&)YhLOIe|O`Hjj_EX6FSg~E^s+28(%(~ms>oAuyOs=OF-FV_AR6<wPwkT!ez<9x00azx5eiX|2qHf{ig6NzKL?%DLe4Bc<'
            'jq_sXYtz!wh?Q3s_s(ARAdIV+sfhkw;?w-w4n-_GC4HXKr7(dDw^ZBtT4f91V<a(Jj2Es`Jlx%LiSow5HEf<rx&US$4AVB~7;T1t'
            'gbUk9>{gA1LMt!%A7*isr*kQ$!CfFE?0M(~SX_z$^XP)WchLpWe&GVhuRBg->^Jt^TGPw5KSIOyPT1Yq;MaIK3crTn?I4E1D3}Le'
            'D=(#hog=P;VE_*?n=Dz#s8ggt)mIPqIC+qX#%o`<Nn4wMm6LTWRDZ^loRV{7r}uB-`+Ww?eRc5j%ihk~PD-o8vs=GP(2om1H3~E%'
            '*~j?nb`pcG*KU^v;Q|JCI^wTLFr7|0yyJfB#THILLjmTz!-n9NyY+C7gr~mhJbdk};)q-~b|p{ze|dL4b$6iVk%E=A^wVZi`aS4*'
            '{UH7TiqZvoJSvo$E+beOK2rayvGh>v;s+$Pu2^8|ZO1B63BrQ(O%yggQ)aR~U-;La3Rk_2ld(6%`N$;jsd73tKPs1JQt8bh)iK8;'
            'QjYxuMe~XVb|r3QYU-(1N~e>2D~jtB_0!|YgsS%sxt6<q2qV%QWMf*9XsWn5bU6y*mi<f@lU2Ey_y$+A0=3sBPb{FFN7xCvfOV_h'
            'Veweu*h26=VR^H1QR;k;7G%G)F8jPXWycf}vPM0O2iFAE{$(3<AM31-=R+kVQD>*DpJK$g!+kk#kdgHEg_EKC?~)h`_3G3`BpV<w'
            'bM$|zW_0d&Sr&YL*a-G#prRjnCd&rsUuhQsUs3=&#Kmd`!DyC#r;LKMs6(FS!z{?q00En%R|8gH%%p5KXOyXPUK(vF?y|Ki^)bQs'
            'd2_2d0*R)62D99TQKRL7|98c!RX}_y|EBmu6~+sVxpB^>9bk6@zz;Qlob9W#NvKrdOKp=L?f~8fT6_110c5ea7|qcVY>Z<s9j#3!'
            '%rk<h5M@Q4uiAsKZeWt))ex=CT$(0p+d!}95AziR2^PIiygUoSw5TS>=Ql-b#(`;BK;2*Ua=QaY6}8aQ#bc1Nf3)=V6Wy&}iJy}o'
            '%P>qnCS`pX@`Vt14F@_=#>gt^Dhl9^NxndqJ7NP{#r=f7#4^KD!OLJJanq~wK{7JHzQ7mbc$QKLX$-m#BD_vwY`GD5AMhcFKVQH^'
            '&7@H3ducwI1fR^jd;A%Q*1<_<S`hLIqnT@#n@G&y{Q~XZ0?+7cb*I7D_l6QZ5U6~(A-a=peDbKM3W@M^X&)gtZh#N*ikH0|`uc|A'
            'vt5bD2P66p$T!{n7Hg{8?PNknYe{>B7oo#Y8eY<#giZ<Ggeu0%8`S`j*Rg7u2GJtV_N^^Mv0!u`2Yad&d4Sj)&FteaR0uQzY0Kio'
            '@Wf+BD>m2_C|MpZb(_>3dV=2ZHfC0Optz92xMS;n@J1d1oy~O7-ue;1*w17@XK{K2X$BiC!7hgAvH%QQhcr}im$b(sf19IPT5Atw'
            'jI#@rs{s8`TT+btn*^qVd7_|Z@j#5K$2&V#j4V!#(u%Vxcv2}ZM+@;Mg$LAqc1;YOq(8?5D3Q(|NR*pMgBLsR?vOT65n37E*2F<e'
            'K?SSb>43nVIHCIhPEIX<y5$w2>zCla!MnaVJ?V>s*9RX4{WJ0N(cm4>wm1-nXZ_dxlflu!v3Lu6;4y#6Bc2r7QE75P$s35b3h;3b'
            'tB4qwP%1XwE8lOR&$`v+2JjXiq_ZQLTp!51pt++8GcM$4CIJHJ{2`Zdr2;XfD%|F+qE;b}t5Tv-S1oj1uF9xAkX;3>DEi1gBZcyT'
            'IsqVcg}prmpOsWk(d0(G<+~on64huf;-ha~z!)^9dmQ9vTtT%avV_xo`9`Hp0!lcTgh#V!KnW&ljBO1TRXAO6hCu0p^Ck5U>tT^H'
            '%hyS8M)5VCja#ap#B!^+kZGKQWu$5mQF=7ii3Q68ws|5e>1+#`rZq1ZBCpXQ+*y6TR=S+xslhGArle`3VM*5>TnA4LZn1qpH7~Vx'
            '2${eZ6OfYBqcIxFZam|Z3q}tkic~2IJnfFLT<4u0U@7(jd8?;99xMFG_>UDd$rtggY@!`5^s8eZs}>`D^jvin60lW=cN<5n>@&aD'
            'i06kj?X^}JxMHrNMi<4u%02Pso1I<Zmbo8nv!P$)T&_DPE&%;_+~3*RZSuzyVMxU+TkB)1@I@ELxsWqhR58nwJk06Zju5LnfCY%P'
            'usc1EasoXMuLoF9+0M$_1MKK08_^ApkXD_Zn(S$UU~dMsocmSl-dhnt^N4A$nl&5vPm2;UJ7`UiNVDB$t2vgIVQJu--yBh*{|(|g'
            'TA!6zUW@5yHCN`ThHQLhJjat`IUF^z)Cvh)@utPD(C#lt$$Z6>LKw41rV$De5GF+7@9HZOtAgl`R#-OG3pl1|anYq1O;%3j?TRN5'
            'qo*;N(%Mj5!mNJF$LH0vFYbZZl&JxK<N|%@Vr1)hB8Le%!N#A<(w{$(wOlsR#0HNOP;rvyj)9@b38n5+anBVY6B~p>Jd_U4k(8-0'
            '3~xse#<yKvL^4))rPxOq{Jbs#by|_Ut>(ueKsu2pcZj^zDoQ0+uB<gMG}9kK>9p2Y)K_f?pt=kykGyi>C7!BYt8(t`X0?FU)kS^2'
            'G~8;Pp<1OWU2^EvhCuuX!*a%4X-*ZNs;zU)SRD{Qs1(RG*RPUte^sFE%9P9<$6&nRt$<^Fn`G$%teD?=b98n-(AUiN#0LYH15)gZ'
            'o9K=;W%9w_te`3Tv#ps0@{=p|^r}>VUfX|E9n`8z&s}%el=!*0LeIWDFOP~F@Hi@j@&nPp0vbo>*$8vNJ#?dAp@qpCNgI4s{#7<x'
            'tm={%m$r7V*Q*|)*?6Z+wWgfYKfw#+yh{Aln}g%y9}f=y&h9CvySDzGs`OI5_@;D`N^||Si%WIk6O~Jasc}FVU7y6EWs%sV$Yobo'
            'xTIEi(@?F-rMiBX>Zk3r>NgFU+LtsION~?qRj?Cm&f0LlnT|s!PPXcmFV_~Aw640@<U*UrN<}KgOY1=nU97nsXEDg!U<RhxTrQ<`'
            'PnuJyXIHDzxly9Ts(YuZvE1bp3EFVQ68*4`%&fiJrPI|$zSgOnZc)+W^_|p<kaBM?FixzN)vMO=`HGQ#UO7@Ndnq5ISrA*v=cZg`'
            '!Ld6<ncWMdqsyk$b1oC?rK71omnNZIy^m{C3`5bijp}mqrb3>&6fl44tSj|t)B2OHXj)hH)G{(h!7Kx<Ll{p=>f=Tb`Eo96ch)t+'
            'GnYD<d7I+eYEA5|^wceHFXyHuW$ooE*;=K7jz$XbnZqXKVr%u4t2JNko*RQc?rRyGH4)@=n{WJTK2a^>=|X+JETx8OVqF1ZT`ACq'
            ')r4JPQr|zBN3CEv{8a+l2_^)2%y3G1QPvfK_|*&H{7>{h(d&=@-9H|@@9&9I(4S`kXYZ~c>|TQ##yNn^fF3!HA$u>A*%*t}BY-)w'
            'bbuH}!QDs-a$$yn_|Nmx6XlBNl>ycQ;*zDsMQihW7KZwnJk<=)v$<<~6?=sjS990Lv})QT1q7^$Z1``}h+_0`Q3P(a86OgtM{QLw'
            '?|hM>VRiO6)3F&3kGaBh46G;c9CjWQWI|9@<_I;jD;sBuU1>eJX%RLJ;b~Jqo?b~@qNlVL&N#pG!wiKF@6fSaC4D-biV|jZ+7QS3'
            '+pyRQn?;SfiOW`KaSg1;`K_W8j|<IX4R~3pZmx<429asmv1*Q|ys1p3<ocD81&V6M7+t1nkAN>g)e1*zbwO^h`$rsbi&c=_LdoE3'
            'tpzY1yw4gh&;73A-EOUA8P(COdE3eId3^xDyn%(&z1l;W$^=Gn_Z{m#;^RMFe?0qm{Qltek-g2I$D}Kl)P7j)U@0mT6mFKHhRm}$'
            'Mb>yVo&sR4Z{_8aK*40=TQ7`9SWfbD$3xA%a-|(SO~>mGT|yM)kR|E%LMC!CO=Rl3_T}^L`ydLA-@C`VFWsA$4Fy82`XE2P?Yr+^'
            'smor#WVGsfa?BEqrhK<O4b2>n8*hldPz8^Zxe$k&E-*(TAPX{E_Dog=C5^FHWFCCd^*&UtMdb>{EB@9aSXbJ+L`V2|0sJSExRdyF'
            '10uSIf~@gm_)8rsqZRJn87LxQ@SD3*8XZNA)GL0l!-R~T_IQb>b-G%UKDMDPRH<O{<@S(~_D1{o15Ber<!G{tFC3e2sa)5@!o%LZ'
            'dTonlzIFLfJTYZ$;HlFHdV^aBR$s7#vbA7TDj%ttigwqWZ&SFYW=x*VnDo*2x%zTYd{@?hzN{!!y~I*Ohi!|Uc631k>bTR(Vggx)'
            'W{{7!{gi78(ea1sj7GGxxnl0nIBe>-2DXtxPL_k}zSDT@YM+bNmCeTTQ7be5&2TrV|FqFm%4f!p*M72#JDI=NfG=$~EEdLc_}&^E'
            '>@frB!TBP!cB54OFQkfRaSl(|J8V?~o8=)WssVZjC@`}jv_pHjek<kuIdt|$y}cmNLOk>JIT4s<X68uHiPD*uwJUm_Dz;lspq<sH'
            'l9{C~KN8>i8D$n5YAtrZKzIJE0$iI0-EP#jh!D~>^2=ejn?Yegh_FgXsO!uWw8Qs5w#%h1l=4iqUwZ4$H(6&UP@ykDSJpw3SZ6^+'
            '$zp`;o)uGS-4$t~NimU{o~SBU{z3&za`lIyFNVNJX8P9e`sIT9=5us@%<+zM%q38<XPCaA??6-mrB;D6RSYii=Q2>&l)`{9<Wz6B'
            'p+BV?`2Ii8!OQ9tQBljLPpxw|#jV?0x;awq5elwkp7B;FcEzI+V(SHy9ZI9|h0mU*trsn9U45hL@QS9Nqrw-h8x{D{xGL&9e)l8|'
            'n;R(d#kXovZw;@n2~<N6G}9(M9$U9sb+fLqbmA%+d%}Gro7#XiYHD;o)ruKyWC}jF<+skzQ_ii<Q!ja2YZaDnXnTZ4^vfGl2qkK6'
            'sGIFs)u}nNgQN#uU>!u}E?bQB(eW~>f(8mbm1xmbQmO=u<PWB9;QhCBdzdr{9+<U7%-`V(nxx3})?%lw6*k-`_kRKAndD~'
        ),
    },
    'narration_presets': {
        'sha256': 'e20513c4b0498c302ef1dc00560e424a09bc306ba8dcd504158b89f2be2034b1',
        'encoding': 'zlib+b85',
        'data': (
            'c-oy<ZExE~68_F#F=bI$3MBQ@0S6nogU?n{UsAiU+!RF-$SZOsZN<0ll2(+!_uKE8S&||pIW3wGt;L<4otI~xncXl9Z&j*e%9ORL'
            'G<im@+$GwblcJ~GM4Fa1PpMpMvU+7)X?Ij4mBVtZ6Z53;J2qBFymt}=zx?nkUD8kY)7kWaE^n{t_U?fmez~JJ_tPo8xtYGdo(1E-'
            '@Gtlvr>iVJaS-KMvZGXOG&!9)O=Z5)u+dH^yjm-pXCZA(xyE+VpCVK0I4G?$iMAaj#!;4+;#;hB0nc;0D|NzBD%<55j&~abXPMiP'
            'SfVIWRxhh0*lST`siHNc%A6kG=*$}XNP{?!s#Iq<m?9XYqEcH#&qU|+YRVR=MWSB~X=zsL(gkdT54Xu&sZ~t*lH6JqCR>pyC-jpt'
            'mj8xeN>vuaM7A$O@+1yU+=$k!Be43(=q;Zx5~uhtC>D_-K}=#D83*INS5cl8$_uH;=FX9=GKXY7u)Nhym&6%xJssE6tDD))-7SvD'
            '*~e_<ZS{ZG2C?Ls3sh0q{7EIWR&Fh4nhZZptjbzYoX?*m@-d($9yZ%pM3uDFRtGDqB&x`XKY^Xi0mv35<1ua5CR%fTEr*w}JX`0s'
            'tQBInfCB*x_PDRn&=*no#IOp24|g|LQ<_cRPp=-}HQmev+Bdf!rne8%>p#PEFcsS##m`5APUNj}I4`kC_YN=xJUGCQt2|#Ndalgm'
            '?exc++nf2r!)z|c&mKqKwIDUt=C+P;wMdL(1isa&$qYfh1{1ymJCCw?e)YrU{pHoe^!{e{aC0?#M3kf4#-b!ZEXG^>IzTJE6=SG@'
            'vy-@M03MZ)#gae)z#e_H&`G`xu#CS*RWVl|A;|EV^9Tq5uUx`Z+DhzepfLAfbA#a{Em5tGlDvvh>1IhPsFA1{B#e94QWPJ4`O^)o'
            ')IN!>DhlL#5`<wG2!YM#%c`ubp3ljog<v6rq@>;jUS`rpzL7afKoER;PFxl**7rU@-nIz(4ZvJOw&fTU%Y>C(<%Qw3hGh?PIL-^P'
            'k%TcWsB(}N_y`Arbbdz=&2onbL7me{!)_UEttoNNh%yNpWZ4=-Ju2JSnRpVM#0R~mF9G3S!>H#J-emA0MbMsfIP!~cK^fI2Ui?vU'
            'fSd1^$x3Ht!3%G#dV<HhX6Z&1x~v>8T_sxC=KH;JDq$WsFPt?~u}Vs{(v}6@W%)MI@k(=G_Qz%>Q7Ny#F*a>J-&ucN_uGtFTZt+Z'
            'XV!e4W|2-*?B)N1;w;y#<(}SUN5kRG(z_hK?@xFRg4@gc`%C7Exo?M^h;qA`ciIg$`W0E-=pAc15*qDA5``*bq3grOHJIKG`;ytJ'
            'LdfP(9*coG8~;4(N(c@l89vYZrElpMAgDyK#`%`ZIZ}Whh1)fNw+3O#)_;f0oeFdUg8!q^&bLMKT}a!Gx~)kP?y0`-Fx=N59%jMJ'
            'LDK7axwQP?(a~os0AQR~sCedgraPv8;c@N^OVkO{IXQ54?kQg-?Dn8t?0}su>DZwhz3Slc6?c?WD+j#CBTzTDImJmJ04FIa?CVSw'
            'Q)u0vM*Hrx<*uf<C`+3odvMrfn$L)OInA_=-Q-Z~Jv&cu{+v86Y0w)jjBgLk+@mvXS9RWMDm_prEyyuVD1<4#pt^>kw<%$$g}#_+'
            'maAPQ$e!;BrT)a_e{rbP^8-|Pdf3|)CAs6&W7dFW<Eo$q@`(A$8e#En{10^2P5TKPh<Q#IXw-jH=pp6qg18R9=W_qPrUD$zHHm|{'
            '(Q+yev?=Gb6>Z%n#nm_2OAb2)<1L<7g^-))zi!b;f9Ot`2zS8~jm(JbyNj$TO14G$d0WOl=4GCuX@<UpiFL0Lf&fWmHFsOkR<x06'
            'ZK2nShaU_AKz)dI+u|o|=b5adr*jG;7>tl`A4dWc^lXmizHDUyfJJNR)K5Dv&07zTYv`T>EkINLqM(aWT^UjeJ#2f07#;VOuk7v!'
            '?XyCw4XFhe`#;A2{Ei$fZjz<oDrz&~D8Xb`mp2tE(&roo6g?ugJ1We!h8cStu7pp(mkGUNX7zW2da#UCl8(C7$|QE6abLiwa&eU@'
            '&*d1|*~hwSWaqj{6_Wj)XPUIYWQ=$%6{HVTZl&8}u&y`5jcr~fIKJfDPst2N2|9dofe4BhZ1O@Qt{TEfITz_+v5QN7&cQ@YGW-Dm'
            'n0||Tz^Y?VFzGu!4Ht!!GYaEz2RbqoX)T;*Vv(tsF(Zy{J#NDo;634)9N1O-A4vH5ZD}}RhZ<}5CUor`LtU>&jh8s7D(j0G6BdYA'
            'tybdpQC`v<s2;a>z&fQb6gc=z3iKqnuTc`XIPG+4+y+CIwi^v#W(jBAT+~R6(KP?}9e_%)S}WSAC*?-8sj*X`u#SAKkpo>Rm%>UV'
            '6Z%luRYj-@ZXnfXy)&Q@zes9>J6xt{VJ!Yk!2bst2OGGF`(gdRC)R~Vm8#D}#1K&0fF4z%EJ2fElRoE`KRNSxLYE5#R!q#H)}>Xe'
            'l&APStLWFecc){B(LuVLXOng;aJ~NHFRm@0O@uk+r*v}WYuC{Tm|*&-FB>gQ3Cr*;-E)KV)GeblDk~`#tL1dmbce!a_k0P$;W=**'
            '0`^&zMMjt-98n1ObXYs|JZi?ar2>h)Fs=EuQ$yMZ1xnf(zBLc{S;69ZY(9T=OuL2HsBye<%?(~<J8DEvOmH%Uj=nbSh)C0d3sV3Z'
            'nLcxbtbmjaD2{4R+P*mnU+8^e<Da#LFnqDF!Wps;)YTy&ZV04y3K-0cy`Z)F3{V5!!lG2|8u<1tI!NU{XPCjSr*AHQe*Z8(oW(JU'
            'w;4Mc)aUP}zw)qNRz;!*M`Qdj2m<V-<6A3@|0P%Ma}Sr08=+&+s|N81e3kihPH37I<xV#6FbOfw$@RU=HMdv*m#?v`5-7#a)WNvl'
            'KFSv!y@0?`dq74<22XGUPhsA98#LUfz5H)Ac0LNBV)N0B_q<pAjR6x?@I=$6MzJUS0&||v>Fi5=!kL_K_VM|wd$;9|!k(iLn9+9m'
            '=749svO%>2QB_Hb!JoHmOp5klhT*UkZV`!~_#-E{Lk>Df9d`ZRi_1e(`|I!d-@3cgF}eVoc=n}L3YW!4?_g~QhPcXRt)?1GFg#D5'
            '^n71L-j{(S_tl7c1o4_nPv91tXb(50PcGx$d+s91qmAICLE(t|JUXGn#Ync^i%I^Tqxj7Zfnde$jSs`eDwp@adVuY{BKikC&(u6X'
            '`=-uMeNc(P6uafA3-jrJXLq+_SNf+(Kb}&)_#JFP2e-8b^76v@nuiz;1zY}7O%Ep!GR0t6mx|QG>l5X`@L;uGu&o6yIgaZ-;}RqK'
            'ppr^Y`Ms?V_hq<1d(2Y3(^=jBQ-0NA1~X_8e8e~Q`2g3z^g?eE+{QA<Fgyk)`Dggtfc$S2>yhcpur>{810>l3Z@KU*zrv#J?{W*H'
            '4hk1O>@68Irt}-uN5MevThD7=eQD=6ulpN^e?0*b<L5T$d((Rg!99f03=_D$ILd|th!G12V6NK$_Lz9S^tM0u3%Aqy{b^`AjcYs$'
            'd_3{Aa*)42ZJLezsnw5PnHpXSF{-Bbscw-q_xN3Nc}I@s0?8NHrq|rAjROtr#{T`fZh|8W@<}ProkA73<NHQgGf<<rZE4b*`u*%{'
            't?Y@haM;WG=6(%qM=j=W0+avoMbKo`9_Zpq(;2?>r)EDT_HnYG5cd}F?@orlilV=q{{wyfDXiJk{d&@ZdH*#ITTi|l3%wqgIy|Yb'
            'JD2u0bXGc+Jom~_f*t%HK~fHW'
        ),
    },
    'source_strategy': {
        'sha256': 'bc7ae7fa6b04eb0e0567370df50bc7fc78f6b67c9ea81f952a8b94baa00bb12d',
        'encoding': 'zlib+b85',
        'data': (
            'c-qx{Yj@j5vfy|AiVpIDL_$)MZ<3uWM>&U<7@OC!<dx)PJX#$RBta4(2ypPQOexymzV+yCbmJjQp5(*&%tR7^epOf3yQ&?>dB5^f'
            'U+jxsPt3e1j)IvNiZsrXnJ>IF_0x3YN15=Vx!5N04E_$HWiQPVFY}i>u?@W_f+zh}>-RSw#nqe9rFeIK^5JwShW{L0UR}0&wf|bL'
            '{6(DjB3lKi*u?WZ#NU0f@zTssL=<QKG>+Hek9@QB=K{wIma9xGl6WH=PtEXCF!$q@|2asr)Dc-MoFtE6vR=4LgS3lXM$qSO>Wg%i'
            '1l!Ddbai#<bj59ut!Qj-mgQd9f@RM0Sr){R(986rSv-e%#8J2t0gNERI0m>)-jCi7Pe*6NF)VsG!3GEnakcU?7-a5?AZrDiZ46+W'
            ';u2%g@m!JlVF-P!@{Jb>gj*c>#b9Y3WGO>t>m^w*%R?`bEAF;BQ&&s_*p3XxnESIJ#i>HixCRNAAQQJ<$}<2scWM7RGgmBgFY%(x'
            '_qn;(nuWo(N6SLUMgAv%eirAmm7l^4w_Y&c`*$8^UT+phX}*EgiNX63Y_~$30H^9Fyq)g`XD5TJ^NWuPVE~fQQFa@Pt)IY_WQ0tx'
            's0BbfJ#4iE{7d{a53_WSCqFLGuK!00OB0Bkw>aHfKoH+c^8~iyd;REQczpiu-SF(h>a4)lF(Rqzs^|>C#XG<o4<I;xbACB&wXSfw'
            'x;dgJtbvdWcNW%)i;fWBK{6K(bU*V~aX9xAhd!od&4a|BWn+ZzIQ5gjPicBWyo2clTR?_Y3^)?wI#bd6MQqgb%T=6YX<wWt!7>0;'
            '7G6FNVjOb|i}RCDuzCQm><*VfbHl05cagUVX1Jo@li$0IVSY1jOUV2^fF^=Difyp<!yuwn<f%XJ%bqUjgovdd`GASo1UC^532Ye5'
            '>gwXd@zsZm!RhHoSX}A>KF<%u1f~>+pZxI_@nO6K6iz3y!?_;?xce(SD)b(XWE$&3?<2{H5DMNJj-Nni!}?$QfM&to`PPpHqdnQz'
            'Aj(!ryoI#^feQNw;mAwy#LUM4f;ic!Rrn#m9ccz11qI=$KZD<ZVx+Lw!bi#x!RtNos+YuY907mTk?f`Ub{hu10b=OC3kM%)6YN38'
            '(7^_h(!eh;^S&6s3A&Y}2Ei*~>Emfl;(WO>HjU`x=<KyzuYmWNFFKQD97A7YFPOmOCfwJ!*p!JYY1kr+Z&TeL^=>1BoCcs4?jih;'
            '`|oW5xjz9AW-AY9Ssua9Xo+~xdVhX8I{qkWcB_Zj4y^!i0K+-28dfzp`E7KGhy48P^kZL4_`r@C;2n1&yztiBrO-0VlL+THN#(AL'
            'aodm1fZk0AyCz<q#bfpM=vOaH{mDzD|0`Sq9X3D`m@3g*kI<$sXNKn*=QE9y2odp9;NOlD|KB+ry*c-fZHc<c?Vs@C@wEXnn4yLm'
            'py{eNjdN&>5QYs=@aoI9V@1C@IDsw@0GreRPwDsj?!?7;4uPQVcriqA8P;h*g7j&;jq`9WPR>OLM>j<-Gr``j9Yy1vn@JaDdb<i{'
            'D_pk!8Rh^>(_QI}#Wjc-Of>;Aa*QML_=-Ptnb!g+u|xiyP6@wYd}o~ICxDvU93~H7$>cX)vh;!f<cU5E3`mJ&z%CZSQoSuWL=eFQ'
            'gE^83d(Mam8~7TJ_YNU<76Sk@FNS^f=DSH3=`McSQw`}!z71n<PQPyRX&9s{`aSh$fP(-l++gHv*hv@2D4ygw&N<CuWF|Phe#>#3'
            '76};R@gfJ*^~Ym@%sWB(2T+pMlD1m%8A3S>rs_{*Fx(thmj!@r@{Rf%AP4`w1LWuLwqAxEsL$TRpDljV*I<$FfFIA+<6_O}%6swS'
            'Ph%~zjf>IyOcxPm#4mvM$xb>%El!5N4nCY-jn4)b7lW(O`PumW#qe@?)up%Pr<YyDRf)=%`|h^fR_pR=aP{GG{M+F4!|+lZiF<0|'
            '_@9C22Z&c*2xkKD_>&*TTYNA!c_zFnon40qb<|<SPXZ7>7K>mOU~_<|hYuHC5c2CXra%`DtrkjU<C76c2v_6d)6x6!<?v!eb9Acw'
            '+2PL_0>hK>)tey_Irz53X`I&ki}T~*<t0#W68x~kPQzGTQP(R#?b}14WGFo0b0P|OwCC|{1j3;2Z6wn~ah1d~l-Q8@eS3(lCm=9J'
            'Oynn03=`)Uqt~Oe!KsuPZj&Gbk&C({4#$8-83>LC5`d7t0pSEX3|P*ONcuqrnP9hgAb~lEQN~0vWrBK3v>C@#0tOP;lqGZ|1<;|E'
            'MuEUDUIO)2fMn7=PrTa{->NQ()|!XnifKc!KZOr0Mab1pC2dO-36*QaUT%{ZU;3Z3e(N=X|NWc6CGVF*%EMT(a6}@ef|4B}#f4`D'
            'kR?`{|C|bYrY2d+NR5=x;jMm$;tufOj5w+-hq0)i#DBo3YPoo3L!XFpzrd5&N5M0r&5AyeEKL%J>Jcc&ON1G(0EM3H*7qZ(*tapW'
            'AeLkq*)9D=AKeC2J`_$YWzsR>*Er3mS&)TrcDhBw5~?2>=Le3Qx8<X$&>#4i2S`uiAPQ0;FkO_FV#Ih>Tu?(IE`+<Bk~B)9<y<WS'
            '_jJs%%a_-1uXnMG0f(Y<mE=IC9mg&9x{4L`-+fCmi5F%&?=Aqw;%`N~C}iD3ga5WuasK<6K=B+|u0mujaAId+Oq`GafEXdBJFPNK'
            'mGbQG2pUrmvjAp6)Guaj;BCXYs<?Y7S%msW;+T!~0FC)km3AKLg#(8CL0SuJEyyr5jWrGgIxAq}5>P%7j@>6rPkkj8&<TjROq+JF'
            '&7kr@Tk&!5uXlrgy%azqr_h6>51@+z@mMAxAc>u~4Z@9gXPi@Wxp*r7lLeX!6YChTGY7>w1=N7~t?_9Hr!Yu>@0XSbl0*hG^Y8TR'
            'Q6PWD)FmaU?&HH_$L{Y!^b%@437{noYM@Kx?SL1c79NFwg&VoEj4ceg0N`8#>DmKLcin(vbAK6x4WO(=`r1Wijc|HwJAkq&0qbgT'
            '@p^bQKKUR8s>|UqP^X{vTki({9G`!<djA0^)A@(9D|mVUPe*5R>&An#*Qdkr@teT~a;g{mL@j>>K8NZj$gU#wfIA@HyM-0Ux2Sk7'
            '@-XZzKpX<)92ATUiI2xdh2HB|;y4by>E1~Kx`n}+nPJXjP(WdK6rTa|1lY|IYsz+|g)uma8lbR)%nu?pcGihJ`ZTzM|E@^H&Ok78'
            'r5z0PsxUMZA75IT3Ud`rJ0KY(y#*$Z96s@T(&~m3G5`&f1a^4L!ahFNst{00Tbn>T1a=qs{yaryGWRn;>VE52klBV8?=MDYSL5Ny'
            '>*4s-$19Qt4)$OCAikp?Er7S^{dNDKw@bCq0*v1y%SYkmFCMHl_ZJedV`IEAs|+2K5xT;kFa7a!mmy)p$sCD#z_=*0fG~Q<0=eUy'
            '02~vvSHP~$to0DSnRZ#));^o~P8cVH2-Qf~u*noK4bkRL#F{^UUNMJ*`TX+d&xkzok4ayYMvRk|vY@~zkhp_Pavm%eAOylp*p#FH'
            '-t*@Hse2%5f#93^KK0Om3E22{wPVIjc0LrciqnN!Utd9kej-ExfT+F^bX|HHET}0y=m+T-+jiVTO+*qefTQ*+JUxrEU(wt!Op-Y1'
            'EEH*D5Ki+j2QYc_0O&dP_26*#2sRj$x=%qf?YQt`j8Eycqya}tN$tzsca69Ge2bbX3`srraXTLVyvIHs{&f0$`Z`XYEqecOTy9Cm'
            '@_6e<5|5l@>bSy7VVr0l`o;R}A^2?{<(N)^da_RmEn3kpCT3bBUv>ezx9)7b&z*fHIHD(NJACFG*i@xjto+aX8^DXt6CPDxS-2L9'
            '&DLLz;p}$BV!^NI4~7XpSVUyzq60J!W|=-g7chZT8&38}v(D*P*4aq0M$_aXwumGlggj0YHOAv}p}{*rw1~T6>dk;=&s|a#gXj-s'
            'B<0iOr=*2Npd9$g3{jJnuo(RrAd4>H=;7YSIi3cf#eqjB?<I<*03Z+e3@RQ0TO=pUBFC_<FzxfKsL@0bZKdWcfOMxcIoAE;8h~VT'
            'N4!4K3nu6cNy)o1U=Rz`{&Tj*pPgU9pP3hu7ONZ&CamJMv@Q~{AZAgxw^B4$rZJj$L2w6B1jKBavrX?QhaGLC$z1Cm)BxXti~z)w'
            'KQsggMf~8kK6$2$0(i~XsXnkzIGz@pt_UAiCJ^6@ZWM}vKsSSqN^ft2InA&1c;%y?t>(FxjtK+dRY}OBcZ9n&?<7Vt@>RQ4d(+s`'
            'N2+MGdr;(6odMc^cwNv7wu6~Cs7zo6Dk9a=U7*6{-GN-X_Wf-t&%@=(+ukCP>>>lHwIEmKkwP;3wBtza{m(NWG?yX$ATJA=a%To>'
            'AY{6>0B{3?0G-ZuTfYMzyZ!MPk!C!80KNbZ9vq`NkuvV3)vETWHUs;Vzt?;Pdx9UJgv$(vEK(_2PH^=oi`_rC{iUCE9Qh770X)6F'
            'q3;)DT_r145UJM1B9XVu#<(8nk>7G(#S6z`4`zcSsnIynUX|&&7ifB8vyYk&%+-}Y<I{0dnj1fL>pSLyQ*I(37i~cQtt>_!p2i9D'
            'duzgpcRSX`ci5BaroaYlyo~omKU7WRge;_1BU*F=2Tf2s=`29%&N}9}pr-$qi+f{DMw22F+dEgVR+4bK;x{kM`FHmTgX^A3c+FyN'
            'zS(#Ql1rt#21`(K)6Oc+<fD{~#Z@*u{0wU2EMv1N8C!&Gtj%r6{(LldcA26Odz5Y>rAteOTv&2J*dmu7d(>0$YWVB<#ZZ25fV4_k'
            'xk<U2lxND$Q?&SopvHm{#(hm0rhPGrq$3IYn|Pmq2cj$1WNy<qlLj?j0iQ?Ik%-TNHpC)DFGP#HUN)I^K|t1KCDzjx-ZTkjsJz>p'
            'P&Kodo}Nz)L6=)ZP9%7}Zz|jIc1&}va)^!ZMNRM0t?$oPV^A%VwA7<~HTR=5$adq|Hm%`l2IyHnu6~1&DDc$$mO}}4@%p7m*KvZu'
            '89OhGg1ea~lwBf-c4#(F(7ZeMc3u=DPKkXX*$Pq=I6RQ{Dj3@(bhS<h;lBhsV~O-_ABl2nNqmZ@j@vM<_WXr_*+eX&A}*mAT@HKA'
            '0KtP<1<5o{p9zwt#W8~M@jDXNl9n<SxWrnBtElzEs>~HxUw4sTWnDq!%XOJKnTUe~JKuCd&}mRsldTOK7Xifq4X83O!;~DY{v0V5'
            '^tIpL*Zhw^Ga4B1%)A8kGkzgw%t|}GBKp_0UV9-x2glLx4Sk2@kec4$#Up*Gw?<DEPbjbNdl&<iul~M-Xa{u)P{Q~RnwmMQCsKbN'
            'CIZR|?9HSuN_lG_F=T&hq%gRpYNr~A4YM{}5@Q>n%MZlTB=XhC)fPqK4iJH42V#%LeR_8#B4+;iC3+?YH<5YI+%CnR%^my&xK!OV'
            'YKdX3o4LwdYv`q-c9wdrkN010U$c!w7(wSvgAgF4blAVc$i_%8t~v`keD)5Bgm=}l`Om5(a*QE(+|ZnP9~kJTIDuv$P0Ap!E}U};'
            'S}E+a$fLbbiRdxKt4WFuN)Bl5x!*&q@g-)7j&`=LyiaId*!VI&1>}NNeCzf#hsWiiCDwyohyJD@Cary<m_Ne#zrMl$w-jIDHA}{$'
            'g;wK`Q+G%~Ui@g#ed-(I9e|K@r12~B$jRcZ(KqtROJ3GEUkFNx=zRa;M^`d8MC)QWHQUGf{&@bj=(|UK!)6HF40y}l2Y=(=8S;C1'
            '#zki!5r|F?C&NazLOM4=V+pv#TJ{wLb&Fq)#KAxIYan;f2)TuWVUglq^>NtWUp%C;Ct%CdAVJs2!n+GSUgRuTgXkP;I35^qEKTO8'
            '+PUmMe=a}aRA3t39ZslW9{UaR5c`ME91*s~dueTHFhgY&=tL3*z{$cp*j*TNzaE7udyRv{;Veg_m<Ot%J^_+vF^AP`$~Cp#oQ&<('
            'mloIRiv5z>i!A_jKiJ%|4kc;<b5A`y2oGc~Ng7@fG#D1s7|+zqkNmZT1BR+9C-9<a0{dSv0o)lv0h-As->mb<+WYEeJactBmNYY1'
            '7a@SL$uc$-8EJVhYS^r`9jcjh3?L6>=YxEXINslHfYcXuY8^*;y77RHEb}PLmpR}wcYD}BSUi9ro$RI2OCakQ<h#}`)MY*kb6DSx'
            'PeWoUl(IY?upaX(9LNJ8Q&Hl<A;G`)+X2EZ0ZAVCKC;zhgHk-wUQi4%<^vuDi9&zL;qOgPD@#bySP#s(bFhzAdwlNN{#2TwbEsL3'
            'VJekW9A2tljkgQ<K44K0x1;6@4qNwA!{a;lwt=mlKC~=`UUF7H#?8AzhU?Biz5=iU#~rKCM+}di#&KBX_-Ar}o{Y4t<@zD=avnNm'
            'zYCfI5*R@l$8|h1hsj(F|M!Q{#qd&iqL_k;E-vOr0r1k4A7nZblzU{fR$%3RQ*Vdj0Y%^3qHBP?##3L$Adxu)L%7ruo*+N`P?8`1'
            'GiLeIejS5)@l^~8IaCI_BIzTb4F>RNPVr*DkxL%D_^a3?^kmOshvKcpu@0WbvL0oO#sG>XE&=uY5J*h~sPRPh#z^ugN25hO6FB$E'
            '<j^*|Yu`L5wOR)5(_8~!%U%KmYMqB6Jg5^V{zkI@^Iu8z=Xsg~Gp-~4?n@meGxwK1NKl}DB_{2!+bc`M<n0h&HCw|fA~yb~S)M5*'
            '+9vU4o8`P2z}dYe2KA^t_zyUfg^)l6f(`ShgEoq@#IYnhDtHmB|52*^9J$(&^m83-Lg2ZuqNNVl@-W;h2sZ(my%;J<oFNCTV&M~&'
            '6g$ePVQkbh_|;o~2O9tD^K*<B8oVEk-wr>zdNQQHY}8~iysQX`dhU8{-A0-b%PJo`R@|rPU>x3{6{nZ#G+R2?|EH2gVU;Qbi+qXk'
            '1Y=I5U{dx)D*^C#Eiw$|pMn%+g&FBg*#3|rS}0o#v+KnF{qOx09oH(yV@8ow=%7a)ALW}Vd=xL(W{J3qmNjid)k$!gvTHv6Oy)e@'
            'nVu=!qyx&G>gcC?d*Z+MUH9pF6LeYFx{V&2xem?ez;(+OX4x2VrW}UXHD}&6#<XKZp4x+}Vn^)sC~21Eh)fO`DV)X>GGcj-<&*EK'
            'vFD;<^q1d$St7-7uWFzs|Ffc3x^nU$3ws&GGKfE!`>|uLD%!rvgK(aTNqsKoM7qn~aTaH4<s?ZP>8tarH!|l^r{>Nr^WMtO6Z(z4'
            'Al@|oBt=zr4r1o2owe3)$Yk^6sXC-^FSgs~5hwp*Gz@#dI3Skt{Wt+Z=Aoe^ZD+PkI2e(#-?Zbhn&)PIMjKJ`tZ~54(G#PNYA{M~'
            'qWZ-(I=*(QVS((oZVVc%4B?hsfKPYCmV7$}S#J#3|B{(ilTL&3*GEJ@s^95u>Q2Q|{ylc?T+I&%$0Tzh-&V{y+A;$o5|!K{?LMGy'
            'Z>|g8baSHvHwwHSOIC-S-zXBRnj9J#2DZ&S3;f;`nywGEN+=!8lFx~=b2B#>!`E*ElcRo%NtRwNW5;GI-$SkxbC65ABBhZJRLHrU'
            'vpB#s)?nMW!T@L~{#;UeYcsTBYCT>1yCaZqrgKkbT6Ao9vAl4@Nd*{i(ST|O{<zf{S-`E8*Fa}6=Kmm#CHHbN#;cp_Ha!GoW)CD6'
            'fpMkzZkZP{Gt3m7s2<v5U}&_N?(%i&uj@p($FRp1^8pj+!*7l(NUJECCNw*&oQP*Y#efcW3}DGIlEPjuh3p@S`}RxG?*9RJVQ*nu'
            '$zSao_ra-jU$0!4n52pEGbiuJgbkQR^Ssfdz4XbqEtZrv2P~q&h-U6zqj<aWH{KYtcR4@dz>qps?lDJ59SWzyXdTmQn0Z-_5^h6='
            'rSXvjLnWg_#Uqox$r&!d;R9S^unq_?(nOJfo`%o?Eq1NK0F)DD9=t54Z~%IW%i;0p-AIhiE(d4g?cic?F@j?$Is`n=b!t;M43Pw8'
            'Yy5>ATIJgIqxiO?(rx>BHgo&H={Y8<V<m(T{G3BwV55=_!(0KsDDwwQZSk3@(0lsDQa>~fl6XLRh<;S}p@os==hlmI(pk1jd|)D>'
            'x5Sh`%v$#nRZnHM3L?LU!L}$=lcLVyy#Oy6wPf_h^jYPjst9DD=}EtFwm~{pa#YnNZ^=D1K|z-IR1@LDdK0`iH7DorQH`ojATWC3'
            ')b(*GJob^S9oxM<R{1BjnqF}Zk^5(UAwv~!3>D8zt#}2Lo{FwuZmg{5LOK3sq_S1n1@^Kqw&v5+&+KR{joZfHcHky|;68@+O>Ijb'
            'x{b%8xr5>>*9?wQ(E_!>2fR-AZ-^myUzB{oE%-Yk25O3>bXa5oM5roLzc=KA7X?&gW_VWzwKIjFlS_HEm?-OoQti5uIW44haj_Mt'
            'Yhb2ol7qH|4Qq1asYFXE?)-#Z4}~Du<2=TYa0NTYRTZUQio!2sgiv*zn9fW|S!5*%4I6%xnJZZ)xE&}w=0V2s`iTtK!~vyi64tYC'
            '>rYC`p-8t#RC0vNSvW;VQfJ{HUynBa&PN3i4N+8{3Pcm^@|)yAbf*3q_@E!bGvw)=vdHB=C|ln)4ucsQn=4abxu#wKAb>&Vj?9I|'
            'BmnB@n%>-8@85KlX{++?;O4=S3+XXJ!AyU*C>CvkBBN;-X9h}kg-Kp1RZPW^jT0|gr(KzPK2GA8DH?)ABk%~&o$~!<0TJw;Y*l2n'
            'l300(4-drP^!LHXOBH~R#hC)E&OrsqWIoB>g!3;z?+&6_n9uR7Vsss4Ir10`csRc)C40q4P&u`}4Gb+v(u@c1QsIG(7X=Go3;dT('
            'XD1{?6y3=nW*PO1c>z2H5Sn&I1r}eWV^d6xRPCp}AN6T@sd&-vpV5hw$A=tmAV6Ya2DLf~>$5XhCm%s7I|OSe41I-F0%N+Uc7rrB'
            'kbe%=NJ@IwSPN<ga+x;7Zz%%;1j9H_VHF#;7Xs&*$eHXPz)W<)=ETGW@ryW^@Uc6S#d@$iS!D&6Lq(4eE*)ZIScAsr7-RbAHUU~t'
            '+Gm|%<1!hLBc>CeRsh5E4_B9?lOZ?q(iMy_QN?j4Y@#J-&1A(JnJ7s9yX}^Im<gPRh>CY9L7E0KtJ2^*Rq@9nD+Z>k3?LKvWmrBO'
            '3V2I|!8|CO86nLkgb_z(rUWJ~e@0YJFiRFARy!uNUDwR4lbC^#`Notm2$p&Q1y_dnnXmUi0h3+c^LO8igTwt7b3>n^i77qk-WJ!X'
            'jdLh;+ftTy=z-)3@0FTY8KsgHu7QJcF2d-VTHF*X`I$l~>0ND#6LlmP1z?fcRvJKV=}QR?br{A*h8FE+(GZ|j{+jHUPPxkk6~id('
            'V$I$0mnqS`&GrYgr&5q4ZbFMvzK>ag6hO_ExwoW}e0B*H<o{e73ln9H+yhS>pPpY1PXq`{hpHaSCuDS7I1EX7Y*t!f4<55=EKe{|'
            'gN|op*i>x!Qyk2wGg)SZDia~PcbV8?!c-6SELjXuMhm0?Y$ei4nVW~X!m`9mguNk!QzLeTE;B^M7MZj$N$`pRK_FOD?7@_?5~&<t'
            'zjb_mb~QY^8lMi%UVj+8wki@uy$_cTvep@E8}doFKwoDFVo{K;MCZe$i!W%$_|r-O7>9v2mbrYYZR2-IP7WpKcO=^;@D*@ZC>z0&'
            'Bhbhbtn(Di1FniCoX4$_&S2vs4UL4H1mHtV5)c~XylBVq8yd!aN7jyTvZQx)VF87om_WJ8lQrlGCOER>i{QXkPXPxP9wd_j0-&9P'
            '$h*YVd+T5p2p(MQyu1Dvq3{ZDf&<IuK`7m@8;=&tHD`XOvX`Y(w8SD=$mg_diAu06wxiG2@beZVEl|Lq<%%mGfEFE5r5=3F;xC#@'
            '<IuOFaaMR{x;mwtm>_-&qBG~<U|%YW7TEz$A!RKcSTuR+xDJCK0*)(XwT$bSmcc09qM&H%kt^n4DNz*jWvQUl2f~If6y(ONFKli7'
            'C&ic{gi-5Tqu@-+g@kAxvM44NhdM*UN*JWF784~&%#&atE0-0eD^u*vo2_p>EKLb$0LU#PwRL_mioG0E^zYo`AZ|a)WQa1?k<j#{'
            '-?|u{oeVD!6u*vMk0oS`L$3s(@hyOVixeG8%TOXM+0S6!s#r+ztvV+71)4CpW|5`CDfM1`+9B%N6@@;h#o*tha52}%PnuSw8rMbX'
            'fZS@zlq<KMq)8A+G(A-L-&GB5Vo1)hRXLOhb3pa#u&-H(sj&D+U`L6Wxex-ICZswwYt53RMFkpq0l7M66qCk<2m_YDQ5c*WHGT<U'
            '6uxLBPZkB_K~TiAOBjNXjeh7_<?frKdM8iW;j1mbGcx`YduBcepl&)xm}(6)siR%gGee)OFqlX5$&!Cyjut4?9;JdggRNU5slxIO'
            'GpS%YrADU0nF96o=bg$F3Y}9f8#}WUyi$?MiaDw)EK{6R|MauzPCohWG51(STeRK9k2S5a%006OQ{0x`Yj*IBsgfnTua9(UO2aga'
            '7Gpg5U-p=r2}~wIsPUIS*1TL=@tMt~xXLa^-qNob`|H{pipG(nDlp43NnNt~q)!9C#<ypN8`M>Mp6IOI;XmJ>j*drHvV0ozV?|0!'
            '4t&x(4YZ#S_k?CZz_}{wT$AOwVz9lQDoVrUD6BMK`0y6QbmU!JrwGZRiZa0N&>A2k3Y={)Tgy6d$P0FH?x^5}jH|q<s<+a}P^o+S'
            'lH?0s+gFsX{TV6mFYxnr760N+(BROLaUmPj*n_M>k4qV)1!a*rtQ0v>1cGpwkLAa5k(}4KvL2zV&eESzOe|#)lv|>D51X?Fsd5Q>'
            '0{TLc0BdcyBI?b4(=X9<ng>V|C3B%Q%2Vxz6(`va3uWb5KFjxFu8+GWm)kC013HwQ!Hf!iBH5gQ#WIDp%+|Pc0ylaL*6)GQj4y|$'
            '!(-&MuRda`FGk#Myt|j|8|BDMzyUzcwU>BFFfRf!3rjziDk!6Z<p(f51QrP88&d}0M%;;x=5wFEbM_uR)iGU++~k(lo?Q|Y=VO)i'
            'Q?}~Vor?yCmkeh!_w7LZd{iC-o|I$K9yNAbvpMM9mm5D|1VV_y<lKY(MiC!iVGsJC1rPbu2>!zrw;ti}k3*o&QCCDg7~)%v&#qOR'
            '3JI?tEqLik{#?qntASG8#DDBTD*@OZ?EJ@pq?$eGfL3WEl(O7C0tugSemVYz1gXw?_@EbL*u{P+A;B~7zvx8=)#E`%_dXnLOPV_>'
            '?|Ow)VVVee{QVSFDA3mh1>{h-Nbs;$V}O*y11-11DMu&-?*dBSqP8k<I!N5gc_Nuik~ULJtD|k?c2*5gfa_EQy9R!6^d-IGTdVfG'
            'DoL#Bd{AL2bt?cC0AaIOCX4*WDA7joG@kE_4ngf=$wX>@q==!qma5{t5{yQ5_FF8fIQ7~3)i*kuR8~4cL50l;zJwy?`j)SI=oWqq'
            'qdOkCosy@*i2o;@E-FO~W07dO4FNPgX)M901T$WN>gmDz%@*}~HW5<2zKw@P#-tJrFmpyq&dOv!!W{Ep94(QU5eri?ye#lK3yvCv'
            ')^M7Z&=$>{SZF86oW7YNVW0>Z7D-32vmYKkKWsds><4egIROHLfY4Aytb-4LX4{!6X8IBXXVO5W<i=F^)ym!=7}?Xqqe_3d;%8A`'
            'D)hONUzF>bb07~bX>S)N`9>~`6L#<GN2Syh5VZWH#yICNIn)^m(;T}5z${S~Pyi8~PyVh9DbSDuq(-jYs|BYL_xr|5GJ4_;mSMm`'
            '0H%$~OdAj}X8{V5G+oR^H_76Gr1*8Sv;)!*<-Q8X>b&tFV<NXP(NB?1lF1m}duYh7)EI7nr=pf1*SGj+oui#nS$%p5kjYOTme*w4'
            '-!i5<DzpQRW2jki99>%FSY>}?)8P`JAlntM`$}`h55~v@_e^=5gwK=V@#qqHJ7&1_?J((z1R<kZ42e<OuB`YN8FOSYfJ$gsO+2)Y'
            '7L%2>svlG%%*IC1KeMrZY&IzcJK1<{mNKagWvYlC)$ktT`pWNSc4+Vp46alBwG@_BVKMlvCRir9e=p6>hz@EGEhjBnCRJ_VOfuGX'
            '$q;DtSN7JDlim!G(GBUIr_xm^2^M2BS-JK?X+9oAmM^Ms1Hk%LaN5Y5LcWx~l?!B<eu_<~nu*H~N`|-ELeqGtI+le;>PaT32YTb#'
            '+773ToOR-RvPUqWh?a2eHp=wX-a78q6eLDb8qEFb&ZWR$XXNph!<4s=4|;J$6?Rs>_l0JLAq;!gtwb<)6?CW~6ZGT_cx!uAC0w=X'
            'qLM0I?UapPR&W}(Vj5EJtJ0OBiqzk#V*a!Isz+aqCT=;f`$=MKsXCt#S}Ri%C?-uD)kdn>6ZlJ_mn-(CzczMRnq;tR%n;5~m~9@;'
            '^OrK@cyx9=JRQCpUXDP1e|tLmczz)|Sdw0NJ7lc$U^DkB#pfHXY81#1D=+IWW>HpknbV=>NoLJcc_oO7hOOSwacynFjoG%?DQHPQ'
            'jkmyK@WM9UPyFC|%TB+ppb#qhz8_@%ChbtosO7beKf5u9QCGh`y(gqIO}#ZIQ9cQsD=d?;IjaF}His|sQ&AJjRu`f?-8qNaTvqo1'
            '1*98)eN{j@S<7V{3H@HuZ)@0bO`LgICaQd54!MYVZ^m|Hcik>LtKY&@2EefHl_;C<RI$E_)MUg{YuW25G1g{Rq+9FDvmEng4EvfX'
            '6`o!^;yq_)Qt{BM7D%?XuKc|{%Szja=LDE5Hd=A$u|>atfOLUXh1h30mYNbxtI?hpGV#pQmwguA3eb@d*Qx4h=7sF?&ZE>@kTC*@'
            'r}l1N@V$ZJ`lblVQ2oHv^BD!lfH;@ZHGPA#6M4r393{HA)To-y_eVAb(8o#N3W}y+1yx^_`*QT?(2P4%lFAOvl*!!N$r~BV#oB40'
            '#HYK?=8o+)p|Z|c<pscW4U4+5()CIn72ewVdZ5>39x5U~h5v!_9N%Fk4&R2NyC(NoqYI|YkBi}W!a9-PYAo8Nc5OMTs~uVF|3|xK'
            'SwiS)R^K()W4Fu<>rm9<dEJ5sbZw&Dt&jlMQp<U6(h-Eyw^ej|<qPKy%g=?m@DSMkIt|u9QK*lH!pT&CSuW0{O3#&R>KXxD-c>IR'
            'JyC8heK5M@T%qL_<7<#MRe;rUNa3whHaC3SI74#tU@*us7rGM{T~kga?Vg(!-XNHvYEd|WiWxr5Ggsc2FNN+q4Q81FRyNAHI>^Kp'
            '!ASCY3`c5AZC1~sbFas2TW{Jf%+uwpIyBSfnMc0G4hPjX8pH6eMMC$-<o7~suP)p;NVIIb!^akDwxx+<LP8d-y)@VX%Hp+H=_c3#'
            'RuLUz8!3_pDj<f{QxJ=%EHhivdzwAo)Tt_HRZ+|<Y7=6aj@YPdTxqGb2GjBw>_gf3+UIQ5)6-vlIhv>m&mWB<QvDbBUI)#ZJ2APQ'
            '(c{2-6c1|bl*ZHq7RCWu^=JsE)2X{<?W@tE2qP}zMPWcNt%OAl44-r9inu$=C{V^B`MuoH$`z_#-cf^csrpVB8?y5fy7z~=w>MPd'
            'Q)!B1SK%XEna4GEbaO4gy7|(?8uGaHO>HP;$a%s^&v$vc$>E^Gl<iryX*{BB4cotPV8Zv!GnPHvRIs}9sI@~Sw%eO;89Cj`e7@X>'
            'O`h)K$DvFs*a%@-?D2FEW&BSMs=c7+9@6T>@Tl%g!9ogFmRzBPttPQTzBdZXqYq0yB?P6KK-NO4F%SE0qx_*JC4I}5Uxv76!FtS+'
            'eK?DIGz(ZmQnU(RS`zXxmfb(lRL!Y3PKez^^WZLss20G~kM1}$NR|np53sJoG)Ulo=nnVqd}4iOoxO_2R<q^Jz6qLdjn>zC!Wp+O'
            'AH%(#d#cJBv25hl9dl;-&{t!Z8Y&<JTZwUaxhLRjDTXOnSlU`qaL=fsn<sS7+tJf30re73qkVzkCx>{JkXKVFVS{*C55|{Yt`K1X'
            'rL2S3)FQOyX~WNBpse=QTb?jkO;lZeKuKBVeFvQ<+%I8fxET<yUd+I~Qvt8y>w*F2(K^Cd1JH*l7RC&UQER-m*LTC0(7-p61?yGb'
            'n42$qeq6ib-6F5<&QljRQ0F0YW7*4t9dE4^9L(gnV2{Sy-^s{%nX~Zwo6+$b@%#D3$tB&_kZ+NDrn7|dL<jm7=Om_^24Mnxg{8{&'
            'Or$#|rOH$DIG_TtGQmpaRTX)klu$%rCCXyKDJ#;PfpDGXne1Pee?z^p-y)yutMd<MCu5mk@pkx;ug$LJNOT`hvTz>-3MIp=;@jdz'
            'N0p#(dVc=)vacFmW^_BA1WE}URA{J3G4AS0M2X+y#UrfLB|Vw)VF?fWQumi*-il8)8_Kjvf|+~_lO=PJncjn7>e1Z2MaH+8VqctK'
            '%o_<%L@DO%iba|-9Oba%G~0y)L-l~>!FSrT<D=33QIp%qYBZY)$(kjdTVu+`k(MOOlPK+kWpCKgqjtgXy7HdFjN8clm}7LS886>#'
            'Bt&4Ij(!{JTE#4=M==#X_7lsPr%i4|zf~~<mEgKp$oVo+$WD1GVid@KiPwSZJSEya!>or%GxBOe)2(T_qx<4jp4s&b0NkL$<F|1t'
            '(@lK5*zncx^!#^)crrs0tEo^Pw{^i&QCg18J)9sD(QG-GTOtsOg%h#J6Uyg9&$qN;vhh;cEtSi=xGajw^k(rk)fcErf@55I`0qUM'
            'i!4nhF#3K7O;04rkegE6Nm+{!6TV&1E0Z@XMn28A+c4mi@s_OVl~R03(;1pbXsT;oXMPcW@5w@dGLW9*{bkx7=j_pGpl~(+Glrt2'
            ';{5FNBLiy@e8x;IL|W(xCSnHcg>2Sldou;^$OymroKw~!|F+1n<E%BjrxLaYtDembV9OGEGB0+DjuZ-pW5`uApZ-FVvE>&<gWb(j'
            '?1zS?jLd_wLs7iP>Wlb2-$8HqU^|Y^0kKF<w<N5{ugub|rixxAPTgj8UiM*IcyW%^Ya*>%hqUf57jmS3v+V<9!hwa8IadW}*wzEx'
            '!~J*P)n_Mm#dqJy&uq&BP4o*5@tdL*9TXi*jFArb-W7i$`gP<W_D#E$`4eeqqUYsOdsMjqPX*!t{)8>()4Gs{MVf8v&+Uy>p2TQh'
            'sZ@vSpzVdXB$Ug==x1MPzPP9LKlJaRkB39*_2|CbH<~Ii2AaYF*8$l$j_$1~!6z>p%o(ivc<tSnv>(iQ#@{FqG{PY0rs{TOIg@TB'
            '4$-NBzT1=yz4F-03NkWZPhB`TRs>tKX!I@;htBe?3QPoQ(~+)vRl1)Q0n{_frHCZy9|%xrbmDNAvM3^pXLmWqm;GofD*vcBYQgcR'
            'SOAOZ8Qw-LwLDFfFnJ2#v@5C#^|eSx9!8a-1(=Li7|SH6q9`TZy~wUb5>5nOq6GY2ay1ON)|2+6s8LFF;&M6?#m$mBPpv7m@b1u}'
            'huzdB1(|&oNuAH5@6w6gsZqy8nez8p4MikioYlWg0W32N`XoDrfxjg+zb^HTVr3Dx8K?VYC7-bvFz``twqCpBd=2Yd*z8Q@W8uZP'
            ')b2s=UgaP(m<E)`4n3kpa6l=}tQpiyX6)D_f<v_N$biBF92rg^e!||LI-=spEuq*X+LzSal_s~)w?YopR?BFs+p^Pqu?S{Za;qqg'
            'oKP`Pbuc$Ms4^tz82!B(ot>C*nw3)unrU%Uh#_hl=U9u}G6qu(*Z~@Wx3u@nW+*GVQpd$SaT)Q+2N7?g=XifBC%~XBD7c9w<`L+O'
            'd#PU*;e<$caJVC>F{$QFNs^UVec8daL+D(+didF>5>R=fwNN>ZDho{w)Mmnrmti#F$`hFneLHK<%WSA$hf-A9?TGz&b$&x5fRbL6'
            'a`q??3_ev?P;xw2BmF8qmEnEV9q@1J`vH>)3^y+tZS*>eICt44#nDh`J2B3=k72)4<sL04P9?$kl5Q8dLpJ6oJ;pkk`5v*NjknbS'
            't4bMcqw?j?a#l#GFm`b#PE)lvitxP)ww(enMUgkR`#7t2UH2J9-~Q^liEog5_7)zgi(4=9LD}g5S5N&J-A#lAwel?P>G&`AZ@G+d'
            'vB^{k4x(+Jj~s<t66j9PK{Vv*52&cfI4wsOmjcwHcMtd6!Gl5<j?y!2#k9;Xm-Ix!szelOqGJCA#?IiCNTsWqjoXrR9;D%c;gW}Y'
            '#U%wOxMTBPx&`tI1YOqKPSq_>W-gY5#U>!_lyu!l)^1yQT&_+B{9&q9zfzF?37Hl}P55e_uU#<IXcUt11oVm`5bnqk---YA4<j`0'
            '*TKI~Q0a-cgO6u}w_<dOp}XSp3SEBxGrAmM6pCSbcg;ZIN_PwAtu)ui>CWS4Kw$}ujQotY%Fk%Ku35uu;auXyN_((_JJ^@T=4NU}'
            '8_scCq*2>_z=~i-U)+wbbG4eI0d<U(RzNmjcNnxh*njZ@=z-!zR~%56?_RY_lRJ*@&~uLQU!r+(be}yg>~r-UUtCYCHu@EySCVt+'
            '=-&ERUH*bggI%w_rAjpAS-!{H2`t`KH%ocdfj3*j#^QuPP*~d|@FzfeQDm;!<E=-x*9CWh9>E%->PBO5x7+UZ;lWQg?nA%dw>o}j'
            '-c>~2K1>q-<R$I$jV5jFjA%<b)Bbq=wrzv;PfCG3y0;oy_g=xW{b(HQf-#oXK9XQEKiBsfS2j)FF;&;7dh-=yDc?6$Y}ifhWi52u'
            'uff3S92dS6fs!WHLQpq(;rAf4ND0dN+EE_Rr&Y6mel>w|5dj)2^ib?`K74%r;dF2|cr`j5T{!y21)1KZ>fp=mbB1THho`5b0>s`n'
            'z+Si#E@#_)9K0V?dTNSxR~?r+csKZ$(*)XU4hXvuj19#g@@4^-Zx#^uxWoq`I`=TpgBeLfel@NZj2FNavsBf`@pitFjis``LVS;@'
            '?%3d=)VHjSQ0iRQX?R-i-v|OWioV>?9HkzAn3sHSGLwaKem1-S{-7_V&%94*u=8C-4N6$lE{C-n)W0a&SwOEbD&DT=URra_dV6JP'
            ')~v>{2+?_j`6j@*Ir;#z@KYj2<t5cumMD{C-QCJZ$%k$a6pwdTlY~fYz{=(^f0vd2hiAjLr?4%D<~vp2)QV6!1i5D&=I8@g4p|Vi'
            '_?ax`dwwzcpV8Ui6gs|CabsP(<NBgebd6ncrz*g3(Odgru<`W;tKTbwK1fWNS)<@70~ndOaJ<{ZDvk%QFGi<uy!&<s<&YtQJVg`w'
            '$`g&13C%n4_T;^3aR}>GQv*pnh@LPvSlv8W2Gx}BNM>kD=*T<a_;NdnWk6-|$&1&C-d**8bzIm(1dIMdLYe-4Lg#luGz&0hX&1?e'
            '-=yDwav-hYO%q5|5$-1=VW$KgRngmgO?H{CC9zre8dRHGuTd(oW=4lhUVl6bHiFaN(X?|id_6in8C^)ToY&?%ZF2Y!W%F5A+U8JU'
            '=XC0@FtRm?yH=fX=l?_w7qTJJ;@T<>shO<YP#}E0TJt_mvqGIynhllzqp}oKk-qpdie*{%+f|_mKi${#bmhQ9?Levp7{LJByz-;`'
            'PMn_&KEmI)^Iy^mF7<Jn7c!~kZE7=0*mU#4q!6lEV~}@AST`wm{g985>^CN*u5bPn60K!g73)&J^djs=ipceqT->%j$&nEWY>WE#'
            '0OgqDirO_O#m5#Mm3q=X!L}s^wmGD<A|nGqNT1o0jA||G99KpprCMq+)b|)dW07=NuQ<Ote>b=qy;W60{F!;xle&cV3-PEeD>fJe'
            '$|`(Ip`xb8l~5n1vM%Or<FJuZQ!6#>@DeM`t`(cyZrPAuh$v|8K_|v;OtW}Z3@gF<16^@Smyt={msle=NZj`PE#U+S$y;O-gF-SX'
            'L_V^tv1yVlNMEpo$}HGI<lA`3npA~j*lSV`Y7C-M8-Q61Of91#F5Y8r57L|T;2>(jV<3U|*r{P8E)As3sXUJ3HmGCOhcN4_k@)!G'
            'w*i7jbj&!`jeqAJ8W~?Ij<zOZm_r305rn^`0Nv|0m4QbqHxeABzN*k&lNga%0JxNFR|NHzw9UsP>1MCY1L)^=2AN9zo*!9eoB|)('
            '_V*>qp}jFS9==ESsCE^hoF%Ux!-`o8AhTrP-e4i@LJU~On1x96<nqm+hqjBI5>mfbCis7y4fad'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
video_analysis = _local_pipeline_modules['video_analysis']
script_generation = _local_pipeline_modules['script_generation']
tts = _local_pipeline_modules['tts']
render_utils = _local_pipeline_modules['render_utils']
youtube_data = _local_pipeline_modules['youtube_data']
viral_prediction = _local_pipeline_modules['viral_prediction']
approval = _local_pipeline_modules['approval']
google_ai = _local_pipeline_modules['google_ai']
provider_adapters = _local_pipeline_modules['provider_adapters']
source_strategy = _local_pipeline_modules['source_strategy']



In [ ]:
paths = youtube_data.resolve_project_paths(
    is_colab=IS_COLAB, use_google_drive=USE_GOOGLE_DRIVE,
    project_root=PROJECT_ROOT, drive_project_root=DRIVE_PROJECT_ROOT,
)
RESULTS_DIR = paths['results_dir']
YOUTUBE_DATA_DIR = paths['youtube_data_dir']
YOUTUBE_CACHE_DIR = paths['cache_dir']
RUNTIME_DIR = paths['base'] / '.local_runtime'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RENDER_DIR = RESULTS_DIR / 'render'

SOURCE_ASSESSMENT_PATH = RESULTS_DIR / 'source_assessment.json'
PROCESSING_PLAN_PATH = RESULTS_DIR / 'processing_plan.json'
HOT_MOMENTS_PATH = RESULTS_DIR / 'hot_moments.json'
MOMENT_SELECTION_PATH = RESULTS_DIR / 'moment_selection.json'
SELECTED_SCRIPT_PATH = RESULTS_DIR / 'selected_script.json'
SELECTED_VOICEOVER_PATH = RESULTS_DIR / 'selected_voiceover.json'
RENDER_MANIFEST_PATH = RENDER_DIR / 'render_manifest.json'
NORMALIZED_VIDEOS_PATH = YOUTUBE_DATA_DIR / 'normalized_videos.csv'
COLLECTION_MANIFEST_PATH = YOUTUBE_DATA_DIR / 'collection_manifest.json'
VIRAL_PREDICTION_PATH = RESULTS_DIR / 'viral_prediction.json'
GOOGLE_MANIFEST_PATH = RESULTS_DIR / google_ai.profile_scoped_filename(
    'google_capability_manifest.json', GOOGLE_CREDENTIAL_PROFILE)
GOOGLE_QUOTA_LEDGER_PATH = RESULTS_DIR / google_ai.profile_scoped_filename(
    'google_quota_usage.json', GOOGLE_CREDENTIAL_PROFILE)

print('Sonuc dizini:', RESULTS_DIR)
print('Drive mount edildi mi:', paths['drive_mounted'])


def find_artifact(filename, primary_dir):
    candidates = [
        Path(primary_dir) / filename, RESULTS_DIR / filename,
        Path('results') / filename, Path('/content/results') / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    return None


# --- Faz 0 durumu ve ORTAK faz kapisi ------------------------------------
# Burada tanimlanir (Faz 0 hucrelerinde degil) ki alt faz hucreleri, kullanici
# Faz 0'i hic calistirmamis olsa bile NameError yerine TEMIZ bir "plan yok"
# mesaji uretsin. Faz 0 ZORUNLUDUR: plan onaylanmadan hicbir alt faz calismaz.
source_assessment = None
processing_plan = None
plan_gates = None


def require_plan_phase(phase):
    """Alt faz hucrelerinin ortak kapisi. Onaylanmis plan yoksa ya da secilen mod
    bu fazi kapatiyorsa faz TEMIZ durur; hicbir istemci kurulmaz, hicbir API
    cagrisi yapilmaz."""
    _plan = globals().get('processing_plan')
    allowed, reason = source_strategy.plan_allows_phase(
        _plan if isinstance(_plan, dict) else None, phase,
    )
    if not allowed:
        record_phase(phase, 'skipped', None, f'Faz 0 kapisi: {reason}')
        print(f'[{phase}] Faz 0 isleme plani calistirmiyor -> {reason}')
    return allowed


def show_review_prompt(kind, artifact, label):
    """Inceleme ciktisi: artefaktin SHA-256'sini yazdirir. Onay hucresine bu
    degeri yapistirmak, icerigi gercekten gordugunun kaydidir."""
    info = approval.describe_for_review(kind, artifact)
    print('=' * 72)
    print(f'{label} -- INCELEME')
    print('=' * 72)
    print('artifact_status:', info['artifact_status'])
    if info['content_sha256']:
        print('Icerik SHA-256:', info['content_sha256'])
    if not info['reviewable']:
        print()
        print('!!! FINAL ONAY ENGELLI !!!')
        print(info['blocker'])
        if info.get('blocking_conditions'):
            print('Saglanmayan kosullar:')
            for condition in info['blocking_conditions']:
                print('  -', condition)
        return info
    print('Onay hucresine yukaridaki SHA-256 degerini yapistir.')
    return info


## Google yetenek kesfi, kota defteri ve ROTA GOSTERIMI (calistirmadan once)

`client.models.list()` bir modelin **sifir olmayan ucretsiz kotasi oldugunu kanitlamaz**; bu yuzden
kesif sonucu acik bir allowlist ile kesistirilir ve kotasi 0 olan modeller (or. `gemini-2.5-pro`,
goruntu modelleri) otomatik rotaya **hic girmez**. Manifest `results/google_capability_manifest.json`
dosyasina yazilir ve TTL suresince yeniden kullanilir -- her calistirmada tum katalog yoklanmaz.

Kota defteri (`results/google_quota_usage.json`) **YEREL TAHMINDIR**, sunucunun yetkili kotasi
degildir; gunluk sayaclar dokumante edilen **Pasifik gece yarisi** sifirlanmasina gore doner.


In [ ]:
google_ai.assert_no_account_quota_rotation()  # politika ihlali varsa burada durur

GOOGLE_CREDENTIAL = google_ai.resolve_profile_secret(
    'google_api', GOOGLE_CREDENTIAL_PROFILE, GOOGLE_CREDENTIAL_PROFILES, get_secret)
google_quota_ledger = google_ai.QuotaLedger(
    GOOGLE_QUOTA_LEDGER_PATH, credential_profile=GOOGLE_CREDENTIAL_PROFILE)

google_manifest = google_ai.load_or_build_capability_manifest(
    RESULTS_DIR,
    client_factory=None,
    credential=GOOGLE_CREDENTIAL,
    credential_profile=GOOGLE_CREDENTIAL_PROFILE,
    run_connection_test=RUN_GOOGLE_CONNECTION_TEST,
) if GOOGLE_CREDENTIAL.configured else None

print('Gemini kimlik bilgisi:', google_ai.describe_credential_state(GOOGLE_CREDENTIAL),
      '| manuel profil:', GOOGLE_CREDENTIAL_PROFILE, '| otomatik profil gecisi: KAPALI')
if google_manifest is None:
    print('Google yetenek kesfi ATLANDI (anahtar yok).')
    print(google_ai.credential_help_text('google_api'))
else:
    print('Kesif dogrulandi mi:', google_manifest['discovery_verified'],
          '| onbellek:', google_manifest.get('cache'))
    if google_manifest.get('discovery_error'):
        print('Kesif hatasi (sanitize):', google_manifest['discovery_error'])
    print('Kullanilabilir modeller:', ', '.join(google_manifest['usable_models']) or '(yok)')
    if google_manifest['excluded_zero_quota']:
        print('0 kota nedeniyle DISLANAN:', ', '.join(google_manifest['excluded_zero_quota']))
    if google_manifest.get('connection_test'):
        print('Baglanti testi:', google_manifest['connection_test']['ok'],
              '| test edilen:', google_manifest['connection_test']['tested_models'])


def show_google_route(task):
    """Calistirmadan ONCE secilen rotayi yazdirir. Gizli deger ICERMEZ."""
    selection = google_ai.select_google_route(
        task, mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
        manifest=google_manifest, ledger=google_quota_ledger,
        enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
        max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
    )
    print(google_ai.format_route_line(selection, GOOGLE_CREDENTIAL,
                                       raw_media=ALLOW_RAW_MEDIA_TO_FREE_TIER))
    if selection['skipped']:
        for skip in selection['skipped']:
            print(f"  atlandi: {skip['model']} -> {skip['reason']}")
    print()
    return selection


for _task in ('hotmoment_review', 'script_generation', 'bulk_classification',
              'grounded_research', 'tts', 'embeddings'):
    show_google_route(_task)

print('Capraz-saglayici (opsiyonel) durum:',
      provider_adapters.describe_provider_states(get_secret))


## Faz 1 — Video Analizi / Hot-Moment Tespiti (`RUN_VIDEO_ANALYSIS`)

Videoyu Drive'daki sabit `source_videos/` klasorune yukle. Dosya adi girmen gerekmez; klasordeki en yeni desteklenen video otomatik secilir. Gerekirse `SOURCE_VIDEO_PATH` ile acik bir dosya yolu vererek otomatik secimi gecersiz kilabilirsin.

In [ ]:
SOURCE_VIDEO_DIR = paths['source_videos_dir']
SOURCE_VIDEO_PATH = ''  # Bos birak: SOURCE_VIDEO_DIR icindeki en yeni video otomatik secilir.
CLIP_START_SEC = 0
CLIP_DURATION_SEC = 240
WHISPER_MODEL_SIZE = 'small'  # 'tiny' | 'small' | 'large-v3-turbo' (GPU onerilir)
SCENE_DETECTOR = 'content'
TOP_N_HOT_MOMENTS = 5


## Faz 0 — Kaynak İşleme Stratejisi (ZORUNLU)

Kaynak video seçildikten sonra **hiçbir alt faz, onaylanmış bir işleme planı olmadan çalışmaz.**
Bu hücre kaynağı ölçer ve iki mod için **yerel** uygunluk skoru üretir; hiçbir API çağrısı yapmaz.

| Mod | Ne yapar | API |
|---|---|---|
| `direct_clip_series` | Birden fazla hot moment → birden fazla Short. **Orijinal ses korunur**, mevcut dinamik aktif-kelime altyazı ve render hattı kullanılır. | script/TTS **çağrılmaz** |
| `narrated_theory` | Seçilen momentler **görsel kanıt/B-roll** olur; senin yazdığın teori/yorum açısına göre mevcut Gemini rotasından yeni script üretilir, script → TTS → render onay zinciri aynen işler. | Gemini script + TTS |

**Öneri bir seçim değildir.** `assess_source()` her zaman `selected_mode=None` ve
`auto_selected=False` döner; modu Faz 0a'da sen seçersin, Faz 0b'de okursun, Faz 0c'de
gösterilen SHA-256 ile onaylarsın. Kaynağı, modu, momentleri, teori açısını, çıktı sayısını,
hedef süreyi veya render ayarını değiştirmek onayı **otomatik geçersiz kılar**.


In [ ]:
source_assessment = None
ASSESSED_SOURCE_PATH = ''

try:
    ASSESSED_SOURCE_PATH = str(video_analysis.resolve_source_video(
        SOURCE_VIDEO_DIR, SOURCE_VIDEO_PATH,
    ))
    print('Degerlendirilen kaynak video:', ASSESSED_SOURCE_PATH)
    _ffmpeg_bin, _ffprobe_bin = render_utils.resolve_ffmpeg_binaries()

    # Varsa mevcut analiz sinyalleri okunur; YOKSA Faz 0 yine calisir ve skorlar
    # yalnizca kaynak sinyallerine dayanir (uydurma sinyal uretilmez).
    _existing_hot_moments = None
    _hm_path = find_artifact('hot_moments.json', RESULTS_DIR)
    if _hm_path is not None:
        with open(_hm_path, 'r', encoding='utf-8') as _f:
            _candidate = json.load(_f)
        if not video_analysis.validate_hot_moments_schema(_candidate):
            _existing_hot_moments = _candidate

    source_assessment = source_strategy.assess_source(
        ASSESSED_SOURCE_PATH, _ffmpeg_bin, _ffprobe_bin,
        hot_moments_data=_existing_hot_moments,
        google_credential_configured=GOOGLE_CREDENTIAL.configured,
    )
    youtube_data.write_json(SOURCE_ASSESSMENT_PATH, source_assessment)
    print(source_strategy.format_assessment_panel(source_assessment))
    record_phase('source_assessment', 'completed', source_assessment['status'],
                 f"oneri={source_assessment['recommended_mode']} (secim YAPILMADI)")
except Exception as exc:
    record_phase('source_assessment', 'failed', None, f'{type(exc).__name__}: {exc}')
    print('Kaynak degerlendirmesi basarisiz:', type(exc).__name__, exc)


### Faz 0a — Modu SEÇ ve işleme planını oluştur

Aşağıdaki formu doldur. `PROCESSING_MODE` boş bırakılamaz ve varsayılanı yoktur —
öneriyi kabul etsen bile modu **açıkça** yazman gerekir.

- `direct_clip_series`: `THEORY_ANGLE` boş kalmalı, `PLAN_OUTPUT_COUNT` kaç Short istediğindir.
- `narrated_theory`: `THEORY_ANGLE` en az 20 karakter olmalı (script bu açıya göre üretilir),
  `PLAN_OUTPUT_COUNT = 1`.
- `PLAN_MOMENT_RANKS` boş bırakılırsa Faz 1'in en iyi adayları kullanılır.


In [ ]:
PROCESSING_MODE = ''  # @param ["", "direct_clip_series", "narrated_theory"]
PLAN_SELECTED_BY = ''  # @param {type:"string"}
PLAN_OUTPUT_COUNT = 0  # @param {type:"integer"}  (0 = degerlendirmenin onerdigi sayi)
PLAN_TARGET_DURATION_SEC = 60  # @param {type:"integer"}
PLAN_MOMENT_RANKS_TEXT = ''  # @param {type:"string"}  ornek: "1,3,4" (bos = Faz 1 en iyileri)
THEORY_ANGLE = ''  # @param {type:"string"}  yalnizca narrated_theory icin
PLAN_NOTES = ''  # @param {type:"string"}
CONFIRM_PROCESSING_PLAN = False  # @param {type:"boolean"}


In [ ]:
if not CONFIRM_PROCESSING_PLAN:
    record_phase('processing_plan', 'skipped', None, 'CONFIRM_PROCESSING_PLAN=False')
    print('Isleme plani olusturulmadi. Faz 0 panelini oku, modu sec, sonra bu hucreyi calistir.')
elif not isinstance(globals().get('source_assessment'), dict):
    record_phase('processing_plan', 'failed', None, 'source_assessment.json yok')
    print("Duzeltme: once Faz 0 degerlendirme hucresini calistir.")
else:
    try:
        _plan_ranks = [int(_p.strip()) for _p in PLAN_MOMENT_RANKS_TEXT.split(',') if _p.strip()]
        # Render/anlatim ayarlari da KARAR blogunun parcasidir: degisirlerse onay duser.
# Render/anlatim ayarlari KARAR blogunun parcasidir: sonradan degisirlerse
        # hem onay hash'i hem de render oncesi drift kontrolu bunu yakalar.
        # Bu hucreler notebook'ta ASAGIDA tanimli oldugu icin globals()'tan okunur.
        _g = globals()
        _plan_render_config = {
            'render_mode': _g.get('RENDER_MODE', 'blurred_background'),
            'preview_mode': _g.get('PREVIEW_MODE', True),
            'caption_style': _g.get('CAPTION_STYLE', 'active_word_yellow'),
            'caption_theme': _g.get('CONTENT_THEME', 'auto'),
        }
        # Anlatim ayarlari (ton, ses, dil) KARAR blogunda TUTULMAZ: bunlar zaten
        # script metninin ve ses dosyasinin kendi hash-bagli onaylarina yansir,
        # ve Faz 0 notebook'ta o hucrelerden ONCE geldigi icin burada okunsalardi
        # ilk gecistee None kaydedilirlerdi. Faz 0 yalnizca kendi sahip oldugu
        # kararlari (mod, teori acisi, cikti sayisi, hedef sure, render ayari)
        # hash'ler. API cagiranlar (or. Creator Console) isterse acikca gecebilir.
        _plan_narration_config = None

        processing_plan = source_strategy.build_processing_plan(
            source_assessment, PROCESSING_MODE, PLAN_SELECTED_BY,
            output_count=PLAN_OUTPUT_COUNT or None,
            target_duration_sec=PLAN_TARGET_DURATION_SEC or None,
            moment_ranks=_plan_ranks, theory_angle=THEORY_ANGLE,
            render_config=_plan_render_config, narration_config=_plan_narration_config,
            render_root=RENDER_DIR, notes=PLAN_NOTES,
        )
        youtube_data.write_json(PROCESSING_PLAN_PATH, processing_plan)
        print(source_strategy.format_plan_panel(processing_plan))
        record_phase('processing_plan', 'completed', processing_plan['status'],
                     f"mod={processing_plan['decision']['mode']} "
                     f"cikti={processing_plan['decision']['output_count']}")
    except Exception as exc:
        record_phase('processing_plan', 'failed', None, f'{type(exc).__name__}: {exc}')
        print('Isleme plani olusturulamadi:', exc)


### Faz 0b — Planı İNCELE

Planı oku ve aşağıda yazdırılan **SHA-256** değerini kopyala. Bu değer, kararın tam olarak
hangi içeriğe (kaynak, mod, momentler, teori açısı, çıktı sayısı, süre, render ayarı)
bağlandığının kaydıdır.


In [ ]:
if processing_plan is None:
    _found_plan = find_artifact('processing_plan.json', RESULTS_DIR)
    if _found_plan is not None:
        with open(_found_plan, 'r', encoding='utf-8') as _f:
            processing_plan = json.load(_f)
        PROCESSING_PLAN_PATH = _found_plan

if processing_plan is None:
    print('Incelenecek isleme plani yok. Once Faz 0a hucresini calistir.')
    plan_review_info = None
else:
    _plan_errors = source_strategy.validate_processing_plan(
        processing_plan, assessment=source_assessment,
        source_fingerprint=(source_assessment or {}).get('source', {}).get('fingerprint_sha256'),
    )
    _plan_errors += source_strategy.check_config_drift(
        processing_plan,
        render_config={
            'render_mode': globals().get('RENDER_MODE', 'blurred_background'),
            'preview_mode': globals().get('PREVIEW_MODE', True),
            'caption_style': globals().get('CAPTION_STYLE', 'active_word_yellow'),
            'caption_theme': globals().get('CONTENT_THEME', 'auto'),
        },
    )
    if _plan_errors:
        print('PLAN GECERSIZ/BAYAT -> onaylanmamali:')
        for _err in _plan_errors:
            print('  -', _err)
    plan_review_info = show_review_prompt('processing_plan', processing_plan, 'ISLEME PLANI')
    print()
    print(source_strategy.format_plan_panel(processing_plan))


### Faz 0c — Planı ONAYLA (ayrı adım)

`PLAN_REVIEWED_SHA256` alanına Faz 0b'de yazdırılan hash'i yapıştır. Onay yalnızca hash
şu anki kararla eşleşirse verilir; kaynağı/modu/momentleri/teori açısını/render ayarını
sonradan değiştirirsen onay otomatik geçersiz olur ve alt fazlar tekrar durur.


In [ ]:
PLAN_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
PLAN_APPROVED_BY = ''       # onayi veren kisi
PLAN_APPROVAL_NOTES = ''

if processing_plan is None:
    print('Onaylanacak isleme plani yok.')
elif not PLAN_REVIEWED_SHA256 or not PLAN_APPROVED_BY:
    print('Onay verilmedi: PLAN_REVIEWED_SHA256 ve PLAN_APPROVED_BY zorunlu.')
    print('Faz 0b inceleme hucresini calistir, gosterilen hash i buraya yapistir.')
else:
    try:
        processing_plan = approval.approve_artifact(
            processing_plan, 'processing_plan', PLAN_REVIEWED_SHA256,
            PLAN_APPROVED_BY, PLAN_APPROVAL_NOTES,
        )
        youtube_data.write_json(PROCESSING_PLAN_PATH, processing_plan)
        record_phase('processing_plan', 'completed', processing_plan['status'],
                     f"onaylayan={PLAN_APPROVED_BY}")
        print('Isleme plani ONAYLANDI ->', PROCESSING_PLAN_PATH)
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)

# Onaylanmis plan alt fazlarin kapisidir; PIPELINE_MODE plandan TURETILIR.
plan_gates = None
if isinstance(processing_plan, dict) and processing_plan.get('status') == 'approved':
    PIPELINE_MODE = processing_plan['pipeline_mode']
    plan_gates = source_strategy.describe_phase_plan(processing_plan)
    print()
    print('Turetilen PIPELINE_MODE:', PIPELINE_MODE)
    print('Faz kapilari:')
    for _phase, _gate in plan_gates.items():
        print(f"  {'ACIK ' if _gate['allowed'] else 'KAPALI'} {_phase}"
              + (f" -- {_gate['reason']}" if _gate['reason'] else ''))
else:
    print()
    print('Plan onayli degil -> tum alt fazlar duracak (Faz 0 ZORUNLUDUR).')


In [ ]:
hot_moments_data = None

if not RUN_VIDEO_ANALYSIS:
    record_phase('video_analysis', 'skipped')
elif not require_plan_phase('video_analysis'):
    pass
else:
    try:
        SOURCE_VIDEO_PATH = str(video_analysis.resolve_source_video(
            SOURCE_VIDEO_DIR, SOURCE_VIDEO_PATH,
        ))
        print('Otomatik secilen kaynak video:', SOURCE_VIDEO_PATH)
        ffmpeg_bin, _ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
        hot_moments_data = video_analysis.run_video_analysis(
            ffmpeg_bin, SOURCE_VIDEO_PATH, RUNTIME_DIR,
            clip_start_sec=CLIP_START_SEC, clip_duration_sec=CLIP_DURATION_SEC,
            whisper_model_size=WHISPER_MODEL_SIZE, scene_detector=SCENE_DETECTOR,
            top_n=TOP_N_HOT_MOMENTS,
            preview_dir=RESULTS_DIR / 'moment_previews',
        )
        youtube_data.write_json(HOT_MOMENTS_PATH, hot_moments_data)
        record_phase('video_analysis', 'completed', hot_moments_data['status'],
                     f"{len(hot_moments_data['hot_moments'])} aday -> {HOT_MOMENTS_PATH.name}")
    except Exception as exc:
        record_phase('video_analysis', 'failed', None, f'{type(exc).__name__}: {exc}')


## Faz 1a — Moment Explorer: adayları İZLE

Bu hücre en yüksek skorlu adayları gerçek MP4 preview olarak gösterir. Zamanı, skoru, nedeni ve transcript'i kontrol et. İzlemeden seçim onayı verme.

In [ ]:
moment_explorer_data = hot_moments_data if isinstance(hot_moments_data, dict) else None
if moment_explorer_data is None:
    _found_hot_moments = find_artifact('hot_moments.json', RESULTS_DIR)
    if _found_hot_moments:
        with open(_found_hot_moments, 'r', encoding='utf-8') as _f:
            moment_explorer_data = json.load(_f)

if moment_explorer_data is None:
    print('Moment Explorer: hot_moments.json yok. Once Faz 1 video analizini calistir.')
else:
    _moment_schema_errors = video_analysis.validate_hot_moments_schema(moment_explorer_data)
    if _moment_schema_errors:
        print('Moment Explorer acilamadi; sema hatalari:', _moment_schema_errors)
    else:
        from IPython.display import Video, Markdown, display
        print('Aday sayisi:', len(moment_explorer_data['hot_moments']))
        for _moment in sorted(moment_explorer_data['hot_moments'], key=lambda m: m['rank']):
            _start = _moment.get('source_start_sec', _moment.get('aligned_start_sec'))
            _end = _moment.get('source_end_sec', _moment.get('aligned_end_sec'))
            display(Markdown(
                f"### Aday #{_moment['rank']} — skor {_moment.get('score')} "
                f"— kaynak zamanı {_start:.2f}s–{_end:.2f}s"
            ))
            print('Secilme nedenleri:', _moment.get('reason') or '(yok)')
            print('Transcript:', _moment.get('transcript') or '(konusma tespit edilmedi)')
            _preview = _moment.get('preview_path')
            if _preview and Path(_preview).is_file() and Path(_preview).stat().st_size > 0:
                display(Video(_preview, embed=True, width=560, html_attributes='controls preload=\"metadata\"'))
            else:
                print('PREVIEW YOK:', _preview, _moment.get('preview_error') or '')
            print('-' * 72)


## Faz 1b — Moment kararını KAYDET

Preview'ları izledikten sonra aşağıdaki formu doldur. `CONFIRM_MOMENT_SELECTION=True` olmadan seçim kaydedilmez. Karar `moment_selection.json` içinde analiz ve preview SHA-256 değerlerine bağlanır; video/analiz/preview değişirse seçim geçersiz olur.

In [ ]:
SELECTED_MOMENT_RANK = 1  # @param {type:"integer"}
REJECTED_MOMENT_RANKS_TEXT = ""  # @param {type:"string"}
MOMENT_REVIEWED_BY = ""  # @param {type:"string"}
MOMENT_REVIEW_NOTES = ""  # @param {type:"string"}
CONFIRM_MOMENT_SELECTION = False  # @param {type:"boolean"}
REQUIRE_MOMENT_PREVIEW = True


In [ ]:
moment_selection = None
_selection_source = moment_explorer_data if isinstance(moment_explorer_data, dict) else None
if not require_plan_phase('moment_selection'):
    pass
elif _selection_source is None:
    record_phase('moment_selection', 'failed', None, 'hot_moments.json bulunamadi')
elif not CONFIRM_MOMENT_SELECTION:
    record_phase('moment_selection', 'skipped', None, 'CONFIRM_MOMENT_SELECTION=False')
    print('Moment secimi kaydedilmedi. Previewlari izle; sonra formu doldurup onayi True yap.')
else:
    try:
        _rejected_ranks = [
            int(_part.strip()) for _part in REJECTED_MOMENT_RANKS_TEXT.split(',')
            if _part.strip()
        ]
        moment_selection = video_analysis.build_moment_selection(
            _selection_source, selected_moment_rank=SELECTED_MOMENT_RANK,
            reviewed_by=MOMENT_REVIEWED_BY, rejected_moment_ranks=_rejected_ranks,
            review_notes=MOMENT_REVIEW_NOTES, require_preview=REQUIRE_MOMENT_PREVIEW,
        )
        youtube_data.write_json(MOMENT_SELECTION_PATH, moment_selection)
        record_phase(
            'moment_selection', 'completed', 'approved',
            f"rank={moment_selection['selected_moment_rank']} reviewed_by={moment_selection['reviewed_by']}",
        )
        print('Moment secimi KAYDEDILDI ->', MOMENT_SELECTION_PATH)
        print('Secilen rank:', moment_selection['selected_moment_rank'])
        print('Preview SHA-256:', moment_selection['selected_moment']['preview_sha256'])
    except Exception as exc:
        record_phase('moment_selection', 'failed', None, f'{type(exc).__name__}: {exc}')
        print('Moment secimi kaydedilemedi:', exc)


## Faz 2 — Script Uretimi (`RUN_SCRIPT_GENERATION`)

Bağımlılık: geçerli ve hash-bağlı `moment_selection.json`. Geçerli insan seçimi yoksa ücretli çağrı sayısı **0**;
aksi halde `ALLOW_SCRIPT_PROVIDER_FALLBACK` kapalıyken tam olarak **1**.

Script hücresi manuel bir onay boolean'ına güvenmez. Faz 1b'nin ürettiği seçimi, güncel `hot_moments.json`
ve seçili preview dosyasının SHA-256 değeriyle doğrular. Script onayı aşağıda ayrı bir hücrededir.

In [ ]:
# --- Ucretli saglayici secimi (fan-out YOK) ---
# Varsayilan: Google/Gemini (GOOGLE_API_KEY). OpenAI/Anthropic fallback KAPALI --
# key'leri eksikse hata degil, sadece atlanir/loglanir.
SCRIPT_PROVIDER = 'google'      # 'google' | 'anthropic' | 'openai' (google = kanonik varsayilan)
SCRIPT_MODEL = None             # None = Google Rota A zinciri (3.6-flash -> 3.5-flash -> 3.5-flash-lite -> gemma-4-31b-it);
                                # bir model adi verirsen router SADECE o modele sabitlenir
ALLOW_SCRIPT_PROVIDER_FALLBACK = False  # True: adaylar SIRAYLA denenir, ilk basarida durur

CONTENT_MODE = 'spoiler-free-recap'
TONE = 'gergin ve merak uyandirici'
TARGET_DURATION_SEC = 60

In [ ]:
selected_script = None
moment_selection = moment_selection if isinstance(globals().get('moment_selection'), dict) else None
SELECTED_MOMENT_RANK = None
MOMENT_APPROVED = False
MOMENT_APPROVED_BY = ''

if not RUN_SCRIPT_GENERATION:
    record_phase('script_generation', 'skipped')
elif not require_plan_phase('script_generation'):
    # direct_clip_series bu noktada durur: istemci KURULMAZ, hicbir script API
    # cagrisi yapilmaz.
    pass
else:
    if hot_moments_data is None:
        found_path = find_artifact('hot_moments.json', RESULTS_DIR)
        if found_path is None:
            record_phase('script_generation', 'failed', None, 'hot_moments.json bulunamadi')
            print("Duzeltme: RUN_VIDEO_ANALYSIS=True ile Faz 1'i calistir veya hot_moments.json "
                  "dosyasini RESULTS_DIR altina yerlestir.")
            hot_moments_data = 'MISSING'
        else:
            with open(found_path, 'r', encoding='utf-8') as f:
                hot_moments_data = json.load(f)
            schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
            if schema_errors:
                record_phase('script_generation', 'failed', None,
                             f'hot_moments.json semasi gecersiz: {schema_errors}')
                hot_moments_data = 'MISSING'
            else:
                HOT_MOMENTS_PATH = found_path
                print('Mevcut hot_moments.json bulundu ve dogrulandi:', found_path)

    if hot_moments_data not in (None, 'MISSING'):
        if moment_selection is None:
            _found_selection = find_artifact('moment_selection.json', RESULTS_DIR)
            if _found_selection:
                with open(_found_selection, 'r', encoding='utf-8') as _f:
                    moment_selection = json.load(_f)
                MOMENT_SELECTION_PATH = _found_selection

        if moment_selection is None:
            print('Gecerli moment_selection.json yok -> ucretli script cagrisi yapilmayacak.')
            print('Duzeltme: Faz 1a previewlarini izle ve Faz 1b karar hucresini calistir.')
        else:
            _selection_errors = video_analysis.validate_moment_selection(
                moment_selection, hot_moments_data, require_preview=True)
            if _selection_errors:
                print('Moment secimi GECERSIZ -> ucretli script cagrisi yapilmayacak:')
                for _error in _selection_errors:
                    print('  -', _error)
            else:
                SELECTED_MOMENT_RANK = moment_selection['selected_moment_rank']
                MOMENT_APPROVED = True
                MOMENT_APPROVED_BY = moment_selection['reviewed_by']
                print('Moment secimi dogrulandi: rank=', SELECTED_MOMENT_RANK,
                      '| reviewed_by=', MOMENT_APPROVED_BY)

        # Istemciler yalnizca AN onaylandiysa kurulur; kurulmalari tek basina
        # cagri yapmaz, ama gereksiz kurulumdan da kaciniyoruz.
        openai_client = anthropic_client = google_client = None
        if MOMENT_APPROVED:
            if OPENAI_API_KEY:
                from openai import OpenAI
                openai_client = OpenAI(api_key=OPENAI_API_KEY)
            if ANTHROPIC_API_KEY:
                import anthropic
                anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
            if GOOGLE_CREDENTIAL.configured and RUN_GOOGLE_SCRIPT_GENERATION:
                # Istemci TEK bir kimlik bilgisinden kurulur (hesap havuzu yok).
                google_client = google_ai.build_google_client(GOOGLE_CREDENTIAL)
                show_google_route('script_generation')
            elif not RUN_GOOGLE_SCRIPT_GENERATION:
                print('RUN_GOOGLE_SCRIPT_GENERATION=False -> Google script uretimi atlandi.')

        try:
            selected_script = script_generation.run_script_generation(
                hot_moments_data, HOT_MOMENTS_PATH, selected_moment_rank=SELECTED_MOMENT_RANK,
                openai_client=openai_client, anthropic_client=anthropic_client,
                google_client=google_client,
                content_mode=CONTENT_MODE, tone=TONE, target_duration_sec=TARGET_DURATION_SEC,
                moment_approved=MOMENT_APPROVED, moment_approved_by=MOMENT_APPROVED_BY,
                script_provider=SCRIPT_PROVIDER, script_model=SCRIPT_MODEL,
                allow_provider_fallback=ALLOW_SCRIPT_PROVIDER_FALLBACK,
                google_ledger=google_quota_ledger, google_manifest=google_manifest,
                google_route_mode=GOOGLE_ROUTE_MODE,
                google_task_model_overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_google_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_google_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                enable_cross_provider_fallback=ENABLE_CROSS_PROVIDER_FALLBACK,
                max_provider_attempts=MAX_PROVIDER_ATTEMPTS,
                cross_provider_secret_getter=get_secret,
            )
            youtube_data.write_json(SELECTED_SCRIPT_PATH, selected_script)
            paid_calls = len([a for a in selected_script.get('provider_attempts', []) if a.get('provider')])
            record_phase('script_generation', 'completed', selected_script['status'],
                         f"ucretli cagri sayisi={paid_calls}, saglayici={selected_script.get('provider')}")
            for attempt in selected_script.get('provider_attempts', []):
                flag = 'OK' if attempt['ok'] else 'BASARISIZ'
                print(f"  deneme: {attempt['provider']}/{attempt['model']} -> {flag}"
                      + (f" ({attempt['error']})" if attempt['error'] else ''))
                for _google_attempt in ((attempt.get('google_route') or {}).get('provider_attempts') or []):
                    _gflag = 'OK' if _google_attempt['ok'] else 'BASARISIZ'
                    print(f"      google: {_google_attempt['model']} -> {_gflag}"
                          + (f" [{_google_attempt.get('error_kind')}] {_google_attempt.get('error')}"
                             if not _google_attempt['ok'] else ''))
        except Exception as exc:
            record_phase('script_generation', 'failed', None, f'{type(exc).__name__}: {exc}')


### Faz 2a — Uretilen scripti INCELE

Asagidaki hucre scripti ve SHA-256'sini yazdirir. **Scripti oku**, sonra bir sonraki onay hucresine
bu SHA-256 degerini yapistir. Onay, uretim cagrisinda degil burada verilir.

In [ ]:
if selected_script is None:
    print('Faz 2 calismadi; incelenecek script yok.')
elif not selected_script.get('script_text'):
    print('Script uretilmedi (sadece prompt paketi). status:', selected_script['status'])
    print('status_reason:', selected_script['status_reason'])
    if selected_script.get('prompt_package'):
        print()
        print('--- PROMPT PAKETI (harici/elle uretim icin) ---')
        print(selected_script['prompt_package']['user_prompt'])
else:
    print('Saglayici/model:', selected_script['provider'], '/', selected_script['model'])
    print('Tahmini maliyet (USD):', selected_script['cost_usd_estimate'])
    print()
    print('--- URETILEN SCRIPT ---')
    print(selected_script['script_text'])
    print()
    script_review_info = show_review_prompt('script', selected_script, 'SCRIPT')


### Faz 2b — Scripti ONAYLA (ayrı adım)

Yukarıdaki scripti okuduktan **sonra** `SCRIPT_REVIEWED_SHA256` alanına inceleme hücresinin yazdırdığı
hash'i yapıştır ve `SCRIPT_APPROVED_BY` doldur. Hash eşleşmezse onay **reddedilir**.

In [ ]:
SCRIPT_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
SCRIPT_APPROVED_BY = ''       # ornek: 'insan: birkan'
SCRIPT_APPROVAL_NOTES = ''

if selected_script is None or not selected_script.get('script_text'):
    print('Onaylanacak bir script yok; bu hucre atlandi.')
elif not SCRIPT_REVIEWED_SHA256 or not SCRIPT_APPROVED_BY:
    print('Onay ATLANDI: SCRIPT_REVIEWED_SHA256 ve SCRIPT_APPROVED_BY doldurulmali.')
    print('selected_script.json status olarak kaliyor:', selected_script['status'])
else:
    try:
        selected_script = approval.approve_artifact(
            selected_script, 'script', SCRIPT_REVIEWED_SHA256,
            SCRIPT_APPROVED_BY, SCRIPT_APPROVAL_NOTES,
        )
        youtube_data.write_json(SELECTED_SCRIPT_PATH, selected_script)
        record_phase('script_generation', 'completed', selected_script['status'],
                     f"onaylayan={SCRIPT_APPROVED_BY}")
        print('Script ONAYLANDI ->', SELECTED_SCRIPT_PATH)
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('selected_script.json status olarak kaliyor:', selected_script['status'])


## Faz 3 — TTS / Seslendirme (`RUN_TTS`)

Bagimlilik: **onaylanmis** `selected_script.json`. Script `approved` degilse hicbir ucretli TTS
cagrisi yapilmaz. **Ucretli cagri sayisi:** fallback kapaliyken tam olarak **1**.

Kalite-oncelikli varsayilan: ElevenLabs `eleven_multilingual_v2`. Opsiyonel uretim fallback'i:
OpenAI `tts-1-hd`. gTTS **yalnizca** `ALLOW_DEV_GTTS_FALLBACK = True` iken ve hicbir uretim
saglayicisi basarili olmadiginda calisir; sonucu her zaman `development_placeholder` kalir ve
**asla onaylanamaz**.

In [ ]:
# Varsayilan: mevcut GeminiKey ile Gemini TTS. Ses uretimi zorunlu ana-pipeline adimidir.
# Piper lokal; ElevenLabs/OpenAI ise kalite karsilastirmasi icin alternatiflerdir.
TTS_PROVIDER = 'gemini'              # 'piper' | 'gemini' | 'elevenlabs' | 'openai'
                                     # 'gemini': RUN_GOOGLE_TTS=True olmali; Gemini TTS
                                     # modelleri PREVIEW oldugu icin cikti HER ZAMAN
                                     # review_required kalir ve yedek ses/model onceki
                                     # seslendirmenin onayini ASLA devralmaz.
ALLOW_TTS_PROVIDER_FALLBACK = False   # True: piper basarisiz olursa siradaki uretim saglayicisi denenir
ALLOW_DEV_GTTS_FALLBACK = False       # True: hicbir uretim saglayicisi (piper dahil) calismazsa gTTS
                                       # (SADECE development_placeholder -- ASLA onaylanamaz, bkz. approval.py)

# Piper ayarlari: binary/ses modeli eksikse Faz 3 temiz sekilde 'insufficient_data'
# doner, notebook COKMEZ (bkz. local_pipeline/tts.py call_piper_tts).
PIPER_BINARY = 'piper'                # PATH icindeki komut veya piper binary'sine tam yol
PIPER_VOICE_MODEL_PATH = None         # ör: '/content/piper-voices/tr_TR-fahrettin-medium.onnx' -- yoksa None birak

In [ ]:
selected_voiceover = None

if not RUN_TTS:
    record_phase('tts', 'skipped')
elif not require_plan_phase('tts'):
    # direct_clip_series bu noktada durur: istemci KURULMAZ, hicbir TTS API
    # cagrisi yapilmaz.
    pass
else:
    if selected_script is None:
        found_path = find_artifact('selected_script.json', RESULTS_DIR)
        if found_path is None:
            record_phase('tts', 'failed', None, 'selected_script.json bulunamadi')
            print("Duzeltme: RUN_SCRIPT_GENERATION=True ile Faz 2'yi calistir veya "
                  "selected_script.json dosyasini RESULTS_DIR altina yerlestir.")
        else:
            with open(found_path, 'r', encoding='utf-8') as f:
                selected_script = json.load(f)
            SELECTED_SCRIPT_PATH = found_path
            print('Mevcut selected_script.json bulundu:', found_path,
                  '(status:', selected_script.get('status'), ')')

    if selected_script is not None:
        # Diskten gelen bir onay hala GECERLI mi? Script onaylandiktan sonra
        # degistirildiyse onay gecersizdir ve ucretli TTS calismamalidir.
        script_check = approval.check_approval(selected_script, 'script')
        if selected_script.get('status') == 'approved' and not script_check['valid']:
            print('UYARI:', script_check['reason'])
            print('Bu script onayi gecersiz sayiliyor; uretim TTS calistirilmayacak.')
            selected_script = dict(selected_script)
            selected_script['status'] = 'review_required'

        if selected_script.get('status') != 'approved':
            print('NOT: script onayli degil ->', selected_script.get('status'),
                  '| uretim TTS (piper dahil) ATLANACAK (Faz 2b onay hucresini calistir).')

        # Istemciler/parametreler yalnizca script ONAYLIYSA kurulur. Piper API key
        # istemez; sadece PIPER_BINARY/PIPER_VOICE_MODEL_PATH gecirilir, eksikse
        # tts.run_tts temiz 'insufficient_data' dondurur.
        elevenlabs_client = openai_tts_client = None
        piper_binary_arg = piper_voice_model_arg = None
        if selected_script.get('status') == 'approved':
            if ELEVENLABS_API_KEY:
                from elevenlabs.client import ElevenLabs
                elevenlabs_client = ElevenLabs(api_key=ELEVENLABS_API_KEY)
            if OPENAI_API_KEY:
                from openai import OpenAI
                openai_tts_client = OpenAI(api_key=OPENAI_API_KEY)
            piper_binary_arg = PIPER_BINARY
            piper_voice_model_arg = PIPER_VOICE_MODEL_PATH

        _, ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
        try:
            if TTS_PROVIDER == 'gemini':
                # Gemini TTS: AYRI bir ses-uretim implementasyonu (metin
                # ayristiricisi degil). Ham PCM gecerli bir WAV'a sarilir ve
                # sure/codec/boyut/bos-olmama dogrulamasindan gecer.
                if not RUN_GOOGLE_TTS:
                    raise RuntimeError(
                        "TTS_PROVIDER='gemini' ama RUN_GOOGLE_TTS=False. "
                        "Google TTS komutunu acikca ac veya baska bir saglayici sec."
                    )
                if not GOOGLE_CREDENTIAL.configured:
                    raise RuntimeError('Gemini anahtari yok. ' + google_ai.credential_help_text('google_api'))
                if selected_script.get('status') != 'approved':
                    raise RuntimeError(
                        'Gemini TTS atlandi: script onayli degil '
                        f"(status={selected_script.get('status')!r}); cagri yapilmadi."
                    )
                show_google_route('tts')
                _gemini_tts = google_ai.run_google_tts(
                    selected_script['script_text'], RESULTS_DIR / 'tts' / 'gemini_tts.wav',
                    client=google_ai.build_google_client(GOOGLE_CREDENTIAL),
                    voice=GEMINI_TTS_VOICE, language_code=GEMINI_TTS_LANGUAGE_CODE,
                    narration_direction=GEMINI_TTS_NARRATION_DIRECTION,
                    manifest=google_manifest, ledger=google_quota_ledger,
                    mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                    enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                    max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                )
                if _gemini_tts.get('status') == 'ok':
                    selected_voiceover = google_ai.build_tts_artifact(
                        _gemini_tts, selected_script['script_text'],
                        script_status=selected_script.get('status', 'review_required'),
                        raw_media_submitted=False,
                    )
                elif ENABLE_CROSS_PROVIDER_FALLBACK:
                    _azure_tts = provider_adapters.run_cross_provider_tts_fallback(
                        selected_script['script_text'], RESULTS_DIR / 'tts' / 'azure_tts.wav',
                        secret_getter=get_secret, enabled=True,
                    )
                    if _azure_tts.get('ok'):
                        _tts_attempts = list(_gemini_tts.get('provider_attempts') or [])
                        _tts_attempts.extend(_azure_tts.get('provider_attempts') or [])
                        selected_voiceover = tts.build_selected_voiceover_artifact(
                            _azure_tts, selected_script['script_text'],
                            selected_script.get('status', 'review_required'), False,
                            ffprobe_bin=ffprobe_bin, language='tr', attempts=_tts_attempts,
                        )
                    else:
                        selected_voiceover = google_ai.build_tts_artifact(
                            _gemini_tts, selected_script['script_text'],
                            script_status=selected_script.get('status', 'review_required'),
                            raw_media_submitted=False,
                            )
                        selected_voiceover['provider_attempts'].extend(
                            _azure_tts.get('provider_attempts') or [])
                else:
                    selected_voiceover = google_ai.build_tts_artifact(
                        _gemini_tts, selected_script['script_text'],
                        script_status=selected_script.get('status', 'review_required'),
                        raw_media_submitted=False,
                    )
            else:
                selected_voiceover = tts.run_tts(
                    selected_script, RESULTS_DIR / 'tts', ffprobe_bin=ffprobe_bin,
                    elevenlabs_client=elevenlabs_client, openai_client=openai_tts_client,
                    piper_binary=piper_binary_arg, piper_voice_model_path=piper_voice_model_arg,
                    tts_provider=TTS_PROVIDER,
                    allow_provider_fallback=ALLOW_TTS_PROVIDER_FALLBACK,
                    allow_dev_gtts_fallback=ALLOW_DEV_GTTS_FALLBACK,
                )
            youtube_data.write_json(SELECTED_VOICEOVER_PATH, selected_voiceover)
            paid = len([a for a in selected_voiceover.get('provider_attempts', [])
                        if a.get('provider') in ('elevenlabs', 'openai')])
            if selected_voiceover.get('audio_validation'):
                print('  ses dogrulamasi:', selected_voiceover['audio_validation']['checks'])
            record_phase('tts', 'completed', selected_voiceover['status'],
                         f"ucretli cagri sayisi={paid}, saglayici={selected_voiceover.get('provider')}")
            for attempt in selected_voiceover.get('provider_attempts', []):
                flag = 'OK' if attempt['ok'] else 'BASARISIZ'
                print(f"  deneme: {attempt['provider']}/{attempt['model']} -> {flag}"
                      + (f" ({attempt['error']})" if attempt['error'] else ''))
        except Exception as exc:
            record_phase('tts', 'failed', None, f'{type(exc).__name__}: {exc}')

### Faz 3a — Seslendirmeyi DINLE

Aşağıdaki hücre sesi çalar ve dosyanın SHA-256'sını yazdırır. **Sesi dinle**, sonra onay hücresine
bu hash'i yapıştır.

In [ ]:
if selected_voiceover is None:
    print('Faz 3 calismadi; incelenecek seslendirme yok.')
elif not selected_voiceover.get('audio_path'):
    print('Seslendirme uretilmedi. status:', selected_voiceover['status'])
    print('status_reason:', selected_voiceover['status_reason'])
else:
    print('Saglayici/model:', selected_voiceover['provider'], '/', selected_voiceover['model'])
    print('Sure (sn):', selected_voiceover.get('duration_sec'))
    print('Tahmini maliyet (USD):', selected_voiceover['cost_usd_estimate'])
    try:
        from IPython.display import Audio, display as _display
        _display(Audio(selected_voiceover['audio_path']))
    except Exception as exc:
        print('Ses oynatici gosterilemedi:', exc)
        print('Dosyayi elle dinle:', selected_voiceover['audio_path'])
    print()
    voiceover_review_info = show_review_prompt('voiceover', selected_voiceover, 'SESLENDIRME')


### Faz 3b — Seslendirmeyi ONAYLA (ayrı adım)

gTTS sonuçları politika gereği **asla onaylanamaz** — onay hücresi bunu reddeder.

In [ ]:
VOICEOVER_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
VOICEOVER_APPROVED_BY = ''
VOICEOVER_APPROVAL_NOTES = ''

if selected_voiceover is None or not selected_voiceover.get('audio_path'):
    print('Onaylanacak bir seslendirme yok; bu hucre atlandi.')
elif not VOICEOVER_REVIEWED_SHA256 or not VOICEOVER_APPROVED_BY:
    print('Onay ATLANDI: VOICEOVER_REVIEWED_SHA256 ve VOICEOVER_APPROVED_BY doldurulmali.')
    print('selected_voiceover.json status olarak kaliyor:', selected_voiceover['status'])
else:
    try:
        selected_voiceover = approval.approve_artifact(
            selected_voiceover, 'voiceover', VOICEOVER_REVIEWED_SHA256,
            VOICEOVER_APPROVED_BY, VOICEOVER_APPROVAL_NOTES,
        )
        youtube_data.write_json(SELECTED_VOICEOVER_PATH, selected_voiceover)
        record_phase('tts', 'completed', selected_voiceover['status'],
                     f'onaylayan={VOICEOVER_APPROVED_BY}')
        print('Seslendirme ONAYLANDI ->', SELECTED_VOICEOVER_PATH)
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('selected_voiceover.json status olarak kaliyor:', selected_voiceover['status'])


## Faz 4 — Render Pipeline (`RUN_RENDER`)

Varsayilan `original_audio_subtitles` modunda bagimlilik yalnizca `hot_moments.json` + gecerli
`moment_selection.json` dosyalaridir. Secilen klibin kendi sesi korunur, Whisper ile zaman kodlu
altyazi uretilir ve videoya gomulur; Gemini script veya TTS cagrisi yapilmaz.

Istege bagli `narrated_recap` modu eski `selected_script.json` + `selected_voiceover.json` zincirini kullanir. Render
**hicbir zaman kendini onaylamaz**: cikti her zaman `review_required` (teknik dogrulama
basarisizsa `failed`) olur; onay asagida ayri bir hucrede, videoyu izledikten sonra verilir.

In [ ]:
RENDER_MODE = 'blurred_background'  # 'blurred_background' | 'center_crop'
PREVIEW_MODE = True  # True = hizli/dusuk kalite onizleme; False = uretim kalitesi
SUBTITLE_WHISPER_MODEL_SIZE = 'small'
CAPTION_STYLE = 'active_word_yellow'
CONTENT_THEME = 'auto'  # GOT/HOTD/Targaryen/Westeros -> game_of_thrones -> Cinzel
CAPTION_FONT_NAME = None       # None = tema profili belirler
CAPTION_FONT_SIZE = None       # None = tema profili belirler
CAPTION_ACTIVE_COLOR_ASS = None  # None = tema profili; GOT profili sari
CAPTION_BASE_COLOR_ASS = None    # None = tema profili; GOT profili beyaz
CAPTION_MAX_WORDS = None
CAPTION_MAX_CHARS = None
CAPTION_MARGIN_V = None
SOURCE_VIDEO_PATH_OVERRIDE = ''


In [ ]:
render_manifest = None
moment_selection = moment_selection if isinstance(globals().get('moment_selection'), dict) else None

if not RUN_RENDER:
    record_phase('render', 'skipped')
elif not require_plan_phase('render'):
    pass
else:
    missing = []
    if hot_moments_data in (None, 'MISSING'):
        found = find_artifact('hot_moments.json', RESULTS_DIR)
        if found is None:
            missing.append('hot_moments.json')
        else:
            with open(found, 'r', encoding='utf-8') as f:
                hot_moments_data = json.load(f)
            HOT_MOMENTS_PATH = found

    if PIPELINE_MODE == 'original_audio_subtitles':
        if moment_selection is None:
            found = find_artifact('moment_selection.json', RESULTS_DIR)
            if found is None:
                missing.append('moment_selection.json')
            else:
                with open(found, 'r', encoding='utf-8') as f:
                    moment_selection = json.load(f)
                MOMENT_SELECTION_PATH = found
    elif PIPELINE_MODE == 'narrated_recap':
        if selected_script is None:
            found = find_artifact('selected_script.json', RESULTS_DIR)
            if found is None:
                missing.append('selected_script.json')
            else:
                with open(found, 'r', encoding='utf-8') as f:
                    selected_script = json.load(f)
                SELECTED_SCRIPT_PATH = found
        if selected_voiceover is None:
            found = find_artifact('selected_voiceover.json', RESULTS_DIR)
            if found is None:
                missing.append('selected_voiceover.json')
            else:
                with open(found, 'r', encoding='utf-8') as f:
                    selected_voiceover = json.load(f)
                SELECTED_VOICEOVER_PATH = found
    else:
        missing.append(f'bilinmeyen PIPELINE_MODE={PIPELINE_MODE!r}')

    _config_drift = source_strategy.check_config_drift(
        processing_plan,
        render_config={
            'render_mode': RENDER_MODE, 'preview_mode': PREVIEW_MODE,
            'caption_style': CAPTION_STYLE, 'caption_theme': CONTENT_THEME,
        },
    )
    if _config_drift:
        missing.append('onaylanmis plan ile suanki render ayari uyusmuyor')
        for _drift in _config_drift:
            print('  DRIFT:', _drift)

    if missing:
        record_phase('render', 'failed', None, f'gerekli girdi(ler) bulunamadi: {missing}')
        print('Duzeltme: original_audio_subtitles icin Faz 1 + moment secimini; '
              'narrated_recap icin ek olarak script + TTS fazlarini tamamla.')
    else:
        try:
            if PIPELINE_MODE == 'original_audio_subtitles':
                render_manifest = render_utils.run_original_audio_subtitle_pipeline(
                    hot_moments_data, moment_selection, HOT_MOMENTS_PATH, MOMENT_SELECTION_PATH,
                    render_dir=RENDER_DIR, render_mode=RENDER_MODE, preview_mode=PREVIEW_MODE,
                    subtitle_whisper_model_size=SUBTITLE_WHISPER_MODEL_SIZE,
                    caption_style=CAPTION_STYLE, caption_theme=CONTENT_THEME,
                    caption_font_name=CAPTION_FONT_NAME,
                    caption_font_size=CAPTION_FONT_SIZE,
                    caption_active_color_ass=CAPTION_ACTIVE_COLOR_ASS,
                    caption_base_color_ass=CAPTION_BASE_COLOR_ASS,
                    caption_max_words=CAPTION_MAX_WORDS, caption_max_chars=CAPTION_MAX_CHARS,
                    caption_margin_v=CAPTION_MARGIN_V,
                    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
                )
            else:
                render_manifest = render_utils.run_render_pipeline(
                    hot_moments_data, selected_script, selected_voiceover,
                    HOT_MOMENTS_PATH, SELECTED_SCRIPT_PATH, SELECTED_VOICEOVER_PATH,
                    render_dir=RENDER_DIR, render_mode=RENDER_MODE, preview_mode=PREVIEW_MODE,
                    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
                )
            youtube_data.write_json(RENDER_MANIFEST_PATH, render_manifest)
            record_phase('render', 'completed', render_manifest['status'],
                         f"mod={PIPELINE_MODE}, teknik passed={render_manifest['output']['validation']['passed']}")
            print('Pipeline modu:', PIPELINE_MODE)
            print('Altyazi kaynagi:', render_manifest['subtitles']['source'],
                  '| cue:', render_manifest['subtitles']['cue_count'])
            print('Dinamik kelime vurgusu:', render_manifest['subtitles'].get('dynamic_word_highlight'),
                  '| aktif renk:', render_manifest['subtitles'].get('active_word_color'))
            print('Caption tema/font:', render_manifest['config'].get('caption_theme_resolved'),
                  '/', render_manifest['config'].get('caption_font_name'),
                  '| neden:', render_manifest['config'].get('caption_theme_resolution_reason'))
            if render_manifest['approval']['failed_conditions']:
                print('Onay oncesi saglanmayan kosullar:',
                      render_manifest['approval']['failed_conditions'])
            print('HATIRLATMA: hicbir yere yukleme yapilmadi (not_uploaded=%s).'
                  % render_manifest['not_uploaded'])

            # Coklu cikti IZOLASYONU: plan birden fazla Short istiyorsa her cikti
            # KENDI dizinine yazilir; hicbiri digerini ezemez. Ilk yuva tarihsel
            # RENDER_DIR'dir, yani tek ciktida yollar aynen korunur.
            output_slot_report = []
            for _slot in ((processing_plan or {}).get('outputs') or [])[1:]:
                output_slot_report.append({
                    'index': _slot['index'], 'moment_rank': _slot.get('moment_rank'),
                    'output_dir': _slot['output_dir'], 'status': 'insufficient_data',
                    'reason': ('bu cikti icin insan onayli ayri moment secimi yok; '
                               "Faz 1b'yi ilgili rank icin calistirip render'i tekrar calistir"),
                })
            if output_slot_report:
                print()
                print('EK CIKTI YUVALARI (henuz render EDILMEDI, izole dizinler):')
                for _entry in output_slot_report:
                    print(f"  #{_entry['index']} rank={_entry['moment_rank']} "
                          f"-> {_entry['output_dir']} [{_entry['status']}] {_entry['reason']}")
        except Exception as exc:
            record_phase('render', 'failed', None, f'{type(exc).__name__}: {exc}')
            print('Render HATASI:', type(exc).__name__, exc)


### Faz 4a — Render'ı İZLE

Aşağıdaki hücre videoyu oynatır ve SHA-256'sını yazdırır. **Videoyu izle**, sonra onay hücresine
bu hash'i yapıştır. Teknik doğrulama geçmediyse onay hücresi reddeder.

In [ ]:
if render_manifest is None:
    print('Faz 4 calismadi; incelenecek render yok.')
else:
    video_path = render_manifest['output']['final_video_path']
    print('Render dosyasi:', video_path)
    print('Cozunurluk:', render_manifest['config']['render_width'], 'x',
          render_manifest['config']['render_height'], '| onizleme modu:',
          render_manifest['config']['preview_mode'])
    print('Teknik dogrulama:', json.dumps(render_manifest['output']['validation']['checks'], indent=2))
    try:
        from IPython.display import Video, display as _display
        _display(Video(video_path, embed=True, width=320))
    except Exception as exc:
        print('Video oynatici gosterilemedi:', exc)
        print('Dosyayi elle izle:', video_path)
    print()
    render_review_info = show_review_prompt('render', render_manifest, 'RENDER')


### Faz 4b — Render'ı ONAYLA (ayrı adım)

Render onayı yalnızca seçili moda ait **tüm** koşullar sağlandığında verilebilir.
`original_audio_subtitles`: teknik doğrulama + onaylı moment seçimi.
`narrated_recap`: teknik doğrulama + onaylı script + onaylı seslendirme + script/ses hash eşleşmesi.
Herhangi biri eksikse onay reddedilir ve artefakt `review_required` kalır — onaylanmamış girdiler
"onaylı" bir çıktıya dönüştürülemez.

Videoyu izlediğini yine de kaydetmek istersen `RECORD_RENDER_REVIEW_ONLY = True` yap: bu, ayrı bir
`render_review` bloğu yazar ve `status`'ü **değiştirmez**.

In [ ]:
RENDER_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
RENDER_APPROVED_BY = ''
RENDER_APPROVAL_NOTES = ''

# True yaparsan SADECE "izledim" kaydi yazilir; status DEGISMEZ.
# Yukari-akis onaylari eksikken videoyu izledigini kaydetmek icin kullan.
RECORD_RENDER_REVIEW_ONLY = False

if render_manifest is None:
    print('Onaylanacak bir render yok; bu hucre atlandi.')
elif not RENDER_REVIEWED_SHA256 or not RENDER_APPROVED_BY:
    print('Onay ATLANDI: RENDER_REVIEWED_SHA256 ve RENDER_APPROVED_BY doldurulmali.')
    print('render_manifest.json status olarak kaliyor:', render_manifest['status'])
elif RECORD_RENDER_REVIEW_ONLY:
    try:
        render_manifest = approval.record_render_review(
            render_manifest, RENDER_REVIEWED_SHA256, RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        youtube_data.write_json(RENDER_MANIFEST_PATH, render_manifest)
        print('Inceleme kaydedildi (render_review). status DEGISMEDI:', render_manifest['status'])
    except ValueError as exc:
        print('INCELEME KAYDI REDDEDILDI:', exc)
else:
    try:
        render_manifest = approval.approve_artifact(
            render_manifest, 'render', RENDER_REVIEWED_SHA256,
            RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        youtube_data.write_json(RENDER_MANIFEST_PATH, render_manifest)
        record_phase('render', 'completed', render_manifest['status'],
                     f'onaylayan={RENDER_APPROVED_BY}')
        print('Render ONAYLANDI ->', RENDER_MANIFEST_PATH)
        print('NOT: onay yalnizca bir kayittir; bu notebook hicbir sey yayinlamaz.')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('render_manifest.json status olarak kaliyor:', render_manifest['status'])
        print("Sadece izledigini kaydetmek istersen RECORD_RENDER_REVIEW_ONLY = True yap.")


## Araştırma A — YouTube Trend/Kanal Keşfi (`RUN_YOUTUBE_DATA`)

**Ana pipeline'dan bağımsızdır.** `YOUTUBE_DISCOVERY_TERMS` ile yeni videoları araştırır, aday kanalları sıralar ve araştırma veri kümesi üretir. `YOUTUBE_CHANNEL_INPUTS` isteğe bağlı manuel ek listedir. Sonuçlar kaynak video veya render girdisi yapılmaz; bu bölüm read-only'dir.

In [ ]:
YOUTUBE_CHANNEL_INPUTS = [
    # "UCxxxxxxxxxxxxxxxxxxxxxx",
    # "@ChannelHandle",
]

AUTO_DISCOVER_CHANNELS = True
YOUTUBE_DISCOVERY_TERMS = ['AI', 'AI tools', 'generative AI', 'yapay zeka']
DISCOVERY_LOOKBACK_DAYS = 14
DISCOVERY_REGION = 'US'
DISCOVERY_LANGUAGE = 'en'
DISCOVERY_ORDERS = ('date', 'viewCount')
MAX_RESULTS_PER_QUERY = 25
MAX_SEARCH_CALLS_PER_RUN = 8
TOP_DISCOVERED_VIDEOS = 50
TOP_DISCOVERED_CHANNELS = 10
MIN_CHANNEL_MATCHED_VIDEOS = 1
DISCOVERY_SHORTS_ONLY = True
YOUTUBE_DISCOVERY_CACHE_TTL_HOURS = 6

MAX_VIDEOS_PER_CHANNEL = 100
YOUTUBE_CACHE_TTL_HOURS = 12
RUN_YOUTUBE_CONNECTION_TEST = False


In [ ]:
from datetime import datetime, timezone

youtube_collection_summary = {'ran': False}

if not RUN_YOUTUBE_DATA:
    record_phase('youtube_data', 'skipped')
else:
    youtube_started_at = datetime.now(timezone.utc).isoformat()
    youtube_cache = youtube_data.Cache(YOUTUBE_CACHE_DIR)
    youtube_quota = youtube_data.QuotaTracker()

    youtube_client = None
    youtube_connection_test_result = None
    youtube_resolved_channels = []
    youtube_failed_channels = []
    youtube_all_videos = []
    youtube_collection_errors = []
    youtube_discovery = {
        'status': 'insufficient_data', 'status_reason': 'otomatik kesif calistirilmadi',
        'videos': [], 'channels': [], 'errors': [],
        'search_calls_planned': 0, 'search_calls_executed': 0,
    }
    effective_channel_inputs = list(YOUTUBE_CHANNEL_INPUTS)
    youtube_status = 'insufficient_data'
    youtube_status_reason = 'baslatilmadi'

    if not YOUTUBE_DATA_API_KEY:
        youtube_status = 'insufficient_data'
        youtube_status_reason = 'YOUTUBE_DATA_API_KEY yok (Colab Secrets kontrol et)'
        print('YouTube Data API key: missing -- toplama yapilamaz.')
    else:
        print('YouTube Data API key: configured')
        try:
            youtube_client = youtube_data.build_client(YOUTUBE_DATA_API_KEY)
        except Exception as exc:
            youtube_status = 'failed'
            youtube_status_reason = 'istemci olusturulamadi: ' + youtube_data.sanitize_error_text(exc, YOUTUBE_DATA_API_KEY)
            youtube_collection_errors.append({'stage': 'build_client', 'error': youtube_status_reason})
            print(youtube_status_reason)

        if youtube_client is not None and RUN_YOUTUBE_CONNECTION_TEST:
            test_input = YOUTUBE_CHANNEL_INPUTS[0] if YOUTUBE_CHANNEL_INPUTS else '@GoogleDevelopers'
            youtube_connection_test_result = youtube_data.run_connection_test(
                youtube_client, YOUTUBE_DATA_API_KEY, test_input)
            print('Baglanti testi ->', youtube_connection_test_result['status'], 'girdi:', test_input)
            if youtube_connection_test_result['status'] != 'ok':
                print('  Aciklama:', youtube_connection_test_result['explanation'])

        if youtube_client is not None and AUTO_DISCOVER_CHANNELS:
            if not YOUTUBE_DISCOVERY_TERMS:
                youtube_discovery['status_reason'] = 'YOUTUBE_DISCOVERY_TERMS bos'
                print('Otomatik kesif atlandi: YOUTUBE_DISCOVERY_TERMS bos.')
            else:
                print('YouTube otomatik kesif basliyor:', YOUTUBE_DISCOVERY_TERMS)
                youtube_discovery = youtube_data.discover_trending_content(
                    youtube_client, youtube_cache, youtube_quota,
                    terms=YOUTUBE_DISCOVERY_TERMS,
                    lookback_days=DISCOVERY_LOOKBACK_DAYS,
                    region_code=DISCOVERY_REGION,
                    relevance_language=DISCOVERY_LANGUAGE,
                    orders=DISCOVERY_ORDERS,
                    max_results_per_query=MAX_RESULTS_PER_QUERY,
                    max_search_calls=MAX_SEARCH_CALLS_PER_RUN,
                    top_videos=TOP_DISCOVERED_VIDEOS,
                    top_channels=TOP_DISCOVERED_CHANNELS,
                    min_channel_matched_videos=MIN_CHANNEL_MATCHED_VIDEOS,
                    shorts_only=DISCOVERY_SHORTS_ONLY,
                    cache_ttl_hours=YOUTUBE_DISCOVERY_CACHE_TTL_HOURS,
                    api_key=YOUTUBE_DATA_API_KEY,
                )
                youtube_collection_errors.extend(youtube_discovery['errors'])
                discovered_channel_ids = [
                    row['channel_id'] for row in youtube_discovery['channels']
                    if row.get('channel_id')
                ]
                effective_channel_inputs.extend(discovered_channel_ids)
                effective_channel_inputs = list(dict.fromkeys(effective_channel_inputs))
                print('Kesif sonucu:', len(youtube_discovery['videos']), 'video,',
                      len(discovered_channel_ids), 'kanal')

        if youtube_client is not None and not effective_channel_inputs:
            youtube_status = youtube_discovery.get('status', 'insufficient_data')
            if youtube_status == 'review_required':
                youtube_status = 'insufficient_data'
            youtube_status_reason = (
                'manuel kanal yok ve otomatik kesif kanal uretmedi: ' +
                youtube_discovery.get('status_reason', 'bilinmeyen neden')
            )
            print(youtube_status_reason)
        elif youtube_client is not None:
            youtube_resolved_channels, youtube_failed_channels = youtube_data.resolve_channels(
                youtube_client, effective_channel_inputs, api_key_for_sanitization=YOUTUBE_DATA_API_KEY,
            )
            for failed in youtube_failed_channels:
                youtube_collection_errors.append(
                    {'stage': 'resolve_channel', 'input': failed['input'], 'error': failed['error']})
                print('Kanal cozumlenemedi:', failed['input'], '->', failed['error'])

            for channel_record in youtube_resolved_channels:
                rows, err = youtube_data.collect_channel_uploads(
                    youtube_client, youtube_cache, youtube_quota, channel_record,
                    MAX_VIDEOS_PER_CHANNEL, channel_cache_ttl_hours=YOUTUBE_CACHE_TTL_HOURS,
                    video_stats_ttl_hours=YOUTUBE_CACHE_TTL_HOURS, api_key=YOUTUBE_DATA_API_KEY,
                )
                if err:
                    youtube_collection_errors.append(
                        {'stage': 'collect_uploads', 'channel_id': channel_record['channel_id'], 'error': err})
                    print('Toplama hatasi:', channel_record['channel_id'], '->', err)
                youtube_all_videos.extend(rows)

            youtube_all_videos = youtube_data.deduplicate_videos(youtube_all_videos)

            if youtube_resolved_channels and youtube_all_videos:
                youtube_status = 'review_required'
                youtube_status_reason = (f'{len(youtube_resolved_channels)} kanal, '
                                         f'{len(youtube_all_videos)} video toplandi -- insan onayi gerekli')
            elif youtube_resolved_channels and not youtube_all_videos:
                youtube_status = 'insufficient_data'
                youtube_status_reason = 'kanallar cozumlendi ama kullanilabilir video toplanamadi'
            else:
                youtube_status = 'failed'
                youtube_status_reason = 'hicbir kanal cozumlenemedi'

    youtube_completed_at = datetime.now(timezone.utc).isoformat()

    youtube_channels_out = [
        {'input': c['input'], 'channel_id': c['channel_id'], 'title': c['title'],
         'uploads_playlist_id': c['uploads_playlist_id'],
         'subscriber_count': c['subscriber_count'], 'video_count': c['video_count']}
        for c in youtube_resolved_channels
    ]
    youtube_data.write_json(YOUTUBE_DATA_DIR / 'channels.json', youtube_channels_out)
    youtube_data.write_normalized_videos_csv(NORMALIZED_VIDEOS_PATH, youtube_all_videos)
    youtube_data.write_discovered_videos_csv(
        YOUTUBE_DATA_DIR / 'discovered_videos.csv', youtube_discovery.get('videos', []))
    youtube_data.write_discovered_channels_csv(
        YOUTUBE_DATA_DIR / 'discovered_channels.csv', youtube_discovery.get('channels', []))
    youtube_data.write_json(
        YOUTUBE_DATA_DIR / 'discovery_manifest.json',
        {k: v for k, v in youtube_discovery.items() if k not in ('videos', 'channels')},
    )

    youtube_manifest = youtube_data.build_collection_manifest(
        status=youtube_status, status_reason=youtube_status_reason,
        started_at=youtube_started_at, completed_at=youtube_completed_at,
        requested_channel_count=len(effective_channel_inputs),
        resolved_channel_count=len(youtube_resolved_channels),
        failed_channel_count=len(youtube_failed_channels),
        video_count=len(youtube_all_videos),
        date_range=youtube_data.compute_date_range(youtube_all_videos),
        cache_statistics=youtube_cache.stats,
        connection_test_status=(youtube_connection_test_result['status']
                                 if youtube_connection_test_result else None),
        discovery={
            'enabled': AUTO_DISCOVER_CHANNELS,
            'status': youtube_discovery.get('status'),
            'status_reason': youtube_discovery.get('status_reason'),
            'discovered_video_count': len(youtube_discovery.get('videos', [])),
            'discovered_channel_count': len(youtube_discovery.get('channels', [])),
            'search_calls_planned': youtube_discovery.get('search_calls_planned', 0),
            'search_calls_executed': youtube_discovery.get('search_calls_executed', 0),
        },
        output_paths={
            'channels_json': YOUTUBE_DATA_DIR / 'channels.json',
            'normalized_videos_csv': NORMALIZED_VIDEOS_PATH,
            'collection_manifest_json': COLLECTION_MANIFEST_PATH,
            'quota_usage_json': YOUTUBE_DATA_DIR / 'quota_usage.json',
            'errors_json': YOUTUBE_DATA_DIR / 'errors.json',
            'discovered_videos_csv': YOUTUBE_DATA_DIR / 'discovered_videos.csv',
            'discovered_channels_csv': YOUTUBE_DATA_DIR / 'discovered_channels.csv',
            'discovery_manifest_json': YOUTUBE_DATA_DIR / 'discovery_manifest.json',
        },
    )
    youtube_data.write_json(COLLECTION_MANIFEST_PATH, youtube_manifest)
    youtube_data.write_json(YOUTUBE_DATA_DIR / 'quota_usage.json', youtube_quota.to_dict())
    youtube_data.write_json(YOUTUBE_DATA_DIR / 'errors.json', youtube_collection_errors)

    youtube_collection_summary = {
        'ran': True, 'pipeline_role': 'research_sidecar',
        'feeds_main_pipeline': False,
        'status': youtube_status, 'status_reason': youtube_status_reason,
        'resolved_channel_count': len(youtube_resolved_channels),
        'failed_channel_count': len(youtube_failed_channels),
        'video_count': len(youtube_all_videos),
        'discovered_video_count': len(youtube_discovery.get('videos', [])),
        'discovered_channel_count': len(youtube_discovery.get('channels', [])),
        'search_calls_used': youtube_quota.calls.get('search.list', 0),
        'quota_estimated_units': youtube_quota.estimated_total(),
        'cache_stats': youtube_cache.stats,
    }

    # Artefakt temiz yazildi; execution_status=completed. Ama artifact_status
    # 'insufficient_data' ise bu ozette AYRICA gorunur -- "ok" diye gecmez.
    record_phase('youtube_data', 'completed', youtube_status, youtube_status_reason)
    print()
    print('=== YouTube toplama ozeti ===')
    for k, v in youtube_collection_summary.items():
        print(f'  {k}: {v}')


## Araştırma B — Viral Potansiyel Tahmini (`RUN_VIRAL_PREDICTION`)

Bu da ana pipeline'dan bağımsız bir karar-destek deneyidir. Bağımlılık: `normalized_videos.csv` + `collection_manifest.json`. Toplama durumu
`failed`/`insufficient_data` ise veya satir sayisi esigin altindaysa artefakt
`insufficient_data` olur ve kendinden emin tahmin URETILMEZ.

In [ ]:
MIN_HISTORY_ROWS = 20


In [ ]:
import pandas as pd

viral_prediction_artifact = None

if not RUN_VIRAL_PREDICTION:
    record_phase('viral_prediction', 'skipped')
else:
    csv_path = NORMALIZED_VIDEOS_PATH if NORMALIZED_VIDEOS_PATH.is_file() else youtube_data.find_normalized_videos_csv(
        [str(YOUTUBE_DATA_DIR), str(RESULTS_DIR / 'youtube_data'),
         'results/youtube_data', '/content/results/youtube_data'],
    )
    if csv_path is None:
        record_phase('viral_prediction', 'failed', None, 'normalized_videos.csv bulunamadi')
        print("Duzeltme: RUN_YOUTUBE_DATA=True ile Faz 5'i calistir veya bu dosyayi "
              "YOUTUBE_DATA_DIR altina yerlestir.")
    else:
        history_load_status, history_load_reason = 'insufficient_data', ''
        history_df = pd.DataFrame()
        try:
            raw_df = pd.read_csv(csv_path)
        except pd.errors.EmptyDataError:
            raw_df = None
            history_load_reason = f'{csv_path} tamamen bos (basliksiz/0 byte) bir CSV dosyasi'
        required_cols = ['video_id', 'channel_id', 'view_count', 'published_at']
        if raw_df is not None:
            missing_cols = [c for c in required_cols if c not in raw_df.columns]
            if missing_cols:
                history_load_reason = f'gerekli sutunlar eksik: {missing_cols}'
            else:
                candidate_df = raw_df.dropna(subset=['view_count']).copy()
                candidate_df['published_at'] = pd.to_datetime(
                    candidate_df['published_at'], errors='coerce', utc=True)
                candidate_df = candidate_df.sort_values('published_at')
                if len(candidate_df) > 0:
                    history_df = candidate_df
                    history_load_status = 'ok'
                else:
                    history_load_reason = f'{csv_path} gecerli semaya sahip ama kullanilabilir satir yok'

        manifest_path = csv_path.parent / 'collection_manifest.json'
        collection_status = resolved_channel_count = collection_timestamp = None
        if manifest_path.is_file():
            with open(manifest_path, 'r', encoding='utf-8') as f:
                collection_manifest_data = json.load(f)
            collection_status = collection_manifest_data.get('status')
            resolved_channel_count = collection_manifest_data.get('resolved_channel_count')
            collection_timestamp = collection_manifest_data.get('completed_at')

        try:
            viral_prediction_artifact = viral_prediction.build_viral_prediction_artifact(
                history_df, history_load_status, history_load_reason, collection_status,
                min_history_rows=MIN_HISTORY_ROWS, resolved_channel_count=resolved_channel_count,
                collection_timestamp=collection_timestamp,
            )
            youtube_data.write_json(VIRAL_PREDICTION_PATH, viral_prediction_artifact)
            record_phase('viral_prediction', 'completed', viral_prediction_artifact['status'],
                         viral_prediction_artifact['status_reason'])
        except Exception as exc:
            record_phase('viral_prediction', 'failed', None, f'{type(exc).__name__}: {exc}')


## Google yetenek komutlari (her biri BAGIMSIZ acilir/kapanir)

Bu bolumdeki her komut ayri bir `RUN_GOOGLE_*` anahtarina baglidir ve kapaliyken **hicbir cagri
yapmaz**. Hepsi ayni router'i, ayni kota defterini ve ayni hata/fallback politikasini kullanir.
Uretilen her artefakt `results/` altina yazilir ve `review_required` / `experimental` /
`insufficient_data` / `failed` disinda bir durum **alamaz** -- `approved` yalnizca ayri,
hash-tabanli insan onayi adiminda verilir.


In [ ]:
google_capability_artifacts = {}


def _google_client_or_none(command_name):
    if not GOOGLE_CREDENTIAL.configured:
        print(f'{command_name}: Gemini anahtari yok, temiz atlandi.')
        print(' ', google_ai.credential_help_text('google_api'))
        return None
    return google_ai.build_google_client(GOOGLE_CREDENTIAL)


def _run_google_command(command_name, task, capability, runner, input_content,
                        enabled, extra=None):
    """Ortak sarmalayici: anahtar kapaliysa atlar, acikken rotayi gosterir,
    artefakti sozlesmeye uygun sekilde yazar. Cikti ASLA uydurulmaz."""
    if not enabled:
        print(f'{command_name}: KAPALI (atlandi).')
        return None
    client = _google_client_or_none(command_name)
    if client is None:
        return None
    show_google_route(task)
    result = runner(client)
    artifact = google_ai.build_google_artifact(
        result, task=task, capability=capability,
        input_content=input_content, output_content=result.get('output'),
        raw_media_submitted=ALLOW_RAW_MEDIA_TO_FREE_TIER,
        extra=extra,
    )
    google_capability_artifacts[command_name] = artifact
    youtube_data.write_json(RESULTS_DIR / f'google_{command_name}.json', artifact)
    print(f"{command_name}: status={artifact['status']} model={artifact['model']} "
          f"fallback={artifact['fallback_used']}")
    if artifact['status_reason']:
        print('  ', artifact['status_reason'])
    return artifact


# --- 1. Hot-moment nihai incelemesi (Rota A) --------------------------------
GOOGLE_HOTMOMENT_PROMPT = ''  # bos birakirsan hot_moments.json ozetinden uretilir
if RUN_GOOGLE_HOTMOMENT_REVIEW:
    _hm_source = hot_moments_data if isinstance(hot_moments_data, dict) else None
    if _hm_source is None:
        _found = find_artifact('hot_moments.json', RESULTS_DIR)
        if _found:
            with open(_found, 'r', encoding='utf-8') as _f:
                _hm_source = json.load(_f)
    if _hm_source is None:
        print('google_hotmoment_review: hot_moments.json yok, atlandi.')
    else:
        # ALLOW_RAW_MEDIA_TO_FREE_TIER=False iken SADECE yerel olarak turetilmis
        # transcript/skor ozetleri gonderilir; ham video ASLA yuklenmez.
        _hm_prompt = GOOGLE_HOTMOMENT_PROMPT or (
            'Asagidaki yerel olarak uretilmis aday anlari degerlendir ve en guclu 3 tanesini sirala. '
            'SADECE JSON don: {"moments": [{"rank": <int>, "score": <0-100>, "reason": "..."}]}\n\n'
            + json.dumps([{k: m.get(k) for k in ('rank', 'source_start_sec', 'source_end_sec',
                                                   'score', 'reason', 'transcript')}
                          for m in _hm_source['hot_moments'][:10]], ensure_ascii=False)
        )
        _run_google_command(
            'hotmoment_review', 'hotmoment_review', 'generate_content',
            lambda c: google_ai.run_google_hotmoment_review(
                _hm_prompt, client=c, manifest=google_manifest, ledger=google_quota_ledger,
                mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS),
            _hm_prompt, RUN_GOOGLE_HOTMOMENT_REVIEW)
else:
    print('google_hotmoment_review: KAPALI (atlandi).')

# --- 2. Toplu siniflama / metadata uretimi (Rota B) -------------------------
GOOGLE_BULK_PROMPT = (
    'Asagidaki script icin YouTube Shorts metadata uret. SADECE JSON don: '
    '{"title": "...", "description": "...", "hashtags": ["#..."]}'
)
if RUN_GOOGLE_BULK_CLASSIFICATION:
    _bulk_input = (selected_script or {}).get('script_text') or ''
    if not _bulk_input:
        print('google_bulk_classification: selected_script.script_text yok, atlandi.')
    else:
        _bulk_prompt = f'{GOOGLE_BULK_PROMPT}\n\nScript:\n{_bulk_input}'
        _run_google_command(
            'bulk_classification', 'bulk_classification', 'generate_content',
            lambda c: google_ai.run_google_bulk_classification(
                _bulk_prompt, client=c, manifest=google_manifest, ledger=google_quota_ledger,
                mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                required_fields=('title', 'description')),
            _bulk_prompt, RUN_GOOGLE_BULK_CLASSIFICATION)
else:
    print('google_bulk_classification: KAPALI (atlandi).')

# --- 3. Arama-temelli arastirma (Rota C, Gemini 2.5 grounding kotasi) -------
GOOGLE_RESEARCH_TOPICS = []   # ornek: ['House of the Dragon 3. sezon']
GOOGLE_RESEARCH_MAX_CANDIDATES = 3
if RUN_GOOGLE_GROUNDED_RESEARCH and ENABLE_GOOGLE_SEARCH_GROUNDING:
    _topics = GOOGLE_RESEARCH_TOPICS[:GOOGLE_RESEARCH_MAX_CANDIDATES]
    if not _topics:
        print('google_grounded_research: GOOGLE_RESEARCH_TOPICS bos, atlandi.')
    else:
        _research_prompt = (
            'Asagidaki konularin SU AN haberlerde/gundemde olup olmadigini arastir ve her biri icin '
            'kisa bir guncellik notu ver. Kaynak baglantilarini belirt.\n\n- '
            + '\n- '.join(_topics)
        )
        _art = _run_google_command(
            'grounded_research', 'grounded_research',
            "generate_content(tools=[google_search])",
            lambda c: google_ai.run_google_grounded_research(
                _research_prompt, client=c, manifest=google_manifest,
                ledger=google_quota_ledger, mode=GOOGLE_ROUTE_MODE,
                overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                max_candidates=GOOGLE_RESEARCH_MAX_CANDIDATES),
            _research_prompt, True)
elif RUN_GOOGLE_GROUNDED_RESEARCH:
    print('google_grounded_research: ENABLE_GOOGLE_SEARCH_GROUNDING=False, atlandi.')
else:
    print('google_grounded_research: KAPALI (atlandi).')

# --- 4. Embeddings: duplikasyon / novelty (Rota E) --------------------------
GOOGLE_EMBEDDING_TEXTS = []          # ornek: gecmis script/baslik metinleri
GOOGLE_EMBEDDING_CANDIDATE_TEXT = '' # yeni aday metin
if RUN_GOOGLE_EMBEDDINGS:
    _candidate_text = GOOGLE_EMBEDDING_CANDIDATE_TEXT or (selected_script or {}).get('script_text') or ''
    if not _candidate_text:
        print('google_embeddings: aday metin yok, atlandi.')
    else:
        _all_texts = [_candidate_text] + list(GOOGLE_EMBEDDING_TEXTS)
        _client = _google_client_or_none('google_embeddings')
        _emb = None
        if _client is not None:
            show_google_route('embeddings')
            _emb = google_ai.run_google_embeddings(
                _all_texts, client=_client, manifest=google_manifest,
                ledger=google_quota_ledger, mode=GOOGLE_ROUTE_MODE,
                overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
            )
        if ((_emb is None or _emb.get('status') != 'ok') and
                ENABLE_CROSS_PROVIDER_FALLBACK):
            _emb = provider_adapters.run_cross_provider_embeddings_fallback(
                _all_texts, secret_getter=get_secret, enabled=True,
                max_provider_attempts=MAX_PROVIDER_ATTEMPTS, allow_local=True)
        if _emb is not None:
            _extra = {'dimension': _emb.get('dimension'),
                      'task_prefix': _emb.get('task_prefix'),
                      'task_prefix_kind': _emb.get('task_prefix_kind'),
                      'input_hashes': _emb.get('input_hashes'),
                      'vector_space': _emb.get('vector_space')}
            if _emb['status'] == 'ok':
                _vectors = _emb['output']
                _novelty = google_ai.novelty_score(_vectors[0], _vectors[1:])
                _extra.update({
                    'novelty_score': _novelty,
                    'duplicate_penalty': google_ai.duplicate_penalty(_novelty),
                    'semantic_duplicates': google_ai.find_semantic_duplicates(
                        _vectors[0], list(zip(
                            [f'previous_{i}' for i in range(len(_vectors) - 1)], _vectors[1:]))),
                    'clusters': google_ai.cluster_by_similarity(
                        list(zip([f'text_{i}' for i in range(len(_vectors))], _vectors))),
                })
                print('  novelty_score:', _novelty,
                      '| duplicate_penalty:', _extra['duplicate_penalty'])
            _art = google_ai.build_google_artifact(
                _emb, task='embeddings', capability='embed_content',
                input_content='\n'.join(_all_texts),
                output_content=None, raw_media_submitted=False, extra=_extra)
            google_capability_artifacts['embeddings'] = _art
            youtube_data.write_json(RESULTS_DIR / 'google_embeddings.json', _art)
            print('google_embeddings: status=' + _art['status'])
else:
    print('google_embeddings: KAPALI (atlandi).')

# --- 5. Antigravity (DENEYSEL, varsayilan KAPALI) --------------------------
GOOGLE_ANTIGRAVITY_PROMPT = 'Bugunun YouTube Shorts trendleri icin kisa bir arastirma notu yaz.'
_antigravity = google_ai.run_antigravity_research_experimental(
    GOOGLE_ANTIGRAVITY_PROMPT,
    enabled=RUN_ANTIGRAVITY_RESEARCH and GOOGLE_CREDENTIAL.configured,
    max_total_tokens=ANTIGRAVITY_MAX_TOTAL_TOKENS,
    client=(google_ai.build_google_client(GOOGLE_CREDENTIAL)
            if (RUN_ANTIGRAVITY_RESEARCH and GOOGLE_CREDENTIAL.configured) else None),
    manifest=google_manifest, ledger=google_quota_ledger,
) if (RUN_ANTIGRAVITY_RESEARCH and GOOGLE_CREDENTIAL.configured) else \
    google_ai.run_antigravity_research_experimental(
        GOOGLE_ANTIGRAVITY_PROMPT, enabled=False,
        max_total_tokens=ANTIGRAVITY_MAX_TOTAL_TOKENS)
_antigravity_artifact = google_ai.build_antigravity_artifact(_antigravity)
google_capability_artifacts['antigravity_research'] = _antigravity_artifact
youtube_data.write_json(RESULTS_DIR / 'google_antigravity_research.json', _antigravity_artifact)
print('antigravity_research: status=' + _antigravity_artifact['status'],
      '| production_ready=' + str(_antigravity_artifact['production_ready']))

# --- Kota defteri ozeti (YEREL TAHMIN) --------------------------------------
google_quota_ledger.save()
print()
print('--- Google kota defteri (YEREL TAHMIN, sunucu yetkilisi degil) ---')
for _model, _record in sorted(google_quota_ledger.data['models'].items()):
    print(f"  {_model}: gunluk {_record['requests_current_day']}/{_record['configured_rpd']} "
          f"(kalan~{_record['estimated_remaining_rpd']}), dakika "
          f"{_record['requests_current_minute']}/{_record['configured_rpm']}, "
          f"devre={_record['circuit_state']}")
print('Defter dosyasi:', GOOGLE_QUOTA_LEDGER_PATH)


## Çalıştırma özeti (execution_status vs artifact_status)

In [ ]:
MAIN_PIPELINE_ORDER = ['source_assessment', 'processing_plan', 'video_analysis',
                       'moment_selection', 'script_generation', 'tts', 'render']
RESEARCH_SIDECAR_ORDER = ['youtube_data', 'viral_prediction']
PHASE_ORDER = MAIN_PIPELINE_ORDER + RESEARCH_SIDECAR_ORDER

def print_status_group(title, names):
    print('=' * 78)
    print(title)
    print(f"{'adim':<20} {'execution':<12} {'artifact':<24} detay")
    print('-' * 78)
    for name in names:
        entry = phase_report.get(name)
        if entry is None:
            print(f"{name:<20} {'-':<12} {'-':<24} hucre calistirilmadi")
            continue
        artifact = entry['artifact_status'] if entry['artifact_status'] is not None else '-'
        detail = (entry['detail'] or '')[:60]
        print(f"{name:<20} {entry['execution_status']:<12} {artifact:<24} {detail}")
    print('=' * 78)

print_status_group('ANA URETIM HATTI: kaynak -> Faz 0 plan -> analiz -> moment -> (script/ses) -> render',
                   MAIN_PIPELINE_ORDER)
if isinstance(globals().get('processing_plan'), dict):
    _pd = processing_plan.get('decision') or {}
    print()
    print(f"Faz 0 plani: mod={_pd.get('mode')} status={processing_plan.get('status')} "
          f"cikti={_pd.get('output_count')} pipeline_mode={processing_plan.get('pipeline_mode')}")
    print('  acik fazlar   :', processing_plan.get('enabled_phases'))
    print('  kapali fazlar :', processing_plan.get('disabled_phases'))
else:
    print()
    print('Faz 0 isleme plani YOK -> ana hatta hicbir faz calistirilamaz.')
print()
print_status_group('BAGIMSIZ ARASTIRMA YAN KOLU: ana pipeline girdilerini degistirmez',
                   RESEARCH_SIDECAR_ORDER)

# Temiz calisip yetersiz veri bildiren fazlari ACIKCA one cikar.
needs_attention = [
    name for name, entry in phase_report.items()
    if entry['execution_status'] == 'completed'
    and entry['artifact_status'] in ('insufficient_data', 'failed', 'development_placeholder')
]
if needs_attention:
    print()
    print('DIKKAT: bu fazlar temiz CALISTI ama artefakt durumu kullanima hazir degil:')
    for name in needs_attention:
        print(f"  - {name}: artifact_status={phase_report[name]['artifact_status']}"
              f" ({phase_report[name]['detail']})")

awaiting_review = [name for name, entry in phase_report.items()
                  if entry['artifact_status'] == 'review_required']
if awaiting_review:
    print()
    print('INSAN INCELEMESI BEKLEYENLER (ilgili inceleme + onay hucrelerini calistir):')
    for name in awaiting_review:
        print(f'  - {name}')

print()
print("Hicbir faz YouTube'a veya baska bir yere yukleme/yayin yapmadi.")
print('production_ready statusu hicbir yerde otomatik atanmaz.')


## Colab yeniden başlatma / kalıcı artefaktlardan devam etme

`USE_GOOGLE_DRIVE = True` iken tüm artefaktlar ve önbellek Drive'da kalır. Colab yeniden başlatılsa
bile bu notebook'u tekrar açıp kurulum + toggle + setup + bootstrap + yol çözümleme hücrelerini
çalıştırman, tamamlanmış fazların toggle'ını `False` yapman ve devam eden fazı çalıştırman yeterlidir.

Onaylar içeriğe SHA-256 ile bağlıdır: bir fazı yeniden çalıştırıp içeriği değiştirirsen, o artefaktın
önceki onayı otomatik olarak geçersiz sayılır (`approval.check_approval`) ve yeniden incelenip
onaylanması gerekir. Faz 3 (TTS), diskten okuduğu script onayının hâlâ geçerli olduğunu ayrıca
doğrular; geçersizse ücretli TTS çalıştırmaz.